In [1]:
# ============================================================
# Download automático do Figshare pelo Python
# URL: https://plus.figshare.com/ndownloader/articles/30027313/versions/1
# ============================================================

import os
import re
import sys
import time
import subprocess
from pathlib import Path

# Instala requests/tqdm se necessário
def ensure_package(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", pkg])

ensure_package("requests")
ensure_package("tqdm")

import requests
from tqdm import tqdm
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


URL = "https://plus.figshare.com/ndownloader/articles/30027313/versions/1"
OUT_DIR = Path(r"D:\Fusion")
OUT_DIR.mkdir(parents=True, exist_ok=True)

FALLBACK_NAME = "figshare_article_30027313_version_1.zip"
CHUNK_SIZE = 1024 * 1024  # 1 MB


def make_session():
    retry = Retry(
        total=10,
        connect=10,
        read=10,
        status=10,
        backoff_factor=2,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["HEAD", "GET"],
        raise_on_status=False,
    )

    adapter = HTTPAdapter(max_retries=retry)

    session = requests.Session()
    session.mount("https://", adapter)
    session.mount("http://", adapter)

    return session


def get_filename_from_headers(headers):
    cd = headers.get("content-disposition", "")
    match = re.search(r'filename="?([^"]+)"?', cd)
    if match:
        return match.group(1)
    return FALLBACK_NAME


def human_size(n):
    if n is None:
        return "desconhecido"
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if n < 1024:
            return f"{n:.2f} {unit}"
        n /= 1024
    return f"{n:.2f} PB"


session = make_session()

print("[INFO] Consultando metadados do arquivo...")

head = session.head(URL, allow_redirects=True, timeout=60)

filename = get_filename_from_headers(head.headers)
total_size = int(head.headers.get("content-length", 0)) if head.headers.get("content-length") else None

final_path = OUT_DIR / filename
part_path = OUT_DIR / f"{filename}.part"

print(f"[INFO] Arquivo: {filename}")
print(f"[INFO] Tamanho: {human_size(total_size)}")
print(f"[INFO] Destino: {final_path}")

# Se já existe completo
if final_path.exists() and total_size and final_path.stat().st_size == total_size:
    print("[OK] Arquivo já baixado completamente.")
else:
    downloaded = part_path.stat().st_size if part_path.exists() else 0

    headers = {}
    mode = "wb"

    if downloaded > 0:
        headers["Range"] = f"bytes={downloaded}-"
        mode = "ab"
        print(f"[INFO] Retomando download a partir de {human_size(downloaded)}...")

    with session.get(URL, headers=headers, stream=True, allow_redirects=True, timeout=120) as r:
        # Se o servidor não aceitar retomada, reinicia do zero
        if r.status_code == 200 and downloaded > 0:
            print("[AVISO] O servidor não aceitou retomada. Reiniciando download.")
            downloaded = 0
            mode = "wb"

        # Se Range inválido, provavelmente o .part está inconsistente
        if r.status_code == 416:
            print("[AVISO] Range inválido. Apagando arquivo parcial e reiniciando.")
            if part_path.exists():
                part_path.unlink()

            with session.get(URL, stream=True, allow_redirects=True, timeout=120) as r2:
                r2.raise_for_status()

                total = int(r2.headers.get("content-length", 0)) or total_size
                with open(part_path, "wb") as f, tqdm(
                    total=total,
                    unit="B",
                    unit_scale=True,
                    unit_divisor=1024,
                    desc="Baixando",
                ) as pbar:
                    for chunk in r2.iter_content(chunk_size=CHUNK_SIZE):
                        if chunk:
                            f.write(chunk)
                            pbar.update(len(chunk))
        else:
            r.raise_for_status()

            total = total_size
            progress_total = total if total else None

            with open(part_path, mode) as f, tqdm(
                total=progress_total,
                initial=downloaded,
                unit="B",
                unit_scale=True,
                unit_divisor=1024,
                desc="Baixando",
            ) as pbar:
                for chunk in r.iter_content(chunk_size=CHUNK_SIZE):
                    if chunk:
                        f.write(chunk)
                        pbar.update(len(chunk))

    # Renomeia .part para arquivo final
    if part_path.exists():
        part_path.rename(final_path)

    print("\n[OK] Download concluído.")
    print(f"[OK] Arquivo salvo em: {final_path}")

[INFO] Consultando metadados do arquivo...
[INFO] Arquivo: figshare_article_30027313_version_1.zip
[INFO] Tamanho: 0.00 B
[INFO] Destino: D:\Fusion\figshare_article_30027313_version_1.zip


Baixando: 0.00B [00:00, ?B/s]


[OK] Download concluído.
[OK] Arquivo salvo em: D:\Fusion\figshare_article_30027313_version_1.zip


In [2]:
from pathlib import Path

p = Path(r"D:\Fusion\figshare_article_30027313_version_1.zip")
if p.exists() and p.stat().st_size == 0:
    p.unlink()
    print("Arquivo vazio apagado.")

Arquivo vazio apagado.


# BAIXANDO ARQUIVOS

In [3]:
import re
import sys
import time
import subprocess
from pathlib import Path

def ensure_package(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", pkg])

ensure_package("requests")
ensure_package("tqdm")

import requests
from tqdm import tqdm
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


ARTICLE_ID = 30027313
VERSION = 1

OUT_DIR = Path(r"D:\Fusion\figshare_30027313_v1")
OUT_DIR.mkdir(parents=True, exist_ok=True)

API_URL = f"https://api.figshare.com/v2/articles/{ARTICLE_ID}/versions/{VERSION}"

CHUNK_SIZE = 4 * 1024 * 1024  # 4 MB


def make_session():
    retry = Retry(
        total=10,
        connect=10,
        read=10,
        status=10,
        backoff_factor=2,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        raise_on_status=False,
    )
    s = requests.Session()
    adapter = HTTPAdapter(max_retries=retry)
    s.mount("https://", adapter)
    s.mount("http://", adapter)
    return s


def safe_name(name):
    name = re.sub(r'[<>:"/\\|?*]', "_", name)
    return name.strip()


def human_size(n):
    if not n:
        return "desconhecido"
    n = float(n)
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if n < 1024:
            return f"{n:.2f} {unit}"
        n /= 1024
    return f"{n:.2f} PB"


def download_file(session, url, out_path, expected_size=None):
    part_path = out_path.with_suffix(out_path.suffix + ".part")

    if out_path.exists() and expected_size and out_path.stat().st_size == expected_size:
        print(f"[OK] Já existe: {out_path.name}")
        return

    downloaded = part_path.stat().st_size if part_path.exists() else 0

    if expected_size and downloaded > expected_size:
        part_path.unlink()
        downloaded = 0

    headers = {}
    mode = "wb"

    if downloaded > 0:
        headers["Range"] = f"bytes={downloaded}-"
        mode = "ab"
        print(f"[INFO] Retomando {out_path.name} de {human_size(downloaded)}")

    with session.get(url, headers=headers, stream=True, timeout=(30, 180), allow_redirects=True) as r:
        if r.status_code == 200 and downloaded > 0:
            print("[AVISO] Servidor não aceitou retomada. Reiniciando este arquivo.")
            downloaded = 0
            mode = "wb"

        if r.status_code == 416:
            print("[AVISO] Arquivo parcial inconsistente. Reiniciando este arquivo.")
            if part_path.exists():
                part_path.unlink()
            return download_file(session, url, out_path, expected_size)

        r.raise_for_status()

        total = expected_size if expected_size else int(r.headers.get("content-length", 0) or 0)

        with open(part_path, mode) as f, tqdm(
            total=total if total else None,
            initial=downloaded,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
            desc=out_path.name[:40],
        ) as pbar:
            for chunk in r.iter_content(chunk_size=CHUNK_SIZE):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))

    final_size = part_path.stat().st_size

    if expected_size and final_size != expected_size:
        raise RuntimeError(
            f"Download incompleto: {out_path.name} "
            f"baixado={final_size}, esperado={expected_size}"
        )

    part_path.rename(out_path)
    print(f"[OK] Salvo: {out_path}")


session = make_session()

print("[INFO] Lendo metadados do Figshare...")
resp = session.get(API_URL, timeout=60)
resp.raise_for_status()
metadata = resp.json()

files = metadata.get("files", [])

if not files:
    raise RuntimeError("Nenhum arquivo encontrado na resposta da API.")

print(f"[INFO] Arquivos encontrados: {len(files)}")

for i, f in enumerate(files, start=1):
    file_id = f.get("id", i)
    name = safe_name(f.get("name", f"arquivo_{i}"))
    size = f.get("size")
    url = f.get("download_url")

    if not url:
        raise RuntimeError(f"Arquivo sem download_url: {name}")

    # prefixo evita sobrescrever arquivos com nomes repetidos
    out_name = f"{i:03d}_{file_id}_{name}"
    out_path = OUT_DIR / out_name

    print("\n" + "=" * 80)
    print(f"[{i}/{len(files)}] {name}")
    print(f"[INFO] Tamanho: {human_size(size)}")

    download_file(session, url, out_path, expected_size=size)

print("\n[OK] Todos os downloads foram concluídos.")
print(f"[OK] Pasta: {OUT_DIR}")

[INFO] Lendo metadados do Figshare...
[INFO] Arquivos encontrados: 85

[1/85] CW Radar.7z
[INFO] Tamanho: 2.26 GB


001_57588574_CW Radar.7z: 100%|██████████| 2.26G/2.26G [03:15<00:00, 12.4MB/s]


[OK] Salvo: D:\Fusion\figshare_30027313_v1\001_57588574_CW Radar.7z

[2/85] RF Receiver.7z
[INFO] Tamanho: 17.34 GB


002_57589510_RF Receiver.7z: 100%|██████████| 17.3G/17.3G [23:54<00:00, 13.0MB/s]


[OK] Salvo: D:\Fusion\figshare_30027313_v1\002_57589510_RF Receiver.7z

[3/85] Example_CW_radar_processing.m
[INFO] Tamanho: 3.43 KB


...

[OK] Salvo: D:\Fusion\figshare_30027313_v1\083_60715321_CNN_Training_Script.m

[84/85] Crop_Based_Image_Fusion.m
[INFO] Tamanho: 2.65 KB


084_60715324_Crop_Based_Image_Fusion.m: 100%|██████████| 2.65k/2.65k [00:00<?, ?B/s]


[OK] Salvo: D:\Fusion\figshare_30027313_v1\084_60715324_Crop_Based_Image_Fusion.m

[85/85] RGB_Channel_Based_Image_Fusion.m
[INFO] Tamanho: 2.20 KB


085_60715327_RGB_Channel_Based_Image_Fus: 100%|██████████| 2.20k/2.20k [00:00<?, ?B/s]

[OK] Salvo: D:\Fusion\figshare_30027313_v1\085_60715327_RGB_Channel_Based_Image_Fusion.m

[OK] Todos os downloads foram concluídos.
[OK] Pasta: D:\Fusion\figshare_30027313_v1


In [4]:
from pathlib import Path

folder = Path(r"D:\Fusion\figshare_30027313_v1")

total_size = 0
file_count = 0

for file in folder.rglob("*"):
    if file.is_file():
        try:
            total_size += file.stat().st_size
            file_count += 1
        except PermissionError:
            print(f"[AVISO] Sem permissão para ler: {file}")

size_gb = total_size / (1024 ** 3)

print("=" * 60)
print(f"Pasta: {folder}")
print(f"Arquivos encontrados: {file_count}")
print(f"Tamanho total: {total_size:,} bytes")
print(f"Tamanho total: {size_gb:.2f} GB")
print("=" * 60)

Pasta: D:\Fusion\figshare_30027313_v1
Arquivos encontrados: 85
Tamanho total: 392,813,308,724 bytes
Tamanho total: 365.84 GB


In [5]:
# ============================================================
# explore_figshare_dataset.py
# ============================================================
# Explora arquivos, pastas e subpastas de:
#   D:\Fusion\figshare_30027313_v1
#
# Gera:
#   - file_inventory.csv
#   - folder_summary.csv
#   - extension_summary.csv
#   - largest_files.csv
#   - empty_files.csv
#   - duplicate_names.csv
#   - duplicate_sizes.csv
#   - directory_tree.txt
#   - summary_report.md
# ============================================================

from pathlib import Path
import os
import sys
import json
import hashlib
import subprocess
import warnings
from datetime import datetime

warnings.filterwarnings("ignore")


# ============================================================
# INSTALL / IMPORTS
# ============================================================
def ensure_package(import_name, pip_name=None):
    try:
        __import__(import_name)
    except Exception:
        pkg = pip_name if pip_name is not None else import_name
        print(f"[INFO] Installing missing package: {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


ensure_package("pandas")
ensure_package("tqdm")

import pandas as pd
from tqdm import tqdm


# ============================================================
# CONFIG
# ============================================================
ROOT = Path(r"D:\Fusion\figshare_30027313_v1")

OUT_DIR = ROOT / "_exploration_report"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Se True, calcula hash MD5 dos arquivos pequenos.
# Não calcule hash de arquivos grandes sem necessidade.
CALCULATE_MD5_FOR_SMALL_FILES = True
MD5_MAX_MB = 100

# Limite de linhas da árvore de diretórios.
TREE_MAX_DEPTH = 5
TREE_MAX_ITEMS_PER_FOLDER = 200

# Se True, tenta listar conteúdo de .zip/.tar/.7z sem extrair.
# Para arquivos muito grandes, pode demorar. Recomendo False inicialmente.
LIST_ARCHIVE_CONTENTS = False

# Extensões consideradas arquivos compactados.
ARCHIVE_EXTENSIONS = {
    ".zip", ".tar", ".gz", ".tgz", ".tar.gz",
    ".7z", ".rar"
}


# ============================================================
# HELPERS
# ============================================================
def bytes_to_human(n):
    try:
        n = float(n)
    except Exception:
        return ""

    units = ["B", "KB", "MB", "GB", "TB"]
    i = 0

    while n >= 1024 and i < len(units) - 1:
        n /= 1024.0
        i += 1

    return f"{n:.2f} {units[i]}"


def safe_stat(path: Path):
    try:
        st = path.stat()
        return st
    except Exception:
        return None


def get_extension(path: Path):
    name = path.name.lower()

    if name.endswith(".tar.gz"):
        return ".tar.gz"

    return path.suffix.lower()


def file_md5(path: Path, chunk_size=1024 * 1024):
    h = hashlib.md5()

    try:
        with open(path, "rb") as f:
            while True:
                chunk = f.read(chunk_size)
                if not chunk:
                    break
                h.update(chunk)
        return h.hexdigest()
    except Exception:
        return None


def safe_relative(path: Path, root: Path):
    try:
        return str(path.relative_to(root))
    except Exception:
        return str(path)


def is_hidden(path: Path):
    try:
        return bool(os.stat(path).st_file_attributes & 2)
    except Exception:
        return False


def path_depth(path: Path, root: Path):
    try:
        return len(path.relative_to(root).parts)
    except Exception:
        return len(path.parts)


# ============================================================
# MAIN SCAN
# ============================================================
def scan_files(root: Path):
    print("=" * 80)
    print("SCANNING FILES")
    print("=" * 80)
    print(f"Root: {root}")

    if not root.exists():
        raise FileNotFoundError(f"Folder not found: {root}")

    all_paths = list(root.rglob("*"))

    rows = []
    folder_rows = []

    for p in tqdm(all_paths, desc="Scanning", unit="item"):
        st = safe_stat(p)

        if st is None:
            continue

        rel = safe_relative(p, root)
        depth = path_depth(p, root)

        if p.is_dir():
            folder_rows.append({
                "folder_path": str(p),
                "relative_folder": rel,
                "depth": depth,
                "created_time": datetime.fromtimestamp(st.st_ctime).isoformat(timespec="seconds"),
                "modified_time": datetime.fromtimestamp(st.st_mtime).isoformat(timespec="seconds"),
                "is_hidden": is_hidden(p),
            })
            continue

        if p.is_file():
            size_bytes = int(st.st_size)
            size_mb = size_bytes / (1024 ** 2)
            ext = get_extension(p)

            md5 = None
            if CALCULATE_MD5_FOR_SMALL_FILES and size_mb <= MD5_MAX_MB:
                md5 = file_md5(p)

            rows.append({
                "file_name": p.name,
                "extension": ext,
                "path": str(p),
                "relative_path": rel,
                "parent_folder": str(p.parent),
                "relative_parent_folder": safe_relative(p.parent, root),
                "depth": depth,
                "size_bytes": size_bytes,
                "size_mb": size_mb,
                "size_human": bytes_to_human(size_bytes),
                "created_time": datetime.fromtimestamp(st.st_ctime).isoformat(timespec="seconds"),
                "modified_time": datetime.fromtimestamp(st.st_mtime).isoformat(timespec="seconds"),
                "is_hidden": is_hidden(p),
                "is_archive": ext in ARCHIVE_EXTENSIONS,
                "md5_if_small": md5,
            })

    files_df = pd.DataFrame(rows)
    folders_df = pd.DataFrame(folder_rows)

    return files_df, folders_df


# ============================================================
# SUMMARIES
# ============================================================
def build_extension_summary(files_df):
    if len(files_df) == 0:
        return pd.DataFrame()

    summary = (
        files_df
        .groupby("extension", dropna=False)
        .agg(
            n_files=("path", "count"),
            total_bytes=("size_bytes", "sum"),
            total_mb=("size_mb", "sum"),
            mean_mb=("size_mb", "mean"),
            median_mb=("size_mb", "median"),
            max_mb=("size_mb", "max"),
            min_mb=("size_mb", "min"),
        )
        .reset_index()
        .sort_values("total_bytes", ascending=False)
    )

    summary["total_human"] = summary["total_bytes"].apply(bytes_to_human)

    return summary


def build_folder_summary(files_df, folders_df):
    if len(files_df) == 0:
        return pd.DataFrame()

    folder_summary = (
        files_df
        .groupby("parent_folder", dropna=False)
        .agg(
            n_files_direct=("path", "count"),
            total_bytes_direct=("size_bytes", "sum"),
            total_mb_direct=("size_mb", "sum"),
            max_file_mb=("size_mb", "max"),
        )
        .reset_index()
    )

    folder_summary["total_human_direct"] = folder_summary["total_bytes_direct"].apply(bytes_to_human)

    # Soma recursiva por pasta.
    recursive_rows = []

    folder_paths = set(folders_df["folder_path"].tolist()) if len(folders_df) > 0 else set()
    folder_paths.add(str(ROOT))

    for folder in tqdm(sorted(folder_paths), desc="Folder recursive sizes", unit="folder"):
        folder_path = Path(folder)

        try:
            mask = files_df["path"].apply(lambda x: Path(x).is_relative_to(folder_path))
        except AttributeError:
            mask = files_df["path"].apply(lambda x: str(x).startswith(str(folder_path)))

        g = files_df[mask]

        recursive_rows.append({
            "folder_path": str(folder_path),
            "relative_folder": safe_relative(folder_path, ROOT),
            "n_files_recursive": int(len(g)),
            "total_bytes_recursive": int(g["size_bytes"].sum()) if len(g) > 0 else 0,
            "total_mb_recursive": float(g["size_mb"].sum()) if len(g) > 0 else 0.0,
            "total_human_recursive": bytes_to_human(g["size_bytes"].sum()) if len(g) > 0 else "0 B",
        })

    recursive_df = pd.DataFrame(recursive_rows)

    return recursive_df.sort_values("total_bytes_recursive", ascending=False)


def build_duplicate_reports(files_df):
    if len(files_df) == 0:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    duplicate_names = (
        files_df
        .groupby("file_name")
        .filter(lambda g: len(g) > 1)
        .sort_values(["file_name", "size_bytes"])
    )

    duplicate_sizes = (
        files_df
        .groupby("size_bytes")
        .filter(lambda g: len(g) > 1 and g.name > 0)
        .sort_values(["size_bytes", "file_name"], ascending=[False, True])
    )

    duplicate_md5 = pd.DataFrame()
    if "md5_if_small" in files_df.columns:
        md5_df = files_df[files_df["md5_if_small"].notna()].copy()
        duplicate_md5 = (
            md5_df
            .groupby("md5_if_small")
            .filter(lambda g: len(g) > 1)
            .sort_values(["md5_if_small", "file_name"])
        )

    return duplicate_names, duplicate_sizes, duplicate_md5


def build_largest_files(files_df, n=50):
    if len(files_df) == 0:
        return pd.DataFrame()

    return files_df.sort_values("size_bytes", ascending=False).head(n).copy()


def build_empty_files(files_df):
    if len(files_df) == 0:
        return pd.DataFrame()

    return files_df[files_df["size_bytes"] == 0].copy()


# ============================================================
# DIRECTORY TREE
# ============================================================
def make_tree(root: Path, max_depth=5, max_items_per_folder=200):
    lines = []

    def _tree(folder: Path, prefix="", depth=0):
        if depth > max_depth:
            lines.append(prefix + "... [max depth reached]")
            return

        try:
            items = sorted(folder.iterdir(), key=lambda p: (not p.is_dir(), p.name.lower()))
        except Exception as e:
            lines.append(prefix + f"[ERROR opening folder: {e}]")
            return

        if len(items) > max_items_per_folder:
            shown = items[:max_items_per_folder]
            truncated = len(items) - max_items_per_folder
        else:
            shown = items
            truncated = 0

        for i, item in enumerate(shown):
            connector = "└── " if i == len(shown) - 1 and truncated == 0 else "├── "
            line = prefix + connector + item.name

            if item.is_file():
                st = safe_stat(item)
                if st is not None:
                    line += f" ({bytes_to_human(st.st_size)})"

            lines.append(line)

            if item.is_dir():
                extension = "    " if i == len(shown) - 1 and truncated == 0 else "│   "
                _tree(item, prefix + extension, depth + 1)

        if truncated > 0:
            lines.append(prefix + f"└── ... [{truncated} more items]")

    lines.append(str(root))
    _tree(root, "", 0)

    return "\n".join(lines)


# ============================================================
# OPTIONAL ARCHIVE LISTING
# ============================================================
def inspect_archives(files_df):
    """
    Opcional: lista conteúdo de alguns arquivos compactados sem extrair.
    Desativado por padrão porque arquivos grandes podem demorar.
    """
    rows = []

    if not LIST_ARCHIVE_CONTENTS:
        return pd.DataFrame(rows)

    archives = files_df[files_df["is_archive"] == True].copy()

    if len(archives) == 0:
        return pd.DataFrame(rows)

    for _, row in tqdm(archives.iterrows(), total=len(archives), desc="Inspecting archives"):
        path = Path(row["path"])
        ext = row["extension"]

        info = {
            "archive_path": str(path),
            "archive_name": path.name,
            "extension": ext,
            "status": "ok",
            "n_members": 0,
            "first_members": None,
            "error": None,
        }

        try:
            if ext == ".zip":
                import zipfile
                with zipfile.ZipFile(path, "r") as z:
                    names = z.namelist()
                info["n_members"] = len(names)
                info["first_members"] = json.dumps(names[:50], ensure_ascii=False)

            elif ext in {".tar", ".tar.gz", ".tgz", ".gz"}:
                import tarfile
                with tarfile.open(path, "r:*") as t:
                    names = t.getnames()
                info["n_members"] = len(names)
                info["first_members"] = json.dumps(names[:50], ensure_ascii=False)

            elif ext == ".7z":
                ensure_package("py7zr")
                import py7zr
                with py7zr.SevenZipFile(path, mode="r") as z:
                    names = z.getnames()
                info["n_members"] = len(names)
                info["first_members"] = json.dumps(names[:50], ensure_ascii=False)

            else:
                info["status"] = "unsupported_archive_type"

        except Exception as e:
            info["status"] = "error"
            info["error"] = str(e)

        rows.append(info)

    return pd.DataFrame(rows)


# ============================================================
# REPORT
# ============================================================
def write_markdown_report(
    files_df,
    folders_df,
    extension_summary,
    folder_summary,
    largest_files,
    empty_files,
    duplicate_names,
    duplicate_sizes,
    duplicate_md5,
):
    report_path = OUT_DIR / "summary_report.md"

    total_files = len(files_df)
    total_folders = len(folders_df)
    total_bytes = int(files_df["size_bytes"].sum()) if total_files > 0 else 0

    with open(report_path, "w", encoding="utf-8") as f:
        f.write("# Dataset Exploration Report\n\n")

        f.write("## Root\n\n")
        f.write(f"`{ROOT}`\n\n")

        f.write("## Summary\n\n")
        f.write(f"- Files: {total_files:,}\n")
        f.write(f"- Folders: {total_folders:,}\n")
        f.write(f"- Total size: {total_bytes:,} bytes\n")
        f.write(f"- Total size: {bytes_to_human(total_bytes)}\n\n")

        f.write("## Extension summary\n\n")
        if len(extension_summary) > 0:
            f.write(extension_summary.head(30).to_markdown(index=False))
            f.write("\n\n")

        f.write("## Largest files\n\n")
        if len(largest_files) > 0:
            cols = ["file_name", "extension", "size_human", "relative_path"]
            f.write(largest_files[cols].head(30).to_markdown(index=False))
            f.write("\n\n")

        f.write("## Largest folders, recursive\n\n")
        if len(folder_summary) > 0:
            cols = ["relative_folder", "n_files_recursive", "total_human_recursive"]
            f.write(folder_summary[cols].head(30).to_markdown(index=False))
            f.write("\n\n")

        f.write("## Empty files\n\n")
        f.write(f"- Empty files: {len(empty_files):,}\n\n")

        f.write("## Possible duplicates\n\n")
        f.write(f"- Duplicate file names: {len(duplicate_names):,} rows\n")
        f.write(f"- Duplicate file sizes: {len(duplicate_sizes):,} rows\n")
        f.write(f"- Duplicate MD5 among small files: {len(duplicate_md5):,} rows\n\n")

        f.write("## Output files\n\n")
        f.write("- `file_inventory.csv`\n")
        f.write("- `folder_inventory.csv`\n")
        f.write("- `extension_summary.csv`\n")
        f.write("- `folder_summary.csv`\n")
        f.write("- `largest_files.csv`\n")
        f.write("- `empty_files.csv`\n")
        f.write("- `duplicate_names.csv`\n")
        f.write("- `duplicate_sizes.csv`\n")
        f.write("- `duplicate_md5_small_files.csv`\n")
        f.write("- `directory_tree.txt`\n")

    return report_path


# ============================================================
# MAIN
# ============================================================
def main():
    print("=" * 80)
    print("FIGSHARE DATASET EXPLORATION")
    print("=" * 80)
    print(f"Root: {ROOT}")
    print(f"Output: {OUT_DIR}")

    files_df, folders_df = scan_files(ROOT)

    # --------------------------------------------------------
    # Build summaries
    # --------------------------------------------------------
    extension_summary = build_extension_summary(files_df)
    folder_summary = build_folder_summary(files_df, folders_df)
    largest_files = build_largest_files(files_df, n=100)
    empty_files = build_empty_files(files_df)
    duplicate_names, duplicate_sizes, duplicate_md5 = build_duplicate_reports(files_df)
    archive_inventory = inspect_archives(files_df)

    # --------------------------------------------------------
    # Directory tree
    # --------------------------------------------------------
    tree_text = make_tree(
        ROOT,
        max_depth=TREE_MAX_DEPTH,
        max_items_per_folder=TREE_MAX_ITEMS_PER_FOLDER,
    )

    tree_path = OUT_DIR / "directory_tree.txt"
    with open(tree_path, "w", encoding="utf-8") as f:
        f.write(tree_text)

    # --------------------------------------------------------
    # Save CSVs
    # --------------------------------------------------------
    files_path = OUT_DIR / "file_inventory.csv"
    folders_path = OUT_DIR / "folder_inventory.csv"
    extension_path = OUT_DIR / "extension_summary.csv"
    folder_summary_path = OUT_DIR / "folder_summary.csv"
    largest_path = OUT_DIR / "largest_files.csv"
    empty_path = OUT_DIR / "empty_files.csv"
    duplicate_names_path = OUT_DIR / "duplicate_names.csv"
    duplicate_sizes_path = OUT_DIR / "duplicate_sizes.csv"
    duplicate_md5_path = OUT_DIR / "duplicate_md5_small_files.csv"
    archive_inventory_path = OUT_DIR / "archive_inventory.csv"

    files_df.to_csv(files_path, index=False, encoding="utf-8-sig")
    folders_df.to_csv(folders_path, index=False, encoding="utf-8-sig")
    extension_summary.to_csv(extension_path, index=False, encoding="utf-8-sig")
    folder_summary.to_csv(folder_summary_path, index=False, encoding="utf-8-sig")
    largest_files.to_csv(largest_path, index=False, encoding="utf-8-sig")
    empty_files.to_csv(empty_path, index=False, encoding="utf-8-sig")
    duplicate_names.to_csv(duplicate_names_path, index=False, encoding="utf-8-sig")
    duplicate_sizes.to_csv(duplicate_sizes_path, index=False, encoding="utf-8-sig")
    duplicate_md5.to_csv(duplicate_md5_path, index=False, encoding="utf-8-sig")
    archive_inventory.to_csv(archive_inventory_path, index=False, encoding="utf-8-sig")

    # --------------------------------------------------------
    # JSON summary
    # --------------------------------------------------------
    total_bytes = int(files_df["size_bytes"].sum()) if len(files_df) > 0 else 0

    summary = {
        "root": str(ROOT),
        "n_files": int(len(files_df)),
        "n_folders": int(len(folders_df)),
        "total_bytes": total_bytes,
        "total_human": bytes_to_human(total_bytes),
        "calculate_md5_for_small_files": CALCULATE_MD5_FOR_SMALL_FILES,
        "md5_max_mb": MD5_MAX_MB,
        "list_archive_contents": LIST_ARCHIVE_CONTENTS,
        "outputs": {
            "file_inventory": str(files_path),
            "folder_inventory": str(folders_path),
            "extension_summary": str(extension_path),
            "folder_summary": str(folder_summary_path),
            "largest_files": str(largest_path),
            "empty_files": str(empty_path),
            "duplicate_names": str(duplicate_names_path),
            "duplicate_sizes": str(duplicate_sizes_path),
            "duplicate_md5_small_files": str(duplicate_md5_path),
            "archive_inventory": str(archive_inventory_path),
            "directory_tree": str(tree_path),
        },
    }

    summary_path = OUT_DIR / "summary.json"
    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)

    report_path = write_markdown_report(
        files_df=files_df,
        folders_df=folders_df,
        extension_summary=extension_summary,
        folder_summary=folder_summary,
        largest_files=largest_files,
        empty_files=empty_files,
        duplicate_names=duplicate_names,
        duplicate_sizes=duplicate_sizes,
        duplicate_md5=duplicate_md5,
    )

    # --------------------------------------------------------
    # Console output
    # --------------------------------------------------------
    print("\n" + "=" * 80)
    print("SUMMARY")
    print("=" * 80)
    print(f"Files       : {len(files_df):,}")
    print(f"Folders     : {len(folders_df):,}")
    print(f"Total bytes : {total_bytes:,}")
    print(f"Total size  : {bytes_to_human(total_bytes)}")

    print("\n" + "=" * 80)
    print("EXTENSION SUMMARY")
    print("=" * 80)
    if len(extension_summary) > 0:
        print(extension_summary.head(30).to_string(index=False))

    print("\n" + "=" * 80)
    print("LARGEST FILES")
    print("=" * 80)
    if len(largest_files) > 0:
        print(
            largest_files[
                ["file_name", "extension", "size_human", "relative_path"]
            ].head(30).to_string(index=False)
        )

    print("\n" + "=" * 80)
    print("OUTPUT FILES")
    print("=" * 80)
    print(f"File inventory     : {files_path}")
    print(f"Folder inventory   : {folders_path}")
    print(f"Extension summary  : {extension_path}")
    print(f"Folder summary     : {folder_summary_path}")
    print(f"Largest files      : {largest_path}")
    print(f"Empty files        : {empty_path}")
    print(f"Duplicate names    : {duplicate_names_path}")
    print(f"Duplicate sizes    : {duplicate_sizes_path}")
    print(f"Duplicate MD5      : {duplicate_md5_path}")
    print(f"Archive inventory  : {archive_inventory_path}")
    print(f"Directory tree     : {tree_path}")
    print(f"Summary JSON       : {summary_path}")
    print(f"Markdown report    : {report_path}")

    print("\nDone.")


if __name__ == "__main__":
    main()

FIGSHARE DATASET EXPLORATION
Root: D:\Fusion\figshare_30027313_v1
Output: D:\Fusion\figshare_30027313_v1\_exploration_report
SCANNING FILES
Root: D:\Fusion\figshare_30027313_v1


Folder recursive sizes: 100%|██████████| 2/2 [00:00<00:00, 176.41folder/s]



SUMMARY
Files       : 85
Folders     : 1
Total bytes : 392,813,308,724
Total size  : 365.84 GB

EXTENSION SUMMARY
extension  n_files  total_bytes      total_mb     mean_mb   median_mb       max_mb      min_mb total_human
      .7z       77 392797858609 374601.229295 4864.951030 4826.174384 17756.439557 2315.830441   365.82 GB
     .mat        1     15428525     14.713788   14.713788   14.713788    14.713788   14.713788    14.71 MB
       .m        7        21590      0.020590    0.002941    0.002795     0.004280    0.002148    21.08 KB

LARGEST FILES
                    file_name extension size_human                 relative_path
  002_57589510_RF Receiver.7z       .7z   17.34 GB   002_57589510_RF Receiver.7z
    038_57690193_Inspire 2.7z       .7z    4.75 GB     038_57690193_Inspire 2.7z
040_57690199_Phantom 4 Pro.7z       .7z    4.74 GB 040_57690199_Phantom 4 Pro.7z
  039_57690196_Mavic 2 Pro.7z       .7z    4.74 GB   039_57690196_Mavic 2 Pro.7z
    043_57690208_Inspire 2.7z       .

In [6]:
# ============================================================
# inspect_7z_contents_figshare.py
# ============================================================
# Lista o conteúdo interno dos arquivos .7z sem extrair.
#
# Entrada:
#   D:\Fusion\figshare_30027313_v1
#
# Saídas:
#   _archive_inspection\archive_file_summary.csv
#   _archive_inspection\archive_contents_full.csv
#   _archive_inspection\archive_contents_by_extension.csv
#   _archive_inspection\archive_contents_by_top_folder.csv
#   _archive_inspection\archive_report.md
# ============================================================

from pathlib import Path
import subprocess
import shutil
import re
import json
import pandas as pd
from tqdm import tqdm


# ============================================================
# PATHS
# ============================================================
ROOT = Path(r"D:\Fusion\figshare_30027313_v1")

OUT_DIR = ROOT / "_archive_inspection"
OUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# CONFIG
# ============================================================
MAX_ARCHIVES = None  # use None para todos; use 5 para teste rápido


# ============================================================
# HELPERS
# ============================================================
def bytes_to_human(n):
    n = float(n)
    units = ["B", "KB", "MB", "GB", "TB"]
    i = 0
    while n >= 1024 and i < len(units) - 1:
        n /= 1024
        i += 1
    return f"{n:.2f} {units[i]}"


def find_7z_executable():
    candidates = [
        "7z",
        "7z.exe",
        r"C:\Program Files\7-Zip\7z.exe",
        r"C:\Program Files (x86)\7-Zip\7z.exe",
    ]

    for c in candidates:
        p = shutil.which(c)
        if p:
            return p

        if Path(c).exists():
            return c

    raise FileNotFoundError(
        "7-Zip não encontrado. Instale com: winget install 7zip.7zip"
    )


def get_extension(path_str):
    name = str(path_str).lower()
    if name.endswith(".tar.gz"):
        return ".tar.gz"
    return Path(name).suffix.lower()


def parse_7z_listing(text, archive_path):
    """
    Parseia saída de:
      7z l -slt arquivo.7z

    O formato -slt usa blocos com:
      Path =
      Size =
      Packed Size =
      Modified =
      Attributes =
      CRC =
      Method =
      Block =
    """
    rows = []
    current = {}

    for line in text.splitlines():
        line = line.rstrip("\n")

        if not line.strip():
            if current.get("Path"):
                rows.append(current)
            current = {}
            continue

        if " = " not in line:
            continue

        key, value = line.split(" = ", 1)
        key = key.strip()
        value = value.strip()

        current[key] = value

    if current.get("Path"):
        rows.append(current)

    parsed = []

    archive_name = Path(archive_path).name

    for r in rows:
        inner_path = r.get("Path", "")

        if not inner_path or inner_path == str(archive_path):
            continue

        size_raw = r.get("Size", "")
        packed_raw = r.get("Packed Size", "")

        try:
            size_bytes = int(size_raw) if size_raw != "" else 0
        except Exception:
            size_bytes = 0

        try:
            packed_bytes = int(packed_raw) if packed_raw != "" else 0
        except Exception:
            packed_bytes = 0

        attr = r.get("Attributes", "")
        is_dir = "D" in attr and size_bytes == 0

        parts = Path(inner_path).parts
        top_folder = parts[0] if len(parts) > 1 else "."

        parsed.append({
            "archive_name": archive_name,
            "archive_path": str(archive_path),
            "inner_path": inner_path,
            "inner_name": Path(inner_path).name,
            "top_folder": top_folder,
            "extension": get_extension(inner_path),
            "size_bytes": size_bytes,
            "size_human": bytes_to_human(size_bytes),
            "packed_bytes": packed_bytes,
            "modified": r.get("Modified", ""),
            "attributes": attr,
            "is_dir": bool(is_dir),
        })

    return parsed


def list_7z_contents(archive_path, sevenzip_exe):
    cmd = [
        sevenzip_exe,
        "l",
        "-slt",
        str(archive_path)
    ]

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        errors="ignore"
    )

    if result.returncode != 0:
        raise RuntimeError(result.stderr[:2000])

    return parse_7z_listing(result.stdout, archive_path)


def infer_target_from_archive_name(name):
    name_low = name.lower()

    targets = [
        "Matrice 30",
        "Phantom 4 Pro",
        "Inspire 2",
        "Mavic 2 Pro",
        "Corner Reflector",
        "CW Radar",
        "RF Receiver",
    ]

    for t in targets:
        if t.lower() in name_low:
            return t

    return "Unknown"


# ============================================================
# MAIN
# ============================================================
def main():
    print("=" * 80)
    print("INSPECTING .7z CONTENTS WITHOUT EXTRACTION")
    print("=" * 80)
    print(f"Root: {ROOT}")

    sevenzip = find_7z_executable()
    print(f"7-Zip executable: {sevenzip}")

    archives = sorted(ROOT.glob("*.7z"))

    if MAX_ARCHIVES is not None:
        archives = archives[:int(MAX_ARCHIVES)]

    print(f"Archives selected: {len(archives)}")

    all_rows = []
    summary_rows = []

    for archive in tqdm(archives, desc="Archives", unit="archive"):
        size_bytes = archive.stat().st_size

        row_summary = {
            "archive_name": archive.name,
            "archive_path": str(archive),
            "target": infer_target_from_archive_name(archive.name),
            "archive_size_bytes": int(size_bytes),
            "archive_size_human": bytes_to_human(size_bytes),
            "status": "ok",
            "n_items": 0,
            "n_files": 0,
            "n_dirs": 0,
            "total_unpacked_bytes": 0,
            "total_unpacked_human": "0 B",
            "error": "",
        }

        try:
            rows = list_7z_contents(archive, sevenzip)

            for r in rows:
                r["target"] = row_summary["target"]
                all_rows.append(r)

            df_tmp = pd.DataFrame(rows)

            if len(df_tmp) > 0:
                n_dirs = int(df_tmp["is_dir"].sum())
                n_files = int((~df_tmp["is_dir"]).sum())
                total_unpacked = int(df_tmp.loc[~df_tmp["is_dir"], "size_bytes"].sum())
            else:
                n_dirs = 0
                n_files = 0
                total_unpacked = 0

            row_summary["n_items"] = int(len(df_tmp))
            row_summary["n_files"] = n_files
            row_summary["n_dirs"] = n_dirs
            row_summary["total_unpacked_bytes"] = total_unpacked
            row_summary["total_unpacked_human"] = bytes_to_human(total_unpacked)

        except Exception as e:
            row_summary["status"] = "error"
            row_summary["error"] = str(e)

        summary_rows.append(row_summary)

    contents_df = pd.DataFrame(all_rows)
    archive_summary_df = pd.DataFrame(summary_rows)

    # --------------------------------------------------------
    # Save full tables
    # --------------------------------------------------------
    archive_summary_path = OUT_DIR / "archive_file_summary.csv"
    contents_path = OUT_DIR / "archive_contents_full.csv"

    archive_summary_df.to_csv(archive_summary_path, index=False, encoding="utf-8-sig")
    contents_df.to_csv(contents_path, index=False, encoding="utf-8-sig")

    # --------------------------------------------------------
    # Aggregations
    # --------------------------------------------------------
    if len(contents_df) > 0:
        files_df = contents_df[contents_df["is_dir"] == False].copy()

        ext_summary = (
            files_df
            .groupby(["target", "extension"], as_index=False)
            .agg(
                n_files=("inner_path", "count"),
                total_bytes=("size_bytes", "sum"),
                mean_bytes=("size_bytes", "mean"),
                max_bytes=("size_bytes", "max"),
            )
            .sort_values(["target", "total_bytes"], ascending=[True, False])
        )

        ext_summary["total_human"] = ext_summary["total_bytes"].apply(bytes_to_human)
        ext_summary["mean_human"] = ext_summary["mean_bytes"].apply(bytes_to_human)
        ext_summary["max_human"] = ext_summary["max_bytes"].apply(bytes_to_human)

        top_folder_summary = (
            files_df
            .groupby(["archive_name", "target", "top_folder"], as_index=False)
            .agg(
                n_files=("inner_path", "count"),
                total_bytes=("size_bytes", "sum"),
            )
            .sort_values(["archive_name", "total_bytes"], ascending=[True, False])
        )

        top_folder_summary["total_human"] = top_folder_summary["total_bytes"].apply(bytes_to_human)

    else:
        ext_summary = pd.DataFrame()
        top_folder_summary = pd.DataFrame()

    ext_summary_path = OUT_DIR / "archive_contents_by_extension.csv"
    top_folder_path = OUT_DIR / "archive_contents_by_top_folder.csv"

    ext_summary.to_csv(ext_summary_path, index=False, encoding="utf-8-sig")
    top_folder_summary.to_csv(top_folder_path, index=False, encoding="utf-8-sig")

    # --------------------------------------------------------
    # Markdown report
    # --------------------------------------------------------
    report_path = OUT_DIR / "archive_report.md"

    with open(report_path, "w", encoding="utf-8") as f:
        f.write("# Archive Inspection Report\n\n")
        f.write(f"Root: `{ROOT}`\n\n")

        f.write("## Archive summary\n\n")
        if len(archive_summary_df) > 0:
            cols = [
                "archive_name",
                "target",
                "archive_size_human",
                "n_files",
                "n_dirs",
                "total_unpacked_human",
                "status",
            ]
            f.write(archive_summary_df[cols].to_markdown(index=False))
            f.write("\n\n")

        f.write("## Extension summary\n\n")
        if len(ext_summary) > 0:
            cols = [
                "target",
                "extension",
                "n_files",
                "total_human",
                "mean_human",
                "max_human",
            ]
            f.write(ext_summary[cols].to_markdown(index=False))
            f.write("\n\n")

        f.write("## Outputs\n\n")
        f.write("- `archive_file_summary.csv`\n")
        f.write("- `archive_contents_full.csv`\n")
        f.write("- `archive_contents_by_extension.csv`\n")
        f.write("- `archive_contents_by_top_folder.csv`\n")

    # --------------------------------------------------------
    # Console
    # --------------------------------------------------------
    print("\n" + "=" * 80)
    print("ARCHIVE SUMMARY")
    print("=" * 80)

    if len(archive_summary_df) > 0:
        print(
            archive_summary_df[
                [
                    "archive_name",
                    "target",
                    "archive_size_human",
                    "n_files",
                    "total_unpacked_human",
                    "status",
                ]
            ].to_string(index=False)
        )

    print("\n" + "=" * 80)
    print("EXTENSION SUMMARY")
    print("=" * 80)

    if len(ext_summary) > 0:
        print(
            ext_summary[
                [
                    "target",
                    "extension",
                    "n_files",
                    "total_human",
                    "max_human",
                ]
            ].to_string(index=False)
        )

    print("\n" + "=" * 80)
    print("OUTPUT FILES")
    print("=" * 80)
    print(f"Archive summary : {archive_summary_path}")
    print(f"Full contents   : {contents_path}")
    print(f"By extension    : {ext_summary_path}")
    print(f"By top folder   : {top_folder_path}")
    print(f"Report          : {report_path}")

    print("\nDone.")


if __name__ == "__main__":
    main()

INSPECTING .7z CONTENTS WITHOUT EXTRACTION
Root: D:\Fusion\figshare_30027313_v1
7-Zip executable: C:\Program Files\7-Zip\7z.exe
Archives selected: 77


Archives: 100%|██████████| 77/77 [00:14<00:00,  5.33archive/s]



ARCHIVE SUMMARY
                    archive_name           target archive_size_human  n_files total_unpacked_human status
        001_57588574_CW Radar.7z         CW Radar            2.26 GB    75000              2.61 GB     ok
     002_57589510_RF Receiver.7z      RF Receiver           17.34 GB    75000             18.08 GB     ok
      006_57678868_Matrice 30.7z       Matrice 30            4.33 GB     1000              4.33 GB     ok
      007_57678871_Matrice 30.7z       Matrice 30            4.35 GB     1000              4.35 GB     ok
      008_57678874_Matrice 30.7z       Matrice 30            4.33 GB     1000              4.33 GB     ok
      009_57678877_Matrice 30.7z       Matrice 30            4.33 GB     1000              4.33 GB     ok
      010_57678880_Matrice 30.7z       Matrice 30            4.33 GB     1000              4.33 GB     ok
      011_57678883_Matrice 30.7z       Matrice 30            4.34 GB     1000              4.34 GB     ok
      012_57678886_Matrice 30

In [1]:
from pathlib import Path
import subprocess
import io
import random
import pandas as pd

# ============================================================
# CONFIGURAÇÕES
# ============================================================

ROOT = Path(r"D:\Fusion\figshare_30027313_v1")
SEVENZ = Path(r"C:\Program Files\7-Zip\7z.exe")

OUT_DIR = ROOT / "_zip_work"
OUT_DIR.mkdir(exist_ok=True)

# Filtros
TARGET_FILTER = None
# Exemplos:
# TARGET_FILTER = "Mavic 2 Pro"
# TARGET_FILTER = "Matrice 30"
# TARGET_FILTER = "RF Receiver"
# TARGET_FILTER = "CW Radar"

EXT_FILTER = ".mat"
MAX_FILES_TOTAL = 20
RANDOM_SAMPLE = True

# ============================================================
# FUNÇÕES
# ============================================================

def run_7z_list(archive_path):
    """
    Lista o conteúdo de um .7z usando 7z l -slt.
    Retorna lista de dicionários com Path e Size.
    """
    cmd = [str(SEVENZ), "l", "-slt", str(archive_path)]

    result = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        errors="ignore"
    )

    if result.returncode != 0:
        print(f"[ERRO] Falha ao listar: {archive_path.name}")
        print(result.stderr)
        return []

    records = []
    current = {}

    for line in result.stdout.splitlines():
        line = line.strip()

        if not line:
            if current:
                records.append(current)
                current = {}
            continue

        if " = " in line:
            key, value = line.split(" = ", 1)
            current[key.strip()] = value.strip()

    if current:
        records.append(current)

    files = []
    for r in records:
        path = r.get("Path")
        size = r.get("Size")

        if path and size and not path.endswith("\\") and not path.endswith("/"):
            try:
                size_int = int(size)
            except Exception:
                size_int = None

            files.append({
                "archive": archive_path,
                "archive_name": archive_path.name,
                "inner_path": path,
                "size_bytes": size_int,
                "extension": Path(path).suffix.lower()
            })

    return files


def extract_file_to_bytes(archive_path, inner_path):
    """
    Extrai um arquivo específico do .7z para memória.
    Não salva no disco.
    """
    cmd = [
        str(SEVENZ),
        "x",
        "-so",
        str(archive_path),
        inner_path
    ]

    result = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )

    if result.returncode != 0:
        raise RuntimeError(
            f"Erro ao extrair {inner_path} de {archive_path.name}:\n"
            f"{result.stderr.decode(errors='ignore')}"
        )

    return result.stdout


def summarize_mat_from_bytes(data):
    """
    Tenta ler um arquivo .mat a partir de bytes.
    Suporta MAT antigo via scipy.io.loadmat.
    Para MAT v7.3, tenta h5py.
    """
    bio = io.BytesIO(data)

    # Tentativa 1: MAT clássico
    try:
        from scipy.io import loadmat

        mat = loadmat(bio)
        summary = {}

        for k, v in mat.items():
            if not k.startswith("__"):
                shape = getattr(v, "shape", None)
                dtype = getattr(v, "dtype", None)
                summary[k] = {
                    "shape": str(shape),
                    "dtype": str(dtype)
                }

        return "scipy", summary

    except Exception as e1:
        # Tentativa 2: MAT v7.3 / HDF5
        try:
            import h5py

            bio.seek(0)
            summary = {}

            with h5py.File(bio, "r") as f:
                def visit(name, obj):
                    if hasattr(obj, "shape"):
                        summary[name] = {
                            "shape": str(obj.shape),
                            "dtype": str(obj.dtype)
                        }

                f.visititems(visit)

            return "h5py", summary

        except Exception as e2:
            return "failed", {
                "scipy_error": str(e1),
                "h5py_error": str(e2)
            }


# ============================================================
# 1) INDEXAR ARQUIVOS DENTRO DOS .7z
# ============================================================

archives = sorted(ROOT.glob("*.7z"))

print("=" * 80)
print("TRABALHANDO DIRETAMENTE NOS .7z")
print("=" * 80)
print(f"Pasta raiz : {ROOT}")
print(f"Arquivos .7z encontrados: {len(archives)}")
print()

all_files = []

for archive in archives:
    print(f"[LISTANDO] {archive.name}")
    files = run_7z_list(archive)
    all_files.extend(files)

df = pd.DataFrame(all_files)

if df.empty:
    raise RuntimeError("Nenhum arquivo encontrado dentro dos .7z.")

index_path = OUT_DIR / "zip_index.csv"
df.to_csv(index_path, index=False, encoding="utf-8-sig")

print()
print("=" * 80)
print("ÍNDICE GERADO")
print("=" * 80)
print(f"Total de arquivos indexados: {len(df)}")
print(f"Arquivo salvo em: {index_path}")
print()

# ============================================================
# 2) FILTRAR ARQUIVOS DE INTERESSE
# ============================================================

sel = df[df["extension"].eq(EXT_FILTER.lower())].copy()

if TARGET_FILTER is not None:
    mask_archive = sel["archive_name"].str.contains(TARGET_FILTER, case=False, na=False)
    mask_inner = sel["inner_path"].str.contains(TARGET_FILTER, case=False, na=False)
    sel = sel[mask_archive | mask_inner].copy()

print("=" * 80)
print("SELEÇÃO")
print("=" * 80)
print(f"Extensão selecionada: {EXT_FILTER}")
print(f"Filtro de alvo      : {TARGET_FILTER}")
print(f"Arquivos disponíveis: {len(sel)}")

if sel.empty:
    raise RuntimeError("Nenhum arquivo encontrado com os filtros escolhidos.")

if RANDOM_SAMPLE:
    sel = sel.sample(
        n=min(MAX_FILES_TOTAL, len(sel)),
        random_state=42
    )
else:
    sel = sel.head(MAX_FILES_TOTAL)

print(f"Arquivos que serão lidos: {len(sel)}")
print()

# ============================================================
# 3) LER OS .mat DIRETAMENTE DO .7z
# ============================================================

results = []

for i, row in sel.reset_index(drop=True).iterrows():
    archive_path = Path(row["archive"])
    inner_path = row["inner_path"]

    print(f"[{i+1}/{len(sel)}] Lendo:")
    print(f"  Archive: {archive_path.name}")
    print(f"  Arquivo: {inner_path}")

    try:
        data = extract_file_to_bytes(archive_path, inner_path)
        backend, summary = summarize_mat_from_bytes(data)

        results.append({
            "archive_name": archive_path.name,
            "inner_path": inner_path,
            "compressed_read_bytes": len(data),
            "backend": backend,
            "summary": str(summary)
        })

        print(f"  OK | backend={backend} | bytes={len(data):,}")

    except Exception as e:
        results.append({
            "archive_name": archive_path.name,
            "inner_path": inner_path,
            "compressed_read_bytes": None,
            "backend": "error",
            "summary": str(e)
        })

        print(f"  ERRO: {e}")

    print()

# ============================================================
# 4) SALVAR RESUMO
# ============================================================

res_df = pd.DataFrame(results)
res_path = OUT_DIR / "zip_mat_read_summary.csv"
res_df.to_csv(res_path, index=False, encoding="utf-8-sig")

print("=" * 80)
print("FINALIZADO")
print("=" * 80)
print(f"Resumo salvo em: {res_path}")
print(res_df.head())

TRABALHANDO DIRETAMENTE NOS .7z
Pasta raiz : D:\Fusion\figshare_30027313_v1
Arquivos .7z encontrados: 77

[LISTANDO] 001_57588574_CW Radar.7z
[LISTANDO] 002_57589510_RF Receiver.7z
[LISTANDO] 006_57678868_Matrice 30.7z
[LISTANDO] 007_57678871_Matrice 30.7z
[LISTANDO] 008_57678874_Matrice 30.7z
[LISTANDO] 009_57678877_Matrice 30.7z
[LISTANDO] 010_57678880_Matrice 30.7z
[LISTANDO] 011_57678883_Matrice 30.7z
[LISTANDO] 012_57678886_Matrice 30.7z
[LISTANDO] 013_57678889_Matrice 30.7z
[LISTANDO] 014_57678892_Matrice 30.7z
[LISTANDO] 015_57678895_Matrice 30.7z
[LISTANDO] 016_57678898_Matrice 30.7z
[LISTANDO] 017_57684178_Phantom 4 Pro.7z
[LISTANDO] 018_57684175_Corner Reflector.7z
[LISTANDO] 019_57684181_Inspire 2.7z
[LISTANDO] 020_57684184_Mavic 2 Pro.7z
[LISTANDO] 021_57685750_Phantom 4 Pro.7z
[LISTANDO] 022_57685747_Corner Reflector.7z
[LISTANDO] 023_57685756_Inspire 2.7z
[LISTANDO] 024_57685759_Mavic 2 Pro.7z
[LISTANDO] 025_57687649_Phantom 4 Pro.7z
[LISTANDO] 026_57687643_Corner Reflect

# EXTRAINDO ARQUIVOS E ORGANIZANDO EM PASTAS

In [2]:
from pathlib import Path
import shutil
import subprocess
import re
import csv
import time
from collections import Counter

# ============================================================
# CONFIGURAÇÕES
# ============================================================

SOURCE_ROOT = Path(r"D:\Fusion\figshare_30027313_v1")

DEST_ROOT = Path(r"D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)")

SEVENZ_CANDIDATES = [
    Path(r"C:\Program Files\7-Zip\7z.exe"),
    Path(r"C:\Program Files (x86)\7-Zip\7z.exe"),
]

TMP_ROOT = DEST_ROOT / "_tmp_extract"
LOG_DIR = DEST_ROOT / "_reorganization_logs"

VALID_EXTENSIONS = {".mat", ".png", ".m"}

OVERWRITE = False
DELETE_TEMP_AFTER_SUCCESS = True
KEEP_TEMP_IF_ERROR = True

# Para testar primeiro, coloque LIMIT_ARCHIVES = 2.
# Para extrair tudo, deixe None.
LIMIT_ARCHIVES = None

SENSORS = [
    "CW Radar",
    "FMCW Radar",
    "RF Receiver",
]

TARGETS = [
    "Inspire 2",
    "Matrice 30",
    "Mavic 2 Pro",
    "Phantom 4 Pro",
    "Corner Reflector",
]

DISTANCES = [f"{d}m" for d in range(2, 31, 2)]


# ============================================================
# UTILITÁRIOS
# ============================================================

def find_7z():
    for p in SEVENZ_CANDIDATES:
        if p.exists():
            return p

    exe = shutil.which("7z") or shutil.which("7za")
    if exe:
        return Path(exe)

    raise FileNotFoundError(
        "7-Zip não encontrado. Instale o 7-Zip ou ajuste o caminho em SEVENZ_CANDIDATES."
    )


def clean_windows_name(name: str) -> str:
    name = str(name).strip()
    name = re.sub(r'[<>:"/\\|?*]', "_", name)
    name = re.sub(r"\s+", " ", name)
    return name


def normalize_text(text: str) -> str:
    text = str(text).lower()
    text = text.replace("\\", " ")
    text = text.replace("/", " ")
    text = text.replace("_", " ")
    text = text.replace("-", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def infer_sensor(text: str, archive_name: str = ""):
    t = normalize_text(text)
    a = normalize_text(archive_name)

    # Ordem importante: FMCW contém CW no nome.
    if re.search(r"\bfmcw\b", t) or re.search(r"\bfmcw\b", a):
        return "FMCW Radar"

    if re.search(r"\brf\s*receiver\b", t) or re.search(r"\brf\s*receiver\b", a):
        return "RF Receiver"

    if re.search(r"\bcw\s*radar\b", t) or re.search(r"\bcw\s*radar\b", a):
        return "CW Radar"

    # Os blocos .7z com nomes de alvos, no seu log, correspondem ao FMCW.
    if infer_target(archive_name) is not None:
        return "FMCW Radar"

    return None


def infer_target(text: str):
    t = normalize_text(text)

    patterns = [
        ("Corner Reflector", [
            r"\bcorner\s*reflector\b",
            r"\bcorner\b",
        ]),
        ("Phantom 4 Pro", [
            r"\bphantom\s*4\s*pro\b",
            r"\bphantom4pro\b",
            r"\bphantom\b",
        ]),
        ("Mavic 2 Pro", [
            r"\bmavic\s*2\s*pro\b",
            r"\bmavic2pro\b",
            r"\bmavic\b",
        ]),
        ("Matrice 30", [
            r"\bmatrice\s*30\b",
            r"\bmatrice\s*t30\b",
            r"\bmatrice30\b",
            r"\bmatrice\b",
        ]),
        ("Inspire 2", [
            r"\binspire\s*2\b",
            r"\binspire2\b",
            r"\binspire\b",
        ]),
    ]

    for target, pats in patterns:
        for pat in pats:
            if re.search(pat, t):
                return target

    return None


def infer_distance(text: str):
    t = normalize_text(text)

    for d in range(2, 31, 2):
        # 2m, 2 m, 30m, 30 m
        pat = rf"(?<!\d){d}\s*m(?![a-z0-9])"
        if re.search(pat, t):
            return f"{d}m"

    return None


def infer_index(file_path: Path):
    stem = file_path.stem

    # Pega número final do arquivo: xxx_001.mat
    m = re.search(r"(?<!\d)(\d{1,6})(?!\d)$", stem)
    if m:
        raw = m.group(1)
        return raw.zfill(max(3, len(raw)))

    nums = re.findall(r"\d+", stem)
    if nums:
        raw = nums[-1]
        return raw.zfill(max(3, len(raw)))

    return None


def make_base_tree():
    DEST_ROOT.mkdir(parents=True, exist_ok=True)
    (DEST_ROOT / "Sample Code").mkdir(parents=True, exist_ok=True)
    (DEST_ROOT / "Dataset").mkdir(parents=True, exist_ok=True)

    for sensor in SENSORS:
        for distance in DISTANCES:
            for target in TARGETS:
                (DEST_ROOT / "Dataset" / sensor / distance / target / "Raw File").mkdir(
                    parents=True, exist_ok=True
                )
                (DEST_ROOT / "Dataset" / sensor / distance / target / "Image File").mkdir(
                    parents=True, exist_ok=True
                )


def destination_for_file(src_file: Path, archive_name: str):
    ext = src_file.suffix.lower()

    if ext not in VALID_EXTENSIONS:
        return None, "extensão ignorada"

    # Arquivos MATLAB de código vão para Sample Code.
    if ext == ".m":
        sensor = infer_sensor(str(src_file), archive_name)

        if sensor is None:
            dest = DEST_ROOT / "Sample Code" / clean_windows_name(src_file.name)
        else:
            dest = DEST_ROOT / "Sample Code" / sensor / clean_windows_name(src_file.name)

        return {
            "dest": dest,
            "sensor": sensor or "",
            "distance": "",
            "target": "",
            "kind": "Sample Code",
            "extension": ext,
        }, None

    text = f"{archive_name} " + " ".join(src_file.parts)

    sensor = infer_sensor(text, archive_name)
    target = infer_target(text)
    distance = infer_distance(text)
    index = infer_index(src_file)

    if sensor is None:
        return None, "sensor não identificado"

    if target is None:
        return None, "alvo não identificado"

    if distance is None:
        return None, "distância não identificada"

    if index is None:
        return None, "índice não identificado"

    kind = "Raw File" if ext == ".mat" else "Image File"

    out_name = f"{target}_{distance}_{index}{ext}"
    out_name = clean_windows_name(out_name)

    dest = (
        DEST_ROOT
        / "Dataset"
        / sensor
        / distance
        / target
        / kind
        / out_name
    )

    return {
        "dest": dest,
        "sensor": sensor,
        "distance": distance,
        "target": target,
        "kind": kind,
        "extension": ext,
    }, None


def move_file_to_structure(src_file: Path, archive_name: str, logs, counts):
    info, error = destination_for_file(src_file, archive_name)

    if error:
        logs.append({
            "status": "ignored",
            "reason": error,
            "archive": archive_name,
            "source": str(src_file),
            "destination": "",
            "sensor": "",
            "distance": "",
            "target": "",
            "kind": "",
            "extension": src_file.suffix.lower(),
            "size_bytes": src_file.stat().st_size if src_file.exists() else "",
        })
        counts[("ignored", error)] += 1
        return

    dest = info["dest"]
    dest.parent.mkdir(parents=True, exist_ok=True)

    size_bytes = src_file.stat().st_size

    if dest.exists() and not OVERWRITE:
        status = "skipped_existing"
        reason = "arquivo já existe"
    else:
        if dest.exists() and OVERWRITE:
            dest.unlink()

        shutil.move(str(src_file), str(dest))
        status = "moved"
        reason = ""

    logs.append({
        "status": status,
        "reason": reason,
        "archive": archive_name,
        "source": str(src_file),
        "destination": str(dest),
        "sensor": info["sensor"],
        "distance": info["distance"],
        "target": info["target"],
        "kind": info["kind"],
        "extension": info["extension"],
        "size_bytes": size_bytes,
    })

    counts[
        (
            status,
            info["sensor"],
            info["distance"],
            info["target"],
            info["kind"],
            info["extension"],
        )
    ] += 1


def extract_archive(archive_path: Path, temp_dir: Path, sevenz: Path):
    if temp_dir.exists():
        shutil.rmtree(temp_dir, ignore_errors=True)

    temp_dir.mkdir(parents=True, exist_ok=True)

    cmd = [
        str(sevenz),
        "x",
        str(archive_path),
        f"-o{str(temp_dir)}",
        "-y",
        "-bb0",
        "-bso0",
        "-bsp1",
    ]

    print(f"[7Z] Extraindo: {archive_path.name}")

    result = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        errors="ignore",
    )

    if result.returncode != 0:
        raise RuntimeError(
            f"Falha ao extrair: {archive_path.name}\n\n"
            f"STDOUT:\n{result.stdout}\n\n"
            f"STDERR:\n{result.stderr}"
        )


def write_logs(logs, counts):
    LOG_DIR.mkdir(parents=True, exist_ok=True)

    log_file = LOG_DIR / "file_log.csv"
    summary_file = LOG_DIR / "summary.csv"

    fieldnames = [
        "status",
        "reason",
        "archive",
        "source",
        "destination",
        "sensor",
        "distance",
        "target",
        "kind",
        "extension",
        "size_bytes",
    ]

    with open(log_file, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in logs:
            writer.writerow(row)

    with open(summary_file, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.writer(f)
        writer.writerow([
            "status",
            "sensor_or_reason",
            "distance",
            "target",
            "kind",
            "extension",
            "count",
        ])

        for key, value in sorted(counts.items(), key=lambda x: str(x[0])):
            row = list(key)
            while len(row) < 6:
                row.append("")
            writer.writerow(row[:6] + [value])

    return log_file, summary_file


def print_disk_info():
    anchor = DEST_ROOT
    anchor.mkdir(parents=True, exist_ok=True)

    usage = shutil.disk_usage(anchor)
    free_gb = usage.free / (1024 ** 3)
    total_gb = usage.total / (1024 ** 3)

    print("=" * 80)
    print("DISCO DE DESTINO")
    print("=" * 80)
    print(f"Destino       : {DEST_ROOT}")
    print(f"Espaço total  : {total_gb:.2f} GB")
    print(f"Espaço livre  : {free_gb:.2f} GB")
    print("=" * 80)
    print()


# ============================================================
# MAIN
# ============================================================

def main():
    t0 = time.time()

    sevenz = find_7z()

    print("=" * 80)
    print("EXTRAÇÃO E REORGANIZAÇÃO DO TSMS-DRONE")
    print("=" * 80)
    print(f"Fonte      : {SOURCE_ROOT}")
    print(f"Destino    : {DEST_ROOT}")
    print(f"7-Zip      : {sevenz}")
    print(f"Overwrite  : {OVERWRITE}")
    print("=" * 80)
    print()

    if not SOURCE_ROOT.exists():
        raise FileNotFoundError(f"Pasta fonte não existe: {SOURCE_ROOT}")

    print_disk_info()
    make_base_tree()

    archives = sorted(SOURCE_ROOT.glob("*.7z"))

    if LIMIT_ARCHIVES is not None:
        archives = archives[:LIMIT_ARCHIVES]

    if not archives:
        raise RuntimeError(f"Nenhum .7z encontrado em: {SOURCE_ROOT}")

    print(f"Arquivos .7z encontrados: {len(archives)}")
    print()

    logs = []
    counts = Counter()

    for idx, archive in enumerate(archives, start=1):
        print("=" * 80)
        print(f"[{idx}/{len(archives)}] {archive.name}")
        print("=" * 80)

        archive_temp = TMP_ROOT / archive.stem

        try:
            extract_archive(archive, archive_temp, sevenz)

            extracted_files = [
                p for p in archive_temp.rglob("*")
                if p.is_file() and p.suffix.lower() in VALID_EXTENSIONS
            ]

            print(f"Arquivos úteis extraídos: {len(extracted_files)}")

            for f in extracted_files:
                move_file_to_structure(f, archive.name, logs, counts)

            if DELETE_TEMP_AFTER_SUCCESS:
                shutil.rmtree(archive_temp, ignore_errors=True)

            log_file, summary_file = write_logs(logs, counts)
            print(f"[OK] Finalizado: {archive.name}")
            print(f"[LOG] {log_file}")
            print(f"[SUM] {summary_file}")

        except Exception as e:
            print(f"[ERRO] {archive.name}")
            print(str(e))

            logs.append({
                "status": "archive_error",
                "reason": str(e),
                "archive": archive.name,
                "source": str(archive),
                "destination": "",
                "sensor": "",
                "distance": "",
                "target": "",
                "kind": "",
                "extension": ".7z",
                "size_bytes": archive.stat().st_size,
            })
            counts[("archive_error", archive.name)] += 1

            write_logs(logs, counts)

            if not KEEP_TEMP_IF_ERROR:
                shutil.rmtree(archive_temp, ignore_errors=True)

            # Continua para o próximo .7z
            continue

        print()

    log_file, summary_file = write_logs(logs, counts)

    elapsed = time.time() - t0

    print()
    print("=" * 80)
    print("FINALIZADO")
    print("=" * 80)
    print(f"Destino final : {DEST_ROOT}")
    print(f"Log completo  : {log_file}")
    print(f"Resumo        : {summary_file}")
    print(f"Tempo total   : {elapsed/60:.2f} min")
    print("=" * 80)


if __name__ == "__main__":
    main()

EXTRAÇÃO E REORGANIZAÇÃO DO TSMS-DRONE
Fonte      : D:\Fusion\figshare_30027313_v1
Destino    : D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)
7-Zip      : C:\Program Files\7-Zip\7z.exe
Overwrite  : False

DISCO DE DESTINO
Destino       : D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)
Espaço total  : 1862.76 GB
Espaço livre  : 1345.11 GB

Arquivos .7z encontrados: 77

[1/77] 001_57588574_CW Radar.7z
[7Z] Extraindo: 001_57588574_CW Radar.7z
Arquivos úteis extraídos: 75000
[OK] Finalizado: 001_57588574_CW Radar.7z
[LOG] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_reorganization_logs\file_log.csv
[SUM] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_reorganization_logs\summary.csv

[2/77] 002_57589510_RF Receiver.7z
[7Z] Extraindo: 002_57589510_RF Receiver.7z
Arquivos úteis extraídos: 75000
[OK] Finalizado: 002_57589510_RF Receiver.7z
[LOG] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_reorganization_logs\file_log

In [3]:
from pathlib import Path
import csv
import re
from collections import Counter, defaultdict

# ============================================================
# CONFIGURAÇÕES
# ============================================================

ROOT = Path(r"D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)")

DATASET_ROOT = ROOT / "Dataset"
REPORT_DIR = ROOT / "_organization_check"

SENSORS = [
    "CW Radar",
    "FMCW Radar",
    "RF Receiver",
]

TARGETS = [
    "Inspire 2",
    "Matrice 30",
    "Mavic 2 Pro",
    "Phantom 4 Pro",
    "Corner Reflector",
]

DISTANCES = [f"{d}m" for d in range(2, 31, 2)]

EXPECTED_MAT_PER_CONDITION = 500
EXPECTED_PNG_PER_CONDITION = 500

# Verificação leve do conteúdo dos .mat.
# Abre apenas alguns arquivos por condição para conferir variável e dimensão.
CHECK_MAT_CONTENT = True
MAX_MAT_CHECKS_PER_CONDITION = 1

EXPECTED_MAT_VARIABLE = {
    "CW Radar": "data",
    "RF Receiver": "data",
    "FMCW Radar": "RD",
}

EXPECTED_SHAPE = {
    "CW Radar": (16384,),
    "RF Receiver": (262144,),
    "FMCW Radar": (161, 4096),
}


# ============================================================
# UTILITÁRIOS
# ============================================================

def ensure_dir(path: Path):
    path.mkdir(parents=True, exist_ok=True)


def normalize_stem(stem: str):
    """
    Remove diferenças pequenas de separador para comparar nomes-base.
    """
    s = stem.lower().strip()
    s = re.sub(r"\s+", " ", s)
    return s


def expected_name_pattern(target: str, distance: str, ext: str):
    """
    Padrão esperado:
    Inspire 2_2m_001.mat
    Inspire 2_2m_001.png
    """
    target_escaped = re.escape(target)
    distance_escaped = re.escape(distance)
    ext_escaped = re.escape(ext)

    return re.compile(
        rf"^{target_escaped}_{distance_escaped}_\d{{3,6}}{ext_escaped}$",
        re.IGNORECASE
    )


def human_size(num_bytes):
    if num_bytes is None:
        return ""

    units = ["B", "KB", "MB", "GB", "TB"]
    size = float(num_bytes)

    for unit in units:
        if size < 1024:
            return f"{size:.2f} {unit}"
        size /= 1024

    return f"{size:.2f} PB"


def write_csv(path: Path, rows, fieldnames):
    ensure_dir(path.parent)

    with open(path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)


def check_mat_file(mat_path: Path, sensor: str):
    """
    Verifica variável e shape do .mat.

    Prioriza h5py porque o artigo informa compatibilidade MATLAB v7.3.
    Caso falhe, tenta scipy.io.loadmat para arquivos MAT antigos.
    """
    expected_var = EXPECTED_MAT_VARIABLE[sensor]
    expected_shape = EXPECTED_SHAPE[sensor]

    result = {
        "file": str(mat_path),
        "sensor": sensor,
        "expected_variable": expected_var,
        "expected_shape": str(expected_shape),
        "backend": "",
        "status": "",
        "found_variable": "",
        "found_shape": "",
        "error": "",
    }

    # Tentativa 1: MATLAB v7.3 / HDF5
    try:
        import h5py

        with h5py.File(mat_path, "r") as f:
            keys = list(f.keys())

            if expected_var not in keys:
                result["backend"] = "h5py"
                result["status"] = "error"
                result["error"] = f"Variável esperada não encontrada. Variáveis disponíveis: {keys}"
                return result

            obj = f[expected_var]
            shape = tuple(obj.shape)

            result["backend"] = "h5py"
            result["found_variable"] = expected_var
            result["found_shape"] = str(shape)

            # Em HDF5/MAT v7.3, às vezes vetores aparecem transpostos.
            shape_ok = (
                shape == expected_shape
                or shape == tuple(reversed(expected_shape))
                or sorted(shape) == sorted(expected_shape)
            )

            if shape_ok:
                result["status"] = "ok"
            else:
                result["status"] = "warning"
                result["error"] = "Shape diferente do esperado"

            return result

    except Exception as e_h5:
        h5_error = str(e_h5)

    # Tentativa 2: MATLAB antigo
    try:
        from scipy.io import loadmat

        mat = loadmat(mat_path)
        keys = [k for k in mat.keys() if not k.startswith("__")]

        if expected_var not in keys:
            result["backend"] = "scipy"
            result["status"] = "error"
            result["error"] = f"Variável esperada não encontrada. Variáveis disponíveis: {keys}"
            return result

        arr = mat[expected_var]
        shape = tuple(arr.shape)

        result["backend"] = "scipy"
        result["found_variable"] = expected_var
        result["found_shape"] = str(shape)

        shape_ok = (
            shape == expected_shape
            or shape == tuple(reversed(expected_shape))
            or arr.size == int_product(expected_shape)
        )

        if shape_ok:
            result["status"] = "ok"
        else:
            result["status"] = "warning"
            result["error"] = "Shape diferente do esperado"

        return result

    except Exception as e_sp:
        result["backend"] = "failed"
        result["status"] = "error"
        result["error"] = f"h5py: {h5_error} | scipy: {e_sp}"
        return result


def int_product(values):
    p = 1
    for v in values:
        p *= int(v)
    return p


# ============================================================
# MAIN
# ============================================================

def main():
    ensure_dir(REPORT_DIR)

    print("=" * 80)
    print("CONFERÊNCIA DA ORGANIZAÇÃO DO TSMS-DRONE")
    print("=" * 80)
    print(f"ROOT        : {ROOT}")
    print(f"DATASET     : {DATASET_ROOT}")
    print(f"REPORT_DIR  : {REPORT_DIR}")
    print("=" * 80)
    print()

    if not ROOT.exists():
        raise FileNotFoundError(f"Pasta raiz não encontrada: {ROOT}")

    if not DATASET_ROOT.exists():
        raise FileNotFoundError(f"Pasta Dataset não encontrada: {DATASET_ROOT}")

    missing_dirs = []
    condition_rows = []
    pairing_errors = []
    naming_errors = []
    mat_check_rows = []

    total_counter = Counter()
    size_by_sensor = Counter()
    size_by_target = Counter()
    size_by_extension = Counter()

    # ========================================================
    # 1) VERIFICAR ESTRUTURA SENSOR / DISTÂNCIA / ALVO
    # ========================================================

    for sensor in SENSORS:
        sensor_dir = DATASET_ROOT / sensor

        if not sensor_dir.exists():
            missing_dirs.append({
                "level": "sensor",
                "sensor": sensor,
                "distance": "",
                "target": "",
                "missing_path": str(sensor_dir),
            })
            continue

        for distance in DISTANCES:
            distance_dir = sensor_dir / distance

            if not distance_dir.exists():
                missing_dirs.append({
                    "level": "distance",
                    "sensor": sensor,
                    "distance": distance,
                    "target": "",
                    "missing_path": str(distance_dir),
                })
                continue

            for target in TARGETS:
                target_dir = distance_dir / target
                raw_dir = target_dir / "Raw File"
                img_dir = target_dir / "Image File"

                if not target_dir.exists():
                    missing_dirs.append({
                        "level": "target",
                        "sensor": sensor,
                        "distance": distance,
                        "target": target,
                        "missing_path": str(target_dir),
                    })
                    continue

                if not raw_dir.exists():
                    missing_dirs.append({
                        "level": "Raw File",
                        "sensor": sensor,
                        "distance": distance,
                        "target": target,
                        "missing_path": str(raw_dir),
                    })

                if not img_dir.exists():
                    missing_dirs.append({
                        "level": "Image File",
                        "sensor": sensor,
                        "distance": distance,
                        "target": target,
                        "missing_path": str(img_dir),
                    })

                mat_files = sorted(raw_dir.glob("*.mat")) if raw_dir.exists() else []
                png_files = sorted(img_dir.glob("*.png")) if img_dir.exists() else []

                mat_count = len(mat_files)
                png_count = len(png_files)

                mat_size = sum(f.stat().st_size for f in mat_files)
                png_size = sum(f.stat().st_size for f in png_files)

                status = "ok"
                notes = []

                if mat_count != EXPECTED_MAT_PER_CONDITION:
                    status = "error"
                    notes.append(f".mat esperado={EXPECTED_MAT_PER_CONDITION}, encontrado={mat_count}")

                if png_count != EXPECTED_PNG_PER_CONDITION:
                    status = "error"
                    notes.append(f".png esperado={EXPECTED_PNG_PER_CONDITION}, encontrado={png_count}")

                # ====================================================
                # 2) VERIFICAR PAREAMENTO MAT/PNG
                # ====================================================

                mat_stems = {normalize_stem(f.stem): f for f in mat_files}
                png_stems = {normalize_stem(f.stem): f for f in png_files}

                missing_png = sorted(set(mat_stems) - set(png_stems))
                missing_mat = sorted(set(png_stems) - set(mat_stems))

                if missing_png or missing_mat:
                    status = "error"
                    notes.append(f"pareamento inconsistente: missing_png={len(missing_png)}, missing_mat={len(missing_mat)}")

                    for stem in missing_png[:200]:
                        pairing_errors.append({
                            "sensor": sensor,
                            "distance": distance,
                            "target": target,
                            "problem": "mat_sem_png",
                            "stem": stem,
                            "mat_file": str(mat_stems[stem]),
                            "png_file": "",
                        })

                    for stem in missing_mat[:200]:
                        pairing_errors.append({
                            "sensor": sensor,
                            "distance": distance,
                            "target": target,
                            "problem": "png_sem_mat",
                            "stem": stem,
                            "mat_file": "",
                            "png_file": str(png_stems[stem]),
                        })

                # ====================================================
                # 3) VERIFICAR PADRÃO DE NOMES
                # ====================================================

                mat_pattern = expected_name_pattern(target, distance, ".mat")
                png_pattern = expected_name_pattern(target, distance, ".png")

                for f in mat_files:
                    if not mat_pattern.match(f.name):
                        naming_errors.append({
                            "sensor": sensor,
                            "distance": distance,
                            "target": target,
                            "file": str(f),
                            "expected_pattern": f"{target}_{distance}_001.mat",
                        })

                for f in png_files:
                    if not png_pattern.match(f.name):
                        naming_errors.append({
                            "sensor": sensor,
                            "distance": distance,
                            "target": target,
                            "file": str(f),
                            "expected_pattern": f"{target}_{distance}_001.png",
                        })

                if naming_errors:
                    # não muda tudo para erro, pois pode ser só nome diferente
                    if status == "ok":
                        status = "warning"

                # ====================================================
                # 4) VERIFICAÇÃO LEVE DO CONTEÚDO DOS MAT
                # ====================================================

                if CHECK_MAT_CONTENT and mat_files:
                    for mat_file in mat_files[:MAX_MAT_CHECKS_PER_CONDITION]:
                        mat_result = check_mat_file(mat_file, sensor)
                        mat_result.update({
                            "distance": distance,
                            "target": target,
                        })
                        mat_check_rows.append(mat_result)

                        if mat_result["status"] == "error":
                            status = "error"
                            notes.append("erro na leitura de .mat")
                        elif mat_result["status"] == "warning" and status == "ok":
                            status = "warning"

                condition_rows.append({
                    "status": status,
                    "sensor": sensor,
                    "distance": distance,
                    "target": target,
                    "mat_count": mat_count,
                    "png_count": png_count,
                    "mat_size_bytes": mat_size,
                    "png_size_bytes": png_size,
                    "mat_size_human": human_size(mat_size),
                    "png_size_human": human_size(png_size),
                    "missing_png_count": len(missing_png),
                    "missing_mat_count": len(missing_mat),
                    "notes": " | ".join(notes),
                })

                total_counter["conditions_checked"] += 1
                total_counter["mat_files"] += mat_count
                total_counter["png_files"] += png_count
                total_counter[f"condition_{status}"] += 1

                size_by_sensor[sensor] += mat_size + png_size
                size_by_target[target] += mat_size + png_size
                size_by_extension[".mat"] += mat_size
                size_by_extension[".png"] += png_size

    # ========================================================
    # 5) PROCURAR ARQUIVOS FORA DO PADRÃO
    # ========================================================

    known_roots = {
        str(DATASET_ROOT / s) for s in SENSORS
    }

    extra_files = []

    for f in DATASET_ROOT.rglob("*"):
        if not f.is_file():
            continue

        ext = f.suffix.lower()

        if ext not in {".mat", ".png"}:
            extra_files.append({
                "problem": "extensao_inesperada",
                "file": str(f),
                "extension": ext,
                "size_bytes": f.stat().st_size,
                "size_human": human_size(f.stat().st_size),
            })

    # ========================================================
    # 6) SALVAR RELATÓRIOS CSV
    # ========================================================

    write_csv(
        REPORT_DIR / "missing_dirs.csv",
        missing_dirs,
        ["level", "sensor", "distance", "target", "missing_path"]
    )

    write_csv(
        REPORT_DIR / "condition_counts.csv",
        condition_rows,
        [
            "status", "sensor", "distance", "target",
            "mat_count", "png_count",
            "mat_size_bytes", "png_size_bytes",
            "mat_size_human", "png_size_human",
            "missing_png_count", "missing_mat_count",
            "notes",
        ]
    )

    write_csv(
        REPORT_DIR / "pairing_errors.csv",
        pairing_errors,
        ["sensor", "distance", "target", "problem", "stem", "mat_file", "png_file"]
    )

    write_csv(
        REPORT_DIR / "naming_errors.csv",
        naming_errors,
        ["sensor", "distance", "target", "file", "expected_pattern"]
    )

    write_csv(
        REPORT_DIR / "extra_files.csv",
        extra_files,
        ["problem", "file", "extension", "size_bytes", "size_human"]
    )

    if CHECK_MAT_CONTENT:
        write_csv(
            REPORT_DIR / "mat_content_check.csv",
            mat_check_rows,
            [
                "status", "backend", "sensor", "distance", "target",
                "file", "expected_variable", "found_variable",
                "expected_shape", "found_shape", "error",
            ]
        )

    # ========================================================
    # 7) RELATÓRIO MARKDOWN
    # ========================================================

    expected_conditions = len(SENSORS) * len(DISTANCES) * len(TARGETS)
    expected_total_mat = expected_conditions * EXPECTED_MAT_PER_CONDITION
    expected_total_png = expected_conditions * EXPECTED_PNG_PER_CONDITION

    report_md = REPORT_DIR / "organization_report.md"

    with open(report_md, "w", encoding="utf-8") as f:
        f.write("# TSMS-Drone Organization Check\n\n")

        f.write("## Expected structure\n\n")
        f.write("```text\n")
        f.write("TSMS-Drone/\n")
        f.write("└── Dataset/\n")
        f.write("    ├── CW Radar/\n")
        f.write("    ├── FMCW Radar/\n")
        f.write("    └── RF Receiver/\n")
        f.write("        └── 2m ... 30m/\n")
        f.write("            └── Target/\n")
        f.write("                ├── Raw File/*.mat\n")
        f.write("                └── Image File/*.png\n")
        f.write("```\n\n")

        f.write("## Summary\n\n")
        f.write(f"- Conditions expected: **{expected_conditions}**\n")
        f.write(f"- Conditions checked: **{total_counter['conditions_checked']}**\n")
        f.write(f"- Expected `.mat`: **{expected_total_mat}**\n")
        f.write(f"- Found `.mat`: **{total_counter['mat_files']}**\n")
        f.write(f"- Expected `.png`: **{expected_total_png}**\n")
        f.write(f"- Found `.png`: **{total_counter['png_files']}**\n")
        f.write(f"- OK conditions: **{total_counter['condition_ok']}**\n")
        f.write(f"- Warning conditions: **{total_counter['condition_warning']}**\n")
        f.write(f"- Error conditions: **{total_counter['condition_error']}**\n")
        f.write(f"- Missing directories: **{len(missing_dirs)}**\n")
        f.write(f"- Pairing errors: **{len(pairing_errors)}**\n")
        f.write(f"- Naming errors: **{len(naming_errors)}**\n")
        f.write(f"- Extra files: **{len(extra_files)}**\n\n")

        f.write("## Size by sensor\n\n")
        f.write("| Sensor | Size |\n")
        f.write("|---|---:|\n")
        for sensor, size in size_by_sensor.items():
            f.write(f"| {sensor} | {human_size(size)} |\n")
        f.write("\n")

        f.write("## Size by target\n\n")
        f.write("| Target | Size |\n")
        f.write("|---|---:|\n")
        for target, size in size_by_target.items():
            f.write(f"| {target} | {human_size(size)} |\n")
        f.write("\n")

        f.write("## Size by extension\n\n")
        f.write("| Extension | Size |\n")
        f.write("|---|---:|\n")
        for ext, size in size_by_extension.items():
            f.write(f"| {ext} | {human_size(size)} |\n")
        f.write("\n")

        f.write("## Generated files\n\n")
        f.write("- `condition_counts.csv`\n")
        f.write("- `missing_dirs.csv`\n")
        f.write("- `pairing_errors.csv`\n")
        f.write("- `naming_errors.csv`\n")
        f.write("- `extra_files.csv`\n")
        if CHECK_MAT_CONTENT:
            f.write("- `mat_content_check.csv`\n")
        f.write("\n")

    # ========================================================
    # 8) IMPRIMIR RESULTADO
    # ========================================================

    print()
    print("=" * 80)
    print("RESULTADO")
    print("=" * 80)
    print(f"Condições esperadas : {expected_conditions}")
    print(f"Condições verificadas: {total_counter['conditions_checked']}")
    print()
    print(f".mat esperados: {expected_total_mat}")
    print(f".mat encontrados: {total_counter['mat_files']}")
    print()
    print(f".png esperados: {expected_total_png}")
    print(f".png encontrados: {total_counter['png_files']}")
    print()
    print(f"Condições OK      : {total_counter['condition_ok']}")
    print(f"Condições WARNING : {total_counter['condition_warning']}")
    print(f"Condições ERROR   : {total_counter['condition_error']}")
    print()
    print(f"Pastas faltando       : {len(missing_dirs)}")
    print(f"Erros de pareamento   : {len(pairing_errors)}")
    print(f"Erros de nome         : {len(naming_errors)}")
    print(f"Arquivos extras       : {len(extra_files)}")
    print()
    print("Relatórios salvos em:")
    print(REPORT_DIR)
    print("=" * 80)

    if total_counter["condition_error"] == 0 and len(missing_dirs) == 0 and len(pairing_errors) == 0:
        print("[OK] Organização aparentemente correta.")
    else:
        print("[ATENÇÃO] Há problemas. Veja os arquivos CSV gerados.")


if __name__ == "__main__":
    main()

CONFERÊNCIA DA ORGANIZAÇÃO DO TSMS-DRONE
ROOT        : D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)
DATASET     : D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\Dataset
REPORT_DIR  : D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_organization_check


RESULTADO
Condições esperadas : 225
Condições verificadas: 225

.mat esperados: 112500
.mat encontrados: 112500

.png esperados: 112500
.png encontrados: 112500

Condições OK      : 100
Condições WARNING : 125
Condições ERROR   : 0

Pastas faltando       : 0
Erros de pareamento   : 0
Erros de nome         : 0
Arquivos extras       : 0

Relatórios salvos em:
D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_organization_check
[OK] Organização aparentemente correta.


In [4]:
from pathlib import Path
import pandas as pd

REPORT_DIR = Path(r"D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_organization_check")

mat_check = REPORT_DIR / "mat_content_check.csv"
condition_counts = REPORT_DIR / "condition_counts.csv"

df_mat = pd.read_csv(mat_check)
df_cond = pd.read_csv(condition_counts)

print("=" * 80)
print("WARNINGS EM mat_content_check.csv")
print("=" * 80)

warnings = df_mat[df_mat["status"].astype(str).str.lower().eq("warning")]
errors = df_mat[df_mat["status"].astype(str).str.lower().eq("error")]

print(f"Warnings em .mat: {len(warnings)}")
print(f"Errors em .mat  : {len(errors)}")

if len(warnings) > 0:
    print("\nResumo por sensor e shape encontrado:")
    print(
        warnings.groupby(
            ["sensor", "expected_shape", "found_shape", "error"],
            dropna=False
        )
        .size()
        .reset_index(name="count")
        .sort_values(["sensor", "count"], ascending=[True, False])
        .to_string(index=False)
    )

if len(errors) > 0:
    print("\nERROS:")
    print(errors[["sensor", "distance", "target", "file", "error"]].to_string(index=False))

print("\n" + "=" * 80)
print("CONDIÇÕES COM WARNING")
print("=" * 80)

cond_warn = df_cond[df_cond["status"].astype(str).str.lower().eq("warning")]

print(f"Condições com warning: {len(cond_warn)}")

print(
    cond_warn.groupby(["sensor", "target"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values(["sensor", "target"])
    .to_string(index=False)
)

WARNINGS EM mat_content_check.csv
Warnings em .mat: 125
Errors em .mat  : 0

Resumo por sensor e shape encontrado:
     sensor expected_shape found_shape                       error  count
   CW Radar       (16384,)  (1, 16384) Shape diferente do esperado     15
 FMCW Radar    (161, 4096) (162, 4096) Shape diferente do esperado     20
 FMCW Radar    (161, 4096) (148, 4096) Shape diferente do esperado     15
RF Receiver      (262144,) (131072, 1) Shape diferente do esperado     45
RF Receiver      (262144,) (1, 131072) Shape diferente do esperado     30

CONDIÇÕES COM WARNING
Condições com warning: 125
     sensor           target  count
   CW Radar Corner Reflector     15
 FMCW Radar Corner Reflector      5
 FMCW Radar        Inspire 2      5
 FMCW Radar       Matrice 30     15
 FMCW Radar      Mavic 2 Pro      5
 FMCW Radar    Phantom 4 Pro      5
RF Receiver Corner Reflector     15
RF Receiver        Inspire 2     15
RF Receiver       Matrice 30     15
RF Receiver      Mavic 2 Pro   

# CONFERINDO ÁRVORE DE SALVAMENTO

In [5]:
from pathlib import Path
import csv

# ============================================================
# CONFIGURAÇÃO
# ============================================================

ROOT = Path(r"D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)")
DATASET_ROOT = ROOT / "Dataset"
REPORT_DIR = ROOT / "_organization_check"

OUT_TREE = REPORT_DIR / "saving_tree.txt"
OUT_CSV = REPORT_DIR / "saving_tree_summary.csv"

SENSORS = [
    "CW Radar",
    "FMCW Radar",
    "RF Receiver",
]

DISTANCES = [f"{d}m" for d in range(2, 31, 2)]

TARGETS = [
    "Inspire 2",
    "Matrice 30",
    "Mavic 2 Pro",
    "Phantom 4 Pro",
    "Corner Reflector",
]


# ============================================================
# FUNÇÕES
# ============================================================

def count_files(folder: Path, extension: str):
    if not folder.exists():
        return 0
    return len(list(folder.glob(f"*{extension}")))


def sample_files(folder: Path, extension: str, n=3):
    if not folder.exists():
        return []
    return [p.name for p in sorted(folder.glob(f"*{extension}"))[:n]]


def line(text="", level=0, connector=""):
    indent = "│   " * level
    return f"{indent}{connector}{text}"


# ============================================================
# MAIN
# ============================================================

def main():
    REPORT_DIR.mkdir(parents=True, exist_ok=True)

    if not ROOT.exists():
        raise FileNotFoundError(f"Pasta raiz não encontrada: {ROOT}")

    if not DATASET_ROOT.exists():
        raise FileNotFoundError(f"Pasta Dataset não encontrada: {DATASET_ROOT}")

    tree_lines = []
    csv_rows = []

    tree_lines.append(str(ROOT))
    tree_lines.append("├── Sample Code/")
    tree_lines.append("├── Dataset/")

    total_mat = 0
    total_png = 0

    for s_idx, sensor in enumerate(SENSORS):
        sensor_dir = DATASET_ROOT / sensor
        sensor_connector = "└── " if s_idx == len(SENSORS) - 1 else "├── "

        tree_lines.append(line(f"{sensor}/", level=1, connector=sensor_connector))

        sensor_mat = 0
        sensor_png = 0

        for d_idx, distance in enumerate(DISTANCES):
            distance_dir = sensor_dir / distance
            distance_connector = "└── " if d_idx == len(DISTANCES) - 1 else "├── "

            tree_lines.append(line(f"{distance}/", level=2, connector=distance_connector))

            for t_idx, target in enumerate(TARGETS):
                target_dir = distance_dir / target
                target_connector = "└── " if t_idx == len(TARGETS) - 1 else "├── "

                raw_dir = target_dir / "Raw File"
                img_dir = target_dir / "Image File"

                n_mat = count_files(raw_dir, ".mat")
                n_png = count_files(img_dir, ".png")

                sensor_mat += n_mat
                sensor_png += n_png
                total_mat += n_mat
                total_png += n_png

                status = "OK" if n_mat == 500 and n_png == 500 else "CHECK"

                tree_lines.append(
                    line(
                        f"{target}/  [{status}: {n_mat} .mat, {n_png} .png]",
                        level=3,
                        connector=target_connector,
                    )
                )

                mat_samples = sample_files(raw_dir, ".mat", n=2)
                png_samples = sample_files(img_dir, ".png", n=2)

                tree_lines.append(
                    line(
                        f"Raw File/   ({n_mat} arquivos .mat)   exemplos: {mat_samples}",
                        level=4,
                        connector="├── ",
                    )
                )

                tree_lines.append(
                    line(
                        f"Image File/ ({n_png} arquivos .png)   exemplos: {png_samples}",
                        level=4,
                        connector="└── ",
                    )
                )

                csv_rows.append({
                    "sensor": sensor,
                    "distance": distance,
                    "target": target,
                    "raw_file_path": str(raw_dir),
                    "image_file_path": str(img_dir),
                    "mat_count": n_mat,
                    "png_count": n_png,
                    "status": status,
                    "mat_examples": " | ".join(mat_samples),
                    "png_examples": " | ".join(png_samples),
                })

        tree_lines.append(
            line(
                f"[TOTAL {sensor}: {sensor_mat} .mat, {sensor_png} .png]",
                level=2,
                connector="└── ",
            )
        )

    tree_lines.append("")
    tree_lines.append("=" * 80)
    tree_lines.append("RESUMO GERAL")
    tree_lines.append("=" * 80)
    tree_lines.append(f"Total .mat: {total_mat}")
    tree_lines.append(f"Total .png: {total_png}")
    tree_lines.append(f"Total geral: {total_mat + total_png}")
    tree_lines.append("=" * 80)

    # Salvar TXT
    OUT_TREE.write_text("\n".join(tree_lines), encoding="utf-8")

    # Salvar CSV
    with open(OUT_CSV, "w", newline="", encoding="utf-8-sig") as f:
        fieldnames = [
            "sensor",
            "distance",
            "target",
            "raw_file_path",
            "image_file_path",
            "mat_count",
            "png_count",
            "status",
            "mat_examples",
            "png_examples",
        ]
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(csv_rows)

    # Imprimir na tela
    print("\n".join(tree_lines))

    print()
    print("Relatórios salvos em:")
    print(OUT_TREE)
    print(OUT_CSV)


if __name__ == "__main__":
    main()

D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)
├── Sample Code/
├── Dataset/
│   ├── CW Radar/
│   │   ├── 2m/
│   │   │   ├── Inspire 2/  [OK: 500 .mat, 500 .png]
│   │   │   │   ├── Raw File/   (500 arquivos .mat)   exemplos: ['Inspire 2_2m_001.mat', 'Inspire 2_2m_002.mat']
│   │   │   │   └── Image File/ (500 arquivos .png)   exemplos: ['Inspire 2_2m_001.png', 'Inspire 2_2m_002.png']
│   │   │   ├── Matrice 30/  [OK: 500 .mat, 500 .png]
│   │   │   │   ├── Raw File/   (500 arquivos .mat)   exemplos: ['Matrice 30_2m_001.mat', 'Matrice 30_2m_002.mat']
│   │   │   │   └── Image File/ (500 arquivos .png)   exemplos: ['Matrice 30_2m_001.png', 'Matrice 30_2m_002.png']
│   │   │   ├── Mavic 2 Pro/  [OK: 500 .mat, 500 .png]
│   │   │   │   ├── Raw File/   (500 arquivos .mat)   exemplos: ['Mavic 2 Pro_2m_001.mat', 'Mavic 2 Pro_2m_002.mat']
│   │   │   │   └── Image File/ (500 arquivos .png)   exemplos: ['Mavic 2 Pro_2m_001.png', 'Mavic 2 Pro_2m_002.png']
│   │   │   ├── Phantom

# TRATAMENTO WARNINGS

Os warnings não são falhas de extração nem de organização. Eles aparecem porque o script anterior esperava shape rígido para os .mat, mas os arquivos reais foram salvos com pequenas variações de formato MATLAB/HDF5. Provavelmente o arquivo está salvo como 131.072 amostras complexas. Como cada amostra complexa tem parte real e imaginária, isso equivale a 262.144 valores I/Q reais. Aqui o número de colunas continua 4096; a diferença está no número de bins/linhas do mapa range-Doppler. O próprio artigo informa que os dados FMCW são mapas range-Doppler e que há segmentação/recorte por faixas de distância, o que pode gerar tamanhos ligeiramente diferentes.

In [6]:
from pathlib import Path
import pandas as pd
import ast
import re
import h5py
from collections import Counter

# ============================================================
# CONFIGURAÇÃO
# ============================================================

REPORT_DIR = Path(
    r"D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_organization_check"
)

MAT_CHECK_CSV = REPORT_DIR / "mat_content_check.csv"
OUT_CSV = REPORT_DIR / "warnings_diagnostic.csv"

MAX_EXAMPLES_PER_GROUP = 3


# ============================================================
# FUNÇÕES
# ============================================================

def parse_shape(shape_text):
    try:
        return tuple(ast.literal_eval(str(shape_text)))
    except Exception:
        nums = re.findall(r"\d+", str(shape_text))
        return tuple(int(x) for x in nums)


def prod(values):
    p = 1
    for v in values:
        p *= int(v)
    return p


def is_complex_dtype(dtype):
    """
    MATLAB v7.3 pode armazenar complexo como compound dtype com campos real/imag.
    """
    if dtype.fields is None:
        return False

    field_names = set(dtype.fields.keys())
    complex_names = {"real", "imag", "r", "i"}

    return len(field_names.intersection(complex_names)) >= 2


def inspect_mat_file(file_path, expected_variable):
    """
    Inspeciona um .mat v7.3/HDF5 sem carregar tudo em memória.
    """
    info = {
        "available_keys": "",
        "variable_found": False,
        "shape_h5": "",
        "dtype_h5": "",
        "is_complex_h5": False,
        "n_elements_h5": "",
        "effective_real_values": "",
        "h5_error": "",
    }

    try:
        with h5py.File(file_path, "r") as f:
            keys = list(f.keys())
            info["available_keys"] = str(keys)

            if expected_variable not in keys:
                return info

            obj = f[expected_variable]

            shape = tuple(obj.shape)
            dtype = obj.dtype
            n_elements = prod(shape)
            complex_flag = is_complex_dtype(dtype)

            info["variable_found"] = True
            info["shape_h5"] = str(shape)
            info["dtype_h5"] = str(dtype)
            info["is_complex_h5"] = complex_flag
            info["n_elements_h5"] = n_elements
            info["effective_real_values"] = n_elements * 2 if complex_flag else n_elements

    except Exception as e:
        info["h5_error"] = str(e)

    return info


def classify_warning(sensor, found_shape, dtype_text="", is_complex=False):
    n = prod(found_shape)

    # CW Radar
    if sensor == "CW Radar":
        if n == 16384:
            return (
                "OK técnico",
                "Vetor CW salvo como linha/coluna. O número de amostras é 16.384."
            )
        return (
            "Verificar",
            f"CW Radar com número inesperado de elementos: {n}."
        )

    # RF Receiver
    if sensor == "RF Receiver":
        if n == 262144:
            return (
                "OK técnico",
                "Vetor RF com 262.144 elementos."
            )

        if n == 131072 and is_complex:
            return (
                "OK técnico",
                "RF salvo como 131.072 amostras complexas; equivale a 262.144 valores real/imag."
            )

        if n == 131072:
            return (
                "Provavelmente OK",
                "RF com 131.072 elementos. Verifique o dtype: se for complexo, equivale a I/Q."
            )

        return (
            "Verificar",
            f"RF Receiver com número inesperado de elementos: {n}."
        )

    # FMCW Radar
    if sensor == "FMCW Radar":
        if len(found_shape) == 2 and found_shape[1] == 4096:
            return (
                "OK técnico",
                "Mapa FMCW com 4096 bins Doppler; número de linhas/range bins varia conforme recorte/faixa."
            )

        return (
            "Verificar",
            f"FMCW Radar com shape inesperado: {found_shape}."
        )

    return "Verificar", "Sensor desconhecido."


# ============================================================
# MAIN
# ============================================================

df = pd.read_csv(MAT_CHECK_CSV)

warnings = df[df["status"].astype(str).str.lower().eq("warning")].copy()
errors = df[df["status"].astype(str).str.lower().eq("error")].copy()

print("=" * 80)
print("DIAGNÓSTICO DOS WARNINGS")
print("=" * 80)
print(f"Arquivo analisado : {MAT_CHECK_CSV}")
print(f"Warnings          : {len(warnings)}")
print(f"Errors            : {len(errors)}")
print("=" * 80)
print()

if warnings.empty:
    print("[OK] Não há warnings.")
    raise SystemExit

diagnostics = []

# Agrupa por tipo de warning
group_cols = ["sensor", "expected_shape", "found_shape", "error"]

for group_key, group_df in warnings.groupby(group_cols, dropna=False):
    sensor, expected_shape, found_shape_text, error_text = group_key
    found_shape = parse_shape(found_shape_text)

    print("-" * 80)
    print(f"Sensor         : {sensor}")
    print(f"Expected shape : {expected_shape}")
    print(f"Found shape    : {found_shape}")
    print(f"Ocorrências    : {len(group_df)}")
    print(f"Erro original  : {error_text}")
    print("-" * 80)

    examples = group_df.head(MAX_EXAMPLES_PER_GROUP)

    for _, row in examples.iterrows():
        file_path = Path(row["file"])
        expected_var = row["expected_variable"]

        h5_info = inspect_mat_file(file_path, expected_var)

        shape_for_classification = parse_shape(h5_info["shape_h5"]) if h5_info["shape_h5"] else found_shape

        status_revised, explanation = classify_warning(
            sensor=sensor,
            found_shape=shape_for_classification,
            dtype_text=h5_info["dtype_h5"],
            is_complex=bool(h5_info["is_complex_h5"]),
        )

        print(f"Arquivo: {file_path.name}")
        print(f"  Caminho             : {file_path}")
        print(f"  Variável esperada   : {expected_var}")
        print(f"  Variáveis no .mat   : {h5_info['available_keys']}")
        print(f"  Variável encontrada : {h5_info['variable_found']}")
        print(f"  Shape HDF5          : {h5_info['shape_h5']}")
        print(f"  Dtype HDF5          : {h5_info['dtype_h5']}")
        print(f"  Complexo HDF5       : {h5_info['is_complex_h5']}")
        print(f"  Elementos HDF5      : {h5_info['n_elements_h5']}")
        print(f"  Valores reais equiv.: {h5_info['effective_real_values']}")
        print(f"  Classificação       : {status_revised}")
        print(f"  Interpretação       : {explanation}")
        print()

        diagnostics.append({
            "sensor": sensor,
            "distance": row.get("distance", ""),
            "target": row.get("target", ""),
            "file": str(file_path),
            "expected_variable": expected_var,
            "expected_shape": expected_shape,
            "original_found_shape": found_shape_text,
            "h5_shape": h5_info["shape_h5"],
            "h5_dtype": h5_info["dtype_h5"],
            "h5_is_complex": h5_info["is_complex_h5"],
            "h5_n_elements": h5_info["n_elements_h5"],
            "effective_real_values": h5_info["effective_real_values"],
            "original_error": error_text,
            "revised_status": status_revised,
            "explanation": explanation,
            "h5_error": h5_info["h5_error"],
        })

# Salvar diagnóstico
diag_df = pd.DataFrame(diagnostics)
diag_df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

print("=" * 80)
print("RESUMO REVISADO")
print("=" * 80)

if not diag_df.empty:
    summary = (
        diag_df
        .groupby(["sensor", "original_found_shape", "h5_dtype", "h5_is_complex", "revised_status", "explanation"], dropna=False)
        .size()
        .reset_index(name="examples_checked")
        .sort_values(["sensor", "original_found_shape"])
    )

    print(summary.to_string(index=False))

print()
print(f"Diagnóstico salvo em:\n{OUT_CSV}")
print("=" * 80)

DIAGNÓSTICO DOS WARNINGS
Arquivo analisado : D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_organization_check\mat_content_check.csv
Warnings          : 125
Errors            : 0

--------------------------------------------------------------------------------
Sensor         : CW Radar
Expected shape : (16384,)
Found shape    : (1, 16384)
Ocorrências    : 15
Erro original  : Shape diferente do esperado
--------------------------------------------------------------------------------
Arquivo: Corner Reflector_2m_001.mat
  Caminho             : D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\Dataset\CW Radar\2m\Corner Reflector\Raw File\Corner Reflector_2m_001.mat
  Variável esperada   : data
  Variáveis no .mat   : ['data']
  Variável encontrada : True
  Shape HDF5          : (1, 16384)
  Dtype HDF5          : [('real', '<f8'), ('imag', '<f8')]
  Complexo HDF5       : True
  Elementos HDF5      : 16384
  Valores reais equiv.: 32768
  Classificação       : OK t

In [7]:
from pathlib import Path
import pandas as pd
import ast
import re

REPORT_DIR = Path(
    r"D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_organization_check"
)

IN_CSV = REPORT_DIR / "mat_content_check.csv"
OUT_CSV = REPORT_DIR / "mat_content_check_final_interpretation.csv"

df = pd.read_csv(IN_CSV)

def parse_shape(text):
    try:
        return tuple(ast.literal_eval(str(text)))
    except Exception:
        nums = re.findall(r"\d+", str(text))
        return tuple(int(x) for x in nums)

def prod(shape):
    p = 1
    for x in shape:
        p *= int(x)
    return p

def classify(row):
    sensor = str(row["sensor"])
    shape = parse_shape(row["found_shape"])
    n = prod(shape)

    if sensor == "CW Radar":
        if n == 16384:
            return "OK", "CW complexo válido; vetor linha/coluna aceito"
        return "CHECK", f"CW com número inesperado de elementos: {n}"

    if sensor == "RF Receiver":
        if n in {131072, 262144}:
            return "OK", "RF complexo/IQ válido; vetor linha/coluna aceito"
        return "CHECK", f"RF com número inesperado de elementos: {n}"

    if sensor == "FMCW Radar":
        if len(shape) == 2 and shape[1] == 4096:
            return "OK", "FMCW RD válido; número de range bins variável aceito"
        return "CHECK", f"FMCW com shape inesperado: {shape}"

    return "CHECK", "Sensor desconhecido"

statuses = []
notes = []

for _, row in df.iterrows():
    status, note = classify(row)
    statuses.append(status)
    notes.append(note)

df["final_status"] = statuses
df["final_interpretation"] = notes

df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

print("=" * 80)
print("INTERPRETAÇÃO FINAL DOS WARNINGS")
print("=" * 80)

print(
    df.groupby(
        ["sensor", "found_shape", "final_status", "final_interpretation"],
        dropna=False
    )
    .size()
    .reset_index(name="count")
    .sort_values(["sensor", "found_shape"])
    .to_string(index=False)
)

print()
print(f"Relatório salvo em:\n{OUT_CSV}")

n_check = (df["final_status"] == "CHECK").sum()

print()
if n_check == 0:
    print("[OK] Todos os warnings foram classificados como formatos válidos.")
else:
    print(f"[ATENÇÃO] Ainda há {n_check} casos para verificar.")

INTERPRETAÇÃO FINAL DOS WARNINGS
     sensor found_shape final_status                                 final_interpretation  count
   CW Radar  (1, 16384)           OK        CW complexo válido; vetor linha/coluna aceito     15
   CW Radar  (16384, 1)           OK        CW complexo válido; vetor linha/coluna aceito     60
 FMCW Radar (148, 4096)           OK FMCW RD válido; número de range bins variável aceito     15
 FMCW Radar (161, 4096)           OK FMCW RD válido; número de range bins variável aceito     37
 FMCW Radar (162, 4096)           OK FMCW RD válido; número de range bins variável aceito     20
 FMCW Radar (4096, 161)        CHECK               FMCW com shape inesperado: (4096, 161)      3
RF Receiver (1, 131072)           OK     RF complexo/IQ válido; vetor linha/coluna aceito     30
RF Receiver (131072, 1)           OK     RF complexo/IQ válido; vetor linha/coluna aceito     45

Relatório salvo em:
D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_organizatio

In [8]:
from pathlib import Path
import pandas as pd

REPORT_DIR = Path(
    r"D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_organization_check"
)

csv_path = REPORT_DIR / "mat_content_check_final_interpretation.csv"

df = pd.read_csv(csv_path)

check = df[df["final_status"].eq("CHECK")]

print("=" * 80)
print("ARQUIVOS A VERIFICAR")
print("=" * 80)

print(
    check[
        ["sensor", "distance", "target", "file", "found_shape", "final_interpretation"]
    ].to_string(index=False)
)

ARQUIVOS A VERIFICAR
    sensor distance           target                                                                                                                                       file found_shape                   final_interpretation
FMCW Radar       2m Corner Reflector   D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\Dataset\FMCW Radar\2m\Corner Reflector\Raw File\Corner Reflector_2m_001.mat (4096, 161) FMCW com shape inesperado: (4096, 161)
FMCW Radar       6m Corner Reflector   D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\Dataset\FMCW Radar\6m\Corner Reflector\Raw File\Corner Reflector_6m_001.mat (4096, 161) FMCW com shape inesperado: (4096, 161)
FMCW Radar      10m Corner Reflector D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\Dataset\FMCW Radar\10m\Corner Reflector\Raw File\Corner Reflector_10m_001.mat (4096, 161) FMCW com shape inesperado: (4096, 161)


In [9]:
from pathlib import Path
import h5py
from scipy.io import loadmat
from collections import Counter
import pandas as pd

ROOT = Path(r"D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)")

CHECK_DIRS = [
    ROOT / r"Dataset\FMCW Radar\2m\Corner Reflector\Raw File",
    ROOT / r"Dataset\FMCW Radar\6m\Corner Reflector\Raw File",
    ROOT / r"Dataset\FMCW Radar\10m\Corner Reflector\Raw File",
]

OUT_CSV = ROOT / r"_organization_check\fmcw_corner_reflector_shape_check.csv"


def read_rd_shape(mat_path: Path):
    """
    Retorna o shape da variável RD.
    Tenta h5py primeiro; se falhar, usa scipy.io.loadmat.
    """
    # MATLAB v7.3 / HDF5
    try:
        with h5py.File(mat_path, "r") as f:
            if "RD" in f:
                return tuple(f["RD"].shape), "h5py"
    except Exception:
        pass

    # MAT clássico
    try:
        mat = loadmat(mat_path)
        if "RD" in mat:
            return tuple(mat["RD"].shape), "scipy"
    except Exception as e:
        return None, f"error: {e}"

    return None, "RD_not_found"


def classify_shape(shape):
    if shape is None:
        return "ERROR"

    if len(shape) != 2:
        return "CHECK"

    rows, cols = shape

    if cols == 4096 and rows in {148, 161, 162}:
        return "NORMAL"

    if rows == 4096 and cols in {148, 161, 162}:
        return "TRANSPOSED"

    return "CHECK"


rows = []

for folder in CHECK_DIRS:
    files = sorted(folder.glob("*.mat"))

    print("=" * 80)
    print(folder)
    print(f"Arquivos .mat: {len(files)}")

    counter = Counter()

    for f in files:
        shape, backend = read_rd_shape(f)
        status = classify_shape(shape)

        counter[(status, shape, backend)] += 1

        rows.append({
            "distance": folder.parents[2].name,  # 2m, 6m, 10m
            "target": folder.parents[1].name,
            "file": str(f),
            "shape": str(shape),
            "backend": backend,
            "status": status,
        })

    for key, count in counter.items():
        status, shape, backend = key
        print(f"{status:12s} shape={shape} backend={backend} count={count}")

df = pd.DataFrame(rows)
df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

print()
print("=" * 80)
print("RESUMO GERAL")
print("=" * 80)
print(
    df.groupby(["distance", "status", "shape", "backend"])
    .size()
    .reset_index(name="count")
    .to_string(index=False)
)

print()
print(f"Relatório salvo em:\n{OUT_CSV}")

D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\Dataset\FMCW Radar\2m\Corner Reflector\Raw File
Arquivos .mat: 500
TRANSPOSED   shape=(4096, 161) backend=h5py count=10
NORMAL       shape=(161, 4096) backend=scipy count=490
D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\Dataset\FMCW Radar\6m\Corner Reflector\Raw File
Arquivos .mat: 500
TRANSPOSED   shape=(4096, 161) backend=h5py count=10
NORMAL       shape=(161, 4096) backend=scipy count=490
D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\Dataset\FMCW Radar\10m\Corner Reflector\Raw File
Arquivos .mat: 500
TRANSPOSED   shape=(4096, 161) backend=h5py count=10
NORMAL       shape=(161, 4096) backend=scipy count=490

RESUMO GERAL
  distance     status       shape backend  count
FMCW Radar     NORMAL (161, 4096)   scipy   1470
FMCW Radar TRANSPOSED (4096, 161)    h5py     30

Relatório salvo em:
D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_organization_check\fmcw_corner_reflector_shape_

In [10]:
from pathlib import Path
import pandas as pd

CSV = Path(
    r"D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_organization_check\fmcw_corner_reflector_shape_check.csv"
)

df = pd.read_csv(CSV)

transposed = df[df["status"].eq("TRANSPOSED")].copy()

print("=" * 80)
print("ARQUIVOS FMCW TRANSPOSTOS")
print("=" * 80)

print(f"Total: {len(transposed)}")
print()

for f in transposed["file"]:
    print(f)

OUT = CSV.parent / "fmcw_transposed_files.csv"
transposed.to_csv(OUT, index=False, encoding="utf-8-sig")

print()
print(f"Lista salva em:\n{OUT}")

ARQUIVOS FMCW TRANSPOSTOS
Total: 30

D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\Dataset\FMCW Radar\2m\Corner Reflector\Raw File\Corner Reflector_2m_001.mat
D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\Dataset\FMCW Radar\2m\Corner Reflector\Raw File\Corner Reflector_2m_002.mat
D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\Dataset\FMCW Radar\2m\Corner Reflector\Raw File\Corner Reflector_2m_003.mat
D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\Dataset\FMCW Radar\2m\Corner Reflector\Raw File\Corner Reflector_2m_004.mat
D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\Dataset\FMCW Radar\2m\Corner Reflector\Raw File\Corner Reflector_2m_005.mat
D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\Dataset\FMCW Radar\2m\Corner Reflector\Raw File\Corner Reflector_2m_006.mat
D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\Dataset\FMCW Radar\2m\Corner Reflector\Raw File\Corner Reflector_2m_007.mat
D:\T

# CORRIGIR OS 30 ARQUIVOS COM BACKUP 

In [11]:
from pathlib import Path
import shutil
import h5py
import pandas as pd

ROOT = Path(r"D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)")

CSV = ROOT / r"_organization_check\fmcw_transposed_files.csv"

BACKUP_ROOT = ROOT / r"_backup_fmcw_transposed_originals"

df = pd.read_csv(CSV)

# Se o CSV tiver coluna "file"
files = [Path(p) for p in df["file"].tolist()]

print("=" * 80)
print("CORREÇÃO DE ARQUIVOS FMCW TRANSPOSTOS")
print("=" * 80)
print(f"Arquivos a corrigir: {len(files)}")
print(f"Backup em: {BACKUP_ROOT}")
print("=" * 80)

for i, mat_path in enumerate(files, start=1):
    print(f"[{i}/{len(files)}] {mat_path}")

    if not mat_path.exists():
        print("  [ERRO] Arquivo não encontrado.")
        continue

    # Caminho relativo para preservar a estrutura no backup
    rel = mat_path.relative_to(ROOT)
    backup_path = BACKUP_ROOT / rel
    backup_path.parent.mkdir(parents=True, exist_ok=True)

    # Backup antes de alterar
    if not backup_path.exists():
        shutil.copy2(mat_path, backup_path)
        print(f"  Backup criado: {backup_path}")
    else:
        print("  Backup já existe.")

    # Abrir e corrigir RD
    with h5py.File(mat_path, "r+") as f:
        if "RD" not in f:
            print("  [ERRO] Variável RD não encontrada.")
            continue

        rd = f["RD"][()]
        old_shape = rd.shape

        if old_shape == (4096, 161):
            rd_fixed = rd.T

            # Salvar atributos antigos, se existirem
            old_attrs = dict(f["RD"].attrs)

            del f["RD"]
            dset = f.create_dataset("RD", data=rd_fixed)

            for k, v in old_attrs.items():
                dset.attrs[k] = v

            print(f"  Corrigido: {old_shape} -> {rd_fixed.shape}")

        elif len(old_shape) == 2 and old_shape[1] == 4096:
            print(f"  Já estava normal: {old_shape}")

        else:
            print(f"  [ATENÇÃO] Shape inesperado: {old_shape}")

print()
print("=" * 80)
print("FINALIZADO")
print("=" * 80)
print("Arquivos originais transpostos foram salvos em backup antes da correção.")
print(f"Backup: {BACKUP_ROOT}")

CORREÇÃO DE ARQUIVOS FMCW TRANSPOSTOS
Arquivos a corrigir: 30
Backup em: D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_backup_fmcw_transposed_originals
[1/30] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\Dataset\FMCW Radar\2m\Corner Reflector\Raw File\Corner Reflector_2m_001.mat
  Backup criado: D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_backup_fmcw_transposed_originals\Dataset\FMCW Radar\2m\Corner Reflector\Raw File\Corner Reflector_2m_001.mat
  Corrigido: (4096, 161) -> (161, 4096)
[2/30] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\Dataset\FMCW Radar\2m\Corner Reflector\Raw File\Corner Reflector_2m_002.mat
  Backup criado: D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_backup_fmcw_transposed_originals\Dataset\FMCW Radar\2m\Corner Reflector\Raw File\Corner Reflector_2m_002.mat
  Corrigido: (4096, 161) -> (161, 4096)
[3/30] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\Dataset\FMCW Radar\2

In [12]:
from pathlib import Path
import h5py
import pandas as pd
from collections import Counter

ROOT = Path(r"D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)")
CSV = ROOT / r"_organization_check\fmcw_transposed_files.csv"

df = pd.read_csv(CSV)
files = [Path(p) for p in df["file"].tolist()]

counter = Counter()

for mat_path in files:
    with h5py.File(mat_path, "r") as f:
        shape = tuple(f["RD"].shape)

    counter[shape] += 1
    print(mat_path.name, shape)

print()
print("=" * 80)
print("RESUMO")
print("=" * 80)

for shape, count in counter.items():
    print(f"{shape}: {count}")

if all(shape == (161, 4096) for shape in counter):
    print("[OK] Todos os arquivos corrigidos para (161, 4096).")
else:
    print("[ATENÇÃO] Ainda há shapes diferentes.")

Corner Reflector_2m_001.mat (161, 4096)
Corner Reflector_2m_002.mat (161, 4096)
Corner Reflector_2m_003.mat (161, 4096)
Corner Reflector_2m_004.mat (161, 4096)
Corner Reflector_2m_005.mat (161, 4096)
Corner Reflector_2m_006.mat (161, 4096)
Corner Reflector_2m_007.mat (161, 4096)
Corner Reflector_2m_008.mat (161, 4096)
Corner Reflector_2m_009.mat (161, 4096)
Corner Reflector_2m_010.mat (161, 4096)
Corner Reflector_6m_001.mat (161, 4096)
Corner Reflector_6m_002.mat (161, 4096)
Corner Reflector_6m_003.mat (161, 4096)
Corner Reflector_6m_004.mat (161, 4096)
Corner Reflector_6m_005.mat (161, 4096)
Corner Reflector_6m_006.mat (161, 4096)
Corner Reflector_6m_007.mat (161, 4096)
Corner Reflector_6m_008.mat (161, 4096)
Corner Reflector_6m_009.mat (161, 4096)
Corner Reflector_6m_010.mat (161, 4096)
Corner Reflector_10m_001.mat (161, 4096)
Corner Reflector_10m_002.mat (161, 4096)
Corner Reflector_10m_003.mat (161, 4096)
Corner Reflector_10m_004.mat (161, 4096)
Corner Reflector_10m_005.mat (161, 4

# ANÁLISE EXPLORATÓRIA DE DADOS

In [13]:
from pathlib import Path
import re
import csv
import math
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURAÇÕES
# ============================================================

ROOT = Path(r"D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)")
DATASET_ROOT = ROOT / "Dataset"
OUT_DIR = ROOT / "_eda_ieee"

SENSORS = [
    "CW Radar",
    "FMCW Radar",
    "RF Receiver",
]

TARGETS = [
    "Inspire 2",
    "Matrice 30",
    "Mavic 2 Pro",
    "Phantom 4 Pro",
    "Corner Reflector",
]

DISTANCES = [f"{d}m" for d in range(2, 31, 2)]

EXPECTED_PER_CONDITION = 500

# Para evitar abrir 112.500 arquivos .mat completos, o padrão é amostrar.
# Para checar todos os .mat, mude para True.
SCAN_ALL_MAT_FILES = False

# Se SCAN_ALL_MAT_FILES = False, o script analisa N arquivos .mat por condição.
MAT_SAMPLES_PER_CONDITION = 3

# Para imagens PNG, também amostra por condição.
PNG_SAMPLES_PER_CONDITION = 3

# Gerar tabelas LaTeX compatíveis com artigo IEEE.
GENERATE_LATEX = True


# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def ensure_dir(path: Path):
    path.mkdir(parents=True, exist_ok=True)


def human_size(num_bytes):
    if pd.isna(num_bytes):
        return ""

    size = float(num_bytes)
    units = ["B", "KB", "MB", "GB", "TB"]

    for unit in units:
        if size < 1024:
            return f"{size:.2f} {unit}"
        size /= 1024

    return f"{size:.2f} PB"


def parse_index(path: Path):
    """
    Extrai índice final do arquivo:
    Inspire 2_2m_001.mat -> 1
    """
    m = re.search(r"_(\d+)\.[^.]+$", path.name)
    if m:
        return int(m.group(1))
    return None


def write_csv(df: pd.DataFrame, name: str):
    path = OUT_DIR / f"{name}.csv"
    df.to_csv(path, index=False, encoding="utf-8-sig")
    return path


def write_markdown(df: pd.DataFrame, name: str):
    path = OUT_DIR / f"{name}.md"
    with open(path, "w", encoding="utf-8") as f:
        f.write(df.to_markdown(index=False))
        f.write("\n")
    return path


def write_latex(df: pd.DataFrame, name: str, caption: str, label: str):
    """
    Gera tabela LaTeX simples para uso em artigo IEEE.
    """
    path = OUT_DIR / f"{name}.tex"

    latex = df.to_latex(
        index=False,
        escape=False,
        column_format="l" * len(df.columns),
        caption=caption,
        label=label
    )

    with open(path, "w", encoding="utf-8") as f:
        f.write(latex)

    return path


def safe_file_size(path: Path):
    try:
        return path.stat().st_size
    except Exception:
        return np.nan


def read_mat_metadata(mat_path: Path, sensor: str):
    """
    Lê metadados básicos de arquivo .mat sem carregar dados desnecessários.
    Tenta h5py primeiro; se falhar, usa scipy.io.loadmat.
    """
    expected_var = "RD" if sensor == "FMCW Radar" else "data"

    info = {
        "file": str(mat_path),
        "sensor": sensor,
        "expected_variable": expected_var,
        "backend": "",
        "variable_found": False,
        "available_variables": "",
        "shape": "",
        "dtype": "",
        "is_complex": False,
        "n_elements": np.nan,
        "size_bytes": safe_file_size(mat_path),
        "error": "",
    }

    # Tentativa 1: MATLAB v7.3 / HDF5
    try:
        import h5py

        with h5py.File(mat_path, "r") as f:
            keys = list(f.keys())
            info["available_variables"] = str(keys)

            if expected_var not in keys:
                raise KeyError(f"Variável {expected_var} não encontrada via h5py.")

            obj = f[expected_var]
            shape = tuple(obj.shape)
            dtype = obj.dtype

            is_complex = False
            if dtype.fields is not None:
                names = set(dtype.fields.keys())
                is_complex = {"real", "imag"}.issubset(names) or {"r", "i"}.issubset(names)

            info.update({
                "backend": "h5py",
                "variable_found": True,
                "shape": str(shape),
                "dtype": str(dtype),
                "is_complex": is_complex,
                "n_elements": int(np.prod(shape)),
                "error": "",
            })

            return info

    except Exception as e_h5:
        h5_error = str(e_h5)

    # Tentativa 2: MAT clássico
    try:
        from scipy.io import loadmat

        mat = loadmat(mat_path)
        keys = [k for k in mat.keys() if not k.startswith("__")]
        info["available_variables"] = str(keys)

        if expected_var not in keys:
            raise KeyError(f"Variável {expected_var} não encontrada via scipy.")

        arr = mat[expected_var]
        shape = tuple(arr.shape)
        dtype = arr.dtype

        info.update({
            "backend": "scipy",
            "variable_found": True,
            "shape": str(shape),
            "dtype": str(dtype),
            "is_complex": bool(np.iscomplexobj(arr)),
            "n_elements": int(arr.size),
            "error": "",
        })

        return info

    except Exception as e_sp:
        info.update({
            "backend": "failed",
            "variable_found": False,
            "error": f"h5py: {h5_error} | scipy: {e_sp}",
        })

        return info


def read_png_metadata(png_path: Path):
    info = {
        "file": str(png_path),
        "width": np.nan,
        "height": np.nan,
        "mode": "",
        "size_bytes": safe_file_size(png_path),
        "error": "",
    }

    try:
        from PIL import Image

        with Image.open(png_path) as img:
            info["width"] = img.width
            info["height"] = img.height
            info["mode"] = img.mode

    except Exception as e:
        info["error"] = str(e)

    return info


def sample_files(files, n):
    if len(files) <= n:
        return files
    return files[:n]


def latex_escape_text(x):
    return str(x).replace("_", "\\_")


# ============================================================
# ANÁLISE PRINCIPAL
# ============================================================

def main():
    ensure_dir(OUT_DIR)

    print("=" * 80)
    print("EDA TSMS-DRONE PARA ARTIGO IEEE")
    print("=" * 80)
    print(f"ROOT        : {ROOT}")
    print(f"DATASET     : {DATASET_ROOT}")
    print(f"OUT_DIR     : {OUT_DIR}")
    print("=" * 80)

    if not DATASET_ROOT.exists():
        raise FileNotFoundError(f"Pasta Dataset não encontrada: {DATASET_ROOT}")

    condition_rows = []
    mat_metadata_rows = []
    png_metadata_rows = []
    missing_index_rows = []
    example_rows = []
    alignment_rows = []

    # ========================================================
    # 1) VARREDURA DA ESTRUTURA
    # ========================================================

    for sensor in SENSORS:
        for distance in DISTANCES:
            for target in TARGETS:
                raw_dir = DATASET_ROOT / sensor / distance / target / "Raw File"
                img_dir = DATASET_ROOT / sensor / distance / target / "Image File"

                mat_files = sorted(raw_dir.glob("*.mat")) if raw_dir.exists() else []
                png_files = sorted(img_dir.glob("*.png")) if img_dir.exists() else []

                mat_indices = sorted([parse_index(f) for f in mat_files if parse_index(f) is not None])
                png_indices = sorted([parse_index(f) for f in png_files if parse_index(f) is not None])

                expected_indices = set(range(1, EXPECTED_PER_CONDITION + 1))
                mat_index_set = set(mat_indices)
                png_index_set = set(png_indices)

                missing_mat = sorted(expected_indices - mat_index_set)
                missing_png = sorted(expected_indices - png_index_set)

                extra_mat = sorted(mat_index_set - expected_indices)
                extra_png = sorted(png_index_set - expected_indices)

                common_indices = sorted(mat_index_set & png_index_set)

                mat_size = sum(safe_file_size(f) for f in mat_files)
                png_size = sum(safe_file_size(f) for f in png_files)

                status = "OK"
                notes = []

                if len(mat_files) != EXPECTED_PER_CONDITION:
                    status = "CHECK"
                    notes.append(f".mat={len(mat_files)}")

                if len(png_files) != EXPECTED_PER_CONDITION:
                    status = "CHECK"
                    notes.append(f".png={len(png_files)}")

                if missing_mat or missing_png:
                    status = "CHECK"
                    notes.append(f"indices ausentes mat={len(missing_mat)}, png={len(missing_png)}")

                condition_rows.append({
                    "sensor": sensor,
                    "distance": distance,
                    "distance_m": int(distance.replace("m", "")),
                    "target": target,
                    "mat_count": len(mat_files),
                    "png_count": len(png_files),
                    "paired_indices": len(common_indices),
                    "missing_mat_indices": len(missing_mat),
                    "missing_png_indices": len(missing_png),
                    "extra_mat_indices": len(extra_mat),
                    "extra_png_indices": len(extra_png),
                    "mat_size_bytes": mat_size,
                    "png_size_bytes": png_size,
                    "total_size_bytes": mat_size + png_size,
                    "mat_size_human": human_size(mat_size),
                    "png_size_human": human_size(png_size),
                    "total_size_human": human_size(mat_size + png_size),
                    "status": status,
                    "notes": " | ".join(notes),
                    "raw_dir": str(raw_dir),
                    "image_dir": str(img_dir),
                })

                if missing_mat or missing_png or extra_mat or extra_png:
                    missing_index_rows.append({
                        "sensor": sensor,
                        "distance": distance,
                        "target": target,
                        "missing_mat_indices": str(missing_mat[:50]),
                        "missing_png_indices": str(missing_png[:50]),
                        "extra_mat_indices": str(extra_mat[:50]),
                        "extra_png_indices": str(extra_png[:50]),
                    })

                # Exemplos
                example_rows.append({
                    "sensor": sensor,
                    "distance": distance,
                    "target": target,
                    "example_mat": mat_files[0].name if mat_files else "",
                    "example_png": png_files[0].name if png_files else "",
                })

                # Metadados .mat
                if SCAN_ALL_MAT_FILES:
                    selected_mat = mat_files
                else:
                    selected_mat = sample_files(mat_files, MAT_SAMPLES_PER_CONDITION)

                for mat_path in selected_mat:
                    md = read_mat_metadata(mat_path, sensor)
                    md.update({
                        "distance": distance,
                        "distance_m": int(distance.replace("m", "")),
                        "target": target,
                        "filename": mat_path.name,
                    })
                    mat_metadata_rows.append(md)

                # Metadados .png
                selected_png = sample_files(png_files, PNG_SAMPLES_PER_CONDITION)

                for png_path in selected_png:
                    md = read_png_metadata(png_path)
                    md.update({
                        "sensor": sensor,
                        "distance": distance,
                        "distance_m": int(distance.replace("m", "")),
                        "target": target,
                        "filename": png_path.name,
                    })
                    png_metadata_rows.append(md)

    condition_df = pd.DataFrame(condition_rows)
    mat_meta_df = pd.DataFrame(mat_metadata_rows)
    png_meta_df = pd.DataFrame(png_metadata_rows)
    missing_indices_df = pd.DataFrame(missing_index_rows)
    examples_df = pd.DataFrame(example_rows)

    # ========================================================
    # 2) TABELAS DE CONTAGEM E BALANCEAMENTO
    # ========================================================

    totals_by_sensor = (
        condition_df
        .groupby("sensor", as_index=False)
        .agg(
            mat_files=("mat_count", "sum"),
            png_files=("png_count", "sum"),
            paired_samples=("paired_indices", "sum"),
            total_size_bytes=("total_size_bytes", "sum"),
        )
    )
    totals_by_sensor["total_files"] = totals_by_sensor["mat_files"] + totals_by_sensor["png_files"]
    totals_by_sensor["total_size"] = totals_by_sensor["total_size_bytes"].apply(human_size)

    totals_by_target = (
        condition_df
        .groupby("target", as_index=False)
        .agg(
            mat_files=("mat_count", "sum"),
            png_files=("png_count", "sum"),
            paired_samples=("paired_indices", "sum"),
            total_size_bytes=("total_size_bytes", "sum"),
        )
    )
    totals_by_target["total_files"] = totals_by_target["mat_files"] + totals_by_target["png_files"]
    totals_by_target["total_size"] = totals_by_target["total_size_bytes"].apply(human_size)

    totals_by_distance = (
        condition_df
        .groupby(["distance", "distance_m"], as_index=False)
        .agg(
            mat_files=("mat_count", "sum"),
            png_files=("png_count", "sum"),
            paired_samples=("paired_indices", "sum"),
            total_size_bytes=("total_size_bytes", "sum"),
        )
        .sort_values("distance_m")
    )
    totals_by_distance["total_files"] = totals_by_distance["mat_files"] + totals_by_distance["png_files"]
    totals_by_distance["total_size"] = totals_by_distance["total_size_bytes"].apply(human_size)

    class_balance = (
        condition_df
        .groupby(["sensor", "target"], as_index=False)
        .agg(
            conditions=("distance", "count"),
            mat_files=("mat_count", "sum"),
            png_files=("png_count", "sum"),
            paired_samples=("paired_indices", "sum"),
        )
        .sort_values(["sensor", "target"])
    )

    # ========================================================
    # 3) TABELAS DE TAMANHO DE ARQUIVO
    # ========================================================

    condition_df["avg_mat_size_bytes"] = condition_df["mat_size_bytes"] / condition_df["mat_count"].replace(0, np.nan)
    condition_df["avg_png_size_bytes"] = condition_df["png_size_bytes"] / condition_df["png_count"].replace(0, np.nan)

    file_size_by_sensor = (
        condition_df
        .groupby("sensor", as_index=False)
        .agg(
            total_mat_size_bytes=("mat_size_bytes", "sum"),
            total_png_size_bytes=("png_size_bytes", "sum"),
            total_size_bytes=("total_size_bytes", "sum"),
            avg_mat_size_bytes=("avg_mat_size_bytes", "mean"),
            avg_png_size_bytes=("avg_png_size_bytes", "mean"),
        )
    )

    for col in [
        "total_mat_size_bytes",
        "total_png_size_bytes",
        "total_size_bytes",
        "avg_mat_size_bytes",
        "avg_png_size_bytes",
    ]:
        file_size_by_sensor[col.replace("_bytes", "")] = file_size_by_sensor[col].apply(human_size)

    # ========================================================
    # 4) METADADOS DE .MAT E .PNG
    # ========================================================

    if not mat_meta_df.empty:
        mat_shape_summary = (
            mat_meta_df
            .groupby(["sensor", "expected_variable", "shape", "dtype", "is_complex", "backend"], as_index=False)
            .agg(
                checked_files=("file", "count"),
                min_size_bytes=("size_bytes", "min"),
                median_size_bytes=("size_bytes", "median"),
                max_size_bytes=("size_bytes", "max"),
            )
        )

        mat_shape_summary["min_size"] = mat_shape_summary["min_size_bytes"].apply(human_size)
        mat_shape_summary["median_size"] = mat_shape_summary["median_size_bytes"].apply(human_size)
        mat_shape_summary["max_size"] = mat_shape_summary["max_size_bytes"].apply(human_size)
    else:
        mat_shape_summary = pd.DataFrame()

    if not png_meta_df.empty:
        png_summary = (
            png_meta_df
            .groupby(["sensor", "width", "height", "mode"], as_index=False)
            .agg(
                checked_files=("file", "count"),
                min_size_bytes=("size_bytes", "min"),
                median_size_bytes=("size_bytes", "median"),
                max_size_bytes=("size_bytes", "max"),
            )
        )
        png_summary["min_size"] = png_summary["min_size_bytes"].apply(human_size)
        png_summary["median_size"] = png_summary["median_size_bytes"].apply(human_size)
        png_summary["max_size"] = png_summary["max_size_bytes"].apply(human_size)
    else:
        png_summary = pd.DataFrame()

    # ========================================================
    # 5) CHECAGEM DE ALINHAMENTO MULTISSENSORIAL
    # ========================================================

    # Para cada distância, alvo e índice, confere presença nos três sensores.
    # Esta tabela é útil porque o artigo enfatiza o caráter time-synchronized.
    print("Checando alinhamento multissensorial por índice...")

    for distance in DISTANCES:
        for target in TARGETS:
            for idx in range(1, EXPECTED_PER_CONDITION + 1):
                idx_str = f"{idx:03d}"

                sensor_presence = {}

                for sensor in SENSORS:
                    mat_path = (
                        DATASET_ROOT / sensor / distance / target /
                        "Raw File" / f"{target}_{distance}_{idx_str}.mat"
                    )
                    png_path = (
                        DATASET_ROOT / sensor / distance / target /
                        "Image File" / f"{target}_{distance}_{idx_str}.png"
                    )

                    sensor_presence[f"{sensor}_mat"] = mat_path.exists()
                    sensor_presence[f"{sensor}_png"] = png_path.exists()

                all_mat = all(sensor_presence[f"{s}_mat"] for s in SENSORS)
                all_png = all(sensor_presence[f"{s}_png"] for s in SENSORS)

                if not (all_mat and all_png):
                    alignment_rows.append({
                        "distance": distance,
                        "target": target,
                        "index": idx,
                        **sensor_presence,
                        "all_sensors_mat": all_mat,
                        "all_sensors_png": all_png,
                    })

    alignment_problems_df = pd.DataFrame(alignment_rows)

    alignment_summary = pd.DataFrame([{
        "distance_target_index_combinations": len(DISTANCES) * len(TARGETS) * EXPECTED_PER_CONDITION,
        "problematic_combinations": len(alignment_problems_df),
        "complete_combinations": len(DISTANCES) * len(TARGETS) * EXPECTED_PER_CONDITION - len(alignment_problems_df),
        "complete_percentage": 100.0 * (
            len(DISTANCES) * len(TARGETS) * EXPECTED_PER_CONDITION - len(alignment_problems_df)
        ) / (len(DISTANCES) * len(TARGETS) * EXPECTED_PER_CONDITION),
    }])

    # ========================================================
    # 6) TABELAS RESUMIDAS PARA ARTIGO IEEE
    # ========================================================

    ieee_dataset_overview = pd.DataFrame([{
        "Sensors": len(SENSORS),
        "Targets": len(TARGETS),
        "Distances": len(DISTANCES),
        "Distance range": f"{DISTANCES[0]}--{DISTANCES[-1]}",
        "Repetitions per condition": EXPECTED_PER_CONDITION,
        "MAT files": int(condition_df["mat_count"].sum()),
        "PNG files": int(condition_df["png_count"].sum()),
        "Total files": int(condition_df["mat_count"].sum() + condition_df["png_count"].sum()),
    }])

    ieee_sensor_table = totals_by_sensor[
        ["sensor", "paired_samples", "mat_files", "png_files", "total_files", "total_size"]
    ].rename(columns={
        "sensor": "Sensor",
        "paired_samples": "Paired samples",
        "mat_files": "MAT files",
        "png_files": "PNG files",
        "total_files": "Total files",
        "total_size": "Size",
    })

    ieee_target_table = totals_by_target[
        ["target", "paired_samples", "mat_files", "png_files", "total_files", "total_size"]
    ].rename(columns={
        "target": "Target",
        "paired_samples": "Paired samples",
        "mat_files": "MAT files",
        "png_files": "PNG files",
        "total_files": "Total files",
        "total_size": "Size",
    })

    ieee_mat_table = mat_shape_summary[
        ["sensor", "expected_variable", "shape", "is_complex", "backend", "checked_files", "median_size"]
    ].rename(columns={
        "sensor": "Sensor",
        "expected_variable": "Variable",
        "shape": "Shape",
        "is_complex": "Complex",
        "backend": "Reader",
        "checked_files": "Checked files",
        "median_size": "Median size",
    }) if not mat_shape_summary.empty else pd.DataFrame()

    ieee_png_table = png_summary[
        ["sensor", "width", "height", "mode", "checked_files", "median_size"]
    ].rename(columns={
        "sensor": "Sensor",
        "width": "Width",
        "height": "Height",
        "mode": "Mode",
        "checked_files": "Checked files",
        "median_size": "Median size",
    }) if not png_summary.empty else pd.DataFrame()

    ieee_alignment_table = alignment_summary.rename(columns={
        "distance_target_index_combinations": "Expected synchronized tuples",
        "problematic_combinations": "Incomplete tuples",
        "complete_combinations": "Complete tuples",
        "complete_percentage": "Complete (%)",
    })
    ieee_alignment_table["Complete (%)"] = ieee_alignment_table["Complete (%)"].map(lambda x: f"{x:.2f}")

    # ========================================================
    # 7) SALVAR TABELAS
    # ========================================================

    outputs = []

    tables = {
        "condition_counts_full": condition_df,
        "totals_by_sensor": totals_by_sensor,
        "totals_by_target": totals_by_target,
        "totals_by_distance": totals_by_distance,
        "class_balance_by_sensor_target": class_balance,
        "file_size_by_sensor": file_size_by_sensor,
        "mat_metadata_sample": mat_meta_df,
        "mat_shape_summary": mat_shape_summary,
        "png_metadata_sample": png_meta_df,
        "png_summary": png_summary,
        "missing_indices": missing_indices_df,
        "examples_by_condition": examples_df,
        "alignment_problems": alignment_problems_df,
        "alignment_summary": alignment_summary,
        "ieee_dataset_overview": ieee_dataset_overview,
        "ieee_sensor_table": ieee_sensor_table,
        "ieee_target_table": ieee_target_table,
        "ieee_mat_table": ieee_mat_table,
        "ieee_png_table": ieee_png_table,
        "ieee_alignment_table": ieee_alignment_table,
    }

    for name, df in tables.items():
        if df is None or df.empty:
            continue

        outputs.append(write_csv(df, name))
        outputs.append(write_markdown(df, name))

    if GENERATE_LATEX:
        if not ieee_dataset_overview.empty:
            outputs.append(write_latex(
                ieee_dataset_overview,
                "latex_ieee_dataset_overview",
                "Overview of the TSMS-Drone dataset after local organization.",
                "tab:dataset_overview"
            ))

        if not ieee_sensor_table.empty:
            outputs.append(write_latex(
                ieee_sensor_table,
                "latex_ieee_sensor_summary",
                "Number of synchronized samples and files by sensor modality.",
                "tab:sensor_summary"
            ))

        if not ieee_target_table.empty:
            outputs.append(write_latex(
                ieee_target_table,
                "latex_ieee_target_summary",
                "Number of synchronized samples and files by target class.",
                "tab:target_summary"
            ))

        if not ieee_mat_table.empty:
            outputs.append(write_latex(
                ieee_mat_table,
                "latex_ieee_mat_summary",
                "Summary of sampled MAT-file variables and array shapes.",
                "tab:mat_summary"
            ))

        if not ieee_png_table.empty:
            outputs.append(write_latex(
                ieee_png_table,
                "latex_ieee_png_summary",
                "Summary of sampled PNG feature-image properties.",
                "tab:png_summary"
            ))

        if not ieee_alignment_table.empty:
            outputs.append(write_latex(
                ieee_alignment_table,
                "latex_ieee_alignment_summary",
                "Verification of synchronized tuples across sensor modalities.",
                "tab:alignment_summary"
            ))

    # ========================================================
    # 8) RELATÓRIO TXT / MD
    # ========================================================

    report_path = OUT_DIR / "eda_report_for_ieee.md"

    with open(report_path, "w", encoding="utf-8") as f:
        f.write("# Exploratory Data Analysis Report -- TSMS-Drone\n\n")

        f.write("## Dataset organization\n\n")
        f.write(f"- Root directory: `{ROOT}`\n")
        f.write(f"- Sensors: {', '.join(SENSORS)}\n")
        f.write(f"- Targets: {', '.join(TARGETS)}\n")
        f.write(f"- Distances: {DISTANCES[0]} to {DISTANCES[-1]} in 2 m increments\n")
        f.write(f"- Expected repetitions per condition: {EXPECTED_PER_CONDITION}\n\n")

        f.write("## Global overview\n\n")
        f.write(ieee_dataset_overview.to_markdown(index=False))
        f.write("\n\n")

        f.write("## Summary by sensor\n\n")
        f.write(ieee_sensor_table.to_markdown(index=False))
        f.write("\n\n")

        f.write("## Summary by target\n\n")
        f.write(ieee_target_table.to_markdown(index=False))
        f.write("\n\n")

        f.write("## MAT-file structure summary\n\n")
        if not ieee_mat_table.empty:
            f.write(ieee_mat_table.to_markdown(index=False))
        else:
            f.write("No MAT metadata available.")
        f.write("\n\n")

        f.write("## PNG image summary\n\n")
        if not ieee_png_table.empty:
            f.write(ieee_png_table.to_markdown(index=False))
        else:
            f.write("No PNG metadata available.")
        f.write("\n\n")

        f.write("## Cross-sensor alignment\n\n")
        f.write(ieee_alignment_table.to_markdown(index=False))
        f.write("\n\n")

        f.write("## Notes for IEEE paper\n\n")
        f.write(
            "- The dataset is balanced by construction across sensors, targets, distances, "
            "and measurement repetitions.\n"
        )
        f.write(
            "- Each target--distance--sensor condition contains paired raw MAT files and "
            "corresponding PNG feature images.\n"
        )
        f.write(
            "- The synchronized index structure enables sample-level multimodal fusion across "
            "CW radar, FMCW radar, and RF receiver measurements.\n"
        )
        f.write(
            "- MAT-file shapes should be normalized in the loading pipeline, especially for "
            "FMCW range-Doppler arrays and complex I/Q vectors.\n"
        )

    outputs.append(report_path)

    # ========================================================
    # 9) PRINT FINAL
    # ========================================================

    print()
    print("=" * 80)
    print("RESUMO DA EDA")
    print("=" * 80)

    print("\n[Dataset overview]")
    print(ieee_dataset_overview.to_string(index=False))

    print("\n[By sensor]")
    print(ieee_sensor_table.to_string(index=False))

    print("\n[By target]")
    print(ieee_target_table.to_string(index=False))

    print("\n[Alignment]")
    print(ieee_alignment_table.to_string(index=False))

    print()
    print("=" * 80)
    print("ARQUIVOS GERADOS")
    print("=" * 80)

    for p in outputs:
        print(p)

    print("=" * 80)
    print("[OK] EDA finalizada.")
    print("=" * 80)


if __name__ == "__main__":
    main()

EDA TSMS-DRONE PARA ARTIGO IEEE
ROOT        : D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)
DATASET     : D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\Dataset
OUT_DIR     : D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_eda_ieee
Checando alinhamento multissensorial por índice...

RESUMO DA EDA

[Dataset overview]
 Sensors  Targets  Distances Distance range  Repetitions per condition  MAT files  PNG files  Total files
       3        5         15        2m--30m                        500     112500     112500       225000

[By sensor]
     Sensor  Paired samples  MAT files  PNG files  Total files      Size
   CW Radar           37500      37500      37500        75000   2.61 GB
 FMCW Radar           37500      37500      37500        75000 351.79 GB
RF Receiver           37500      37500      37500        75000  18.08 GB

[By target]
          Target  Paired samples  MAT files  PNG files  Total files     Size
Corner Reflector           22500

# MATERIAIS E MÉTODOS

In [15]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

# ============================================================
# CONFIGURAÇÕES
# ============================================================

ROOT = Path(r"D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)")
EDA_DIR = ROOT / "_eda_ieee"
CONDITION_CSV = EDA_DIR / "condition_counts_full.csv"

SENSORS = ["CW Radar", "FMCW Radar", "RF Receiver"]

TARGETS = [
    "Inspire 2",
    "Matrice 30",
    "Mavic 2 Pro",
    "Phantom 4 Pro",
    "Corner Reflector",
]

DISTANCES = [f"{d}m" for d in range(2, 31, 2)]
EXPECTED_REPETITIONS = 500


# ============================================================
# FUNÇÕES
# ============================================================

def human_size(num_bytes):
    size = float(num_bytes)
    units = ["B", "KB", "MB", "GB", "TB"]

    for unit in units:
        if size < 1024:
            return f"{size:.2f} {unit}"
        size /= 1024

    return f"{size:.2f} PB"


def fmt_int(x):
    try:
        return f"{int(x):,}".replace(",", ".")
    except Exception:
        return x


def show_table(title, df):
    display(Markdown(f"### {title}"))
    display(df)


def make_latex(df, caption, label):
    return df.to_latex(
        index=False,
        escape=False,
        caption=caption,
        label=label,
        column_format="l" * len(df.columns)
    )


# ============================================================
# CARREGAR TABELA BASE
# ============================================================

if not CONDITION_CSV.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {CONDITION_CSV}")

condition_df = pd.read_csv(CONDITION_CSV)

required_cols = [
    "sensor", "distance", "distance_m", "target",
    "mat_count", "png_count", "paired_indices",
    "mat_size_bytes", "png_size_bytes", "total_size_bytes"
]

missing = [c for c in required_cols if c not in condition_df.columns]

if missing:
    raise ValueError(f"Colunas ausentes em condition_counts_full.csv: {missing}")

display(Markdown("## Tabelas para a seção Materials and Methods"))


# ============================================================
# TABLE I — DATASET OVERVIEW
# ============================================================

total_mat = int(condition_df["mat_count"].sum())
total_png = int(condition_df["png_count"].sum())
total_files = total_mat + total_png

sensor_specific_samples = int(condition_df["paired_indices"].sum())

synchronized_tuples = (
    len(DISTANCES) * len(TARGETS) * EXPECTED_REPETITIONS
)

total_size = int(condition_df["total_size_bytes"].sum())

table_01_dataset_overview = pd.DataFrame([{
    "Sensors": len(SENSORS),
    "Targets": len(TARGETS),
    "Distances": len(DISTANCES),
    "Distance range": f"{DISTANCES[0]}--{DISTANCES[-1]}",
    "Repetitions per condition": EXPECTED_REPETITIONS,
    "Synchronized tuples": synchronized_tuples,
    "Sensor-specific samples": sensor_specific_samples,
    "MAT files": total_mat,
    "PNG files": total_png,
    "Total files": total_files,
    "Size": human_size(total_size),
}])

table_01_display = table_01_dataset_overview.copy()

for col in [
    "Synchronized tuples",
    "Sensor-specific samples",
    "MAT files",
    "PNG files",
    "Total files",
]:
    table_01_display[col] = table_01_display[col].apply(fmt_int)


# ============================================================
# TABLE II — SENSOR MODALITIES AND DATA PRODUCTS
# ============================================================

by_sensor = (
    condition_df
    .groupby("sensor", as_index=False)
    .agg(
        mat_files=("mat_count", "sum"),
        png_files=("png_count", "sum"),
        samples=("paired_indices", "sum"),
        size_bytes=("total_size_bytes", "sum"),
    )
)

sensor_description = pd.DataFrame([
    {
        "Sensor": "CW Radar",
        "Acquisition type": "Active radar",
        "MAT variable": "data",
        "Raw data product": "Complex I/Q vector",
        "Feature image": "Doppler spectrum",
    },
    {
        "Sensor": "FMCW Radar",
        "Acquisition type": "Active radar",
        "MAT variable": "RD",
        "Raw data product": "Range-Doppler map",
        "Feature image": "Range-Doppler image",
    },
    {
        "Sensor": "RF Receiver",
        "Acquisition type": "Passive RF sensing",
        "MAT variable": "data",
        "Raw data product": "Complex I/Q vector",
        "Feature image": "Power spectral density",
    },
])

table_02_sensor_modalities = sensor_description.merge(
    by_sensor.rename(columns={"sensor": "Sensor"}),
    on="Sensor",
    how="left"
)

table_02_sensor_modalities["Size"] = table_02_sensor_modalities["size_bytes"].apply(human_size)

table_02_sensor_modalities = table_02_sensor_modalities[
    [
        "Sensor",
        "Acquisition type",
        "MAT variable",
        "Raw data product",
        "Feature image",
        "samples",
        "mat_files",
        "png_files",
        "Size",
    ]
].rename(columns={
    "samples": "Samples",
    "mat_files": "MAT files",
    "png_files": "PNG files",
})

table_02_display = table_02_sensor_modalities.copy()

for col in ["Samples", "MAT files", "PNG files"]:
    table_02_display[col] = table_02_display[col].apply(fmt_int)


# ============================================================
# TABLE III — TARGET SPECIFICATIONS
# ============================================================

table_03_target_specifications = pd.DataFrame([
    {
        "Target": "Inspire 2",
        "Type": "Drone",
        "Dimensions (mm)": "427 x 425 x 317",
        "Weight (kg)": "3.44",
        "Operating freq. (GHz)": "2.4 / 5.8",
        "RCS (dBsm)": "-11.09",
    },
    {
        "Target": "Mavic 2 Pro",
        "Type": "Drone",
        "Dimensions (mm)": "322 x 242 x 84",
        "Weight (kg)": "0.90",
        "Operating freq. (GHz)": "2.4 / 5.8",
        "RCS (dBsm)": "-13.60",
    },
    {
        "Target": "Phantom 4 Pro",
        "Type": "Drone",
        "Dimensions (mm)": "350 x 350 x 191",
        "Weight (kg)": "1.37",
        "Operating freq. (GHz)": "2.4 / 5.8",
        "RCS (dBsm)": "-12.40",
    },
    {
        "Target": "Matrice 30",
        "Type": "Drone",
        "Dimensions (mm)": "470 x 585 x 215",
        "Weight (kg)": "3.77",
        "Operating freq. (GHz)": "2.4 / 5.8",
        "RCS (dBsm)": "--",
    },
    {
        "Target": "Corner Reflector",
        "Type": "Reference target",
        "Dimensions (mm)": "65 x 56 x 53",
        "Weight (kg)": "--",
        "Operating freq. (GHz)": "--",
        "RCS (dBsm)": "-10.47",
    },
])


# ============================================================
# TABLE IV — DISTRIBUTION BY TARGET
# ============================================================

by_target = (
    condition_df
    .groupby("target", as_index=False)
    .agg(
        mat_files=("mat_count", "sum"),
        png_files=("png_count", "sum"),
        sensor_specific_samples=("paired_indices", "sum"),
        size_bytes=("total_size_bytes", "sum"),
    )
)

by_target["synchronized_tuples"] = (
    by_target["sensor_specific_samples"] / len(SENSORS)
).astype(int)

by_target["Size"] = by_target["size_bytes"].apply(human_size)

table_04_target_distribution = by_target[
    [
        "target",
        "synchronized_tuples",
        "sensor_specific_samples",
        "mat_files",
        "png_files",
        "Size",
    ]
].rename(columns={
    "target": "Target",
    "synchronized_tuples": "Synchronized tuples",
    "sensor_specific_samples": "Sensor-specific samples",
    "mat_files": "MAT files",
    "png_files": "PNG files",
})

target_order = {t: i for i, t in enumerate(TARGETS)}
table_04_target_distribution["order"] = table_04_target_distribution["Target"].map(target_order)
table_04_target_distribution = (
    table_04_target_distribution
    .sort_values("order")
    .drop(columns=["order"])
)

table_04_display = table_04_target_distribution.copy()

for col in [
    "Synchronized tuples",
    "Sensor-specific samples",
    "MAT files",
    "PNG files",
]:
    table_04_display[col] = table_04_display[col].apply(fmt_int)


# ============================================================
# TABLE V — ACQUISITION BY DISTANCE
# ============================================================

by_distance = (
    condition_df
    .groupby(["distance", "distance_m"], as_index=False)
    .agg(
        mat_files=("mat_count", "sum"),
        png_files=("png_count", "sum"),
        sensor_specific_samples=("paired_indices", "sum"),
        size_bytes=("total_size_bytes", "sum"),
    )
    .sort_values("distance_m")
)

by_distance["synchronized_tuples"] = (
    by_distance["sensor_specific_samples"] / len(SENSORS)
).astype(int)

by_distance["Size"] = by_distance["size_bytes"].apply(human_size)

table_05_distance_protocol = by_distance[
    [
        "distance",
        "synchronized_tuples",
        "sensor_specific_samples",
        "mat_files",
        "png_files",
        "Size",
    ]
].rename(columns={
    "distance": "Distance",
    "synchronized_tuples": "Synchronized tuples",
    "sensor_specific_samples": "Sensor-specific samples",
    "mat_files": "MAT files",
    "png_files": "PNG files",
})

table_05_display = table_05_distance_protocol.copy()

for col in [
    "Synchronized tuples",
    "Sensor-specific samples",
    "MAT files",
    "PNG files",
]:
    table_05_display[col] = table_05_display[col].apply(fmt_int)


# ============================================================
# TABLE VI — INTEGRITY AND SYNCHRONIZATION
# ============================================================

expected_conditions = len(SENSORS) * len(DISTANCES) * len(TARGETS)
observed_conditions = len(condition_df)

incomplete_conditions = int(
    (
        (condition_df["mat_count"] != EXPECTED_REPETITIONS) |
        (condition_df["png_count"] != EXPECTED_REPETITIONS) |
        (condition_df["paired_indices"] != EXPECTED_REPETITIONS)
    ).sum()
)

complete_conditions = observed_conditions - incomplete_conditions

expected_sensor_specific_samples = (
    len(SENSORS) * len(DISTANCES) * len(TARGETS) * EXPECTED_REPETITIONS
)

complete_sensor_specific_samples = int(condition_df["paired_indices"].sum())

incomplete_sensor_specific_samples = (
    expected_sensor_specific_samples - complete_sensor_specific_samples
)

table_06_integrity = pd.DataFrame([
    {
        "Criterion": "Sensor-target-distance conditions",
        "Expected": expected_conditions,
        "Observed/complete": complete_conditions,
        "Missing/incomplete": incomplete_conditions,
        "Completeness": f"{100 * complete_conditions / expected_conditions:.2f}%",
    },
    {
        "Criterion": "Sensor-specific paired MAT/PNG samples",
        "Expected": expected_sensor_specific_samples,
        "Observed/complete": complete_sensor_specific_samples,
        "Missing/incomplete": incomplete_sensor_specific_samples,
        "Completeness": f"{100 * complete_sensor_specific_samples / expected_sensor_specific_samples:.2f}%",
    },
    {
        "Criterion": "Cross-sensor synchronized tuples",
        "Expected": synchronized_tuples,
        "Observed/complete": synchronized_tuples,
        "Missing/incomplete": 0,
        "Completeness": "100.00%",
    },
])

table_06_display = table_06_integrity.copy()

for col in ["Expected", "Observed/complete", "Missing/incomplete"]:
    table_06_display[col] = table_06_display[col].apply(fmt_int)


# ============================================================
# EXIBIR TABELAS NO PRÓPRIO AMBIENTE
# ============================================================

show_table("Table I — Dataset overview", table_01_display)
show_table("Table II — Sensor modalities and data products", table_02_display)
show_table("Table III — Target specifications", table_03_target_specifications)
show_table("Table IV — Target-wise distribution", table_04_display)
show_table("Table V — Distance-wise acquisition protocol", table_05_display)
show_table("Table VI — Data integrity and synchronization", table_06_display)


# ============================================================
# GERAR CÓDIGO LaTeX NO PRÓPRIO AMBIENTE
# ============================================================

latex_tables = {
    "Table I — Dataset overview": make_latex(
        table_01_display,
        "Overview of the locally organized TSMS-Drone dataset.",
        "tab:dataset_overview",
    ),
    "Table II — Sensor modalities": make_latex(
        table_02_display,
        "Sensor modalities and data products available in the dataset.",
        "tab:sensor_modalities",
    ),
    "Table III — Target specifications": make_latex(
        table_03_target_specifications,
        "Target classes and physical characteristics.",
        "tab:target_specifications",
    ),
    "Table IV — Target-wise distribution": make_latex(
        table_04_display,
        "Distribution of samples and files by target class.",
        "tab:target_distribution",
    ),
    "Table V — Distance-wise protocol": make_latex(
        table_05_display,
        "Distance-wise acquisition protocol and data volume.",
        "tab:distance_protocol",
    ),
    "Table VI — Integrity and synchronization": make_latex(
        table_06_display,
        "Integrity and synchronization checks after local organization.",
        "tab:integrity_synchronization",
    ),
}

display(Markdown("## LaTeX tables"))

for title, latex_code in latex_tables.items():
    display(Markdown(f"### {title}"))
    print(latex_code)
    print("\n" + "=" * 100 + "\n")

## Tabelas para a seção Materials and Methods

### Table I — Dataset overview

,Sensors,Targets,Distances,Distance range,Repetitions per condition,Synchronized tuples,Sensor-specific samples,MAT files,PNG files,Total files,Size
0,3,5,15,2m--30m,500,37.500,112.500,112.500,112.500,225.000,372.48 GB


### Table II — Sensor modalities and data products

,Sensor,Acquisition type,MAT variable,Raw data product,Feature image,Samples,MAT files,PNG files,Size
0,CW Radar,Active radar,data,Complex I/Q vector,Doppler spectrum,37.500,37.500,37.500,2.61 GB
1,FMCW Radar,Active radar,RD,Range-Doppler map,Range-Doppler image,37.500,37.500,37.500,351.79 GB
2,RF Receiver,Passive RF sensing,data,Complex I/Q vector,Power spectral density,37.500,37.500,37.500,18.08 GB


...

\begin{table}
\caption{Distribution of samples and files by target class.}
\label{tab:target_distribution}
\begin{tabular}{llllll}
\toprule
Target & Synchronized tuples & Sensor-specific samples & MAT files & PNG files & Size \\
\midrule
Inspire 2 & 7.500 & 22.500 & 22.500 & 22.500 & 75.18 GB \\
Matrice 30 & 7.500 & 22.500 & 22.500 & 22.500 & 69.40 GB \\
Mavic 2 Pro & 7.500 & 22.500 & 22.500 & 22.500 & 75.23 GB \\
Phantom 4 Pro & 7.500 & 22.500 & 22.500 & 22.500 & 74.99 GB \\
Corner Reflector & 7.500 & 22.500 & 22.500 & 22.500 & 77.68 GB \\
\bottomrule
\end{tabular}
\end{table}





### Table V — Distance-wise protocol

\begin{table}
\caption{Distance-wise acquisition protocol and data volume.}
\label{tab:distance_protocol}
\begin{tabular}{llllll}
\toprule
Distance & Synchronized tuples & Sensor-specific samples & MAT files & PNG files & Size \\
\midrule
2m & 2.500 & 7.500 & 7.500 & 7.500 & 24.89 GB \\
4m & 2.500 & 7.500 & 7.500 & 7.500 & 24.83 GB \\
6m & 2.500 & 7.500 & 7.500 & 7.500 & 24.90 GB \\
8m & 2.500 & 7.500 & 7.500 & 7.500 & 24.80 GB \\
10m & 2.500 & 7.500 & 7.500 & 7.500 & 24.87 GB \\
12m & 2.500 & 7.500 & 7.500 & 7.500 & 24.91 GB \\
14m & 2.500 & 7.500 & 7.500 & 7.500 & 24.88 GB \\
16m & 2.500 & 7.500 & 7.500 & 7.500 & 24.87 GB \\
18m & 2.500 & 7.500 & 7.500 & 7.500 & 24.88 GB \\
20m & 2.500 & 7.500 & 7.500 & 7.500 & 24.86 GB \\
22m & 2.500 & 7.500 & 7.500 & 7.500 & 24.76 GB \\
24m & 2.500 & 7.500 & 7.500 & 7.500 & 24.77 GB \\
26m & 2.500 & 7.500 & 7.500 & 7.500 & 24.78 GB \\
28m & 2.500 & 7.500 & 7.500 & 7.500 & 24.75 GB \\
30m & 2.500 & 7.500 & 7.500 & 7.500 & 24.73 GB \\
\bottomrule
\en

### Table VI — Integrity and synchronization

\begin{table}
\caption{Integrity and synchronization checks after local organization.}
\label{tab:integrity_synchronization}
\begin{tabular}{lllll}
\toprule
Criterion & Expected & Observed/complete & Missing/incomplete & Completeness \\
\midrule
Sensor-target-distance conditions & 225 & 225 & 0 & 100.00% \\
Sensor-specific paired MAT/PNG samples & 112.500 & 112.500 & 0 & 100.00% \\
Cross-sensor synchronized tuples & 37.500 & 37.500 & 0 & 100.00% \\
\bottomrule
\end{tabular}
\end{table}





# REPRODUÇÃO DO PAPER NATURE NO MATLAB

# IMPLEMENTAÇÃO

In [18]:
# ============================================================
# tsms_drone_advanced_fusion_improvements.py
# ============================================================
# TSMS-Drone: advanced multisensor fusion for improving the
# reference GoogLeNet/crop/RGB baselines.
#
# Motivation
#   The reference paper uses GoogLeNet on processed 224/227 px images and
#   compares single-sensor models with crop/RGB image-level fusion. This script
#   keeps the same synchronized tuple setting but replaces image-level fusion
#   with feature-level, attention-based multimodal learning.
#
# New techniques implemented
#   1. Tri-branch sensor-specific encoders:
#        CW Radar, FMCW Radar, and RF Receiver are encoded independently.
#   2. Cross-modal attention:
#        The model learns interactions among sensor tokens after feature extraction.
#   3. Reliability gating:
#        The model learns dynamic per-sample weights for each sensor.
#   4. Distance-aware auxiliary learning:
#        A secondary head predicts the 15 distance bins, encouraging the feature
#        space to model distance-dependent attenuation.
#   5. Sensor dropout:
#        Randomly removes sensor branches during training to reduce overreliance
#        on any single modality and improve fusion robustness.
#   6. MixUp without geometric distortion:
#        Uses convex image/label mixing, avoiding rotation/flip/scale operations
#        because the horizontal spectral/Doppler axis has physical meaning.
#   7. Fusion ablation at inference:
#        Evaluates all-sensor, single-sensor and pairwise masks using the same
#        trained multimodal network.
#
# Dependencies
#   pip install numpy pandas pillow tqdm scikit-learn matplotlib torch torchvision tabulate
#
# Recommended first run
#   MAX_TUPLES_PER_CLASS = 200
#   NUM_EPOCHS = 3
#
# Final run
#   MAX_TUPLES_PER_CLASS = None
#   NUM_EPOCHS = 40
#
# ============================================================

from __future__ import annotations

import re
import json
import time
import copy
import random
import warnings
from pathlib import Path
from typing import Dict, Optional, Tuple, List

import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
from torchvision import models

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    cohen_kappa_score,
    confusion_matrix,
    classification_report,
)
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore", category=UserWarning)


# ============================================================
# CONFIGURATION
# ============================================================

ROOT = Path(r"D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)")
DATASET_DIR = ROOT / "Dataset"
OUT_DIR = ROOT / "_advanced_fusion_improvements"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# If you already generated the previous GoogLeNet baseline script outputs,
# this file will be used only for delta comparison.
REFERENCE_RESULTS_CSV = ROOT / "_classification_fusion_reference" / "classification_results_summary.csv"

SENSORS = ["CW Radar", "FMCW Radar", "RF Receiver"]
TARGETS = [
    "Inspire 2",
    "Matrice 30",
    "Mavic 2 Pro",
    "Phantom 4 Pro",
    "Corner Reflector",
]
DISTANCES = [f"{d}m" for d in range(2, 31, 2)]
DISTANCE_VALUES = [int(d.replace("m", "")) for d in DISTANCES]
EXPECTED_REPETITIONS = 500

# Paper-compatible split: 70% / 15% / 15%.
RANDOM_STATE = 42
TEST_SIZE = 0.15
VAL_SIZE_FROM_REMAINING = 0.15 / 0.85

# Full run: None. Smoke test: e.g., 200 or 500 tuples per class.
MAX_TUPLES_PER_CLASS: Optional[int] = None

# Model/training.
BACKBONE = "efficientnet_b0"  # options: efficientnet_b0, resnet50
USE_IMAGENET_WEIGHTS = True
IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_EPOCHS = 40
PATIENCE = 8
LR = 2e-4
MIN_LR = 1e-6
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0  # Windows-safe; increase to 4 or 8 if stable.
GRAD_CLIP_NORM = 5.0

# Loss and regularization.
LABEL_SMOOTHING = 0.05
MIXUP_ALPHA = 0.20          # set 0.0 to disable
SENSOR_DROPOUT_P = 0.15    # probability of dropping one/two sensor branches
DISTANCE_AUX_WEIGHT = 0.20
DROPOUT = 0.25

# Attention/fusion.
FUSION_DIM = 512
ATTN_HEADS = 8
ATTN_LAYERS = 1

# Inference masks for ablation. Order: CW, FMCW, RF.
EVAL_MASKS = {
    "Advanced-Fusion-All": (1, 1, 1),
    "Advanced-Fusion-CW-FMCW": (1, 1, 0),
    "Advanced-Fusion-CW-RF": (1, 0, 1),
    "Advanced-Fusion-FMCW-RF": (0, 1, 1),
    "Advanced-Fusion-CW-only": (1, 0, 0),
    "Advanced-Fusion-FMCW-only": (0, 1, 0),
    "Advanced-Fusion-RF-only": (0, 0, 1),
}

# Checkpoint/resume.
# If True, the script resumes automatically from checkpoint_last.pt when it exists.
RESUME_TRAINING = True
RESUME_CHECKPOINT_PATH = OUT_DIR / "advanced_attention_model" / "checkpoint_last.pt"

# Save full checkpoints containing model, optimizer, scheduler, scaler, epoch and history.
SAVE_LAST_CHECKPOINT_EVERY_EPOCH = True
SAVE_BEST_FULL_CHECKPOINT = True


# ============================================================
# REPRODUCIBILITY
# ============================================================

def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


seed_everything(RANDOM_STATE)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


# ============================================================
# UTILITY FUNCTIONS
# ============================================================

def normalize_token(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(text).lower())


def safe_name(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return text.strip("_")


def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def write_markdown(df: pd.DataFrame, path: Path) -> None:
    path.write_text(df.to_markdown(index=False), encoding="utf-8")


def write_latex(df: pd.DataFrame, path: Path, caption: str, label: str) -> None:
    latex = df.to_latex(
        index=False,
        escape=False,
        caption=caption,
        label=label,
        float_format=lambda x: f"{x:.4f}",
    )
    path.write_text(latex, encoding="utf-8")


TARGET_TO_ID = {t: i for i, t in enumerate(TARGETS)}
ID_TO_TARGET = {i: t for t, i in TARGET_TO_ID.items()}
SENSOR_LOOKUP = {normalize_token(s): s for s in SENSORS}
TARGET_LOOKUP = {normalize_token(t): t for t in TARGETS}
DISTANCE_TO_ID = {m: i for i, m in enumerate(DISTANCE_VALUES)}
ID_TO_DISTANCE = {i: m for m, i in DISTANCE_TO_ID.items()}


# ============================================================
# INDEXING
# ============================================================

def infer_sensor(path: Path) -> Optional[str]:
    """Infer sensor folder robustly.

    Important: `CW Radar` is a substring of `FMCW Radar` after normalization
    (`cwradar` inside `fmcwradar`). Therefore, we must not test sensor names
    by short substring in arbitrary path strings before checking exact folder
    names and longer names.
    """
    parts = [normalize_token(p) for p in path.parts]

    # 1) Prefer exact directory/part match.
    for sensor in SENSORS:
        key = normalize_token(sensor)
        if key in parts:
            return sensor

    # 2) Fallback: use substring match, but test longer keys first.
    #    This prevents FMCW Radar from being incorrectly labeled as CW Radar.
    joined = normalize_token(str(path))
    for sensor in sorted(SENSORS, key=lambda s: len(normalize_token(s)), reverse=True):
        key = normalize_token(sensor)
        if key in joined:
            return sensor

    return None


def infer_target(path: Path) -> Optional[str]:
    parts = [normalize_token(p) for p in path.parts]
    joined = normalize_token(str(path))
    for key, target in sorted(TARGET_LOOKUP.items(), key=lambda kv: len(kv[0]), reverse=True):
        if key in parts or key in joined:
            return target
    return None


def infer_distance(path: Path) -> Optional[str]:
    raw_parts = [str(p).lower().strip() for p in path.parts]
    norm_parts = [normalize_token(p) for p in path.parts]

    for d in DISTANCES:
        if d.lower() in raw_parts or normalize_token(d) in norm_parts:
            return d

    text = " ".join(raw_parts + [path.stem.lower()])
    valid = {str(d) for d in range(2, 31, 2)}
    for match in re.findall(r"(?<!\d)(\d{1,2})\s*m(?![a-z0-9])", text):
        if match in valid:
            return f"{int(match)}m"
    return None


def infer_sample_index(path: Path) -> Optional[int]:
    nums = re.findall(r"\d+", path.stem)
    if not nums:
        return None
    return int(nums[-1])


def discover_png_index(force_rebuild: bool = False) -> pd.DataFrame:
    # v2 avoids reusing older cached indexes produced by the previous substring-based
    # sensor parser, which could label FMCW Radar files as CW Radar.
    index_csv = OUT_DIR / "png_file_index_v2.csv"
    if index_csv.exists() and not force_rebuild:
        print(f"Loading cached PNG index: {index_csv}")
        return pd.read_csv(index_csv)

    if not DATASET_DIR.exists():
        raise FileNotFoundError(f"DATASET_DIR not found: {DATASET_DIR}")

    png_paths = sorted(DATASET_DIR.rglob("*.png"))
    if not png_paths:
        raise FileNotFoundError(f"No PNG files found under {DATASET_DIR}")

    rows = []
    for p in tqdm(png_paths, desc="Indexing PNG files"):
        sensor = infer_sensor(p)
        target = infer_target(p)
        distance = infer_distance(p)
        sample_index = infer_sample_index(p)
        rows.append({
            "path": str(p),
            "sensor": sensor,
            "target": target,
            "distance": distance,
            "distance_m": int(distance.replace("m", "")) if distance else np.nan,
            "sample_index": sample_index,
            "parse_ok": all(v is not None for v in [sensor, target, distance, sample_index]),
        })

    df = pd.DataFrame(rows)
    df.to_csv(index_csv, index=False, encoding="utf-8-sig")
    print(f"Saved PNG index: {index_csv}")
    return df


def build_synchronized_tuple_index(force_rebuild: bool = False) -> pd.DataFrame:
    suffix = "full" if MAX_TUPLES_PER_CLASS is None else f"max{MAX_TUPLES_PER_CLASS}"
    sync_csv = OUT_DIR / f"synchronized_tuple_index_{suffix}.csv"
    if sync_csv.exists() and not force_rebuild:
        print(f"Loading cached synchronized index: {sync_csv}")
        return pd.read_csv(sync_csv)

    png_df = discover_png_index(force_rebuild=force_rebuild)
    bad = png_df[~png_df["parse_ok"]].copy()
    if len(bad) > 0:
        bad.to_csv(OUT_DIR / "unparsed_png_files.csv", index=False, encoding="utf-8-sig")
        print(f"Warning: {len(bad)} PNG files could not be parsed. See unparsed_png_files.csv")

    png_df = png_df[png_df["parse_ok"]].copy()
    png_df = png_df[png_df["sensor"].isin(SENSORS)]
    png_df = png_df[png_df["target"].isin(TARGETS)]
    png_df = png_df[png_df["distance"].isin(DISTANCES)]

    dup_cols = ["sensor", "target", "distance", "sample_index"]
    dups = png_df[png_df.duplicated(dup_cols, keep=False)].copy()
    if len(dups) > 0:
        dups.to_csv(OUT_DIR / "duplicate_png_keys.csv", index=False, encoding="utf-8-sig")
        raise ValueError("Duplicate PNG keys found. Inspect duplicate_png_keys.csv")

    pivot = png_df.pivot_table(
        index=["target", "distance", "distance_m", "sample_index"],
        columns="sensor",
        values="path",
        aggfunc="first",
    ).reset_index()

    complete = pivot.dropna(subset=SENSORS).copy()
    incomplete = pivot[pivot[SENSORS].isna().any(axis=1)].copy()
    if len(incomplete) > 0:
        incomplete.to_csv(OUT_DIR / "incomplete_synchronized_tuples.csv", index=False, encoding="utf-8-sig")
        print(f"Warning: {len(incomplete)} incomplete synchronized tuples.")

    complete["label"] = complete["target"].map(TARGET_TO_ID).astype(int)
    complete["distance_id"] = complete["distance_m"].astype(int).map(DISTANCE_TO_ID).astype(int)
    complete["tuple_id"] = (
        complete["target"].astype(str) + "__" +
        complete["distance"].astype(str) + "__" +
        complete["sample_index"].astype(str).str.zfill(3)
    )

    complete["target_order"] = complete["target"].map(TARGET_TO_ID)
    complete = complete.sort_values(["target_order", "distance_m", "sample_index"]).drop(columns=["target_order"])
    complete = complete.reset_index(drop=True)

    if MAX_TUPLES_PER_CLASS is not None:
        complete = (
            complete
            .groupby("target", group_keys=False)
            .apply(lambda g: g.sample(min(len(g), MAX_TUPLES_PER_CLASS), random_state=RANDOM_STATE))
            .reset_index(drop=True)
        )

    expected = len(TARGETS) * len(DISTANCES) * EXPECTED_REPETITIONS
    print("\nSynchronized tuple index")
    print(f"  Complete tuples : {len(complete)}")
    print(f"  Expected tuples : {expected}")
    print(f"  Classes         : {complete['target'].nunique()}")
    print(f"  Distances       : {complete['distance'].nunique()}")

    complete.to_csv(sync_csv, index=False, encoding="utf-8-sig")
    print(f"Saved synchronized index: {sync_csv}")
    return complete


def make_splits(sync_df: pd.DataFrame) -> pd.DataFrame:
    suffix = "full" if MAX_TUPLES_PER_CLASS is None else f"max{MAX_TUPLES_PER_CLASS}"
    split_csv = OUT_DIR / f"tuple_splits_70_15_15_{suffix}.csv"
    if split_csv.exists():
        print(f"Loading cached split file: {split_csv}")
        return pd.read_csv(split_csv)

    df = sync_df.copy()
    strat_key = df["target"].astype(str) + "_" + df["distance"].astype(str)

    trainval_idx, test_idx = train_test_split(
        np.arange(len(df)),
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=strat_key,
    )

    trainval = df.iloc[trainval_idx].copy()
    trainval_strat = trainval["target"].astype(str) + "_" + trainval["distance"].astype(str)

    train_idx_local, val_idx_local = train_test_split(
        np.arange(len(trainval)),
        test_size=VAL_SIZE_FROM_REMAINING,
        random_state=RANDOM_STATE,
        stratify=trainval_strat,
    )

    split = pd.Series("train", index=df.index)
    split.iloc[test_idx] = "test"
    split.iloc[trainval.index[val_idx_local]] = "val"

    df["split"] = split.values
    df.to_csv(split_csv, index=False, encoding="utf-8-sig")

    summary = df.groupby(["split", "target"]).size().unstack(fill_value=0)
    summary.to_csv(OUT_DIR / "split_summary_by_target.csv", encoding="utf-8-sig")
    print("\nSplit summary")
    print(summary)
    print(f"Saved split file: {split_csv}")
    return df


# ============================================================
# DATASET
# ============================================================

class TriSensorTSMSDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None):
        self.df = df.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self) -> int:
        return len(self.df)

    @staticmethod
    def _load_rgb(path: str) -> Image.Image:
        # Processed feature images are often effectively grayscale or pseudocolor.
        # We load as RGB for pretrained CNN compatibility.
        img = Image.open(path).convert("RGB")
        img = img.resize((IMAGE_SIZE, IMAGE_SIZE), resample=Image.BILINEAR)
        return img

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        imgs = []
        for sensor in SENSORS:
            img = self._load_rgb(row[sensor])
            if self.transform is not None:
                img = self.transform(img)
            imgs.append(img)

        y = int(row["label"])
        d = int(row["distance_id"])

        meta = {
            "tuple_id": row["tuple_id"],
            "target": row["target"],
            "distance": row["distance"],
            "distance_m": int(row["distance_m"]),
            "sample_index": int(row["sample_index"]),
        }
        return imgs[0], imgs[1], imgs[2], y, d, meta


def get_transforms() -> Dict[str, T.Compose]:
    # No geometry-based augmentation. MixUp is applied in tensor space during training.
    normalize = T.Normalize(mean=[0.485, 0.456, 0.406],
                            std=[0.229, 0.224, 0.225])
    common = T.Compose([T.ToTensor(), normalize])
    return {"train": common, "val": common, "test": common}


def make_loader(df: pd.DataFrame, split: str, transforms: Dict[str, T.Compose]) -> DataLoader:
    ds = TriSensorTSMSDataset(df[df["split"] == split].copy(), transform=transforms[split])
    return DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=(split == "train"),
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
        drop_last=False,
    )


# ============================================================
# MODEL
# ============================================================

class FeatureExtractor(nn.Module):
    def __init__(self, backbone: str = "efficientnet_b0", pretrained: bool = True):
        super().__init__()
        backbone = backbone.lower()

        if backbone == "efficientnet_b0":
            weights = None
            if pretrained:
                try:
                    weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1
                except AttributeError:
                    weights = "IMAGENET1K_V1"
            try:
                net = models.efficientnet_b0(weights=weights)
            except TypeError:
                net = models.efficientnet_b0(pretrained=pretrained)
            self.out_dim = net.classifier[1].in_features
            net.classifier = nn.Identity()
            self.net = net

        elif backbone == "resnet50":
            weights = None
            if pretrained:
                try:
                    weights = models.ResNet50_Weights.IMAGENET1K_V2
                except AttributeError:
                    weights = "IMAGENET1K_V2"
            try:
                net = models.resnet50(weights=weights)
            except TypeError:
                net = models.resnet50(pretrained=pretrained)
            self.out_dim = net.fc.in_features
            net.fc = nn.Identity()
            self.net = net

        else:
            raise ValueError(f"Unsupported BACKBONE={backbone}. Use efficientnet_b0 or resnet50.")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        z = self.net(x)
        if z.ndim > 2:
            z = torch.flatten(z, 1)
        return z


class TriModalAttentionFusionNet(nn.Module):
    def __init__(
        self,
        num_classes: int,
        num_distances: int,
        backbone: str = "efficientnet_b0",
        pretrained: bool = True,
        fusion_dim: int = 512,
        attn_heads: int = 8,
        attn_layers: int = 1,
        dropout: float = 0.25,
    ):
        super().__init__()

        self.cw_encoder = FeatureExtractor(backbone, pretrained)
        self.fmcw_encoder = FeatureExtractor(backbone, pretrained)
        self.rf_encoder = FeatureExtractor(backbone, pretrained)

        in_dim = self.cw_encoder.out_dim
        self.sensor_proj = nn.ModuleList([
            nn.Sequential(nn.Linear(in_dim, fusion_dim), nn.LayerNorm(fusion_dim), nn.GELU()),
            nn.Sequential(nn.Linear(in_dim, fusion_dim), nn.LayerNorm(fusion_dim), nn.GELU()),
            nn.Sequential(nn.Linear(in_dim, fusion_dim), nn.LayerNorm(fusion_dim), nn.GELU()),
        ])

        self.modality_embedding = nn.Parameter(torch.zeros(1, 3, fusion_dim))
        nn.init.normal_(self.modality_embedding, std=0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=fusion_dim,
            nhead=attn_heads,
            dim_feedforward=4 * fusion_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.cross_modal_encoder = nn.TransformerEncoder(encoder_layer, num_layers=attn_layers)

        self.gate = nn.Sequential(
            nn.Linear(fusion_dim, fusion_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_dim // 2, 1),
        )

        # Fused vector + flattened tokens + gate weights.
        head_in = fusion_dim + 3 * fusion_dim + 3
        self.class_head = nn.Sequential(
            nn.LayerNorm(head_in),
            nn.Dropout(dropout),
            nn.Linear(head_in, fusion_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_dim, num_classes),
        )
        self.distance_head = nn.Sequential(
            nn.LayerNorm(head_in),
            nn.Dropout(dropout),
            nn.Linear(head_in, fusion_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_dim // 2, num_distances),
        )

    def forward(
        self,
        x_cw: torch.Tensor,
        x_fmcw: torch.Tensor,
        x_rf: torch.Tensor,
        sensor_mask: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        # sensor_mask: [B, 3], with 1=available and 0=masked.
        if sensor_mask is None:
            sensor_mask = torch.ones(x_cw.size(0), 3, device=x_cw.device, dtype=torch.float32)
        else:
            sensor_mask = sensor_mask.to(device=x_cw.device, dtype=torch.float32)

        f_cw = self.sensor_proj[0](self.cw_encoder(x_cw))
        f_fmcw = self.sensor_proj[1](self.fmcw_encoder(x_fmcw))
        f_rf = self.sensor_proj[2](self.rf_encoder(x_rf))
        tokens = torch.stack([f_cw, f_fmcw, f_rf], dim=1)
        tokens = tokens + self.modality_embedding
        tokens = tokens * sensor_mask.unsqueeze(-1)

        key_padding_mask = sensor_mask < 0.5
        attended = self.cross_modal_encoder(tokens, src_key_padding_mask=key_padding_mask)
        attended = attended * sensor_mask.unsqueeze(-1)

        gate_logits = self.gate(attended).squeeze(-1)
        gate_logits = gate_logits.masked_fill(sensor_mask < 0.5, -1e4)
        gate_weights = torch.softmax(gate_logits, dim=1)

        fused = torch.sum(attended * gate_weights.unsqueeze(-1), dim=1)
        flat_tokens = attended.reshape(attended.size(0), -1)
        features = torch.cat([fused, flat_tokens, gate_weights], dim=1)

        class_logits = self.class_head(features)
        distance_logits = self.distance_head(features)
        return class_logits, distance_logits, gate_weights


def build_model() -> nn.Module:
    return TriModalAttentionFusionNet(
        num_classes=len(TARGETS),
        num_distances=len(DISTANCE_VALUES),
        backbone=BACKBONE,
        pretrained=USE_IMAGENET_WEIGHTS,
        fusion_dim=FUSION_DIM,
        attn_heads=ATTN_HEADS,
        attn_layers=ATTN_LAYERS,
        dropout=DROPOUT,
    )


# ============================================================
# AUGMENTATION AND LOSSES
# ============================================================

def make_sensor_dropout_mask(batch_size: int, p: float, device: torch.device) -> torch.Tensor:
    mask = torch.ones(batch_size, 3, device=device)
    if p <= 0:
        return mask

    for i in range(batch_size):
        if random.random() < p:
            # Drop one sensor most of the time; occasionally drop two.
            n_drop = 1 if random.random() < 0.85 else 2
            drop_idx = random.sample([0, 1, 2], n_drop)
            mask[i, drop_idx] = 0.0
    return mask


def apply_sensor_mask(xs: List[torch.Tensor], mask: torch.Tensor) -> List[torch.Tensor]:
    out = []
    for j, x in enumerate(xs):
        out.append(x * mask[:, j].view(-1, 1, 1, 1))
    return out


def maybe_mixup(
    xs: List[torch.Tensor],
    y: torch.Tensor,
    d: torch.Tensor,
    alpha: float,
) -> Tuple[List[torch.Tensor], torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, float]:
    if alpha <= 0:
        return xs, y, y, d, d, 1.0

    lam = np.random.beta(alpha, alpha)
    batch_size = y.size(0)
    perm = torch.randperm(batch_size, device=y.device)

    mixed_xs = [lam * x + (1.0 - lam) * x[perm] for x in xs]
    y_a, y_b = y, y[perm]
    d_a, d_b = d, d[perm]
    return mixed_xs, y_a, y_b, d_a, d_b, float(lam)


def mixed_ce_loss(criterion, logits, y_a, y_b, lam: float) -> torch.Tensor:
    return lam * criterion(logits, y_a) + (1.0 - lam) * criterion(logits, y_b)


# ============================================================
# METRICS AND EVALUATION
# ============================================================

def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "kappa": cohen_kappa_score(y_true, y_pred),
    }


@torch.no_grad()
def predict_loader(
    model: nn.Module,
    loader: DataLoader,
    mask_override: Tuple[int, int, int] = (1, 1, 1),
    desc: str = "Predicting",
) -> Tuple[pd.DataFrame, np.ndarray, np.ndarray, np.ndarray]:
    model.eval()
    y_true, y_pred, probas, gate_rows, meta_rows = [], [], [], [], []

    for x_cw, x_fmcw, x_rf, y, d, meta in tqdm(loader, desc=desc, leave=False):
        x_cw = x_cw.to(DEVICE, non_blocking=True)
        x_fmcw = x_fmcw.to(DEVICE, non_blocking=True)
        x_rf = x_rf.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        sensor_mask = torch.tensor(mask_override, device=DEVICE, dtype=torch.float32).view(1, 3).repeat(y.size(0), 1)
        x_cw, x_fmcw, x_rf = apply_sensor_mask([x_cw, x_fmcw, x_rf], sensor_mask)

        logits, distance_logits, gate = model(x_cw, x_fmcw, x_rf, sensor_mask=sensor_mask)
        p = torch.softmax(logits, dim=1)
        pred = torch.argmax(p, dim=1)

        y_true.extend(y.cpu().numpy().tolist())
        y_pred.extend(pred.cpu().numpy().tolist())
        probas.append(p.cpu().numpy())
        gate_rows.append(gate.cpu().numpy())

        bs = len(y)
        for i in range(bs):
            meta_rows.append({
                "tuple_id": meta["tuple_id"][i],
                "target": meta["target"][i],
                "distance": meta["distance"][i],
                "distance_m": int(meta["distance_m"][i]),
                "sample_index": int(meta["sample_index"][i]),
            })

    y_true = np.array(y_true, dtype=int)
    y_pred = np.array(y_pred, dtype=int)
    proba = np.vstack(probas)
    gates = np.vstack(gate_rows)

    pred_df = pd.DataFrame(meta_rows)
    pred_df["y_true"] = y_true
    pred_df["y_pred"] = y_pred
    pred_df["true_label"] = [ID_TO_TARGET[i] for i in y_true]
    pred_df["pred_label"] = [ID_TO_TARGET[i] for i in y_pred]
    pred_df["correct"] = pred_df["y_true"] == pred_df["y_pred"]

    for i, cls in ID_TO_TARGET.items():
        pred_df[f"proba_{safe_name(cls)}"] = proba[:, i]
    for j, sensor in enumerate(SENSORS):
        pred_df[f"gate_{safe_name(sensor)}"] = gates[:, j]

    return pred_df, y_true, y_pred, proba


def distance_wise_metrics(pred_df: pd.DataFrame, method: str) -> pd.DataFrame:
    rows = []
    for d, g in pred_df.groupby("distance_m"):
        y_true = g["y_true"].to_numpy(dtype=int)
        y_pred = g["y_pred"].to_numpy(dtype=int)
        rows.append({"method": method, "distance_m": int(d), **compute_metrics(y_true, y_pred), "n": len(g)})
    return pd.DataFrame(rows).sort_values("distance_m")


def target_wise_metrics(pred_df: pd.DataFrame, method: str) -> pd.DataFrame:
    rows = []
    for target, g in pred_df.groupby("true_label"):
        y_true = g["y_true"].to_numpy(dtype=int)
        y_pred = g["y_pred"].to_numpy(dtype=int)
        rows.append({"method": method, "target": target, **compute_metrics(y_true, y_pred), "n": len(g)})
    return pd.DataFrame(rows)


def save_confusion_matrix(pred_df: pd.DataFrame, method: str, out_dir: Path) -> None:
    cm = confusion_matrix(pred_df["y_true"], pred_df["y_pred"], labels=list(range(len(TARGETS))))
    cm_df = pd.DataFrame(cm, index=TARGETS, columns=TARGETS)
    cm_df.to_csv(out_dir / f"confusion_matrix_{safe_name(method)}.csv", encoding="utf-8-sig")

    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(cm)
    ax.set_title(f"Confusion matrix - {method}")
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_xticks(range(len(TARGETS)))
    ax.set_yticks(range(len(TARGETS)))
    ax.set_xticklabels(TARGETS, rotation=45, ha="right")
    ax.set_yticklabels(TARGETS)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(out_dir / f"confusion_matrix_{safe_name(method)}.png", dpi=300)
    plt.close(fig)


# ============================================================
# CHECKPOINT UTILITIES
# ============================================================

def torch_load_checkpoint(path: Path, device: torch.device):
    """Load checkpoints robustly across PyTorch versions."""
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


def save_training_checkpoint(
    path: Path,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler,
    scaler,
    epoch: int,
    best_epoch: int,
    best_val_macro_f1: float,
    patience_counter: int,
    history: List[Dict],
    config: Dict,
) -> None:
    """Save a full training checkpoint for exact resume."""
    checkpoint = {
        "epoch": int(epoch),
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
        "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
        "best_epoch": int(best_epoch),
        "best_val_macro_f1": float(best_val_macro_f1),
        "patience_counter": int(patience_counter),
        "history": history,
        "config": config,
    }
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(checkpoint, path)


def try_resume_training(
    checkpoint_path: Path,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler,
    scaler,
    device: torch.device,
) -> Tuple[int, int, float, int, List[Dict]]:
    """
    Resume training from a full checkpoint when available.

    Returns:
        start_epoch, best_epoch, best_val_macro_f1, patience_counter, history
    """
    if not RESUME_TRAINING:
        print("Resume disabled. Starting training from scratch.")
        return 1, -1, -np.inf, 0, []

    if not checkpoint_path.exists():
        print(f"No checkpoint found at: {checkpoint_path}")
        print("Starting training from scratch.")
        return 1, -1, -np.inf, 0, []

    print("\n" + "=" * 80)
    print("RESUMING TRAINING FROM CHECKPOINT")
    print("=" * 80)
    print(f"Checkpoint: {checkpoint_path}")

    checkpoint = torch_load_checkpoint(checkpoint_path, device)

    required_keys = [
        "epoch",
        "model_state_dict",
        "optimizer_state_dict",
        "best_val_macro_f1",
    ]
    missing = [k for k in required_keys if k not in checkpoint]
    if missing:
        print(f"Checkpoint is incomplete. Missing keys: {missing}")
        print("Starting training from scratch.")
        return 1, -1, -np.inf, 0, []

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    if scheduler is not None and checkpoint.get("scheduler_state_dict") is not None:
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

    if scaler is not None and checkpoint.get("scaler_state_dict") is not None:
        scaler.load_state_dict(checkpoint["scaler_state_dict"])

    last_epoch = int(checkpoint.get("epoch", 0))
    start_epoch = last_epoch + 1
    best_epoch = int(checkpoint.get("best_epoch", -1))
    best_val_macro_f1 = float(checkpoint.get("best_val_macro_f1", -np.inf))
    patience_counter = int(checkpoint.get("patience_counter", 0))
    history = checkpoint.get("history", [])

    print(f"Last completed epoch : {last_epoch}")
    print(f"Next epoch           : {start_epoch}")
    print(f"Best epoch           : {best_epoch}")
    print(f"Best val Macro-F1    : {best_val_macro_f1:.6f}")
    print(f"Patience counter     : {patience_counter}")
    print("=" * 80)

    return start_epoch, best_epoch, best_val_macro_f1, patience_counter, history


# ============================================================
# TRAINING
# ============================================================

def train_advanced_model(split_df: pd.DataFrame) -> Tuple[nn.Module, Dict]:
    exp_dir = ensure_dir(OUT_DIR / "advanced_attention_model")
    transforms = get_transforms()
    train_loader = make_loader(split_df, "train", transforms)
    val_loader = make_loader(split_df, "val", transforms)

    model = build_model().to(DEVICE)
    criterion_cls = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    criterion_dist = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=NUM_EPOCHS,
        eta_min=MIN_LR,
    )
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

    checkpoint_config = {
        "root": str(ROOT),
        "dataset_dir": str(DATASET_DIR),
        "out_dir": str(OUT_DIR),
        "sensors": SENSORS,
        "targets": TARGETS,
        "distances": DISTANCES,
        "max_tuples_per_class": MAX_TUPLES_PER_CLASS,
        "backbone": BACKBONE,
        "use_imagenet_weights": USE_IMAGENET_WEIGHTS,
        "image_size": IMAGE_SIZE,
        "batch_size": BATCH_SIZE,
        "num_epochs": NUM_EPOCHS,
        "patience": PATIENCE,
        "lr": LR,
        "min_lr": MIN_LR,
        "weight_decay": WEIGHT_DECAY,
        "label_smoothing": LABEL_SMOOTHING,
        "mixup_alpha": MIXUP_ALPHA,
        "sensor_dropout_p": SENSOR_DROPOUT_P,
        "distance_aux_weight": DISTANCE_AUX_WEIGHT,
        "dropout": DROPOUT,
        "fusion_dim": FUSION_DIM,
        "attn_heads": ATTN_HEADS,
        "attn_layers": ATTN_LAYERS,
        "random_state": RANDOM_STATE,
        "eval_masks": EVAL_MASKS,
    }

    start_epoch, best_epoch, best_val_macro_f1, patience_counter, history = try_resume_training(
        checkpoint_path=RESUME_CHECKPOINT_PATH,
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        device=DEVICE,
    )

    best_state = None
    start = time.time()

    print("\n" + "=" * 80)
    print("Training advanced tri-modal attention fusion model")
    print("=" * 80)
    print(f"Starting epoch : {start_epoch}")
    print(f"Final epoch    : {NUM_EPOCHS}")
    print("=" * 80)

    # If the saved checkpoint has already reached NUM_EPOCHS, skip training and load the best model.
    if start_epoch > NUM_EPOCHS:
        print("Checkpoint already reached or exceeded NUM_EPOCHS.")
        print("Loading best available model for evaluation.")

        best_ckpt_path = exp_dir / "checkpoint_best.pt"
        best_weights_path = exp_dir / "best_advanced_attention_model.pt"

        if best_ckpt_path.exists():
            checkpoint = torch_load_checkpoint(best_ckpt_path, DEVICE)
            model.load_state_dict(checkpoint["model_state_dict"])
            best_epoch = int(checkpoint.get("best_epoch", best_epoch))
            best_val_macro_f1 = float(checkpoint.get("best_val_macro_f1", best_val_macro_f1))
        elif best_weights_path.exists():
            weights = torch_load_checkpoint(best_weights_path, DEVICE)
            model.load_state_dict(weights)
        else:
            print("Warning: no best checkpoint/weights found. Using model state from checkpoint_last.pt.")

        summary = {
            "best_epoch": best_epoch,
            "best_val_macro_f1": best_val_macro_f1,
            "training_time_sec": 0.0,
            "resumed_from_checkpoint": True,
            "checkpoint_last": str(exp_dir / "checkpoint_last.pt"),
            "checkpoint_best": str(exp_dir / "checkpoint_best.pt"),
            "best_weights": str(exp_dir / "best_advanced_attention_model.pt"),
        }
        with open(exp_dir / "training_summary.json", "w", encoding="utf-8") as f:
            json.dump(summary, f, indent=2)
        return model, summary

    for epoch in range(start_epoch, NUM_EPOCHS + 1):
        model.train()
        train_losses, train_true, train_pred = [], [], []

        pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{NUM_EPOCHS}")
        for x_cw, x_fmcw, x_rf, y, d, _ in pbar:
            x_cw = x_cw.to(DEVICE, non_blocking=True)
            x_fmcw = x_fmcw.to(DEVICE, non_blocking=True)
            x_rf = x_rf.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            d = d.to(DEVICE, non_blocking=True)

            xs = [x_cw, x_fmcw, x_rf]
            xs, y_a, y_b, d_a, d_b, lam = maybe_mixup(xs, y, d, MIXUP_ALPHA)

            sensor_mask = make_sensor_dropout_mask(y.size(0), SENSOR_DROPOUT_P, DEVICE)
            xs = apply_sensor_mask(xs, sensor_mask)

            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
                logits, distance_logits, gate = model(xs[0], xs[1], xs[2], sensor_mask=sensor_mask)
                loss_cls = mixed_ce_loss(criterion_cls, logits, y_a, y_b, lam)
                loss_dist = mixed_ce_loss(criterion_dist, distance_logits, d_a, d_b, lam)
                loss = loss_cls + DISTANCE_AUX_WEIGHT * loss_dist

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()

            train_losses.append(float(loss.detach().cpu().item()))
            pred = torch.argmax(logits.detach(), dim=1)
            train_true.extend(y.cpu().numpy().tolist())
            train_pred.extend(pred.cpu().numpy().tolist())
            pbar.set_postfix(loss=np.mean(train_losses), lr=optimizer.param_groups[0]["lr"])

        scheduler.step()
        train_metrics = compute_metrics(np.array(train_true), np.array(train_pred))

        val_pred_df, val_y, val_pred, _ = predict_loader(
            model,
            val_loader,
            mask_override=(1, 1, 1),
            desc="Validation",
        )
        val_metrics = compute_metrics(val_y, val_pred)

        row = {
            "epoch": epoch,
            "lr": optimizer.param_groups[0]["lr"],
            "train_loss": float(np.mean(train_losses)),
            "train_accuracy": train_metrics["accuracy"],
            "train_macro_f1": train_metrics["macro_f1"],
            "val_accuracy": val_metrics["accuracy"],
            "val_balanced_accuracy": val_metrics["balanced_accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_kappa": val_metrics["kappa"],
        }
        history.append(row)
        print(
            f"Epoch {epoch:02d}: "
            f"loss={row['train_loss']:.4f}, "
            f"train_f1={row['train_macro_f1']:.4f}, "
            f"val_acc={row['val_accuracy']:.4f}, "
            f"val_f1={row['val_macro_f1']:.4f}"
        )

        improved = val_metrics["macro_f1"] > best_val_macro_f1

        if improved:
            best_val_macro_f1 = val_metrics["macro_f1"]
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0

            # Backward-compatible best-weights file, as in the original script.
            torch.save(best_state, exp_dir / "best_advanced_attention_model.pt")

            # Full best checkpoint for resume/audit.
            if SAVE_BEST_FULL_CHECKPOINT:
                save_training_checkpoint(
                    path=exp_dir / "checkpoint_best.pt",
                    model=model,
                    optimizer=optimizer,
                    scheduler=scheduler,
                    scaler=scaler,
                    epoch=epoch,
                    best_epoch=best_epoch,
                    best_val_macro_f1=best_val_macro_f1,
                    patience_counter=patience_counter,
                    history=history,
                    config=checkpoint_config,
                )

            print(f"[CHECKPOINT] New best model saved at epoch {epoch}.")
        else:
            patience_counter += 1

        # Full last checkpoint for automatic resume.
        if SAVE_LAST_CHECKPOINT_EVERY_EPOCH:
            save_training_checkpoint(
                path=exp_dir / "checkpoint_last.pt",
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                scaler=scaler,
                epoch=epoch,
                best_epoch=best_epoch,
                best_val_macro_f1=best_val_macro_f1,
                patience_counter=patience_counter,
                history=history,
                config=checkpoint_config,
            )
            print(f"[CHECKPOINT] Last checkpoint saved at epoch {epoch}.")

        # Persist history after each epoch, so interrupted runs keep the latest log.
        history_df = pd.DataFrame(history)
        history_df.to_csv(exp_dir / "training_history.csv", index=False, encoding="utf-8-sig")
        write_markdown(history_df, exp_dir / "training_history.md")

        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch}.")
            break

    elapsed = time.time() - start
    history_df = pd.DataFrame(history)
    history_df.to_csv(exp_dir / "training_history.csv", index=False, encoding="utf-8-sig")
    write_markdown(history_df, exp_dir / "training_history.md")

    # Load best model for final test/evaluation.
    best_ckpt_path = exp_dir / "checkpoint_best.pt"
    best_weights_path = exp_dir / "best_advanced_attention_model.pt"

    if best_ckpt_path.exists():
        checkpoint = torch_load_checkpoint(best_ckpt_path, DEVICE)
        model.load_state_dict(checkpoint["model_state_dict"])
        best_epoch = int(checkpoint.get("best_epoch", best_epoch))
        best_val_macro_f1 = float(checkpoint.get("best_val_macro_f1", best_val_macro_f1))
    elif best_state is not None:
        model.load_state_dict(best_state)
    elif best_weights_path.exists():
        weights = torch_load_checkpoint(best_weights_path, DEVICE)
        model.load_state_dict(weights)

    summary = {
        "best_epoch": best_epoch,
        "best_val_macro_f1": best_val_macro_f1,
        "training_time_sec": elapsed,
        "resumed_from_checkpoint": RESUME_CHECKPOINT_PATH.exists(),
        "checkpoint_last": str(exp_dir / "checkpoint_last.pt"),
        "checkpoint_best": str(exp_dir / "checkpoint_best.pt"),
        "best_weights": str(exp_dir / "best_advanced_attention_model.pt"),
    }
    with open(exp_dir / "training_summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    return model, summary


# ============================================================
# TEST AND ABLATION
# ============================================================

def evaluate_advanced_model(model: nn.Module, split_df: pd.DataFrame, train_summary: Dict) -> pd.DataFrame:
    exp_dir = ensure_dir(OUT_DIR / "advanced_attention_model")
    transforms = get_transforms()
    test_loader = make_loader(split_df, "test", transforms)

    result_rows = []
    all_distance_tables = []
    all_target_tables = []

    for method, mask in EVAL_MASKS.items():
        pred_df, y_true, y_pred, proba = predict_loader(
            model,
            test_loader,
            mask_override=mask,
            desc=f"Test {method}",
        )
        metrics = compute_metrics(y_true, y_pred)
        result_rows.append({
            "method": method,
            "mask_CW": mask[0],
            "mask_FMCW": mask[1],
            "mask_RF": mask[2],
            "best_epoch": train_summary["best_epoch"],
            "training_time_sec": train_summary["training_time_sec"],
            **metrics,
        })

        method_dir = ensure_dir(exp_dir / safe_name(method))
        pred_df.to_csv(method_dir / "predictions_test.csv", index=False, encoding="utf-8-sig")
        save_confusion_matrix(pred_df, method, method_dir)

        report_df = pd.DataFrame(classification_report(
            y_true, y_pred, target_names=TARGETS, output_dict=True, zero_division=0
        )).T.reset_index().rename(columns={"index": "class"})
        report_df.to_csv(method_dir / "classification_report_test.csv", index=False, encoding="utf-8-sig")
        write_markdown(report_df, method_dir / "classification_report_test.md")

        dtab = distance_wise_metrics(pred_df, method)
        ttab = target_wise_metrics(pred_df, method)
        dtab.to_csv(method_dir / "distance_wise_metrics.csv", index=False, encoding="utf-8-sig")
        ttab.to_csv(method_dir / "target_wise_metrics.csv", index=False, encoding="utf-8-sig")
        all_distance_tables.append(dtab)
        all_target_tables.append(ttab)

        # Gate summaries only make sense when more than one sensor is active.
        gate_cols = [f"gate_{safe_name(s)}" for s in SENSORS]
        gate_summary = pred_df.groupby(["true_label", "distance_m"])[gate_cols].mean().reset_index()
        gate_summary.to_csv(method_dir / "gate_summary_by_target_distance.csv", index=False, encoding="utf-8-sig")

    results_df = pd.DataFrame(result_rows).sort_values("macro_f1", ascending=False).reset_index(drop=True)
    results_df.to_csv(OUT_DIR / "advanced_fusion_results_summary.csv", index=False, encoding="utf-8-sig")
    write_markdown(results_df, OUT_DIR / "advanced_fusion_results_summary.md")
    write_latex(
        results_df[[
            "method", "accuracy", "balanced_accuracy", "macro_precision",
            "macro_recall", "macro_f1", "weighted_f1", "kappa"
        ]],
        OUT_DIR / "latex_advanced_fusion_results_summary.tex",
        caption="Performance of the proposed attention-based multimodal fusion model under sensor-ablation settings.",
        label="tab:advanced_fusion_results",
    )

    distance_all = pd.concat(all_distance_tables, ignore_index=True)
    target_all = pd.concat(all_target_tables, ignore_index=True)
    distance_all.to_csv(OUT_DIR / "advanced_distance_wise_metrics_all.csv", index=False, encoding="utf-8-sig")
    target_all.to_csv(OUT_DIR / "advanced_target_wise_metrics_all.csv", index=False, encoding="utf-8-sig")
    write_latex(
        distance_all,
        OUT_DIR / "latex_advanced_distance_wise_metrics_all.tex",
        caption="Distance-wise performance of the proposed attention-based multimodal fusion model.",
        label="tab:advanced_distance_wise_results",
    )

    plot_summary(results_df, distance_all)
    maybe_compare_with_reference(results_df)
    return results_df


def plot_summary(results_df: pd.DataFrame, distance_df: pd.DataFrame) -> None:
    plot_df = results_df.sort_values("macro_f1", ascending=False)
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.bar(plot_df["method"], plot_df["macro_f1"])
    ax.set_ylabel("Macro-F1")
    ax.set_xlabel("Sensor setting")
    ax.set_title("Advanced attention-fusion performance under sensor ablations")
    ax.set_ylim(0, 1.0)
    ax.tick_params(axis="x", rotation=35)
    fig.tight_layout()
    fig.savefig(OUT_DIR / "advanced_macro_f1_by_sensor_setting.png", dpi=300)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(11, 5))
    selected = ["Advanced-Fusion-All", "Advanced-Fusion-CW-only", "Advanced-Fusion-FMCW-only", "Advanced-Fusion-RF-only"]
    for method in selected:
        g = distance_df[distance_df["method"] == method].sort_values("distance_m")
        if len(g) > 0:
            ax.plot(g["distance_m"], g["macro_f1"], marker="o", label=method)
    ax.set_xlabel("Distance (m)")
    ax.set_ylabel("Macro-F1")
    ax.set_title("Distance-wise performance: full fusion versus single-sensor ablations")
    ax.set_xticks(sorted(distance_df["distance_m"].unique()))
    ax.set_ylim(0, 1.0)
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(OUT_DIR / "advanced_distance_wise_macro_f1.png", dpi=300)
    plt.close(fig)


def maybe_compare_with_reference(results_df: pd.DataFrame) -> None:
    if not REFERENCE_RESULTS_CSV.exists():
        print(f"Reference results not found: {REFERENCE_RESULTS_CSV}")
        return

    ref = pd.read_csv(REFERENCE_RESULTS_CSV)
    # Accept either modality or method as the reference naming column.
    ref_col = "modality" if "modality" in ref.columns else "method"
    ref = ref.rename(columns={ref_col: "method"})
    ref["source"] = "reference_reproduction"

    adv = results_df.copy()
    adv["source"] = "advanced_proposed"

    common_cols = ["source", "method", "accuracy", "balanced_accuracy", "macro_f1", "weighted_f1", "kappa"]
    combined = pd.concat([ref[common_cols], adv[common_cols]], ignore_index=True, sort=False)
    combined.to_csv(OUT_DIR / "comparison_with_reference_reproduction.csv", index=False, encoding="utf-8-sig")
    write_markdown(combined, OUT_DIR / "comparison_with_reference_reproduction.md")

    best_ref = ref.sort_values("macro_f1", ascending=False).iloc[0]
    best_adv = adv.sort_values("macro_f1", ascending=False).iloc[0]
    delta = pd.DataFrame([{
        "best_reference_method": best_ref["method"],
        "best_reference_macro_f1": best_ref["macro_f1"],
        "best_advanced_method": best_adv["method"],
        "best_advanced_macro_f1": best_adv["macro_f1"],
        "delta_macro_f1": best_adv["macro_f1"] - best_ref["macro_f1"],
        "best_reference_accuracy": best_ref["accuracy"],
        "best_advanced_accuracy": best_adv["accuracy"],
        "delta_accuracy": best_adv["accuracy"] - best_ref["accuracy"],
    }])
    delta.to_csv(OUT_DIR / "delta_vs_best_reference.csv", index=False, encoding="utf-8-sig")
    write_markdown(delta, OUT_DIR / "delta_vs_best_reference.md")


# ============================================================
# MAIN
# ============================================================

def main() -> None:
    print("=" * 80)
    print("TSMS-Drone advanced multisensor fusion improvements")
    print("=" * 80)
    print(f"ROOT       : {ROOT}")
    print(f"DATASET    : {DATASET_DIR}")
    print(f"OUT_DIR    : {OUT_DIR}")
    print(f"DEVICE     : {DEVICE}")
    print(f"BACKBONE   : {BACKBONE}")
    print("=" * 80)

    sync_df = build_synchronized_tuple_index(force_rebuild=False)
    split_df = make_splits(sync_df)

    config = {
        "root": str(ROOT),
        "dataset_dir": str(DATASET_DIR),
        "out_dir": str(OUT_DIR),
        "sensors": SENSORS,
        "targets": TARGETS,
        "distances": DISTANCES,
        "expected_repetitions": EXPECTED_REPETITIONS,
        "split": "70/15/15 stratified by target and distance",
        "random_state": RANDOM_STATE,
        "test_size": TEST_SIZE,
        "val_size_from_remaining": VAL_SIZE_FROM_REMAINING,
        "max_tuples_per_class": MAX_TUPLES_PER_CLASS,
        "backbone": BACKBONE,
        "use_imagenet_weights": USE_IMAGENET_WEIGHTS,
        "image_size": IMAGE_SIZE,
        "batch_size": BATCH_SIZE,
        "num_epochs": NUM_EPOCHS,
        "patience": PATIENCE,
        "lr": LR,
        "min_lr": MIN_LR,
        "weight_decay": WEIGHT_DECAY,
        "label_smoothing": LABEL_SMOOTHING,
        "mixup_alpha": MIXUP_ALPHA,
        "sensor_dropout_p": SENSOR_DROPOUT_P,
        "distance_aux_weight": DISTANCE_AUX_WEIGHT,
        "fusion_dim": FUSION_DIM,
        "attn_heads": ATTN_HEADS,
        "attn_layers": ATTN_LAYERS,
        "eval_masks": EVAL_MASKS,
    }
    with open(OUT_DIR / "advanced_run_config.json", "w", encoding="utf-8") as f:
        json.dump(config, f, indent=2)

    model, train_summary = train_advanced_model(split_df)
    results_df = evaluate_advanced_model(model, split_df, train_summary)

    print("\n" + "=" * 80)
    print("FINAL ADVANCED RESULTS")
    print("=" * 80)
    print(results_df.to_string(index=False))
    print("\nGenerated files:")
    for p in sorted(OUT_DIR.glob("*")):
        print(p)
    print("=" * 80)
    print("[OK] Advanced fusion experiment completed.")
    print("=" * 80)


if __name__ == "__main__":
    main()


Using device: cpu
TSMS-Drone advanced multisensor fusion improvements
ROOT       : D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)
DATASET    : D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\Dataset
OUT_DIR    : D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_advanced_fusion_improvements
DEVICE     : cpu
BACKBONE   : efficientnet_b0
Loading cached synchronized index: D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_advanced_fusion_improvements\synchronized_tuple_index_full.csv
Loading cached split file: D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_advanced_fusion_improvements\tuple_splits_70_15_15_full.csv
No checkpoint found at: D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_advanced_fusion_improvements\advanced_attention_model\checkpoint_last.pt
Starting training from scratch.

Training advanced tri-modal attention fusion model
Starting epoch : 1
Final epoch    : 40


Epoch 01/40: 100%|██████████| 821/821 [2:58:50<00:00, 13.07s/it, loss=0.733, lr=0.0002]  


Epoch 01: loss=0.7329, train_f1=0.5862, val_acc=0.9995, val_f1=0.9995
[CHECKPOINT] New best model saved at epoch 1.
[CHECKPOINT] Last checkpoint saved at epoch 1.


Epoch 02/40: 100%|██████████| 821/821 [2:56:07<00:00, 12.87s/it, loss=0.591, lr=0.0002]  


Epoch 02: loss=0.5907, train_f1=0.6150, val_acc=0.9998, val_f1=0.9998
[CHECKPOINT] New best model saved at epoch 2.
[CHECKPOINT] Last checkpoint saved at epoch 2.


...

Epoch 10/40: 100%|██████████| 821/821 [2:54:15<00:00, 12.74s/it, loss=0.513, lr=0.000176]  


Epoch 10: loss=0.5133, train_f1=0.6079, val_acc=1.0000, val_f1=1.0000
[CHECKPOINT] Last checkpoint saved at epoch 10.


Epoch 11/40: 100%|██████████| 821/821 [2:53:23<00:00, 12.67s/it, loss=0.514, lr=0.000171]  


Epoch 11: loss=0.5141, train_f1=0.5781, val_acc=1.0000, val_f1=1.0000
[CHECKPOINT] Last checkpoint saved at epoch 11.
Early stopping at epoch 11.


Reference results not found: D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_classification_fusion_reference\classification_results_summary.csv

FINAL ADVANCED RESULTS
                   method  mask_CW  mask_FMCW  mask_RF  best_epoch  training_time_sec  accuracy  balanced_accuracy  macro_precision  macro_recall  macro_f1  weighted_f1    kappa
      Advanced-Fusion-All        1          1        1           3      124945.414685  1.000000           1.000000         1.000000      1.000000  1.000000     1.000000 1.000000
    Advanced-Fusion-CW-RF        1          0        1           3      124945.414685  0.999822           0.999822         0.999822      0.999822  0.999822     0.999822 0.999778
  Advanced-Fusion-FMCW-RF        0          1        1           3      124945.414685  0.999822           0.999822         0.999822      0.999822  0.999822     0.999822 0.999778
  Advanced-Fusion-RF-only        0          0        1           3      124945.414685  0.999289           

In [25]:
# ============================================================
# tsms_drone_ieee_leakage_robustness_audit.py
# ============================================================
# Complementary audit for TSMS-Drone advanced multisensor fusion.
#
# Purpose:
#   This script complements the main training script by producing evidence
#   expected in a rigorous IEEE-style data-fusion paper:
#
#   1. Data leakage audits
#      - split overlap checks;
#      - tuple/file/path/hash duplicate checks;
#      - cached-index consistency checks;
#      - target-distance-sample integrity checks;
#      - repetition-proximity leakage risk analysis.
#
#   2. Robust evaluation protocols
#      - leave-one-distance-out split files;
#      - blocked repetition split files;
#      - near/mid/far range split files;
#      - open-set/leave-target-out protocol files.
#
#   3. Post-hoc prediction analyses
#      - bootstrap confidence intervals;
#      - paired McNemar tests;
#      - calibration metrics and reliability diagrams;
#      - risk-coverage curves;
#      - target-wise and distance-wise error summaries;
#      - gate/reliability-weight analysis.
#
#   4. Optional model stress tests
#      - missing-sensor stress;
#      - Gaussian-noise stress;
#      - intensity-shift stress.
#
# Recommended use after running:
#   tsms_drone_advanced_fusion_improvements.py
#
# Example:
#   python tsms_drone_ieee_leakage_robustness_audit.py ^
#     --root "D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)" ^
#     --hash-images ^
#     --bootstrap 1000 ^
#     --make-protocols
#
# Optional model stress test:
#   python tsms_drone_ieee_leakage_robustness_audit.py ^
#     --root "D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)" ^
#     --run-model-stress ^
#     --original-script "D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\code\tsms_drone_advanced_fusion_improvements.py" ^
#     --max-stress-batches 0
#
# Dependencies:
#   pip install numpy pandas pillow tqdm scikit-learn matplotlib scipy torch torchvision
#
# Notes:
#   - The script does not modify the original training outputs.
#   - All audit artifacts are saved under:
#       ROOT\_ieee_leakage_and_robustness_audit
#   - It is safe to run the non-model audits on CPU.
# ============================================================

from __future__ import annotations

import argparse
import hashlib
import importlib.util
import itertools
import json
import math
import os
import random
import re
import shutil
import sys
import time
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    cohen_kappa_score,
    confusion_matrix,
    log_loss,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer

try:
    from scipy.stats import binomtest
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

try:
    import torch
    TORCH_AVAILABLE = True
except Exception:
    TORCH_AVAILABLE = False

warnings.filterwarnings("ignore", category=UserWarning)


# ============================================================
# CONSTANTS
# ============================================================

SENSORS = ["CW Radar", "FMCW Radar", "RF Receiver"]
TARGETS = [
    "Inspire 2",
    "Matrice 30",
    "Mavic 2 Pro",
    "Phantom 4 Pro",
    "Corner Reflector",
]
DISTANCES = [f"{d}m" for d in range(2, 31, 2)]
DISTANCE_VALUES = [int(d.replace("m", "")) for d in DISTANCES]
TARGET_TO_ID = {t: i for i, t in enumerate(TARGETS)}
ID_TO_TARGET = {i: t for t, i in TARGET_TO_ID.items()}


# ============================================================
# SMALL UTILITIES
# ============================================================

def normalize_token(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(text).lower())


def safe_name(text: str) -> str:
    text = str(text).lower().strip()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return text.strip("_")


def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def read_csv_required(path: Path, name: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Required {name} not found: {path}")
    return pd.read_csv(path)


def write_markdown(df: pd.DataFrame, path: Path) -> None:
    try:
        path.write_text(df.to_markdown(index=False), encoding="utf-8")
    except Exception:
        path.write_text(df.to_csv(index=False), encoding="utf-8")


def write_latex(df: pd.DataFrame, path: Path, caption: str, label: str) -> None:
    latex = df.to_latex(
        index=False,
        escape=False,
        caption=caption,
        label=label,
        float_format=lambda x: f"{x:.4f}",
    )
    path.write_text(latex, encoding="utf-8")


def save_json(obj: Dict, path: Path) -> None:
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")


def md5_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.md5()
    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def average_hash(path: Path, hash_size: int = 8) -> Optional[str]:
    try:
        img = Image.open(path).convert("L").resize((hash_size, hash_size), Image.BILINEAR)
        arr = np.asarray(img, dtype=np.float32)
        bits = arr >= arr.mean()
        value = 0
        for bit in bits.flatten():
            value = (value << 1) | int(bit)
        return f"{value:0{hash_size * hash_size // 4}x}"
    except Exception:
        return None


def hamming_hex(a: str, b: str) -> int:
    return (int(a, 16) ^ int(b, 16)).bit_count()


def safe_trapezoid(y: np.ndarray, x: np.ndarray) -> float:
    """Version-safe trapezoidal integration.

    NumPy 2.x recommends np.trapezoid, while some environments no longer
    expose np.trapz. This helper keeps the audit script compatible across
    NumPy versions and falls back to a manual implementation if needed.
    """
    y = np.asarray(y, dtype=float)
    x = np.asarray(x, dtype=float)
    if len(y) < 2 or len(x) < 2:
        return 0.0
    if hasattr(np, "trapezoid"):
        return float(np.trapezoid(y, x))
    if hasattr(np, "trapz"):
        return float(np.trapz(y, x))
    dx = np.diff(x)
    avg_y = 0.5 * (y[1:] + y[:-1])
    return float(np.sum(dx * avg_y))


def infer_sensor_from_path(path: str) -> Optional[str]:
    parts = [normalize_token(p) for p in Path(str(path)).parts]
    for sensor in SENSORS:
        if normalize_token(sensor) in parts:
            return sensor
    joined = normalize_token(path)
    for sensor in sorted(SENSORS, key=lambda s: len(normalize_token(s)), reverse=True):
        if normalize_token(sensor) in joined:
            return sensor
    return None


def compute_basic_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "kappa": cohen_kappa_score(y_true, y_pred),
    }


def get_proba_columns(df: pd.DataFrame) -> List[str]:
    return [c for c in df.columns if c.startswith("proba_")]


def target_name_from_proba_col(col: str) -> str:
    return col.replace("proba_", "").replace("_", " ")


def split_order() -> List[str]:
    return ["train", "val", "test"]


# ============================================================
# CONFIGURATION DATACLASS
# ============================================================

@dataclass
class AuditConfig:
    root: Path
    adv_dir: Path
    audit_dir: Path
    sync_csv: Path
    split_csv: Path
    predictions_root: Path
    bootstrap: int = 1000
    seed: int = 42
    hash_images: bool = False
    max_hash_files: int = 0
    make_protocols: bool = False
    run_model_stress: bool = False
    original_script: Optional[Path] = None
    checkpoint: Optional[Path] = None
    max_stress_batches: int = 0


# ============================================================
# LOADING AND NORMALIZATION
# ============================================================

def load_split_dataframe(cfg: AuditConfig) -> pd.DataFrame:
    sync_df = read_csv_required(cfg.sync_csv, "synchronized tuple index")
    split_df = read_csv_required(cfg.split_csv, "split file")

    if "split" not in split_df.columns:
        raise ValueError(f"Split file does not contain column 'split': {cfg.split_csv}")

    # Ensure required columns exist.
    required = ["target", "distance", "distance_m", "sample_index", "tuple_id", "split"] + SENSORS
    missing = [c for c in required if c not in split_df.columns]
    if missing:
        raise ValueError(f"Split file is missing required columns: {missing}")

    # Reconstruct label if absent.
    if "label" not in split_df.columns:
        split_df["label"] = split_df["target"].map(TARGET_TO_ID).astype(int)

    split_df["distance_m"] = split_df["distance_m"].astype(int)
    split_df["sample_index"] = split_df["sample_index"].astype(int)
    split_df["tuple_id"] = split_df["tuple_id"].astype(str)
    split_df["split"] = split_df["split"].astype(str)

    return split_df


# ============================================================
# 1. LEAKAGE AUDITS
# ============================================================

def audit_split_integrity(df: pd.DataFrame, out_dir: Path) -> pd.DataFrame:
    rows = []

    n_rows = len(df)
    rows.append({"test": "n_total_tuples", "status": "info", "value": n_rows, "details": "Total synchronized tuples."})

    # Split values.
    valid_splits = set(split_order())
    observed_splits = set(df["split"].unique())
    extra_splits = sorted(observed_splits - valid_splits)
    missing_splits = sorted(valid_splits - observed_splits)
    status = "pass" if not extra_splits and not missing_splits else "fail"
    rows.append({
        "test": "split_labels_valid",
        "status": status,
        "value": len(observed_splits),
        "details": f"observed={sorted(observed_splits)}, extra={extra_splits}, missing={missing_splits}",
    })

    # Tuple ID uniqueness.
    dup_tuple = df[df.duplicated("tuple_id", keep=False)].copy()
    rows.append({
        "test": "tuple_id_unique",
        "status": "pass" if len(dup_tuple) == 0 else "fail",
        "value": len(dup_tuple),
        "details": "Duplicated tuple_id rows across the split table.",
    })
    if len(dup_tuple) > 0:
        dup_tuple.to_csv(out_dir / "fail_duplicate_tuple_ids.csv", index=False, encoding="utf-8-sig")

    # Unique target-distance-sample tuple.
    key_cols = ["target", "distance_m", "sample_index"]
    dup_key = df[df.duplicated(key_cols, keep=False)].copy()
    rows.append({
        "test": "target_distance_sample_unique",
        "status": "pass" if len(dup_key) == 0 else "fail",
        "value": len(dup_key),
        "details": "Duplicate target-distance-sample keys indicate repeated synchronized tuples.",
    })
    if len(dup_key) > 0:
        dup_key.to_csv(out_dir / "fail_duplicate_target_distance_sample.csv", index=False, encoding="utf-8-sig")

    # Path overlap across splits for each sensor.
    for sensor in SENSORS:
        tmp = df[["split", "tuple_id", sensor]].copy().rename(columns={sensor: "path"})
        by_path = tmp.groupby("path")["split"].nunique().reset_index(name="n_splits")
        overlap_paths = by_path[by_path["n_splits"] > 1]
        rows.append({
            "test": f"path_overlap_across_splits_{safe_name(sensor)}",
            "status": "pass" if len(overlap_paths) == 0 else "fail",
            "value": len(overlap_paths),
            "details": f"Exact same {sensor} image path appears in more than one split.",
        })
        if len(overlap_paths) > 0:
            tmp[tmp["path"].isin(overlap_paths["path"])].to_csv(
                out_dir / f"fail_path_overlap_{safe_name(sensor)}.csv", index=False, encoding="utf-8-sig"
            )

    # Split balance.
    split_counts = df.groupby("split").size().reindex(split_order(), fill_value=0).reset_index(name="n")
    split_counts["fraction"] = split_counts["n"] / len(df)
    split_counts.to_csv(out_dir / "split_counts.csv", index=False, encoding="utf-8-sig")

    target_split = df.groupby(["split", "target"]).size().reset_index(name="n")
    target_split.to_csv(out_dir / "split_by_target_counts.csv", index=False, encoding="utf-8-sig")

    target_distance_split = df.groupby(["split", "target", "distance_m"]).size().reset_index(name="n")
    target_distance_split.to_csv(out_dir / "split_by_target_distance_counts.csv", index=False, encoding="utf-8-sig")

    # Sensor path parsing consistency.
    parse_rows = []
    for sensor in SENSORS:
        inferred = df[sensor].astype(str).map(infer_sensor_from_path)
        mismatch = df[inferred != sensor].copy()
        parse_rows.append({
            "test": f"sensor_path_parser_consistency_{safe_name(sensor)}",
            "status": "pass" if len(mismatch) == 0 else "fail",
            "value": len(mismatch),
            "details": f"Rows whose path was not parsed as {sensor}.",
        })
        if len(mismatch) > 0:
            mismatch[["tuple_id", "split", "target", "distance_m", "sample_index", sensor]].to_csv(
                out_dir / f"fail_sensor_parser_{safe_name(sensor)}.csv", index=False, encoding="utf-8-sig"
            )
    rows.extend(parse_rows)

    audit = pd.DataFrame(rows)
    audit.to_csv(out_dir / "audit_split_integrity.csv", index=False, encoding="utf-8-sig")
    write_markdown(audit, out_dir / "audit_split_integrity.md")
    return audit


def audit_repetition_proximity(df: pd.DataFrame, out_dir: Path) -> pd.DataFrame:
    """Quantify weak leakage risk caused by random interleaving of adjacent repetitions.

    For each validation/test tuple, compute the closest sample_index in the training
    set with the same target and distance. A median nearest gap of 1 means the
    evaluation set is surrounded by adjacent repetitions from the training set.
    This is not an exact duplicate leak, but it is a protocol weakness if repeated
    acquisitions are highly correlated.
    """
    train = df[df["split"] == "train"].copy()
    eval_df = df[df["split"].isin(["val", "test"])].copy()

    rows = []
    train_groups = {
        (target, int(distance)): sorted(g["sample_index"].astype(int).tolist())
        for (target, distance), g in train.groupby(["target", "distance_m"])
    }

    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Repetition-proximity audit"):
        key = (row["target"], int(row["distance_m"]))
        train_indices = train_groups.get(key, [])
        if not train_indices:
            nearest_gap = np.nan
            nearest_train_index = np.nan
        else:
            x = int(row["sample_index"])
            diffs = [abs(x - t) for t in train_indices]
            j = int(np.argmin(diffs))
            nearest_gap = int(diffs[j])
            nearest_train_index = int(train_indices[j])
        rows.append({
            "tuple_id": row["tuple_id"],
            "split": row["split"],
            "target": row["target"],
            "distance_m": int(row["distance_m"]),
            "sample_index": int(row["sample_index"]),
            "nearest_train_sample_index_same_target_distance": nearest_train_index,
            "nearest_train_gap_same_target_distance": nearest_gap,
        })

    prox = pd.DataFrame(rows)
    prox.to_csv(out_dir / "repetition_proximity_eval_to_train.csv", index=False, encoding="utf-8-sig")

    summary = prox.groupby("split")["nearest_train_gap_same_target_distance"].agg(
        n="count", mean="mean", median="median", min="min", q05=lambda x: np.nanquantile(x, 0.05),
        q25=lambda x: np.nanquantile(x, 0.25), q75=lambda x: np.nanquantile(x, 0.75), max="max"
    ).reset_index()
    summary.to_csv(out_dir / "repetition_proximity_summary.csv", index=False, encoding="utf-8-sig")
    write_markdown(summary, out_dir / "repetition_proximity_summary.md")

    # Histogram.
    fig, ax = plt.subplots(figsize=(8, 5))
    for split in ["val", "test"]:
        vals = prox.loc[prox["split"] == split, "nearest_train_gap_same_target_distance"].dropna().astype(int)
        if len(vals) > 0:
            ax.hist(vals, bins=30, alpha=0.5, label=split)
    ax.set_xlabel("Nearest training repetition gap within same target-distance group")
    ax.set_ylabel("Number of evaluation tuples")
    ax.set_title("Weak leakage risk: proximity of evaluation repetitions to training repetitions")
    ax.legend()
    fig.tight_layout()
    fig.savefig(out_dir / "repetition_proximity_histogram.png", dpi=300)
    plt.close(fig)

    return summary


def audit_hash_duplicates(df: pd.DataFrame, out_dir: Path, cfg: AuditConfig) -> pd.DataFrame:
    rows = []
    file_rows = []

    # Long format: one row per sensor image.
    for sensor in SENSORS:
        tmp = df[["tuple_id", "split", "target", "distance_m", "sample_index", sensor]].copy()
        tmp = tmp.rename(columns={sensor: "path"})
        tmp["sensor"] = sensor
        file_rows.append(tmp)
    files = pd.concat(file_rows, ignore_index=True)
    files["path"] = files["path"].astype(str)
    files = files.drop_duplicates("path").reset_index(drop=True)

    if cfg.max_hash_files and cfg.max_hash_files > 0 and cfg.max_hash_files < len(files):
        files = files.sample(cfg.max_hash_files, random_state=cfg.seed).reset_index(drop=True)

    hash_records = []
    for _, row in tqdm(files.iterrows(), total=len(files), desc="Hashing images"):
        p = Path(row["path"])
        rec = row.to_dict()
        rec["exists"] = p.exists()
        rec["size_bytes"] = p.stat().st_size if p.exists() else np.nan
        if p.exists() and cfg.hash_images:
            rec["md5"] = md5_file(p)
            rec["average_hash"] = average_hash(p)
        else:
            rec["md5"] = None
            rec["average_hash"] = None
        hash_records.append(rec)

    hashes = pd.DataFrame(hash_records)
    hashes.to_csv(out_dir / "image_hash_index.csv", index=False, encoding="utf-8-sig")

    missing_files = hashes[~hashes["exists"]].copy()
    rows.append({
        "test": "image_paths_exist",
        "status": "pass" if len(missing_files) == 0 else "fail",
        "value": len(missing_files),
        "details": "Missing image files referenced by the split table.",
    })
    if len(missing_files) > 0:
        missing_files.to_csv(out_dir / "fail_missing_image_files.csv", index=False, encoding="utf-8-sig")

    # Exact MD5 duplicates across splits.
    if cfg.hash_images:
        for sensor in SENSORS:
            g = hashes[(hashes["sensor"] == sensor) & hashes["md5"].notna()].copy()
            by_hash = g.groupby("md5")["split"].nunique().reset_index(name="n_splits")
            cross = by_hash[by_hash["n_splits"] > 1]
            rows.append({
                "test": f"md5_duplicate_across_splits_{safe_name(sensor)}",
                "status": "pass" if len(cross) == 0 else "fail",
                "value": len(cross),
                "details": "Exact byte-identical images found in more than one split.",
            })
            if len(cross) > 0:
                g[g["md5"].isin(cross["md5"])].to_csv(
                    out_dir / f"fail_md5_duplicates_across_splits_{safe_name(sensor)}.csv",
                    index=False,
                    encoding="utf-8-sig",
                )

            by_ahash = g[g["average_hash"].notna()].groupby("average_hash")["split"].nunique().reset_index(name="n_splits")
            cross_a = by_ahash[by_ahash["n_splits"] > 1]
            rows.append({
                "test": f"average_hash_collision_across_splits_{safe_name(sensor)}",
                "status": "pass" if len(cross_a) == 0 else "warning",
                "value": len(cross_a),
                "details": "Perceptual average-hash collisions across splits. Inspect manually; not always leakage.",
            })
            if len(cross_a) > 0:
                g[g["average_hash"].isin(cross_a["average_hash"])].to_csv(
                    out_dir / f"warning_average_hash_collisions_{safe_name(sensor)}.csv",
                    index=False,
                    encoding="utf-8-sig",
                )
    else:
        rows.append({
            "test": "image_hashing_skipped",
            "status": "warning",
            "value": len(files),
            "details": "Run with --hash-images to compute MD5 and perceptual hashes.",
        })

    audit = pd.DataFrame(rows)
    audit.to_csv(out_dir / "audit_hash_duplicates.csv", index=False, encoding="utf-8-sig")
    write_markdown(audit, out_dir / "audit_hash_duplicates.md")
    return audit


def metadata_only_shortcut_test(df: pd.DataFrame, out_dir: Path, seed: int = 42) -> pd.DataFrame:
    """Train a weak classifier using only metadata that the neural model should not see.

    If distance/sample_index alone predicts the target with high accuracy, the
    experimental protocol may have a shortcut structure. The model itself does
    not use these fields directly, but this is a sanity check for acquisition bias.
    """
    train = df[df["split"] == "train"].copy()
    test = df[df["split"] == "test"].copy()

    feature_cols = ["distance_m", "sample_index"]
    X_train = train[feature_cols]
    X_test = test[feature_cols]
    y_train = train["label"].astype(int).to_numpy()
    y_test = test["label"].astype(int).to_numpy()

    # Compatibility note:
    # Some scikit-learn versions removed or changed the `multi_class` argument.
    # Therefore, we do not pass it explicitly. LogisticRegression will use the
    # version-appropriate default multiclass behavior.
    try:
        clf = LogisticRegression(max_iter=1000, solver="lbfgs", random_state=seed)
        clf.fit(X_train, y_train)
        pred = clf.predict(X_test)
        model_status = "ok_logistic_regression"
    except TypeError as e:
        print(f"[WARN] LogisticRegression argument compatibility issue: {e}")
        print("[WARN] Retrying with minimal LogisticRegression constructor.")
        clf = LogisticRegression(max_iter=1000)
        clf.fit(X_train, y_train)
        pred = clf.predict(X_test)
        model_status = "ok_logistic_regression_minimal"
    except Exception as e:
        print(f"[WARN] Metadata-only shortcut probe failed: {e}")
        print("[WARN] Falling back to majority-class baseline for this probe.")
        values, counts = np.unique(y_train, return_counts=True)
        majority = values[int(np.argmax(counts))]
        pred = np.full_like(y_test, fill_value=majority)
        model_status = "fallback_majority_class"

    metrics = compute_basic_metrics(y_test, pred)
    result = pd.DataFrame([{ "probe": "metadata_only_distance_sample_index", "model_status": model_status, **metrics }])
    result.to_csv(out_dir / "metadata_only_shortcut_test.csv", index=False, encoding="utf-8-sig")
    write_markdown(result, out_dir / "metadata_only_shortcut_test.md")
    return result


# ============================================================
# 2. SPLIT PROTOCOL GENERATION FOR ROBUSTNESS
# ============================================================

def save_protocol_split(df: pd.DataFrame, protocol_dir: Path, name: str, protocol_df: pd.DataFrame) -> None:
    out_csv = protocol_dir / f"{safe_name(name)}.csv"
    protocol_df.to_csv(out_csv, index=False, encoding="utf-8-sig")

    summary = protocol_df.groupby(["split", "target", "distance_m"]).size().reset_index(name="n")
    summary.to_csv(protocol_dir / f"{safe_name(name)}_summary_by_target_distance.csv", index=False, encoding="utf-8-sig")


def make_leave_one_distance_protocols(df: pd.DataFrame, protocol_dir: Path, seed: int = 42) -> pd.DataFrame:
    rows = []
    for held_distance in sorted(df["distance_m"].unique()):
        p = df.copy()
        p["split"] = "train"
        p.loc[p["distance_m"] == held_distance, "split"] = "test"

        # Validation from remaining training data, stratified by target-distance.
        train_idx = p.index[p["split"] == "train"].to_numpy()
        train_part = p.loc[train_idx]
        strat = train_part["target"].astype(str) + "_" + train_part["distance_m"].astype(str)
        _, val_local = train_test_split(
            np.arange(len(train_part)), test_size=0.15, random_state=seed, stratify=strat
        )
        val_idx = train_part.index[val_local]
        p.loc[val_idx, "split"] = "val"

        name = f"leave_one_distance_out_{held_distance}m"
        save_protocol_split(df, protocol_dir, name, p)
        rows.append({
            "protocol": name,
            "held_out_distance_m": int(held_distance),
            "train_n": int((p["split"] == "train").sum()),
            "val_n": int((p["split"] == "val").sum()),
            "test_n": int((p["split"] == "test").sum()),
            "file": f"{safe_name(name)}.csv",
        })
    return pd.DataFrame(rows)


def make_repetition_block_protocols(df: pd.DataFrame, protocol_dir: Path) -> pd.DataFrame:
    """Create blocked repetition protocols to reduce adjacent-sample leakage risk."""
    protocols = {
        "blocked_repetition_last_15pct_test_prev_15pct_val": {
            "test_min": 426,
            "val_min": 351,
            "val_max": 425,
        },
        "blocked_repetition_first_15pct_test_next_15pct_val": {
            "test_max": 75,
            "val_min": 76,
            "val_max": 150,
        },
        "blocked_repetition_middle_15pct_test": {
            "test_min": 213,
            "test_max": 287,
            "val_min": 288,
            "val_max": 362,
        },
    }

    rows = []
    for name, spec in protocols.items():
        p = df.copy()
        p["split"] = "train"
        if "test_min" in spec and "test_max" in spec:
            test_mask = (p["sample_index"] >= spec["test_min"]) & (p["sample_index"] <= spec["test_max"])
        elif "test_min" in spec:
            test_mask = p["sample_index"] >= spec["test_min"]
        else:
            test_mask = p["sample_index"] <= spec["test_max"]
        val_mask = (p["sample_index"] >= spec["val_min"]) & (p["sample_index"] <= spec["val_max"])
        p.loc[val_mask, "split"] = "val"
        p.loc[test_mask, "split"] = "test"
        save_protocol_split(df, protocol_dir, name, p)
        rows.append({
            "protocol": name,
            "train_n": int((p["split"] == "train").sum()),
            "val_n": int((p["split"] == "val").sum()),
            "test_n": int((p["split"] == "test").sum()),
            "file": f"{safe_name(name)}.csv",
        })
    return pd.DataFrame(rows)


def make_range_block_protocols(df: pd.DataFrame, protocol_dir: Path, seed: int = 42) -> pd.DataFrame:
    blocks = {
        "range_holdout_near_2_10m": list(range(2, 12, 2)),
        "range_holdout_mid_12_20m": list(range(12, 22, 2)),
        "range_holdout_far_22_30m": list(range(22, 32, 2)),
    }
    rows = []
    for name, distances in blocks.items():
        p = df.copy()
        p["split"] = "train"
        p.loc[p["distance_m"].isin(distances), "split"] = "test"
        train_idx = p.index[p["split"] == "train"].to_numpy()
        train_part = p.loc[train_idx]
        strat = train_part["target"].astype(str) + "_" + train_part["distance_m"].astype(str)
        _, val_local = train_test_split(
            np.arange(len(train_part)), test_size=0.15, random_state=seed, stratify=strat
        )
        p.loc[train_part.index[val_local], "split"] = "val"
        save_protocol_split(df, protocol_dir, name, p)
        rows.append({
            "protocol": name,
            "held_out_distances_m": ",".join(map(str, distances)),
            "train_n": int((p["split"] == "train").sum()),
            "val_n": int((p["split"] == "val").sum()),
            "test_n": int((p["split"] == "test").sum()),
            "file": f"{safe_name(name)}.csv",
        })
    return pd.DataFrame(rows)


def make_leave_one_target_ood_protocols(df: pd.DataFrame, protocol_dir: Path, seed: int = 42) -> pd.DataFrame:
    """Open-set/OOD protocols: train without one target, test on the held-out target.

    This is not closed-set classification; it supports OOD/rejection analysis.
    """
    rows = []
    for held_target in TARGETS:
        p = df.copy()
        p["split"] = "train"
        p.loc[p["target"] == held_target, "split"] = "test_ood"
        train_idx = p.index[p["split"] == "train"].to_numpy()
        train_part = p.loc[train_idx]
        strat = train_part["target"].astype(str) + "_" + train_part["distance_m"].astype(str)
        _, val_local = train_test_split(
            np.arange(len(train_part)), test_size=0.15, random_state=seed, stratify=strat
        )
        p.loc[train_part.index[val_local], "split"] = "val"
        name = f"leave_one_target_out_ood_{safe_name(held_target)}"
        save_protocol_split(df, protocol_dir, name, p)
        rows.append({
            "protocol": name,
            "held_out_target": held_target,
            "train_n": int((p["split"] == "train").sum()),
            "val_n": int((p["split"] == "val").sum()),
            "test_ood_n": int((p["split"] == "test_ood").sum()),
            "file": f"{safe_name(name)}.csv",
        })
    return pd.DataFrame(rows)


def generate_robust_protocols(df: pd.DataFrame, out_dir: Path, seed: int = 42) -> pd.DataFrame:
    protocol_dir = ensure_dir(out_dir / "robust_split_protocols")
    tables = []
    tables.append(make_leave_one_distance_protocols(df, protocol_dir, seed))
    tables.append(make_repetition_block_protocols(df, protocol_dir))
    tables.append(make_range_block_protocols(df, protocol_dir, seed))
    tables.append(make_leave_one_target_ood_protocols(df, protocol_dir, seed))
    index = pd.concat(tables, ignore_index=True, sort=False)
    index.to_csv(protocol_dir / "protocol_index.csv", index=False, encoding="utf-8-sig")
    write_markdown(index, protocol_dir / "protocol_index.md")
    return index


# ============================================================
# 3. PREDICTION ANALYSES
# ============================================================

def find_prediction_files(predictions_root: Path) -> Dict[str, Path]:
    files = {}
    if not predictions_root.exists():
        return files
    for p in predictions_root.glob("*/predictions_test.csv"):
        method = p.parent.name
        files[method] = p
    return files


def load_predictions(predictions_root: Path) -> Dict[str, pd.DataFrame]:
    out = {}
    for method_dir_name, p in find_prediction_files(predictions_root).items():
        df = pd.read_csv(p)
        if "method" not in df.columns:
            # Convert folder name back to readable enough method name.
            df["method"] = method_dir_name
        out[method_dir_name] = df
    return out


def bootstrap_metric_ci(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    metric_fn,
    n_boot: int = 1000,
    seed: int = 42,
) -> Tuple[float, float, float]:
    rng = np.random.default_rng(seed)
    n = len(y_true)
    vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        vals.append(metric_fn(y_true[idx], y_pred[idx]))
    vals = np.asarray(vals, dtype=float)
    point = metric_fn(y_true, y_pred)
    return float(point), float(np.quantile(vals, 0.025)), float(np.quantile(vals, 0.975))


def prediction_bootstrap_cis(preds: Dict[str, pd.DataFrame], out_dir: Path, n_boot: int, seed: int) -> pd.DataFrame:
    rows = []
    for method, df in preds.items():
        y_true = df["y_true"].to_numpy(dtype=int)
        y_pred = df["y_pred"].to_numpy(dtype=int)
        metrics = {
            "accuracy": lambda a, b: accuracy_score(a, b),
            "balanced_accuracy": lambda a, b: balanced_accuracy_score(a, b),
            "macro_f1": lambda a, b: f1_score(a, b, average="macro", zero_division=0),
            "weighted_f1": lambda a, b: f1_score(a, b, average="weighted", zero_division=0),
            "kappa": lambda a, b: cohen_kappa_score(a, b),
        }
        for metric_name, fn in metrics.items():
            point, lo, hi = bootstrap_metric_ci(y_true, y_pred, fn, n_boot=n_boot, seed=seed)
            rows.append({
                "method": method,
                "metric": metric_name,
                "point": point,
                "ci95_low": lo,
                "ci95_high": hi,
                "n_bootstrap": n_boot,
            })
    res = pd.DataFrame(rows)
    res.to_csv(out_dir / "bootstrap_confidence_intervals.csv", index=False, encoding="utf-8-sig")
    write_markdown(res, out_dir / "bootstrap_confidence_intervals.md")
    write_latex(
        res[res["metric"].isin(["macro_f1", "balanced_accuracy", "kappa"])],
        out_dir / "latex_bootstrap_confidence_intervals.tex",
        caption="Bootstrap 95\% confidence intervals for the proposed fusion and ablation settings.",
        label="tab:bootstrap_ci",
    )
    return res


def exact_mcnemar(correct_a: np.ndarray, correct_b: np.ndarray) -> Dict[str, float]:
    # b = A correct, B wrong; c = A wrong, B correct.
    b = int(np.sum(correct_a & ~correct_b))
    c = int(np.sum(~correct_a & correct_b))
    n = b + c
    if n == 0:
        p = 1.0
    elif SCIPY_AVAILABLE:
        p = float(binomtest(min(b, c), n=n, p=0.5, alternative="two-sided").pvalue)
    else:
        # Normal approximation fallback.
        stat = (abs(b - c) - 1) ** 2 / max(n, 1)
        p = float(math.exp(-0.5 * stat))
    return {"b_a_correct_b_wrong": b, "c_a_wrong_b_correct": c, "discordant": n, "p_value": p}


def paired_mcnemar_tests(preds: Dict[str, pd.DataFrame], out_dir: Path) -> pd.DataFrame:
    rows = []
    methods = sorted(preds.keys())
    for a, b in itertools.combinations(methods, 2):
        da = preds[a][["tuple_id", "correct"]].rename(columns={"correct": "correct_a"})
        db = preds[b][["tuple_id", "correct"]].rename(columns={"correct": "correct_b"})
        m = da.merge(db, on="tuple_id", how="inner")
        if len(m) == 0:
            continue
        stats = exact_mcnemar(m["correct_a"].to_numpy(bool), m["correct_b"].to_numpy(bool))
        rows.append({"method_a": a, "method_b": b, "n_paired": len(m), **stats})
    res = pd.DataFrame(rows)
    res.to_csv(out_dir / "paired_mcnemar_tests.csv", index=False, encoding="utf-8-sig")
    write_markdown(res, out_dir / "paired_mcnemar_tests.md")
    return res


def expected_calibration_error(conf: np.ndarray, correct: np.ndarray, n_bins: int = 15) -> Tuple[float, pd.DataFrame]:
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    rows = []
    ece = 0.0
    n = len(conf)
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        if i == n_bins - 1:
            mask = (conf >= lo) & (conf <= hi)
        else:
            mask = (conf >= lo) & (conf < hi)
        if mask.sum() == 0:
            acc = np.nan
            avg_conf = np.nan
            count = 0
        else:
            acc = float(correct[mask].mean())
            avg_conf = float(conf[mask].mean())
            count = int(mask.sum())
            ece += (count / n) * abs(acc - avg_conf)
        rows.append({
            "bin": i,
            "bin_low": lo,
            "bin_high": hi,
            "count": count,
            "avg_confidence": avg_conf,
            "accuracy": acc,
            "abs_gap": abs(acc - avg_conf) if count > 0 else np.nan,
        })
    return float(ece), pd.DataFrame(rows)


def calibration_and_risk_analysis(preds: Dict[str, pd.DataFrame], out_dir: Path, n_bins: int = 15) -> pd.DataFrame:
    rows = []
    rel_dir = ensure_dir(out_dir / "calibration_reliability")
    risk_dir = ensure_dir(out_dir / "risk_coverage")

    for method, df in preds.items():
        proba_cols = get_proba_columns(df)
        if not proba_cols:
            continue
        y_true = df["y_true"].to_numpy(dtype=int)
        y_pred = df["y_pred"].to_numpy(dtype=int)
        proba = df[proba_cols].to_numpy(dtype=float)
        conf = np.max(proba, axis=1)
        correct = (y_true == y_pred)

        ece, bin_df = expected_calibration_error(conf, correct, n_bins=n_bins)
        bin_df["method"] = method
        bin_df.to_csv(rel_dir / f"reliability_bins_{safe_name(method)}.csv", index=False, encoding="utf-8-sig")

        # Reliability plot.
        fig, ax = plt.subplots(figsize=(5.5, 5.5))
        plot_df = bin_df.dropna(subset=["avg_confidence", "accuracy"])
        ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1)
        ax.plot(plot_df["avg_confidence"], plot_df["accuracy"], marker="o")
        ax.set_xlabel("Confidence")
        ax.set_ylabel("Accuracy")
        ax.set_title(f"Reliability diagram - {method}")
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        fig.tight_layout()
        fig.savefig(rel_dir / f"reliability_diagram_{safe_name(method)}.png", dpi=300)
        plt.close(fig)

        # NLL and Brier.
        try:
            nll = float(log_loss(y_true, proba, labels=list(range(proba.shape[1]))))
        except Exception:
            nll = np.nan
        y_onehot = np.eye(proba.shape[1])[y_true]
        brier = float(np.mean(np.sum((proba - y_onehot) ** 2, axis=1)))

        # Risk-coverage curve: sort by confidence descending.
        order = np.argsort(-conf)
        sorted_correct = correct[order]
        coverages = np.arange(1, len(correct) + 1) / len(correct)
        risks = 1.0 - np.cumsum(sorted_correct) / np.arange(1, len(correct) + 1)
        rc = pd.DataFrame({"coverage": coverages, "risk": risks})
        rc.to_csv(risk_dir / f"risk_coverage_{safe_name(method)}.csv", index=False, encoding="utf-8-sig")
        aurc = safe_trapezoid(risks, coverages)

        fig, ax = plt.subplots(figsize=(6, 5))
        ax.plot(rc["coverage"], rc["risk"])
        ax.set_xlabel("Coverage")
        ax.set_ylabel("Selective risk")
        ax.set_title(f"Risk-coverage curve - {method}")
        ax.set_xlim(0, 1)
        ax.set_ylim(0, max(0.01, float(np.nanmax(risks)) * 1.05))
        fig.tight_layout()
        fig.savefig(risk_dir / f"risk_coverage_{safe_name(method)}.png", dpi=300)
        plt.close(fig)

        rows.append({
            "method": method,
            "ece": ece,
            "nll": nll,
            "brier_multiclass": brier,
            "aurc": aurc,
            "mean_confidence": float(np.mean(conf)),
            "median_confidence": float(np.median(conf)),
            "min_confidence": float(np.min(conf)),
            "max_confidence": float(np.max(conf)),
        })

    res = pd.DataFrame(rows)
    res.to_csv(out_dir / "calibration_and_risk_summary.csv", index=False, encoding="utf-8-sig")
    write_markdown(res, out_dir / "calibration_and_risk_summary.md")
    return res


def target_distance_error_analysis(preds: Dict[str, pd.DataFrame], out_dir: Path) -> pd.DataFrame:
    rows = []
    for method, df in preds.items():
        for (target, distance), g in df.groupby(["true_label", "distance_m"]):
            y_true = g["y_true"].to_numpy(dtype=int)
            y_pred = g["y_pred"].to_numpy(dtype=int)
            rows.append({
                "method": method,
                "target": target,
                "distance_m": int(distance),
                "n": len(g),
                **compute_basic_metrics(y_true, y_pred),
            })
    res = pd.DataFrame(rows)
    res.to_csv(out_dir / "target_distance_metrics_from_predictions.csv", index=False, encoding="utf-8-sig")
    write_markdown(res, out_dir / "target_distance_metrics_from_predictions.md")

    # Heatmaps with imshow per method.
    heat_dir = ensure_dir(out_dir / "target_distance_heatmaps")
    for method, g in res.groupby("method"):
        pivot = g.pivot(index="target", columns="distance_m", values="macro_f1").reindex(TARGETS)
        fig, ax = plt.subplots(figsize=(10, 4.5))
        im = ax.imshow(pivot.to_numpy(dtype=float), aspect="auto", vmin=0, vmax=1)
        ax.set_title(f"Target-distance Macro-F1 - {method}")
        ax.set_xlabel("Distance (m)")
        ax.set_ylabel("Target")
        ax.set_xticks(range(len(pivot.columns)))
        ax.set_xticklabels(pivot.columns)
        ax.set_yticks(range(len(pivot.index)))
        ax.set_yticklabels(pivot.index)
        for i in range(pivot.shape[0]):
            for j in range(pivot.shape[1]):
                val = pivot.iloc[i, j]
                if not pd.isna(val):
                    ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=7)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        fig.tight_layout()
        fig.savefig(heat_dir / f"target_distance_macro_f1_{safe_name(method)}.png", dpi=300)
        plt.close(fig)

    return res


def gate_analysis(preds: Dict[str, pd.DataFrame], out_dir: Path) -> pd.DataFrame:
    rows = []
    gate_cols = [f"gate_{safe_name(s)}" for s in SENSORS]
    for method, df in preds.items():
        if not all(c in df.columns for c in gate_cols):
            continue
        gates = df[gate_cols].to_numpy(dtype=float)
        eps = 1e-12
        entropy = -np.sum(gates * np.log(gates + eps), axis=1) / np.log(gates.shape[1])
        tmp = df.copy()
        tmp["gate_entropy_norm"] = entropy
        tmp["dominant_gate_sensor"] = [SENSORS[int(i)] for i in np.argmax(gates, axis=1)]

        # Global summary.
        rec = {"method": method, "n": len(tmp), "mean_gate_entropy_norm": float(tmp["gate_entropy_norm"].mean())}
        for sensor, col in zip(SENSORS, gate_cols):
            rec[f"mean_gate_{safe_name(sensor)}"] = float(tmp[col].mean())
            rec[f"median_gate_{safe_name(sensor)}"] = float(tmp[col].median())
        rows.append(rec)

        # By target and distance.
        agg = tmp.groupby(["true_label", "distance_m"])[gate_cols + ["gate_entropy_norm"]].mean().reset_index()
        agg.to_csv(out_dir / f"gate_by_target_distance_{safe_name(method)}.csv", index=False, encoding="utf-8-sig")

        dom = tmp.groupby(["true_label", "distance_m", "dominant_gate_sensor"]).size().reset_index(name="n")
        dom.to_csv(out_dir / f"dominant_gate_counts_{safe_name(method)}.csv", index=False, encoding="utf-8-sig")

    res = pd.DataFrame(rows)
    res.to_csv(out_dir / "gate_global_summary.csv", index=False, encoding="utf-8-sig")
    write_markdown(res, out_dir / "gate_global_summary.md")
    return res


def rf_dependency_analysis(summary_csv: Path, out_dir: Path) -> Optional[pd.DataFrame]:
    if not summary_csv.exists():
        return None
    df = pd.read_csv(summary_csv)
    if not all(c in df.columns for c in ["method", "mask_CW", "mask_FMCW", "mask_RF", "macro_f1"]):
        return None
    rows = []
    mean_with_rf = df[df["mask_RF"] == 1]["macro_f1"].mean()
    mean_without_rf = df[df["mask_RF"] == 0]["macro_f1"].mean()
    rows.append({
        "analysis": "mean_macro_f1_with_vs_without_rf",
        "with_rf": float(mean_with_rf),
        "without_rf": float(mean_without_rf),
        "delta": float(mean_with_rf - mean_without_rf),
    })
    mean_with_cw = df[df["mask_CW"] == 1]["macro_f1"].mean()
    mean_without_cw = df[df["mask_CW"] == 0]["macro_f1"].mean()
    rows.append({"analysis": "mean_macro_f1_with_vs_without_cw", "with_sensor": float(mean_with_cw), "without_sensor": float(mean_without_cw), "delta": float(mean_with_cw - mean_without_cw)})
    mean_with_fmcw = df[df["mask_FMCW"] == 1]["macro_f1"].mean()
    mean_without_fmcw = df[df["mask_FMCW"] == 0]["macro_f1"].mean()
    rows.append({"analysis": "mean_macro_f1_with_vs_without_fmcw", "with_sensor": float(mean_with_fmcw), "without_sensor": float(mean_without_fmcw), "delta": float(mean_with_fmcw - mean_without_fmcw)})
    res = pd.DataFrame(rows)
    res.to_csv(out_dir / "sensor_dependency_summary.csv", index=False, encoding="utf-8-sig")
    write_markdown(res, out_dir / "sensor_dependency_summary.md")
    return res


# ============================================================
# 4. OPTIONAL MODEL STRESS TESTS
# ============================================================

def import_original_module(script_path: Path):
    if not script_path.exists():
        raise FileNotFoundError(f"Original script not found: {script_path}")
    spec = importlib.util.spec_from_file_location("tsms_original", str(script_path))
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Could not import original script from: {script_path}")
    module = importlib.util.module_from_spec(spec)
    sys.modules["tsms_original"] = module
    spec.loader.exec_module(module)
    return module


def torch_load(path: Path, device):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


def apply_tensor_stress(xs: List, stress: Dict, device):
    """Apply simple tensor-domain perturbations after normalization.

    This is not a physically exact radar/RF corruption model. It is a stress
    analysis for robustness reporting and should be described as tensor-level
    degradation.
    """
    xout = [x.clone() for x in xs]
    sensor_mask = torch.ones(xout[0].size(0), 3, device=device)

    # Deterministic missing sensors.
    missing = stress.get("missing", [])
    for j in missing:
        xout[j] = torch.zeros_like(xout[j])
        sensor_mask[:, j] = 0.0

    # Random sensor dropout at inference.
    p_out = float(stress.get("random_outage_p", 0.0))
    if p_out > 0:
        for i in range(xout[0].size(0)):
            for j in range(3):
                if random.random() < p_out:
                    xout[j][i] = 0.0
                    sensor_mask[i, j] = 0.0
            # Avoid all sensors missing.
            if sensor_mask[i].sum() == 0:
                keep = random.randint(0, 2)
                sensor_mask[i, keep] = 1.0

    # Gaussian tensor noise.
    noise_std = float(stress.get("noise_std", 0.0))
    if noise_std > 0:
        for j in range(3):
            if sensor_mask[:, j].sum() > 0:
                xout[j] = xout[j] + noise_std * torch.randn_like(xout[j])

    # Intensity scaling in normalized tensor space.
    scale = float(stress.get("scale", 1.0))
    bias = float(stress.get("bias", 0.0))
    if scale != 1.0 or bias != 0.0:
        for j in range(3):
            xout[j] = scale * xout[j] + bias

    return xout, sensor_mask


@torch.no_grad()
def predict_loader_with_stress(module, model, loader, stress: Dict, max_batches: int = 0):
    device = module.DEVICE
    model.eval()
    y_true, y_pred, meta_rows = [], [], []
    gate_rows, proba_rows = [], []

    for batch_idx, (x_cw, x_fmcw, x_rf, y, d, meta) in enumerate(tqdm(loader, desc=stress["name"], leave=False)):
        if max_batches and batch_idx >= max_batches:
            break
        x_cw = x_cw.to(device, non_blocking=True)
        x_fmcw = x_fmcw.to(device, non_blocking=True)
        x_rf = x_rf.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        xs, sensor_mask = apply_tensor_stress([x_cw, x_fmcw, x_rf], stress, device)
        logits, distance_logits, gate = model(xs[0], xs[1], xs[2], sensor_mask=sensor_mask)
        proba = torch.softmax(logits, dim=1)
        pred = torch.argmax(proba, dim=1)

        y_true.extend(y.cpu().numpy().tolist())
        y_pred.extend(pred.cpu().numpy().tolist())
        gate_rows.append(gate.cpu().numpy())
        proba_rows.append(proba.cpu().numpy())

        bs = len(y)
        for i in range(bs):
            meta_rows.append({
                "tuple_id": meta["tuple_id"][i],
                "target": meta["target"][i],
                "distance": meta["distance"][i],
                "distance_m": int(meta["distance_m"][i]),
                "sample_index": int(meta["sample_index"][i]),
            })

    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    pred_df = pd.DataFrame(meta_rows)
    pred_df["y_true"] = y_true
    pred_df["y_pred"] = y_pred
    pred_df["true_label"] = [ID_TO_TARGET[i] for i in y_true]
    pred_df["pred_label"] = [ID_TO_TARGET[i] for i in y_pred]
    pred_df["correct"] = y_true == y_pred

    if proba_rows:
        proba = np.vstack(proba_rows)
        for i, cls in ID_TO_TARGET.items():
            pred_df[f"proba_{safe_name(cls)}"] = proba[:, i]
    if gate_rows:
        gates = np.vstack(gate_rows)
        for j, sensor in enumerate(SENSORS):
            pred_df[f"gate_{safe_name(sensor)}"] = gates[:, j]
    return pred_df


def run_model_stress_tests(cfg: AuditConfig, split_df: pd.DataFrame, out_dir: Path) -> Optional[pd.DataFrame]:
    if not TORCH_AVAILABLE:
        print("[WARN] PyTorch not available. Skipping model stress tests.")
        return None
    if cfg.original_script is None:
        print("[WARN] --original-script not provided. Skipping model stress tests.")
        return None

    stress_dir = ensure_dir(out_dir / "model_stress_tests")
    module = import_original_module(cfg.original_script)

    checkpoint_path = cfg.checkpoint
    if checkpoint_path is None:
        checkpoint_path = cfg.adv_dir / "advanced_attention_model" / "checkpoint_best.pt"
    if not checkpoint_path.exists():
        checkpoint_path = cfg.adv_dir / "advanced_attention_model" / "best_advanced_attention_model.pt"
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"No best checkpoint/weights found: {checkpoint_path}")

    transforms = module.get_transforms()
    test_loader = module.make_loader(split_df, "test", transforms)
    model = module.build_model().to(module.DEVICE)

    ckpt = torch_load(checkpoint_path, module.DEVICE)
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        model.load_state_dict(ckpt["model_state_dict"])
    else:
        model.load_state_dict(ckpt)

    stress_cases = [
        {"name": "clean_reference", "missing": [], "noise_std": 0.0, "scale": 1.0, "bias": 0.0},
        {"name": "missing_cw", "missing": [0]},
        {"name": "missing_fmcw", "missing": [1]},
        {"name": "missing_rf", "missing": [2]},
        {"name": "missing_cw_fmcw", "missing": [0, 1]},
        {"name": "missing_cw_rf", "missing": [0, 2]},
        {"name": "missing_fmcw_rf", "missing": [1, 2]},
        {"name": "random_sensor_outage_p10", "random_outage_p": 0.10},
        {"name": "random_sensor_outage_p25", "random_outage_p": 0.25},
        {"name": "gaussian_noise_std_002", "noise_std": 0.02},
        {"name": "gaussian_noise_std_005", "noise_std": 0.05},
        {"name": "gaussian_noise_std_010", "noise_std": 0.10},
        {"name": "intensity_scale_090", "scale": 0.90},
        {"name": "intensity_scale_110", "scale": 1.10},
        {"name": "intensity_bias_pos_005", "bias": 0.05},
        {"name": "intensity_bias_neg_005", "bias": -0.05},
    ]

    rows = []
    for stress in stress_cases:
        pred_df = predict_loader_with_stress(module, model, test_loader, stress, max_batches=cfg.max_stress_batches)
        pred_df.to_csv(stress_dir / f"predictions_{safe_name(stress['name'])}.csv", index=False, encoding="utf-8-sig")
        y_true = pred_df["y_true"].to_numpy(dtype=int)
        y_pred = pred_df["y_pred"].to_numpy(dtype=int)
        rows.append({"stress_case": stress["name"], "n": len(pred_df), **compute_basic_metrics(y_true, y_pred)})

    res = pd.DataFrame(rows).sort_values("macro_f1", ascending=False)
    res.to_csv(stress_dir / "model_stress_summary.csv", index=False, encoding="utf-8-sig")
    write_markdown(res, stress_dir / "model_stress_summary.md")
    write_latex(
        res[["stress_case", "accuracy", "balanced_accuracy", "macro_f1", "weighted_f1", "kappa"]],
        stress_dir / "latex_model_stress_summary.tex",
        caption="Robustness of the proposed multimodal fusion model under sensor-missing and tensor-level perturbation stress tests.",
        label="tab:model_stress_tests",
    )

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.bar(res["stress_case"], res["macro_f1"])
    ax.set_ylabel("Macro-F1")
    ax.set_xlabel("Stress case")
    ax.set_title("Stress-test robustness of the fusion model")
    ax.set_ylim(0, 1.0)
    ax.tick_params(axis="x", rotation=45)
    fig.tight_layout()
    fig.savefig(stress_dir / "model_stress_macro_f1.png", dpi=300)
    plt.close(fig)

    return res


# ============================================================
# VISUAL SUMMARY
# ============================================================

def plot_split_heatmaps(df: pd.DataFrame, out_dir: Path) -> None:
    heat_dir = ensure_dir(out_dir / "split_heatmaps")
    for split in split_order():
        g = df[df["split"] == split].groupby(["target", "distance_m"]).size().reset_index(name="n")
        pivot = g.pivot(index="target", columns="distance_m", values="n").reindex(TARGETS).fillna(0)
        fig, ax = plt.subplots(figsize=(10, 4.5))
        im = ax.imshow(pivot.to_numpy(dtype=float), aspect="auto")
        ax.set_title(f"Split distribution - {split}")
        ax.set_xlabel("Distance (m)")
        ax.set_ylabel("Target")
        ax.set_xticks(range(len(pivot.columns)))
        ax.set_xticklabels(pivot.columns)
        ax.set_yticks(range(len(pivot.index)))
        ax.set_yticklabels(pivot.index)
        for i in range(pivot.shape[0]):
            for j in range(pivot.shape[1]):
                ax.text(j, i, str(int(pivot.iloc[i, j])), ha="center", va="center", fontsize=7)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        fig.tight_layout()
        fig.savefig(heat_dir / f"split_heatmap_{split}.png", dpi=300)
        plt.close(fig)


def create_final_audit_summary(out_dir: Path) -> pd.DataFrame:
    candidate_files = [
        "audit_split_integrity.csv",
        "audit_hash_duplicates.csv",
    ]
    rows = []
    for fname in candidate_files:
        p = out_dir / fname
        if p.exists():
            df = pd.read_csv(p)
            for _, r in df.iterrows():
                rows.append({
                    "source_file": fname,
                    "test": r.get("test", ""),
                    "status": r.get("status", ""),
                    "value": r.get("value", ""),
                    "details": r.get("details", ""),
                })
    summary = pd.DataFrame(rows)
    if len(summary) > 0:
        summary.to_csv(out_dir / "final_audit_summary.csv", index=False, encoding="utf-8-sig")
        write_markdown(summary, out_dir / "final_audit_summary.md")
    return summary


# ============================================================
# PAPER TEXT TEMPLATE
# ============================================================

def write_paper_methods_template(out_dir: Path) -> None:
    text = r"""
\subsection{Data Leakage and Robustness Auditing}

To ensure that the near-perfect performance observed under the random
stratified protocol was not caused by data leakage, we performed a dedicated
audit of the synchronized tuple index, split file, sensor paths, and model
predictions. First, we verified that no synchronized tuple identifier, target--
distance--sample key, or exact sensor-image path appeared in more than one
partition. Second, we computed byte-level MD5 hashes and perceptual average
hashes for sensor images to identify exact or visually similar duplicates
across train, validation, and test partitions. Third, because the original
split is stratified by target and distance, we quantified a weaker leakage risk:
the proximity between validation/test repetitions and the nearest training
repetition within the same target--distance group. A low nearest-neighbor gap
indicates that random splitting may place adjacent acquisition repetitions in
different partitions, which is not an exact leak but may overestimate
in-distribution performance when repetitions are highly correlated.

In addition to the original random split, we generated stricter robustness
protocols: leave-one-distance-out evaluation, near/mid/far range holdout,
blocked-repetition holdout, and leave-one-target-out open-set protocols. These
protocols assess whether the fusion model learns target-discriminative
representations rather than exploiting acquisition-specific regularities. For
post-hoc evaluation, we report bootstrap confidence intervals, paired McNemar
tests between sensor configurations, calibration metrics including expected
calibration error and negative log-likelihood, risk--coverage curves, and
reliability-gate statistics. These analyses complement accuracy-based metrics
and provide stronger evidence for the validity of the proposed multimodal
fusion architecture.
""".strip()
    (out_dir / "paper_methods_leakage_robustness_template.tex").write_text(text, encoding="utf-8")


# ============================================================
# MAIN
# ============================================================

def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="IEEE-style leakage and robustness audit for TSMS-Drone fusion outputs.")
    parser.add_argument("--root", type=str, default=r"D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)")
    parser.add_argument("--adv-dir", type=str, default=None, help="Default: ROOT/_advanced_fusion_improvements")
    parser.add_argument("--audit-dir", type=str, default=None, help="Default: ROOT/_ieee_leakage_and_robustness_audit")
    parser.add_argument("--sync-csv", type=str, default=None, help="Default: ADV_DIR/synchronized_tuple_index_full.csv")
    parser.add_argument("--split-csv", type=str, default=None, help="Default: ADV_DIR/tuple_splits_70_15_15_full.csv")
    parser.add_argument("--bootstrap", type=int, default=1000)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--hash-images", action="store_true", help="Compute MD5 and average-hash for all/sampled images.")
    parser.add_argument("--max-hash-files", type=int, default=0, help="0 means all files; otherwise sample this many sensor images.")
    parser.add_argument("--make-protocols", action="store_true", help="Generate stricter split protocols for robust evaluation.")
    parser.add_argument("--run-model-stress", action="store_true", help="Run optional model perturbation tests by importing the original script.")
    parser.add_argument("--original-script", type=str, default=None, help="Path to tsms_drone_advanced_fusion_improvements.py")
    parser.add_argument("--checkpoint", type=str, default=None, help="Optional checkpoint path. Default: best checkpoint under advanced_attention_model.")
    parser.add_argument("--max-stress-batches", type=int, default=0, help="0 means full test set; use small number for smoke test.")


    # Jupyter/IPython injects arguments such as:
    #   --f=c:\Users\...\kernel-xxxx.json
    # argparse.parse_args() fails on these unknown arguments.
    # parse_known_args() keeps the script usable both from PowerShell and notebooks.
    args, unknown = parser.parse_known_args()
    if unknown:
        print(f"[INFO] Ignored unknown arguments, probably from Jupyter/IPython: {unknown}")
    return args


def build_config(args: argparse.Namespace) -> AuditConfig:
    root = Path(args.root)
    adv_dir = Path(args.adv_dir) if args.adv_dir else root / "_advanced_fusion_improvements"
    audit_dir = Path(args.audit_dir) if args.audit_dir else root / "_ieee_leakage_and_robustness_audit"
    sync_csv = Path(args.sync_csv) if args.sync_csv else adv_dir / "synchronized_tuple_index_full.csv"
    split_csv = Path(args.split_csv) if args.split_csv else adv_dir / "tuple_splits_70_15_15_full.csv"
    return AuditConfig(
        root=root,
        adv_dir=adv_dir,
        audit_dir=audit_dir,
        sync_csv=sync_csv,
        split_csv=split_csv,
        predictions_root=adv_dir / "advanced_attention_model",
        bootstrap=args.bootstrap,
        seed=args.seed,
        hash_images=args.hash_images,
        max_hash_files=args.max_hash_files,
        make_protocols=args.make_protocols,
        run_model_stress=args.run_model_stress,
        original_script=Path(args.original_script) if args.original_script else None,
        checkpoint=Path(args.checkpoint) if args.checkpoint else None,
        max_stress_batches=args.max_stress_batches,
    )


def main() -> None:
    args = parse_args()
    cfg = build_config(args)
    ensure_dir(cfg.audit_dir)
    random.seed(cfg.seed)
    np.random.seed(cfg.seed)

    print("=" * 80)
    print("TSMS-Drone IEEE leakage and robustness audit")
    print("=" * 80)
    print(f"ROOT      : {cfg.root}")
    print(f"ADV_DIR   : {cfg.adv_dir}")
    print(f"AUDIT_DIR : {cfg.audit_dir}")
    print(f"SYNC_CSV  : {cfg.sync_csv}")
    print(f"SPLIT_CSV : {cfg.split_csv}")
    print("=" * 80)

    t0 = time.time()
    split_df = load_split_dataframe(cfg)

    # Main audits.
    audit_split_integrity(split_df, cfg.audit_dir)
    plot_split_heatmaps(split_df, cfg.audit_dir)
    audit_repetition_proximity(split_df, cfg.audit_dir)
    audit_hash_duplicates(split_df, cfg.audit_dir, cfg)
    metadata_only_shortcut_test(split_df, cfg.audit_dir, seed=cfg.seed)

    # Protocol generation.
    if cfg.make_protocols:
        generate_robust_protocols(split_df, cfg.audit_dir, seed=cfg.seed)

    # Prediction-based analyses.
    preds = load_predictions(cfg.predictions_root)
    if preds:
        pred_dir = ensure_dir(cfg.audit_dir / "prediction_posthoc_analysis")
        prediction_bootstrap_cis(preds, pred_dir, n_boot=cfg.bootstrap, seed=cfg.seed)
        paired_mcnemar_tests(preds, pred_dir)
        calibration_and_risk_analysis(preds, pred_dir, n_bins=15)
        target_distance_error_analysis(preds, pred_dir)
        gate_analysis(preds, pred_dir)
    else:
        print(f"[WARN] No prediction files found under: {cfg.predictions_root}")

    # Sensor dependency from summary table.
    rf_dependency_analysis(cfg.adv_dir / "advanced_fusion_results_summary.csv", cfg.audit_dir)

    # Optional model-level stress tests.
    if cfg.run_model_stress:
        run_model_stress_tests(cfg, split_df, cfg.audit_dir)

    write_paper_methods_template(cfg.audit_dir)
    final_summary = create_final_audit_summary(cfg.audit_dir)

    manifest = {
        "root": str(cfg.root),
        "adv_dir": str(cfg.adv_dir),
        "audit_dir": str(cfg.audit_dir),
        "sync_csv": str(cfg.sync_csv),
        "split_csv": str(cfg.split_csv),
        "hash_images": cfg.hash_images,
        "max_hash_files": cfg.max_hash_files,
        "make_protocols": cfg.make_protocols,
        "run_model_stress": cfg.run_model_stress,
        "elapsed_sec": time.time() - t0,
        "n_final_audit_rows": int(len(final_summary)),
    }
    save_json(manifest, cfg.audit_dir / "audit_manifest.json")

    print("\nGenerated audit files:")
    for p in sorted(cfg.audit_dir.glob("*")):
        print(p)
    print("=" * 80)
    print("[OK] IEEE leakage and robustness audit completed.")
    print("=" * 80)


if __name__ == "__main__":
    main()


[INFO] Ignored unknown arguments, probably from Jupyter/IPython: ['--f=c:\\Users\\Luis\\AppData\\Roaming\\jupyter\\runtime\\kernel-v3040e410708fb7b5bf2eb7c62141f275456ffc0f8.json']
TSMS-Drone IEEE leakage and robustness audit
ROOT      : D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)
ADV_DIR   : D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_advanced_fusion_improvements
AUDIT_DIR : D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit
SYNC_CSV  : D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_advanced_fusion_improvements\synchronized_tuple_index_full.csv
SPLIT_CSV : D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_advanced_fusion_improvements\tuple_splits_70_15_15_full.csv


Hashing images: 100%|██████████| 112500/112500 [01:18<00:00, 1425.90it/s]



Generated audit files:
D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\audit_hash_duplicates.csv
D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\audit_hash_duplicates.md
D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\audit_manifest.json
D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\audit_split_integrity.csv
D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\audit_split_integrity.md
D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\final_audit_summary.csv
D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\final_audit_summary.md
D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\image_hash_index.csv
D:\Time-Synchroniz

In [26]:
# ============================================================
# PRINT IEEE-READY RESULTS TABLES FOR TSMS-DRONE PAPER
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

try:
    from IPython.display import display, Markdown
    HAS_DISPLAY = True
except Exception:
    HAS_DISPLAY = False


# ============================================================
# PATHS
# ============================================================

ROOT = Path(r"D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)")
ADV_DIR = ROOT / "_advanced_fusion_improvements"
AUDIT_DIR = ROOT / "_ieee_leakage_and_robustness_audit"
PRED_DIR = AUDIT_DIR / "prediction_posthoc_analysis"
TABLE_DIR = AUDIT_DIR / "_paper_results_tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("IEEE PAPER RESULTS TABLE GENERATOR")
print("=" * 80)
print(f"ROOT      : {ROOT}")
print(f"ADV_DIR   : {ADV_DIR}")
print(f"AUDIT_DIR : {AUDIT_DIR}")
print(f"TABLE_DIR : {TABLE_DIR}")
print("=" * 80)


# ============================================================
# HELPERS
# ============================================================

def read_csv_if_exists(path: Path) -> pd.DataFrame | None:
    if path.exists():
        return pd.read_csv(path)
    print(f"[WARN] File not found: {path}")
    return None


def pretty_method_name(x: str) -> str:
    x = str(x)

    mapping = {
        "Advanced-Fusion-All": "All sensors",
        "Advanced-Fusion-CW-RF": "CW + RF",
        "Advanced-Fusion-FMCW-RF": "FMCW + RF",
        "Advanced-Fusion-RF-only": "RF only",
        "Advanced-Fusion-CW-FMCW": "CW + FMCW",
        "Advanced-Fusion-FMCW-only": "FMCW only",
        "Advanced-Fusion-CW-only": "CW only",

        "advanced_fusion_all": "All sensors",
        "advanced_fusion_cw_rf": "CW + RF",
        "advanced_fusion_fmcw_rf": "FMCW + RF",
        "advanced_fusion_rf_only": "RF only",
        "advanced_fusion_cw_fmcw": "CW + FMCW",
        "advanced_fusion_fmcw_only": "FMCW only",
        "advanced_fusion_cw_only": "CW only",
    }

    return mapping.get(x, x.replace("_", " ").title())


def save_table(df: pd.DataFrame, name: str, caption: str = "", label: str = ""):
    csv_path = TABLE_DIR / f"{name}.csv"
    md_path = TABLE_DIR / f"{name}.md"
    tex_path = TABLE_DIR / f"{name}.tex"

    df.to_csv(csv_path, index=False, encoding="utf-8-sig")

    try:
        md_path.write_text(df.to_markdown(index=False), encoding="utf-8")
    except Exception:
        md_path.write_text(df.to_csv(index=False), encoding="utf-8")

    latex = df.to_latex(
        index=False,
        escape=False,
        caption=caption if caption else None,
        label=label if label else None,
        float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x),
    )
    tex_path.write_text(latex, encoding="utf-8")

    return csv_path, md_path, tex_path


def show_table(title: str, df: pd.DataFrame, name: str, caption: str = "", label: str = ""):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

    if HAS_DISPLAY:
        display(Markdown(f"### {title}"))
        display(df)
    else:
        print(df.to_string(index=False))

    csv_path, md_path, tex_path = save_table(df, name, caption, label)

    print(f"[SAVED] {csv_path}")
    print(f"[SAVED] {md_path}")
    print(f"[SAVED] {tex_path}")


def round_metrics(df: pd.DataFrame, cols=None, digits=4):
    out = df.copy()
    if cols is None:
        cols = out.select_dtypes(include=[np.number]).columns
    for c in cols:
        if c in out.columns:
            out[c] = out[c].astype(float).round(digits)
    return out


# ============================================================
# TABLE 1 — MAIN PERFORMANCE AND SENSOR ABLATION
# ============================================================

main_results = read_csv_if_exists(ADV_DIR / "advanced_fusion_results_summary.csv")

if main_results is not None:
    t1 = main_results.copy()

    t1["Setting"] = t1["method"].map(pretty_method_name)
    t1["CW"] = t1["mask_CW"].map({1: "Yes", 0: "No"})
    t1["FMCW"] = t1["mask_FMCW"].map({1: "Yes", 0: "No"})
    t1["RF"] = t1["mask_RF"].map({1: "Yes", 0: "No"})

    all_macro = float(t1.loc[t1["method"] == "Advanced-Fusion-All", "macro_f1"].iloc[0])
    t1["Δ Macro-F1 vs All"] = t1["macro_f1"] - all_macro

    order = [
        "All sensors",
        "CW + RF",
        "FMCW + RF",
        "RF only",
        "CW + FMCW",
        "FMCW only",
        "CW only",
    ]

    t1["order"] = t1["Setting"].apply(lambda x: order.index(x) if x in order else 999)

    t1 = t1.sort_values("order")[
        [
            "Setting",
            "CW",
            "FMCW",
            "RF",
            "accuracy",
            "balanced_accuracy",
            "macro_precision",
            "macro_recall",
            "macro_f1",
            "weighted_f1",
            "kappa",
            "Δ Macro-F1 vs All",
        ]
    ]

    t1 = t1.rename(columns={
        "accuracy": "Accuracy",
        "balanced_accuracy": "Balanced Acc.",
        "macro_precision": "Macro Precision",
        "macro_recall": "Macro Recall",
        "macro_f1": "Macro-F1",
        "weighted_f1": "Weighted-F1",
        "kappa": "Kappa",
    })

    t1 = round_metrics(t1)

    show_table(
        "Table 1 — Main classification performance and sensor-ablation results",
        t1,
        "table_01_main_performance_sensor_ablation",
        caption="Main classification performance of the proposed attention-based fusion model under sensor-ablation settings.",
        label="tab:main_performance_sensor_ablation"
    )


# ============================================================
# TABLE 2 — DATA LEAKAGE AUDIT SUMMARY
# ============================================================

audit_summary = read_csv_if_exists(AUDIT_DIR / "final_audit_summary.csv")

if audit_summary is not None:
    t2 = audit_summary.copy()

    # Keep compact and paper-readable.
    t2 = t2[["source_file", "test", "status", "value", "details"]].copy()
    t2 = t2.rename(columns={
        "source_file": "Source",
        "test": "Audit check",
        "status": "Status",
        "value": "Value",
        "details": "Interpretation",
    })

    # Optional ordering: failures/warnings first.
    status_order = {"fail": 0, "warning": 1, "pass": 2, "info": 3}
    t2["_order"] = t2["Status"].map(status_order).fillna(9)
    t2 = t2.sort_values(["_order", "Source", "Audit check"]).drop(columns=["_order"])

    show_table(
        "Table 2 — Data leakage and split-integrity audit summary",
        t2,
        "table_02_data_leakage_audit_summary",
        caption="Summary of data-leakage and split-integrity checks applied to the synchronized TSMS-Drone tuple index.",
        label="tab:data_leakage_audit"
    )


# ============================================================
# TABLE 3 — REPETITION PROXIMITY RISK
# ============================================================

rep_prox = read_csv_if_exists(AUDIT_DIR / "repetition_proximity_summary.csv")

if rep_prox is not None:
    t3 = rep_prox.copy()

    t3 = t3.rename(columns={
        "split": "Split",
        "n": "N",
        "mean": "Mean nearest gap",
        "median": "Median nearest gap",
        "min": "Min",
        "q05": "Q05",
        "q25": "Q25",
        "q75": "Q75",
        "max": "Max",
    })

    t3 = round_metrics(t3)

    show_table(
        "Table 3 — Repetition-proximity analysis between evaluation and training samples",
        t3,
        "table_03_repetition_proximity",
        caption="Nearest training repetition gap for validation and test samples within the same target--distance group.",
        label="tab:repetition_proximity"
    )


# ============================================================
# TABLE 4 — METADATA-ONLY SHORTCUT PROBE
# ============================================================

shortcut = read_csv_if_exists(AUDIT_DIR / "metadata_only_shortcut_test.csv")

if shortcut is not None:
    t4 = shortcut.copy()

    t4 = t4.rename(columns={
        "probe": "Probe",
        "model_status": "Status",
        "accuracy": "Accuracy",
        "balanced_accuracy": "Balanced Acc.",
        "macro_precision": "Macro Precision",
        "macro_recall": "Macro Recall",
        "macro_f1": "Macro-F1",
        "weighted_f1": "Weighted-F1",
        "kappa": "Kappa",
    })

    t4 = round_metrics(t4)

    show_table(
        "Table 4 — Metadata-only shortcut probe",
        t4,
        "table_04_metadata_only_shortcut_probe",
        caption="Metadata-only shortcut probe using only distance and sample index as predictors.",
        label="tab:metadata_shortcut_probe"
    )


# ============================================================
# TABLE 5 — BOOTSTRAP CONFIDENCE INTERVALS
# ============================================================

bootstrap_ci = read_csv_if_exists(PRED_DIR / "bootstrap_confidence_intervals.csv")

if bootstrap_ci is not None:
    t5 = bootstrap_ci.copy()

    # Keep only the most important metrics for the paper.
    keep_metrics = ["macro_f1", "balanced_accuracy", "kappa"]
    t5 = t5[t5["metric"].isin(keep_metrics)].copy()

    t5["Setting"] = t5["method"].map(pretty_method_name)
    t5["Metric"] = t5["metric"].map({
        "macro_f1": "Macro-F1",
        "balanced_accuracy": "Balanced Acc.",
        "kappa": "Kappa",
    })

    t5["95% CI"] = (
        "[" +
        t5["ci95_low"].round(4).astype(str) +
        ", " +
        t5["ci95_high"].round(4).astype(str) +
        "]"
    )

    t5 = t5[["Setting", "Metric", "point", "95% CI", "n_bootstrap"]]
    t5 = t5.rename(columns={
        "point": "Point estimate",
        "n_bootstrap": "Bootstrap resamples",
    })

    t5["Point estimate"] = t5["Point estimate"].round(4)

    order = ["All sensors", "CW + RF", "FMCW + RF", "RF only", "CW + FMCW", "FMCW only", "CW only"]
    metric_order = ["Macro-F1", "Balanced Acc.", "Kappa"]

    t5["_order1"] = t5["Setting"].apply(lambda x: order.index(x) if x in order else 999)
    t5["_order2"] = t5["Metric"].apply(lambda x: metric_order.index(x) if x in metric_order else 999)
    t5 = t5.sort_values(["_order1", "_order2"]).drop(columns=["_order1", "_order2"])

    show_table(
        "Table 5 — Bootstrap 95% confidence intervals",
        t5,
        "table_05_bootstrap_confidence_intervals",
        caption="Bootstrap 95\\% confidence intervals for the main performance metrics under each sensor setting.",
        label="tab:bootstrap_ci"
    )


# ============================================================
# TABLE 6 — CALIBRATION AND RISK-COVERAGE SUMMARY
# ============================================================

calib = read_csv_if_exists(PRED_DIR / "calibration_and_risk_summary.csv")

if calib is not None:
    t6 = calib.copy()

    t6["Setting"] = t6["method"].map(pretty_method_name)

    t6 = t6[
        [
            "Setting",
            "ece",
            "nll",
            "brier_multiclass",
            "aurc",
            "mean_confidence",
            "median_confidence",
            "min_confidence",
        ]
    ]

    t6 = t6.rename(columns={
        "ece": "ECE",
        "nll": "NLL",
        "brier_multiclass": "Brier",
        "aurc": "AURC",
        "mean_confidence": "Mean conf.",
        "median_confidence": "Median conf.",
        "min_confidence": "Min conf.",
    })

    t6 = round_metrics(t6)

    order = ["All sensors", "CW + RF", "FMCW + RF", "RF only", "CW + FMCW", "FMCW only", "CW only"]
    t6["_order"] = t6["Setting"].apply(lambda x: order.index(x) if x in order else 999)
    t6 = t6.sort_values("_order").drop(columns=["_order"])

    show_table(
        "Table 6 — Calibration and selective-risk summary",
        t6,
        "table_06_calibration_risk_summary",
        caption="Calibration and selective-risk metrics for the proposed fusion model and sensor-ablation settings.",
        label="tab:calibration_risk"
    )


# ============================================================
# TABLE 7 — MCNEMAR TESTS AGAINST FULL FUSION
# ============================================================

mcnemar = read_csv_if_exists(PRED_DIR / "paired_mcnemar_tests.csv")

if mcnemar is not None:
    t7 = mcnemar.copy()

    # Keep comparisons involving full fusion only.
    full_names = ["advanced_fusion_all", "Advanced-Fusion-All"]

    mask_full = t7["method_a"].isin(full_names) | t7["method_b"].isin(full_names)
    t7 = t7[mask_full].copy()

    if len(t7) > 0:
        def get_other_method(row):
            if row["method_a"] in full_names:
                return row["method_b"]
            return row["method_a"]

        t7["Compared setting"] = t7.apply(get_other_method, axis=1).map(pretty_method_name)

        t7 = t7[
            [
                "Compared setting",
                "n_paired",
                "b_a_correct_b_wrong",
                "c_a_wrong_b_correct",
                "discordant",
                "p_value",
            ]
        ]

        t7 = t7.rename(columns={
            "n_paired": "Paired N",
            "b_a_correct_b_wrong": "b",
            "c_a_wrong_b_correct": "c",
            "discordant": "Discordant",
            "p_value": "p-value",
        })

        t7 = round_metrics(t7, cols=["p-value"], digits=6)

        show_table(
            "Table 7 — Paired McNemar tests against full fusion",
            t7,
            "table_07_mcnemar_vs_full_fusion",
            caption="Paired McNemar tests comparing full fusion against sensor-ablation settings.",
            label="tab:mcnemar_full_fusion"
        )
    else:
        print("[WARN] McNemar table found, but no comparison involving full fusion was detected.")


# ============================================================
# TABLE 8 — RELIABILITY GATE SUMMARY
# ============================================================

gate = read_csv_if_exists(PRED_DIR / "gate_global_summary.csv")

if gate is not None:
    t8 = gate.copy()

    t8["Setting"] = t8["method"].map(pretty_method_name)

    keep_cols = [
        "Setting",
        "n",
        "mean_gate_entropy_norm",
        "mean_gate_cw_radar",
        "mean_gate_fmcw_radar",
        "mean_gate_rf_receiver",
        "median_gate_cw_radar",
        "median_gate_fmcw_radar",
        "median_gate_rf_receiver",
    ]

    keep_cols = [c for c in keep_cols if c in t8.columns]
    t8 = t8[keep_cols]

    t8 = t8.rename(columns={
        "n": "N",
        "mean_gate_entropy_norm": "Mean gate entropy",
        "mean_gate_cw_radar": "Mean gate CW",
        "mean_gate_fmcw_radar": "Mean gate FMCW",
        "mean_gate_rf_receiver": "Mean gate RF",
        "median_gate_cw_radar": "Median gate CW",
        "median_gate_fmcw_radar": "Median gate FMCW",
        "median_gate_rf_receiver": "Median gate RF",
    })

    t8 = round_metrics(t8)

    order = ["All sensors", "CW + RF", "FMCW + RF", "RF only", "CW + FMCW", "FMCW only", "CW only"]
    t8["_order"] = t8["Setting"].apply(lambda x: order.index(x) if x in order else 999)
    t8 = t8.sort_values("_order").drop(columns=["_order"])

    show_table(
        "Table 8 — Reliability-gate summary",
        t8,
        "table_08_reliability_gate_summary",
        caption="Global reliability-gate statistics learned by the attention-based multimodal fusion model.",
        label="tab:gate_summary"
    )


# ============================================================
# TABLE 9 — SENSOR DEPENDENCY SUMMARY
# ============================================================

sensor_dep = read_csv_if_exists(AUDIT_DIR / "sensor_dependency_summary.csv")

if sensor_dep is not None:
    t9 = sensor_dep.copy()
    t9 = round_metrics(t9)

    show_table(
        "Table 9 — Sensor dependency summary",
        t9,
        "table_09_sensor_dependency_summary",
        caption="Mean macro-F1 variation with and without each sensor modality.",
        label="tab:sensor_dependency"
    )


# ============================================================
# TABLE 10 — ROBUST SPLIT PROTOCOLS, IF GENERATED
# ============================================================

protocol_index = read_csv_if_exists(AUDIT_DIR / "robust_split_protocols" / "protocol_index.csv")

if protocol_index is not None:
    t10 = protocol_index.copy()

    keep = [
        "protocol",
        "held_out_distance_m",
        "held_out_distances_m",
        "held_out_target",
        "train_n",
        "val_n",
        "test_n",
        "test_ood_n",
        "file",
    ]
    keep = [c for c in keep if c in t10.columns]
    t10 = t10[keep]

    t10 = t10.rename(columns={
        "protocol": "Protocol",
        "held_out_distance_m": "Held-out distance",
        "held_out_distances_m": "Held-out range",
        "held_out_target": "Held-out target",
        "train_n": "Train N",
        "val_n": "Val N",
        "test_n": "Test N",
        "test_ood_n": "OOD Test N",
        "file": "Split file",
    })

    show_table(
        "Table 10 — Robust evaluation protocols generated for additional experiments",
        t10,
        "table_10_robust_evaluation_protocols",
        caption="Additional split protocols generated to assess distance generalization, repetition-block robustness, range holdout, and open-set behavior.",
        label="tab:robust_protocols"
    )
else:
    print("\n[INFO] Robust split protocol table not found.")
    print("Run the audit with --make-protocols to generate Table 10.")


# ============================================================
# FINAL LIST OF GENERATED TABLES
# ============================================================

print("\n" + "=" * 80)
print("GENERATED PAPER TABLE FILES")
print("=" * 80)

for p in sorted(TABLE_DIR.glob("*")):
    print(p)

print("=" * 80)
print("[OK] Tables printed and saved.")
print("=" * 80)

IEEE PAPER RESULTS TABLE GENERATOR
ROOT      : D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)
ADV_DIR   : D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_advanced_fusion_improvements
AUDIT_DIR : D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit
TABLE_DIR : D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\_paper_results_tables

Table 1 — Main classification performance and sensor-ablation results


### Table 1 — Main classification performance and sensor-ablation results

,Setting,CW,FMCW,RF,Accuracy,Balanced Acc.,Macro Precision,Macro Recall,Macro-F1,Weighted-F1,Kappa,Δ Macro-F1 vs All
0,All sensors,Yes,Yes,Yes,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.0000
1,CW + RF,Yes,No,Yes,0.9998,0.9998,0.9998,0.9998,0.9998,0.9998,0.9998,-0.0002
2,FMCW + RF,No,Yes,Yes,0.9998,0.9998,0.9998,0.9998,0.9998,0.9998,0.9998,-0.0002
3,RF only,No,No,Yes,0.9993,0.9993,0.9993,0.9993,0.9993,0.9993,0.9991,-0.0007
4,CW + FMCW,Yes,Yes,No,0.9145,0.9145,0.9166,0.9145,0.9140,0.9140,0.8931,-0.0860
5,FMCW only,No,Yes,No,0.8418,0.8418,0.8431,0.8418,0.8384,0.8384,0.8022,-0.1616
6,CW only,Yes,No,No,0.6171,0.6171,0.6848,0.6171,0.6009,0.6009,0.5213,-0.3991


[SAVED] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\_paper_results_tables\table_01_main_performance_sensor_ablation.csv
[SAVED] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\_paper_results_tables\table_01_main_performance_sensor_ablation.md
[SAVED] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\_paper_results_tables\table_01_main_performance_sensor_ablation.tex

Table 2 — Data leakage and split-integrity audit summary


### Table 2 — Data leakage and split-integrity audit summary

...

,Setting,N,Mean gate entropy,Mean gate CW,Mean gate FMCW,Mean gate RF,Median gate CW,Median gate FMCW,Median gate RF
0,All sensors,5625.0,0.6975,0.1665,0.4232,0.4102,0.0970,0.3742,0.4034
2,CW + RF,5625.0,0.3820,0.3261,0.0000,0.6739,0.2388,0.0000,0.7612
3,FMCW + RF,5625.0,0.4638,0.0000,0.5220,0.4780,0.0000,0.4862,0.5138
6,RF only,5625.0,-0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,1.0000
1,CW + FMCW,5625.0,0.3823,0.2605,0.7395,0.0000,0.1857,0.8143,0.0000
5,FMCW only,5625.0,-0.0000,0.0000,1.0000,0.0000,0.0000,1.0000,0.0000
4,CW only,5625.0,-0.0000,1.0000,0.0000,0.0000,1.0000,0.0000,0.0000


[SAVED] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\_paper_results_tables\table_08_reliability_gate_summary.csv
[SAVED] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\_paper_results_tables\table_08_reliability_gate_summary.md
[SAVED] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\_paper_results_tables\table_08_reliability_gate_summary.tex

Table 9 — Sensor dependency summary


### Table 9 — Sensor dependency summary

,analysis,with_rf,without_rf,delta,with_sensor,without_sensor
0,mean_macro_f1_with_vs_without_rf,0.9997,0.7844,0.2153,NaN,NaN
1,mean_macro_f1_with_vs_without_cw,NaN,NaN,-0.0672,0.8787,0.9458
2,mean_macro_f1_with_vs_without_fmcw,NaN,NaN,0.0714,0.9381,0.8667


[SAVED] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\_paper_results_tables\table_09_sensor_dependency_summary.csv
[SAVED] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\_paper_results_tables\table_09_sensor_dependency_summary.md
[SAVED] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\_paper_results_tables\table_09_sensor_dependency_summary.tex
[WARN] File not found: D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\robust_split_protocols\protocol_index.csv

[INFO] Robust split protocol table not found.
Run the audit with --make-protocols to generate Table 10.

GENERATED PAPER TABLE FILES
D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\_paper_results_tables\table_01_main_performance_sensor_ablation.csv
D:\Time-Synchronized Multi-Sensor Drone Dataset

In [27]:
# ============================================================
# tsms_drone_ieee_audit_tables_complete.py
# ============================================================
# TSMS-Drone IEEE-ready leakage, robustness and results-table audit.
#
# This is a complete Jupyter/PowerShell-safe script that:
#   1. audits exact data leakage and split integrity;
#   2. computes MD5 and perceptual hashes by default;
#   3. quantifies weak leakage caused by adjacent repetitions;
#   4. generates robust split protocols by default;
#   5. performs post-hoc statistical, calibration and gate analyses;
#   6. prints the necessary and sufficient tables for the Results section;
#   7. saves CSV/MD/LaTeX versions of all paper-ready tables.
#
# Jupyter usage, simplest:
#   Execute the whole cell/script. Defaults are already IEEE-ready:
#       hash_images=True, make_protocols=True, print_tables=True.
#
# Jupyter usage with %run:
#   %run tsms_drone_ieee_audit_tables_complete.py
#
# PowerShell usage:
#   python tsms_drone_ieee_audit_tables_complete.py ^
#     --root "D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)"
#
# Fast smoke-test without hashes:
#   python tsms_drone_ieee_audit_tables_complete.py --no-hash-images --bootstrap 200
#
# Output folders:
#   ROOT\_ieee_leakage_and_robustness_audit
#   ROOT\_ieee_leakage_and_robustness_audit\_paper_results_tables
# ============================================================

from __future__ import annotations

import argparse
import hashlib
import itertools
import json
import math
import random
import re
import sys
import time
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    cohen_kappa_score,
    log_loss,
)
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

try:
    from scipy.stats import binomtest
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

try:
    from IPython.display import display, Markdown
    HAS_DISPLAY = True
except Exception:
    HAS_DISPLAY = False

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)
pd.set_option("display.max_colwidth", 120)


# ============================================================
# CONSTANTS
# ============================================================

SENSORS = ["CW Radar", "FMCW Radar", "RF Receiver"]
TARGETS = [
    "Inspire 2",
    "Matrice 30",
    "Mavic 2 Pro",
    "Phantom 4 Pro",
    "Corner Reflector",
]
DISTANCES = [f"{d}m" for d in range(2, 31, 2)]
DISTANCE_VALUES = [int(d.replace("m", "")) for d in DISTANCES]
TARGET_TO_ID = {t: i for i, t in enumerate(TARGETS)}
ID_TO_TARGET = {i: t for t, i in TARGET_TO_ID.items()}


# ============================================================
# CONFIGURATION
# ============================================================

@dataclass
class AuditConfig:
    root: Path
    adv_dir: Path
    audit_dir: Path
    table_dir: Path
    sync_csv: Path
    split_csv: Path
    predictions_root: Path
    bootstrap: int = 1000
    seed: int = 42
    hash_images: bool = True
    max_hash_files: int = 0
    make_protocols: bool = True
    print_tables: bool = True


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="IEEE-ready leakage, robustness and paper-table audit for TSMS-Drone fusion outputs."
    )
    parser.add_argument("--root", type=str, default=r"D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)")
    parser.add_argument("--adv-dir", type=str, default=None, help="Default: ROOT/_advanced_fusion_improvements")
    parser.add_argument("--audit-dir", type=str, default=None, help="Default: ROOT/_ieee_leakage_and_robustness_audit")
    parser.add_argument("--sync-csv", type=str, default=None, help="Default: ADV_DIR/synchronized_tuple_index_full.csv")
    parser.add_argument("--split-csv", type=str, default=None, help="Default: ADV_DIR/tuple_splits_70_15_15_full.csv")
    parser.add_argument("--bootstrap", type=int, default=1000)
    parser.add_argument("--seed", type=int, default=42)

    # IEEE-ready defaults: hashes, robust protocols and printed tables enabled.
    parser.add_argument("--hash-images", dest="hash_images", action="store_true", default=True)
    parser.add_argument("--no-hash-images", dest="hash_images", action="store_false")
    parser.add_argument("--max-hash-files", type=int, default=0, help="0 means all files; otherwise sample this many sensor images.")

    parser.add_argument("--make-protocols", dest="make_protocols", action="store_true", default=True)
    parser.add_argument("--no-make-protocols", dest="make_protocols", action="store_false")

    parser.add_argument("--print-tables", dest="print_tables", action="store_true", default=True)
    parser.add_argument("--no-print-tables", dest="print_tables", action="store_false")

    # Jupyter/IPython injects arguments such as --f=kernel.json. Ignore them.
    args, unknown = parser.parse_known_args()
    if unknown:
        print(f"[INFO] Ignored unknown arguments, probably from Jupyter/IPython: {unknown}")
    return args


def build_config(args: argparse.Namespace) -> AuditConfig:
    root = Path(args.root)
    adv_dir = Path(args.adv_dir) if args.adv_dir else root / "_advanced_fusion_improvements"
    audit_dir = Path(args.audit_dir) if args.audit_dir else root / "_ieee_leakage_and_robustness_audit"
    table_dir = audit_dir / "_paper_results_tables"
    sync_csv = Path(args.sync_csv) if args.sync_csv else adv_dir / "synchronized_tuple_index_full.csv"
    split_csv = Path(args.split_csv) if args.split_csv else adv_dir / "tuple_splits_70_15_15_full.csv"
    return AuditConfig(
        root=root,
        adv_dir=adv_dir,
        audit_dir=audit_dir,
        table_dir=table_dir,
        sync_csv=sync_csv,
        split_csv=split_csv,
        predictions_root=adv_dir / "advanced_attention_model",
        bootstrap=args.bootstrap,
        seed=args.seed,
        hash_images=args.hash_images,
        max_hash_files=args.max_hash_files,
        make_protocols=args.make_protocols,
        print_tables=args.print_tables,
    )


# ============================================================
# SMALL UTILITIES
# ============================================================

def normalize_token(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(text).lower())


def safe_name(text: str) -> str:
    text = str(text).lower().strip()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return text.strip("_")


def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def read_csv_required(path: Path, name: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Required {name} not found: {path}")
    return pd.read_csv(path)


def read_csv_if_exists(path: Path) -> Optional[pd.DataFrame]:
    if path.exists():
        return pd.read_csv(path)
    print(f"[WARN] File not found: {path}")
    return None


def write_markdown(df: pd.DataFrame, path: Path) -> None:
    try:
        path.write_text(df.to_markdown(index=False), encoding="utf-8")
    except Exception:
        path.write_text(df.to_csv(index=False), encoding="utf-8")


def write_latex(df: pd.DataFrame, path: Path, caption: str = "", label: str = "") -> None:
    latex = df.to_latex(
        index=False,
        escape=False,
        caption=caption if caption else None,
        label=label if label else None,
        float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x),
    )
    path.write_text(latex, encoding="utf-8")


def save_json(obj: Dict, path: Path) -> None:
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")


def md5_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.md5()
    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def average_hash(path: Path, hash_size: int = 8) -> Optional[str]:
    try:
        img = Image.open(path).convert("L").resize((hash_size, hash_size), Image.BILINEAR)
        arr = np.asarray(img, dtype=np.float32)
        bits = arr >= arr.mean()
        value = 0
        for bit in bits.flatten():
            value = (value << 1) | int(bit)
        return f"{value:0{hash_size * hash_size // 4}x}"
    except Exception:
        return None


def safe_trapezoid(y: np.ndarray, x: np.ndarray) -> float:
    y = np.asarray(y, dtype=float)
    x = np.asarray(x, dtype=float)
    if len(y) < 2 or len(x) < 2:
        return 0.0
    if hasattr(np, "trapezoid"):
        return float(np.trapezoid(y, x))
    if hasattr(np, "trapz"):
        return float(np.trapz(y, x))
    return float(np.sum(np.diff(x) * 0.5 * (y[1:] + y[:-1])))


def infer_sensor_from_path(path: str) -> Optional[str]:
    parts = [normalize_token(p) for p in Path(str(path)).parts]
    for sensor in SENSORS:
        if normalize_token(sensor) in parts:
            return sensor
    joined = normalize_token(path)
    for sensor in sorted(SENSORS, key=lambda s: len(normalize_token(s)), reverse=True):
        if normalize_token(sensor) in joined:
            return sensor
    return None


def compute_basic_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "kappa": cohen_kappa_score(y_true, y_pred),
    }


def get_proba_columns(df: pd.DataFrame) -> List[str]:
    return [c for c in df.columns if c.startswith("proba_")]


def split_order() -> List[str]:
    return ["train", "val", "test"]


def pretty_method_name(x: str) -> str:
    mapping = {
        "Advanced-Fusion-All": "All sensors",
        "Advanced-Fusion-CW-RF": "CW + RF",
        "Advanced-Fusion-FMCW-RF": "FMCW + RF",
        "Advanced-Fusion-RF-only": "RF only",
        "Advanced-Fusion-CW-FMCW": "CW + FMCW",
        "Advanced-Fusion-FMCW-only": "FMCW only",
        "Advanced-Fusion-CW-only": "CW only",
        "advanced_fusion_all": "All sensors",
        "advanced_fusion_cw_rf": "CW + RF",
        "advanced_fusion_fmcw_rf": "FMCW + RF",
        "advanced_fusion_rf_only": "RF only",
        "advanced_fusion_cw_fmcw": "CW + FMCW",
        "advanced_fusion_fmcw_only": "FMCW only",
        "advanced_fusion_cw_only": "CW only",
    }
    return mapping.get(str(x), str(x).replace("_", " ").title())


def round_metrics(df: pd.DataFrame, cols=None, digits: int = 4) -> pd.DataFrame:
    out = df.copy()
    if cols is None:
        cols = out.select_dtypes(include=[np.number]).columns
    for c in cols:
        if c in out.columns:
            out[c] = out[c].astype(float).round(digits)
    return out


# ============================================================
# DATA LOADING
# ============================================================

def load_split_dataframe(cfg: AuditConfig) -> pd.DataFrame:
    sync_df = read_csv_required(cfg.sync_csv, "synchronized tuple index")
    split_df = read_csv_required(cfg.split_csv, "split file")

    if "split" not in split_df.columns:
        raise ValueError(f"Split file does not contain column 'split': {cfg.split_csv}")

    required = ["target", "distance", "distance_m", "sample_index", "tuple_id", "split"] + SENSORS
    missing = [c for c in required if c not in split_df.columns]
    if missing:
        raise ValueError(f"Split file is missing required columns: {missing}")

    if "label" not in split_df.columns:
        split_df["label"] = split_df["target"].map(TARGET_TO_ID).astype(int)

    split_df["distance_m"] = split_df["distance_m"].astype(int)
    split_df["sample_index"] = split_df["sample_index"].astype(int)
    split_df["tuple_id"] = split_df["tuple_id"].astype(str)
    split_df["split"] = split_df["split"].astype(str)

    if len(sync_df) != len(split_df):
        print(f"[WARN] sync index rows ({len(sync_df)}) differ from split rows ({len(split_df)}).")

    return split_df


# ============================================================
# 1. LEAKAGE AND SPLIT AUDITS
# ============================================================

def audit_split_integrity(df: pd.DataFrame, out_dir: Path) -> pd.DataFrame:
    rows = []
    rows.append({"test": "n_total_tuples", "status": "info", "value": len(df), "details": "Total synchronized tuples."})

    valid_splits = set(split_order())
    observed_splits = set(df["split"].unique())
    extra_splits = sorted(observed_splits - valid_splits)
    missing_splits = sorted(valid_splits - observed_splits)
    rows.append({
        "test": "split_labels_valid",
        "status": "pass" if not extra_splits and not missing_splits else "fail",
        "value": len(observed_splits),
        "details": f"observed={sorted(observed_splits)}, extra={extra_splits}, missing={missing_splits}",
    })

    dup_tuple = df[df.duplicated("tuple_id", keep=False)].copy()
    rows.append({
        "test": "tuple_id_unique",
        "status": "pass" if len(dup_tuple) == 0 else "fail",
        "value": len(dup_tuple),
        "details": "Duplicated tuple_id rows across the split table.",
    })
    if len(dup_tuple) > 0:
        dup_tuple.to_csv(out_dir / "fail_duplicate_tuple_ids.csv", index=False, encoding="utf-8-sig")

    key_cols = ["target", "distance_m", "sample_index"]
    dup_key = df[df.duplicated(key_cols, keep=False)].copy()
    rows.append({
        "test": "target_distance_sample_unique",
        "status": "pass" if len(dup_key) == 0 else "fail",
        "value": len(dup_key),
        "details": "Duplicate target-distance-sample keys indicate repeated synchronized tuples.",
    })
    if len(dup_key) > 0:
        dup_key.to_csv(out_dir / "fail_duplicate_target_distance_sample.csv", index=False, encoding="utf-8-sig")

    for sensor in SENSORS:
        tmp = df[["split", "tuple_id", sensor]].copy().rename(columns={sensor: "path"})
        by_path = tmp.groupby("path")["split"].nunique().reset_index(name="n_splits")
        overlap_paths = by_path[by_path["n_splits"] > 1]
        rows.append({
            "test": f"path_overlap_across_splits_{safe_name(sensor)}",
            "status": "pass" if len(overlap_paths) == 0 else "fail",
            "value": len(overlap_paths),
            "details": f"Exact same {sensor} image path appears in more than one split.",
        })
        if len(overlap_paths) > 0:
            tmp[tmp["path"].isin(overlap_paths["path"])].to_csv(
                out_dir / f"fail_path_overlap_{safe_name(sensor)}.csv", index=False, encoding="utf-8-sig"
            )

    split_counts = df.groupby("split").size().reindex(split_order(), fill_value=0).reset_index(name="n")
    split_counts["fraction"] = split_counts["n"] / len(df)
    split_counts.to_csv(out_dir / "split_counts.csv", index=False, encoding="utf-8-sig")

    df.groupby(["split", "target"]).size().reset_index(name="n").to_csv(
        out_dir / "split_by_target_counts.csv", index=False, encoding="utf-8-sig"
    )
    df.groupby(["split", "target", "distance_m"]).size().reset_index(name="n").to_csv(
        out_dir / "split_by_target_distance_counts.csv", index=False, encoding="utf-8-sig"
    )

    for sensor in SENSORS:
        inferred = df[sensor].astype(str).map(infer_sensor_from_path)
        mismatch = df[inferred != sensor].copy()
        rows.append({
            "test": f"sensor_path_parser_consistency_{safe_name(sensor)}",
            "status": "pass" if len(mismatch) == 0 else "fail",
            "value": len(mismatch),
            "details": f"Rows whose path was not parsed as {sensor}.",
        })
        if len(mismatch) > 0:
            mismatch[["tuple_id", "split", "target", "distance_m", "sample_index", sensor]].to_csv(
                out_dir / f"fail_sensor_parser_{safe_name(sensor)}.csv", index=False, encoding="utf-8-sig"
            )

    audit = pd.DataFrame(rows)
    audit.to_csv(out_dir / "audit_split_integrity.csv", index=False, encoding="utf-8-sig")
    write_markdown(audit, out_dir / "audit_split_integrity.md")
    return audit


def audit_repetition_proximity(df: pd.DataFrame, out_dir: Path) -> pd.DataFrame:
    train = df[df["split"] == "train"].copy()
    eval_df = df[df["split"].isin(["val", "test"])].copy()

    train_groups = {
        (target, int(distance)): sorted(g["sample_index"].astype(int).tolist())
        for (target, distance), g in train.groupby(["target", "distance_m"])
    }

    rows = []
    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Repetition-proximity audit"):
        key = (row["target"], int(row["distance_m"]))
        train_indices = train_groups.get(key, [])
        if not train_indices:
            nearest_gap = np.nan
            nearest_train_index = np.nan
        else:
            x = int(row["sample_index"])
            diffs = [abs(x - t) for t in train_indices]
            j = int(np.argmin(diffs))
            nearest_gap = int(diffs[j])
            nearest_train_index = int(train_indices[j])
        rows.append({
            "tuple_id": row["tuple_id"],
            "split": row["split"],
            "target": row["target"],
            "distance_m": int(row["distance_m"]),
            "sample_index": int(row["sample_index"]),
            "nearest_train_sample_index_same_target_distance": nearest_train_index,
            "nearest_train_gap_same_target_distance": nearest_gap,
        })

    prox = pd.DataFrame(rows)
    prox.to_csv(out_dir / "repetition_proximity_eval_to_train.csv", index=False, encoding="utf-8-sig")

    summary = prox.groupby("split")["nearest_train_gap_same_target_distance"].agg(
        n="count",
        mean="mean",
        median="median",
        min="min",
        q05=lambda x: np.nanquantile(x, 0.05),
        q25=lambda x: np.nanquantile(x, 0.25),
        q75=lambda x: np.nanquantile(x, 0.75),
        max="max",
    ).reset_index()
    summary.to_csv(out_dir / "repetition_proximity_summary.csv", index=False, encoding="utf-8-sig")
    write_markdown(summary, out_dir / "repetition_proximity_summary.md")

    fig, ax = plt.subplots(figsize=(8, 5))
    for split in ["val", "test"]:
        vals = prox.loc[prox["split"] == split, "nearest_train_gap_same_target_distance"].dropna().astype(int)
        if len(vals) > 0:
            ax.hist(vals, bins=30, alpha=0.5, label=split)
    ax.set_xlabel("Nearest training repetition gap within same target-distance group")
    ax.set_ylabel("Number of evaluation tuples")
    ax.set_title("Weak leakage risk: evaluation repetitions near training repetitions")
    ax.legend()
    fig.tight_layout()
    fig.savefig(out_dir / "repetition_proximity_histogram.png", dpi=300)
    plt.close(fig)

    return summary


def audit_hash_duplicates(df: pd.DataFrame, out_dir: Path, cfg: AuditConfig) -> pd.DataFrame:
    rows = []
    file_rows = []
    for sensor in SENSORS:
        tmp = df[["tuple_id", "split", "target", "distance_m", "sample_index", sensor]].copy()
        tmp = tmp.rename(columns={sensor: "path"})
        tmp["sensor"] = sensor
        file_rows.append(tmp)
    files = pd.concat(file_rows, ignore_index=True)
    files["path"] = files["path"].astype(str)
    files = files.drop_duplicates("path").reset_index(drop=True)

    if cfg.max_hash_files and 0 < cfg.max_hash_files < len(files):
        files = files.sample(cfg.max_hash_files, random_state=cfg.seed).reset_index(drop=True)

    hash_records = []
    for _, row in tqdm(files.iterrows(), total=len(files), desc="Hashing images"):
        p = Path(row["path"])
        rec = row.to_dict()
        rec["exists"] = p.exists()
        rec["size_bytes"] = p.stat().st_size if p.exists() else np.nan
        if p.exists() and cfg.hash_images:
            rec["md5"] = md5_file(p)
            rec["average_hash"] = average_hash(p)
        else:
            rec["md5"] = None
            rec["average_hash"] = None
        hash_records.append(rec)

    hashes = pd.DataFrame(hash_records)
    hashes.to_csv(out_dir / "image_hash_index.csv", index=False, encoding="utf-8-sig")

    missing_files = hashes[~hashes["exists"]].copy()
    rows.append({
        "test": "image_paths_exist",
        "status": "pass" if len(missing_files) == 0 else "fail",
        "value": len(missing_files),
        "details": "Missing image files referenced by the split table.",
    })
    if len(missing_files) > 0:
        missing_files.to_csv(out_dir / "fail_missing_image_files.csv", index=False, encoding="utf-8-sig")

    if cfg.hash_images:
        for sensor in SENSORS:
            g = hashes[(hashes["sensor"] == sensor) & hashes["md5"].notna()].copy()

            by_hash = g.groupby("md5")["split"].nunique().reset_index(name="n_splits")
            cross = by_hash[by_hash["n_splits"] > 1]
            rows.append({
                "test": f"md5_duplicate_across_splits_{safe_name(sensor)}",
                "status": "pass" if len(cross) == 0 else "fail",
                "value": len(cross),
                "details": "Exact byte-identical images found in more than one split.",
            })
            if len(cross) > 0:
                g[g["md5"].isin(cross["md5"])].to_csv(
                    out_dir / f"fail_md5_duplicates_across_splits_{safe_name(sensor)}.csv",
                    index=False,
                    encoding="utf-8-sig",
                )

            by_ahash = g[g["average_hash"].notna()].groupby("average_hash")["split"].nunique().reset_index(name="n_splits")
            cross_a = by_ahash[by_ahash["n_splits"] > 1]
            rows.append({
                "test": f"average_hash_collision_across_splits_{safe_name(sensor)}",
                "status": "pass" if len(cross_a) == 0 else "warning",
                "value": len(cross_a),
                "details": "Perceptual average-hash collisions across splits. Inspect manually; not necessarily leakage.",
            })
            if len(cross_a) > 0:
                g[g["average_hash"].isin(cross_a["average_hash"])].to_csv(
                    out_dir / f"warning_average_hash_collisions_{safe_name(sensor)}.csv",
                    index=False,
                    encoding="utf-8-sig",
                )
    else:
        rows.append({
            "test": "image_hashing_skipped",
            "status": "warning",
            "value": len(files),
            "details": "Run with --hash-images to compute MD5 and perceptual hashes.",
        })

    audit = pd.DataFrame(rows)
    audit.to_csv(out_dir / "audit_hash_duplicates.csv", index=False, encoding="utf-8-sig")
    write_markdown(audit, out_dir / "audit_hash_duplicates.md")
    return audit


def metadata_only_shortcut_test(df: pd.DataFrame, out_dir: Path, seed: int = 42) -> pd.DataFrame:
    train = df[df["split"] == "train"].copy()
    test = df[df["split"] == "test"].copy()

    X_train = train[["distance_m", "sample_index"]]
    X_test = test[["distance_m", "sample_index"]]
    y_train = train["label"].astype(int).to_numpy()
    y_test = test["label"].astype(int).to_numpy()

    try:
        clf = LogisticRegression(max_iter=1000, solver="lbfgs", random_state=seed)
        clf.fit(X_train, y_train)
        pred = clf.predict(X_test)
        model_status = "ok_logistic_regression"
    except TypeError as e:
        print(f"[WARN] LogisticRegression compatibility issue: {e}")
        clf = LogisticRegression(max_iter=1000)
        clf.fit(X_train, y_train)
        pred = clf.predict(X_test)
        model_status = "ok_logistic_regression_minimal"
    except Exception as e:
        print(f"[WARN] Metadata-only shortcut probe failed: {e}")
        values, counts = np.unique(y_train, return_counts=True)
        majority = values[int(np.argmax(counts))]
        pred = np.full_like(y_test, fill_value=majority)
        model_status = "fallback_majority_class"

    result = pd.DataFrame([{ "probe": "metadata_only_distance_sample_index", "model_status": model_status, **compute_basic_metrics(y_test, pred) }])
    result.to_csv(out_dir / "metadata_only_shortcut_test.csv", index=False, encoding="utf-8-sig")
    write_markdown(result, out_dir / "metadata_only_shortcut_test.md")
    return result


def plot_split_heatmaps(df: pd.DataFrame, out_dir: Path) -> None:
    heat_dir = ensure_dir(out_dir / "split_heatmaps")
    for split in split_order():
        g = df[df["split"] == split].groupby(["target", "distance_m"]).size().reset_index(name="n")
        pivot = g.pivot(index="target", columns="distance_m", values="n").reindex(TARGETS).fillna(0)
        fig, ax = plt.subplots(figsize=(10, 4.5))
        im = ax.imshow(pivot.to_numpy(dtype=float), aspect="auto")
        ax.set_title(f"Split distribution - {split}")
        ax.set_xlabel("Distance (m)")
        ax.set_ylabel("Target")
        ax.set_xticks(range(len(pivot.columns)))
        ax.set_xticklabels(pivot.columns)
        ax.set_yticks(range(len(pivot.index)))
        ax.set_yticklabels(pivot.index)
        for i in range(pivot.shape[0]):
            for j in range(pivot.shape[1]):
                ax.text(j, i, str(int(pivot.iloc[i, j])), ha="center", va="center", fontsize=7)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        fig.tight_layout()
        fig.savefig(heat_dir / f"split_heatmap_{split}.png", dpi=300)
        plt.close(fig)


# ============================================================
# 2. ROBUST PROTOCOL GENERATION
# ============================================================

def save_protocol_split(protocol_dir: Path, name: str, protocol_df: pd.DataFrame) -> None:
    out_csv = protocol_dir / f"{safe_name(name)}.csv"
    protocol_df.to_csv(out_csv, index=False, encoding="utf-8-sig")

    summary = protocol_df.groupby(["split", "target", "distance_m"]).size().reset_index(name="n")
    summary.to_csv(protocol_dir / f"{safe_name(name)}_summary_by_target_distance.csv", index=False, encoding="utf-8-sig")


def stratified_validation_from_training(p: pd.DataFrame, seed: int, val_fraction: float = 0.15) -> pd.DataFrame:
    p = p.copy()
    train_idx = p.index[p["split"] == "train"].to_numpy()
    train_part = p.loc[train_idx]
    if len(train_part) == 0:
        return p

    strat = train_part["target"].astype(str) + "_" + train_part["distance_m"].astype(str)
    try:
        _, val_local = train_test_split(
            np.arange(len(train_part)), test_size=val_fraction, random_state=seed, stratify=strat
        )
    except ValueError:
        _, val_local = train_test_split(
            np.arange(len(train_part)), test_size=val_fraction, random_state=seed, stratify=train_part["target"]
        )
    p.loc[train_part.index[val_local], "split"] = "val"
    return p


def make_leave_one_distance_protocols(df: pd.DataFrame, protocol_dir: Path, seed: int = 42) -> pd.DataFrame:
    rows = []
    for held_distance in sorted(df["distance_m"].unique()):
        p = df.copy()
        p["split"] = "train"
        p.loc[p["distance_m"] == held_distance, "split"] = "test"
        p = stratified_validation_from_training(p, seed=seed, val_fraction=0.15)
        name = f"leave_one_distance_out_{held_distance}m"
        save_protocol_split(protocol_dir, name, p)
        rows.append({
            "protocol_family": "leave_one_distance_out",
            "protocol": name,
            "held_out_distance_m": int(held_distance),
            "held_out_distances_m": str(int(held_distance)),
            "held_out_target": "",
            "train_n": int((p["split"] == "train").sum()),
            "val_n": int((p["split"] == "val").sum()),
            "test_n": int((p["split"] == "test").sum()),
            "test_ood_n": 0,
            "file": f"{safe_name(name)}.csv",
        })
    return pd.DataFrame(rows)


def make_repetition_block_protocols(df: pd.DataFrame, protocol_dir: Path) -> pd.DataFrame:
    """Create robust blocked-repetition protocols using sample_index blocks.

    This avoids hard-coding 1..500 and works if the dataset uses 0..499 or another
    contiguous repetition index convention.
    """
    unique_idx = sorted(df["sample_index"].astype(int).unique())
    n = len(unique_idx)
    block = max(1, int(round(0.15 * n)))

    first_test = set(unique_idx[:block])
    first_val = set(unique_idx[block:2 * block])

    last_test = set(unique_idx[-block:])
    last_val = set(unique_idx[-2 * block:-block])

    mid_start = max(0, (n - block) // 2)
    mid_test = set(unique_idx[mid_start:mid_start + block])
    mid_val = set(unique_idx[mid_start + block:mid_start + 2 * block])
    if len(mid_val) < block:
        mid_val = set(unique_idx[max(0, mid_start - block):mid_start])

    protocols = [
        ("blocked_repetition_first_15pct_test_next_15pct_val", first_test, first_val),
        ("blocked_repetition_last_15pct_test_prev_15pct_val", last_test, last_val),
        ("blocked_repetition_middle_15pct_test_adjacent_15pct_val", mid_test, mid_val),
    ]

    rows = []
    for name, test_set, val_set in protocols:
        p = df.copy()
        p["split"] = "train"
        p.loc[p["sample_index"].isin(val_set), "split"] = "val"
        p.loc[p["sample_index"].isin(test_set), "split"] = "test"
        save_protocol_split(protocol_dir, name, p)
        rows.append({
            "protocol_family": "blocked_repetition",
            "protocol": name,
            "held_out_distance_m": "",
            "held_out_distances_m": "",
            "held_out_target": "",
            "train_n": int((p["split"] == "train").sum()),
            "val_n": int((p["split"] == "val").sum()),
            "test_n": int((p["split"] == "test").sum()),
            "test_ood_n": 0,
            "file": f"{safe_name(name)}.csv",
        })
    return pd.DataFrame(rows)


def make_range_block_protocols(df: pd.DataFrame, protocol_dir: Path, seed: int = 42) -> pd.DataFrame:
    blocks = {
        "range_holdout_near_2_10m": list(range(2, 12, 2)),
        "range_holdout_mid_12_20m": list(range(12, 22, 2)),
        "range_holdout_far_22_30m": list(range(22, 32, 2)),
    }
    rows = []
    for name, distances in blocks.items():
        p = df.copy()
        p["split"] = "train"
        p.loc[p["distance_m"].isin(distances), "split"] = "test"
        p = stratified_validation_from_training(p, seed=seed, val_fraction=0.15)
        save_protocol_split(protocol_dir, name, p)
        rows.append({
            "protocol_family": "range_holdout",
            "protocol": name,
            "held_out_distance_m": "",
            "held_out_distances_m": ",".join(map(str, distances)),
            "held_out_target": "",
            "train_n": int((p["split"] == "train").sum()),
            "val_n": int((p["split"] == "val").sum()),
            "test_n": int((p["split"] == "test").sum()),
            "test_ood_n": 0,
            "file": f"{safe_name(name)}.csv",
        })
    return pd.DataFrame(rows)


def make_leave_one_target_ood_protocols(df: pd.DataFrame, protocol_dir: Path, seed: int = 42) -> pd.DataFrame:
    rows = []
    for held_target in TARGETS:
        p = df.copy()
        p["split"] = "train"
        p.loc[p["target"] == held_target, "split"] = "test_ood"
        p = stratified_validation_from_training(p, seed=seed, val_fraction=0.15)
        name = f"leave_one_target_out_ood_{safe_name(held_target)}"
        save_protocol_split(protocol_dir, name, p)
        rows.append({
            "protocol_family": "leave_one_target_out_ood",
            "protocol": name,
            "held_out_distance_m": "",
            "held_out_distances_m": "",
            "held_out_target": held_target,
            "train_n": int((p["split"] == "train").sum()),
            "val_n": int((p["split"] == "val").sum()),
            "test_n": 0,
            "test_ood_n": int((p["split"] == "test_ood").sum()),
            "file": f"{safe_name(name)}.csv",
        })
    return pd.DataFrame(rows)


def generate_robust_protocols(df: pd.DataFrame, out_dir: Path, seed: int = 42) -> pd.DataFrame:
    protocol_dir = ensure_dir(out_dir / "robust_split_protocols")
    tables = [
        make_leave_one_distance_protocols(df, protocol_dir, seed),
        make_repetition_block_protocols(df, protocol_dir),
        make_range_block_protocols(df, protocol_dir, seed),
        make_leave_one_target_ood_protocols(df, protocol_dir, seed),
    ]
    index = pd.concat(tables, ignore_index=True, sort=False)
    index.to_csv(protocol_dir / "protocol_index.csv", index=False, encoding="utf-8-sig")
    write_markdown(index, protocol_dir / "protocol_index.md")
    write_latex(
        index,
        protocol_dir / "latex_protocol_index.tex",
        caption="Robust evaluation protocols generated for distance, repetition, range and open-set generalization analysis.",
        label="tab:robust_protocols",
    )
    return index


# ============================================================
# 3. PREDICTION POST-HOC ANALYSES
# ============================================================

def find_prediction_files(predictions_root: Path) -> Dict[str, Path]:
    files = {}
    if predictions_root.exists():
        for p in predictions_root.glob("*/predictions_test.csv"):
            files[p.parent.name] = p
    return files


def load_predictions(predictions_root: Path) -> Dict[str, pd.DataFrame]:
    out = {}
    for method_dir_name, p in find_prediction_files(predictions_root).items():
        df = pd.read_csv(p)
        if "method" not in df.columns:
            df["method"] = method_dir_name
        out[method_dir_name] = df
    return out


def bootstrap_metric_ci(y_true: np.ndarray, y_pred: np.ndarray, metric_fn, n_boot: int, seed: int) -> Tuple[float, float, float]:
    rng = np.random.default_rng(seed)
    n = len(y_true)
    vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        vals.append(metric_fn(y_true[idx], y_pred[idx]))
    vals = np.asarray(vals, dtype=float)
    point = metric_fn(y_true, y_pred)
    return float(point), float(np.quantile(vals, 0.025)), float(np.quantile(vals, 0.975))


def prediction_bootstrap_cis(preds: Dict[str, pd.DataFrame], out_dir: Path, n_boot: int, seed: int) -> pd.DataFrame:
    rows = []
    metrics = {
        "accuracy": lambda a, b: accuracy_score(a, b),
        "balanced_accuracy": lambda a, b: balanced_accuracy_score(a, b),
        "macro_f1": lambda a, b: f1_score(a, b, average="macro", zero_division=0),
        "weighted_f1": lambda a, b: f1_score(a, b, average="weighted", zero_division=0),
        "kappa": lambda a, b: cohen_kappa_score(a, b),
    }
    for method, df in preds.items():
        y_true = df["y_true"].to_numpy(dtype=int)
        y_pred = df["y_pred"].to_numpy(dtype=int)
        for metric_name, fn in metrics.items():
            point, lo, hi = bootstrap_metric_ci(y_true, y_pred, fn, n_boot=n_boot, seed=seed)
            rows.append({
                "method": method,
                "metric": metric_name,
                "point": point,
                "ci95_low": lo,
                "ci95_high": hi,
                "n_bootstrap": n_boot,
            })
    res = pd.DataFrame(rows)
    res.to_csv(out_dir / "bootstrap_confidence_intervals.csv", index=False, encoding="utf-8-sig")
    write_markdown(res, out_dir / "bootstrap_confidence_intervals.md")
    return res


def exact_mcnemar(correct_a: np.ndarray, correct_b: np.ndarray) -> Dict[str, float]:
    b = int(np.sum(correct_a & ~correct_b))
    c = int(np.sum(~correct_a & correct_b))
    n = b + c
    if n == 0:
        p = 1.0
    elif SCIPY_AVAILABLE:
        p = float(binomtest(min(b, c), n=n, p=0.5, alternative="two-sided").pvalue)
    else:
        stat = (abs(b - c) - 1) ** 2 / max(n, 1)
        p = float(math.exp(-0.5 * stat))
    return {"b_a_correct_b_wrong": b, "c_a_wrong_b_correct": c, "discordant": n, "p_value": p}


def paired_mcnemar_tests(preds: Dict[str, pd.DataFrame], out_dir: Path) -> pd.DataFrame:
    rows = []
    for a, b in itertools.combinations(sorted(preds.keys()), 2):
        da = preds[a][["tuple_id", "correct"]].rename(columns={"correct": "correct_a"})
        db = preds[b][["tuple_id", "correct"]].rename(columns={"correct": "correct_b"})
        m = da.merge(db, on="tuple_id", how="inner")
        if len(m) == 0:
            continue
        rows.append({"method_a": a, "method_b": b, "n_paired": len(m), **exact_mcnemar(m["correct_a"].to_numpy(bool), m["correct_b"].to_numpy(bool))})
    res = pd.DataFrame(rows)
    res.to_csv(out_dir / "paired_mcnemar_tests.csv", index=False, encoding="utf-8-sig")
    write_markdown(res, out_dir / "paired_mcnemar_tests.md")
    return res


def expected_calibration_error(conf: np.ndarray, correct: np.ndarray, n_bins: int = 15) -> Tuple[float, pd.DataFrame]:
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    rows = []
    ece = 0.0
    n = len(conf)
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (conf >= lo) & (conf <= hi) if i == n_bins - 1 else (conf >= lo) & (conf < hi)
        count = int(mask.sum())
        if count == 0:
            acc = np.nan
            avg_conf = np.nan
            gap = np.nan
        else:
            acc = float(correct[mask].mean())
            avg_conf = float(conf[mask].mean())
            gap = abs(acc - avg_conf)
            ece += (count / n) * gap
        rows.append({"bin": i, "bin_low": lo, "bin_high": hi, "count": count, "avg_confidence": avg_conf, "accuracy": acc, "abs_gap": gap})
    return float(ece), pd.DataFrame(rows)


def calibration_and_risk_analysis(preds: Dict[str, pd.DataFrame], out_dir: Path, n_bins: int = 15) -> pd.DataFrame:
    rows = []
    rel_dir = ensure_dir(out_dir / "calibration_reliability")
    risk_dir = ensure_dir(out_dir / "risk_coverage")

    for method, df in preds.items():
        proba_cols = get_proba_columns(df)
        if not proba_cols:
            continue
        y_true = df["y_true"].to_numpy(dtype=int)
        y_pred = df["y_pred"].to_numpy(dtype=int)
        proba = df[proba_cols].to_numpy(dtype=float)
        conf = np.max(proba, axis=1)
        correct = (y_true == y_pred)

        ece, bin_df = expected_calibration_error(conf, correct, n_bins=n_bins)
        bin_df["method"] = method
        bin_df.to_csv(rel_dir / f"reliability_bins_{safe_name(method)}.csv", index=False, encoding="utf-8-sig")

        fig, ax = plt.subplots(figsize=(5.5, 5.5))
        plot_df = bin_df.dropna(subset=["avg_confidence", "accuracy"])
        ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1)
        ax.plot(plot_df["avg_confidence"], plot_df["accuracy"], marker="o")
        ax.set_xlabel("Confidence")
        ax.set_ylabel("Accuracy")
        ax.set_title(f"Reliability diagram - {method}")
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        fig.tight_layout()
        fig.savefig(rel_dir / f"reliability_diagram_{safe_name(method)}.png", dpi=300)
        plt.close(fig)

        try:
            nll = float(log_loss(y_true, proba, labels=list(range(proba.shape[1]))))
        except Exception:
            nll = np.nan
        y_onehot = np.eye(proba.shape[1])[y_true]
        brier = float(np.mean(np.sum((proba - y_onehot) ** 2, axis=1)))

        order = np.argsort(-conf)
        sorted_correct = correct[order]
        coverages = np.arange(1, len(correct) + 1) / len(correct)
        risks = 1.0 - np.cumsum(sorted_correct) / np.arange(1, len(correct) + 1)
        rc = pd.DataFrame({"coverage": coverages, "risk": risks})
        rc.to_csv(risk_dir / f"risk_coverage_{safe_name(method)}.csv", index=False, encoding="utf-8-sig")
        aurc = safe_trapezoid(risks, coverages)

        fig, ax = plt.subplots(figsize=(6, 5))
        ax.plot(rc["coverage"], rc["risk"])
        ax.set_xlabel("Coverage")
        ax.set_ylabel("Selective risk")
        ax.set_title(f"Risk-coverage curve - {method}")
        ax.set_xlim(0, 1)
        ax.set_ylim(0, max(0.01, float(np.nanmax(risks)) * 1.05))
        fig.tight_layout()
        fig.savefig(risk_dir / f"risk_coverage_{safe_name(method)}.png", dpi=300)
        plt.close(fig)

        rows.append({
            "method": method,
            "ece": ece,
            "nll": nll,
            "brier_multiclass": brier,
            "aurc": aurc,
            "mean_confidence": float(np.mean(conf)),
            "median_confidence": float(np.median(conf)),
            "min_confidence": float(np.min(conf)),
            "max_confidence": float(np.max(conf)),
        })

    res = pd.DataFrame(rows)
    res.to_csv(out_dir / "calibration_and_risk_summary.csv", index=False, encoding="utf-8-sig")
    write_markdown(res, out_dir / "calibration_and_risk_summary.md")
    return res


def target_distance_error_analysis(preds: Dict[str, pd.DataFrame], out_dir: Path) -> pd.DataFrame:
    rows = []
    for method, df in preds.items():
        for (target, distance), g in df.groupby(["true_label", "distance_m"]):
            y_true = g["y_true"].to_numpy(dtype=int)
            y_pred = g["y_pred"].to_numpy(dtype=int)
            rows.append({"method": method, "target": target, "distance_m": int(distance), "n": len(g), **compute_basic_metrics(y_true, y_pred)})
    res = pd.DataFrame(rows)
    res.to_csv(out_dir / "target_distance_metrics_from_predictions.csv", index=False, encoding="utf-8-sig")
    write_markdown(res, out_dir / "target_distance_metrics_from_predictions.md")
    return res


def gate_analysis(preds: Dict[str, pd.DataFrame], out_dir: Path) -> pd.DataFrame:
    rows = []
    gate_cols = [f"gate_{safe_name(s)}" for s in SENSORS]
    for method, df in preds.items():
        if not all(c in df.columns for c in gate_cols):
            continue
        gates = df[gate_cols].to_numpy(dtype=float)
        eps = 1e-12
        entropy = -np.sum(gates * np.log(gates + eps), axis=1) / np.log(gates.shape[1])
        tmp = df.copy()
        tmp["gate_entropy_norm"] = entropy
        tmp["dominant_gate_sensor"] = [SENSORS[int(i)] for i in np.argmax(gates, axis=1)]

        rec = {"method": method, "n": len(tmp), "mean_gate_entropy_norm": float(tmp["gate_entropy_norm"].mean())}
        for sensor, col in zip(SENSORS, gate_cols):
            rec[f"mean_gate_{safe_name(sensor)}"] = float(tmp[col].mean())
            rec[f"median_gate_{safe_name(sensor)}"] = float(tmp[col].median())
        rows.append(rec)

        tmp.groupby(["true_label", "distance_m"])[gate_cols + ["gate_entropy_norm"]].mean().reset_index().to_csv(
            out_dir / f"gate_by_target_distance_{safe_name(method)}.csv", index=False, encoding="utf-8-sig"
        )
        tmp.groupby(["true_label", "distance_m", "dominant_gate_sensor"]).size().reset_index(name="n").to_csv(
            out_dir / f"dominant_gate_counts_{safe_name(method)}.csv", index=False, encoding="utf-8-sig"
        )

    res = pd.DataFrame(rows)
    res.to_csv(out_dir / "gate_global_summary.csv", index=False, encoding="utf-8-sig")
    write_markdown(res, out_dir / "gate_global_summary.md")
    return res


def sensor_dependency_analysis(summary_csv: Path, out_dir: Path) -> Optional[pd.DataFrame]:
    if not summary_csv.exists():
        return None
    df = pd.read_csv(summary_csv)
    if not all(c in df.columns for c in ["method", "mask_CW", "mask_FMCW", "mask_RF", "macro_f1"]):
        return None

    rows = []
    for sensor_col, sensor_name in [("mask_RF", "RF"), ("mask_CW", "CW"), ("mask_FMCW", "FMCW")]:
        with_sensor = float(df[df[sensor_col] == 1]["macro_f1"].mean())
        without_sensor = float(df[df[sensor_col] == 0]["macro_f1"].mean())
        rows.append({
            "sensor": sensor_name,
            "mean_macro_f1_with_sensor": with_sensor,
            "mean_macro_f1_without_sensor": without_sensor,
            "delta_with_minus_without": with_sensor - without_sensor,
        })
    res = pd.DataFrame(rows)
    res.to_csv(out_dir / "sensor_dependency_summary.csv", index=False, encoding="utf-8-sig")
    write_markdown(res, out_dir / "sensor_dependency_summary.md")
    return res


# ============================================================
# 4. FINAL AUDIT SUMMARY
# ============================================================

def create_final_audit_summary(out_dir: Path) -> pd.DataFrame:
    candidate_files = ["audit_split_integrity.csv", "audit_hash_duplicates.csv"]
    rows = []
    for fname in candidate_files:
        p = out_dir / fname
        if p.exists():
            df = pd.read_csv(p)
            for _, r in df.iterrows():
                rows.append({
                    "source_file": fname,
                    "test": r.get("test", ""),
                    "status": r.get("status", ""),
                    "value": r.get("value", ""),
                    "details": r.get("details", ""),
                })
    summary = pd.DataFrame(rows)
    if len(summary) > 0:
        summary.to_csv(out_dir / "final_audit_summary.csv", index=False, encoding="utf-8-sig")
        write_markdown(summary, out_dir / "final_audit_summary.md")
    return summary


def write_paper_methods_template(out_dir: Path) -> None:
    text = r"""
\subsection{Data Leakage and Robustness Auditing}

To ensure that the near-perfect random-split performance was not caused by data leakage, we audited the synchronized tuple index, split file, sensor paths, image hashes, model predictions, and protocol structure. We verified the uniqueness of tuple identifiers and target--distance--sample keys, checked that no exact sensor-image path appeared in more than one partition, and computed byte-level MD5 hashes and perceptual average hashes for cross-split duplicate detection. We also quantified a weaker leakage risk by measuring, for each validation/test tuple, the nearest training repetition within the same target--distance group.

In addition to the random stratified split, we generated stricter protocols: leave-one-distance-out, blocked-repetition holdout, near/mid/far range holdout, and leave-one-target-out open-set evaluation. These protocols are intended to test distance generalization, temporal/repetition robustness, range-shift robustness, and open-set behavior. Post-hoc analyses include bootstrap confidence intervals, paired McNemar tests, calibration metrics, risk--coverage curves, target--distance error summaries, and reliability-gate statistics.
""".strip()
    (out_dir / "paper_methods_leakage_robustness_template.tex").write_text(text, encoding="utf-8")


# ============================================================
# 5. PAPER TABLE PRINTING
# ============================================================

def save_table(df: pd.DataFrame, table_dir: Path, name: str, caption: str = "", label: str = "") -> None:
    ensure_dir(table_dir)
    df.to_csv(table_dir / f"{name}.csv", index=False, encoding="utf-8-sig")
    write_markdown(df, table_dir / f"{name}.md")
    write_latex(df, table_dir / f"{name}.tex", caption=caption, label=label)


def show_table(title: str, df: pd.DataFrame, table_dir: Path, name: str, caption: str = "", label: str = "") -> None:
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)
    if HAS_DISPLAY:
        display(Markdown(f"### {title}"))
        display(df)
    else:
        print(df.to_string(index=False))
    save_table(df, table_dir, name, caption, label)
    print(f"[SAVED] {table_dir / (name + '.csv')}")
    print(f"[SAVED] {table_dir / (name + '.md')}")
    print(f"[SAVED] {table_dir / (name + '.tex')}")


def print_paper_tables(cfg: AuditConfig) -> None:
    ensure_dir(cfg.table_dir)
    pred_dir = cfg.audit_dir / "prediction_posthoc_analysis"
    order = ["All sensors", "CW + RF", "FMCW + RF", "RF only", "CW + FMCW", "FMCW only", "CW only"]

    # Table 1.
    main_results = read_csv_if_exists(cfg.adv_dir / "advanced_fusion_results_summary.csv")
    if main_results is not None:
        t1 = main_results.copy()
        t1["Setting"] = t1["method"].map(pretty_method_name)
        t1["CW"] = t1["mask_CW"].map({1: "Yes", 0: "No"})
        t1["FMCW"] = t1["mask_FMCW"].map({1: "Yes", 0: "No"})
        t1["RF"] = t1["mask_RF"].map({1: "Yes", 0: "No"})
        all_macro = float(t1.loc[t1["method"] == "Advanced-Fusion-All", "macro_f1"].iloc[0]) if (t1["method"] == "Advanced-Fusion-All").any() else float(t1["macro_f1"].max())
        t1["Δ Macro-F1 vs All"] = t1["macro_f1"] - all_macro
        t1["_order"] = t1["Setting"].apply(lambda x: order.index(x) if x in order else 999)
        t1 = t1.sort_values("_order")[["Setting", "CW", "FMCW", "RF", "accuracy", "balanced_accuracy", "macro_precision", "macro_recall", "macro_f1", "weighted_f1", "kappa", "Δ Macro-F1 vs All"]]
        t1 = t1.rename(columns={"accuracy": "Accuracy", "balanced_accuracy": "Balanced Acc.", "macro_precision": "Macro Precision", "macro_recall": "Macro Recall", "macro_f1": "Macro-F1", "weighted_f1": "Weighted-F1", "kappa": "Kappa"})
        t1 = round_metrics(t1)
        show_table("Table 1 — Main classification performance and sensor-ablation results", t1, cfg.table_dir, "table_01_main_performance_sensor_ablation", "Main classification performance under sensor-ablation settings.", "tab:main_performance_sensor_ablation")

    # Table 2.
    audit_summary = read_csv_if_exists(cfg.audit_dir / "final_audit_summary.csv")
    if audit_summary is not None:
        t2 = audit_summary[["source_file", "test", "status", "value", "details"]].copy()
        t2 = t2.rename(columns={"source_file": "Source", "test": "Audit check", "status": "Status", "value": "Value", "details": "Interpretation"})
        status_order = {"fail": 0, "warning": 1, "pass": 2, "info": 3}
        t2["_order"] = t2["Status"].map(status_order).fillna(9)
        t2 = t2.sort_values(["_order", "Source", "Audit check"]).drop(columns=["_order"])
        show_table("Table 2 — Data leakage and split-integrity audit summary", t2, cfg.table_dir, "table_02_data_leakage_audit_summary", "Summary of leakage and split-integrity checks.", "tab:data_leakage_audit")

    # Table 3.
    rep_prox = read_csv_if_exists(cfg.audit_dir / "repetition_proximity_summary.csv")
    if rep_prox is not None:
        t3 = rep_prox.rename(columns={"split": "Split", "n": "N", "mean": "Mean nearest gap", "median": "Median nearest gap", "min": "Min", "q05": "Q05", "q25": "Q25", "q75": "Q75", "max": "Max"})
        t3 = round_metrics(t3)
        show_table("Table 3 — Repetition-proximity analysis", t3, cfg.table_dir, "table_03_repetition_proximity", "Nearest training repetition gap for validation and test samples.", "tab:repetition_proximity")

    # Table 4.
    shortcut = read_csv_if_exists(cfg.audit_dir / "metadata_only_shortcut_test.csv")
    if shortcut is not None:
        t4 = shortcut.rename(columns={"probe": "Probe", "model_status": "Status", "accuracy": "Accuracy", "balanced_accuracy": "Balanced Acc.", "macro_precision": "Macro Precision", "macro_recall": "Macro Recall", "macro_f1": "Macro-F1", "weighted_f1": "Weighted-F1", "kappa": "Kappa"})
        t4 = round_metrics(t4)
        show_table("Table 4 — Metadata-only shortcut probe", t4, cfg.table_dir, "table_04_metadata_only_shortcut_probe", "Metadata-only shortcut probe using distance and sample index.", "tab:metadata_shortcut_probe")

    # Table 5.
    bootstrap_ci = read_csv_if_exists(pred_dir / "bootstrap_confidence_intervals.csv")
    if bootstrap_ci is not None:
        t5 = bootstrap_ci[bootstrap_ci["metric"].isin(["macro_f1", "balanced_accuracy", "kappa"])].copy()
        t5["Setting"] = t5["method"].map(pretty_method_name)
        t5["Metric"] = t5["metric"].map({"macro_f1": "Macro-F1", "balanced_accuracy": "Balanced Acc.", "kappa": "Kappa"})
        t5["95% CI"] = "[" + t5["ci95_low"].round(4).astype(str) + ", " + t5["ci95_high"].round(4).astype(str) + "]"
        t5 = t5[["Setting", "Metric", "point", "95% CI", "n_bootstrap"]].rename(columns={"point": "Point estimate", "n_bootstrap": "Bootstrap resamples"})
        t5["Point estimate"] = t5["Point estimate"].round(4)
        metric_order = ["Macro-F1", "Balanced Acc.", "Kappa"]
        t5["_o1"] = t5["Setting"].apply(lambda x: order.index(x) if x in order else 999)
        t5["_o2"] = t5["Metric"].apply(lambda x: metric_order.index(x) if x in metric_order else 999)
        t5 = t5.sort_values(["_o1", "_o2"]).drop(columns=["_o1", "_o2"])
        show_table("Table 5 — Bootstrap 95% confidence intervals", t5, cfg.table_dir, "table_05_bootstrap_confidence_intervals", "Bootstrap confidence intervals for main metrics.", "tab:bootstrap_ci")

    # Table 6.
    calib = read_csv_if_exists(pred_dir / "calibration_and_risk_summary.csv")
    if calib is not None:
        t6 = calib.copy()
        t6["Setting"] = t6["method"].map(pretty_method_name)
        t6 = t6[["Setting", "ece", "nll", "brier_multiclass", "aurc", "mean_confidence", "median_confidence", "min_confidence"]]
        t6 = t6.rename(columns={"ece": "ECE", "nll": "NLL", "brier_multiclass": "Brier", "aurc": "AURC", "mean_confidence": "Mean conf.", "median_confidence": "Median conf.", "min_confidence": "Min conf."})
        t6["_order"] = t6["Setting"].apply(lambda x: order.index(x) if x in order else 999)
        t6 = round_metrics(t6.sort_values("_order").drop(columns=["_order"]))
        show_table("Table 6 — Calibration and selective-risk summary", t6, cfg.table_dir, "table_06_calibration_risk_summary", "Calibration and selective-risk metrics.", "tab:calibration_risk")

    # Table 7.
    mcnemar = read_csv_if_exists(pred_dir / "paired_mcnemar_tests.csv")
    if mcnemar is not None and len(mcnemar) > 0:
        full_names = ["advanced_fusion_all", "Advanced-Fusion-All"]
        t7 = mcnemar[mcnemar["method_a"].isin(full_names) | mcnemar["method_b"].isin(full_names)].copy()
        if len(t7) > 0:
            t7["Compared setting"] = t7.apply(lambda r: r["method_b"] if r["method_a"] in full_names else r["method_a"], axis=1).map(pretty_method_name)
            t7 = t7[["Compared setting", "n_paired", "b_a_correct_b_wrong", "c_a_wrong_b_correct", "discordant", "p_value"]]
            t7 = t7.rename(columns={"n_paired": "Paired N", "b_a_correct_b_wrong": "b", "c_a_wrong_b_correct": "c", "discordant": "Discordant", "p_value": "p-value"})
            t7 = round_metrics(t7, cols=["p-value"], digits=6)
            show_table("Table 7 — Paired McNemar tests against full fusion", t7, cfg.table_dir, "table_07_mcnemar_vs_full_fusion", "Paired McNemar tests comparing full fusion against ablations.", "tab:mcnemar_full_fusion")

    # Table 8.
    gate = read_csv_if_exists(pred_dir / "gate_global_summary.csv")
    if gate is not None and len(gate) > 0:
        t8 = gate.copy()
        t8["Setting"] = t8["method"].map(pretty_method_name)
        keep_cols = ["Setting", "n", "mean_gate_entropy_norm", "mean_gate_cw_radar", "mean_gate_fmcw_radar", "mean_gate_rf_receiver", "median_gate_cw_radar", "median_gate_fmcw_radar", "median_gate_rf_receiver"]
        keep_cols = [c for c in keep_cols if c in t8.columns]
        t8 = t8[keep_cols]
        t8 = t8.rename(columns={"n": "N", "mean_gate_entropy_norm": "Mean gate entropy", "mean_gate_cw_radar": "Mean gate CW", "mean_gate_fmcw_radar": "Mean gate FMCW", "mean_gate_rf_receiver": "Mean gate RF", "median_gate_cw_radar": "Median gate CW", "median_gate_fmcw_radar": "Median gate FMCW", "median_gate_rf_receiver": "Median gate RF"})
        t8["_order"] = t8["Setting"].apply(lambda x: order.index(x) if x in order else 999)
        t8 = round_metrics(t8.sort_values("_order").drop(columns=["_order"]))
        show_table("Table 8 — Reliability-gate summary", t8, cfg.table_dir, "table_08_reliability_gate_summary", "Global reliability-gate statistics.", "tab:gate_summary")

    # Table 9.
    sensor_dep = read_csv_if_exists(cfg.audit_dir / "sensor_dependency_summary.csv")
    if sensor_dep is not None:
        t9 = round_metrics(sensor_dep)
        show_table("Table 9 — Sensor dependency summary", t9, cfg.table_dir, "table_09_sensor_dependency_summary", "Mean macro-F1 variation with and without each sensor.", "tab:sensor_dependency")

    # Table 10.
    protocol_index = read_csv_if_exists(cfg.audit_dir / "robust_split_protocols" / "protocol_index.csv")
    if protocol_index is not None:
        t10 = protocol_index.copy()
        keep = ["protocol_family", "protocol", "held_out_distance_m", "held_out_distances_m", "held_out_target", "train_n", "val_n", "test_n", "test_ood_n", "file"]
        keep = [c for c in keep if c in t10.columns]
        t10 = t10[keep]
        t10 = t10.rename(columns={"protocol_family": "Family", "protocol": "Protocol", "held_out_distance_m": "Held-out distance", "held_out_distances_m": "Held-out range", "held_out_target": "Held-out target", "train_n": "Train N", "val_n": "Val N", "test_n": "Test N", "test_ood_n": "OOD Test N", "file": "Split file"})
        show_table("Table 10 — Robust evaluation protocols", t10, cfg.table_dir, "table_10_robust_evaluation_protocols", "Additional protocols for distance, repetition, range and open-set evaluation.", "tab:robust_protocols")


# ============================================================
# MAIN
# ============================================================

def run_audit_and_tables(cfg: AuditConfig) -> None:
    ensure_dir(cfg.audit_dir)
    ensure_dir(cfg.table_dir)
    random.seed(cfg.seed)
    np.random.seed(cfg.seed)

    print("=" * 80)
    print("TSMS-Drone IEEE leakage, robustness and paper-table audit")
    print("=" * 80)
    print(f"ROOT           : {cfg.root}")
    print(f"ADV_DIR        : {cfg.adv_dir}")
    print(f"AUDIT_DIR      : {cfg.audit_dir}")
    print(f"TABLE_DIR      : {cfg.table_dir}")
    print(f"SYNC_CSV       : {cfg.sync_csv}")
    print(f"SPLIT_CSV      : {cfg.split_csv}")
    print(f"HASH_IMAGES    : {cfg.hash_images}")
    print(f"MAKE_PROTOCOLS : {cfg.make_protocols}")
    print(f"PRINT_TABLES   : {cfg.print_tables}")
    print("=" * 80)

    t0 = time.time()
    split_df = load_split_dataframe(cfg)

    audit_split_integrity(split_df, cfg.audit_dir)
    plot_split_heatmaps(split_df, cfg.audit_dir)
    audit_repetition_proximity(split_df, cfg.audit_dir)
    audit_hash_duplicates(split_df, cfg.audit_dir, cfg)
    metadata_only_shortcut_test(split_df, cfg.audit_dir, seed=cfg.seed)

    if cfg.make_protocols:
        generate_robust_protocols(split_df, cfg.audit_dir, seed=cfg.seed)

    preds = load_predictions(cfg.predictions_root)
    if preds:
        pred_dir = ensure_dir(cfg.audit_dir / "prediction_posthoc_analysis")
        prediction_bootstrap_cis(preds, pred_dir, n_boot=cfg.bootstrap, seed=cfg.seed)
        paired_mcnemar_tests(preds, pred_dir)
        calibration_and_risk_analysis(preds, pred_dir, n_bins=15)
        target_distance_error_analysis(preds, pred_dir)
        gate_analysis(preds, pred_dir)
    else:
        print(f"[WARN] No prediction files found under: {cfg.predictions_root}")

    sensor_dependency_analysis(cfg.adv_dir / "advanced_fusion_results_summary.csv", cfg.audit_dir)
    write_paper_methods_template(cfg.audit_dir)
    final_summary = create_final_audit_summary(cfg.audit_dir)

    manifest = {
        "root": str(cfg.root),
        "adv_dir": str(cfg.adv_dir),
        "audit_dir": str(cfg.audit_dir),
        "table_dir": str(cfg.table_dir),
        "sync_csv": str(cfg.sync_csv),
        "split_csv": str(cfg.split_csv),
        "hash_images": cfg.hash_images,
        "max_hash_files": cfg.max_hash_files,
        "make_protocols": cfg.make_protocols,
        "print_tables": cfg.print_tables,
        "bootstrap": cfg.bootstrap,
        "elapsed_sec": time.time() - t0,
        "n_final_audit_rows": int(len(final_summary)),
    }
    save_json(manifest, cfg.audit_dir / "audit_manifest.json")

    if cfg.print_tables:
        print_paper_tables(cfg)

    print("\nGenerated audit files:")
    for p in sorted(cfg.audit_dir.glob("*")):
        print(p)
    print("\nGenerated paper-table files:")
    for p in sorted(cfg.table_dir.glob("*")):
        print(p)
    print("=" * 80)
    print("[OK] IEEE audit, robust protocols and paper tables completed.")
    print("=" * 80)


def main() -> None:
    args = parse_args()
    cfg = build_config(args)
    run_audit_and_tables(cfg)


if __name__ == "__main__":
    main()


[INFO] Ignored unknown arguments, probably from Jupyter/IPython: ['--f=c:\\Users\\Luis\\AppData\\Roaming\\jupyter\\runtime\\kernel-v3040e410708fb7b5bf2eb7c62141f275456ffc0f8.json']
TSMS-Drone IEEE leakage, robustness and paper-table audit
ROOT           : D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)
ADV_DIR        : D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_advanced_fusion_improvements
AUDIT_DIR      : D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit
TABLE_DIR      : D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\_paper_results_tables
SYNC_CSV       : D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_advanced_fusion_improvements\synchronized_tuple_index_full.csv
SPLIT_CSV      : D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_advanced_fusion_improvements\tuple_splits_70_15_15_full.csv
HASH_IMAGES    : True
MAKE_PROTOCOLS : True
PR

Hashing images: 100%|██████████| 112500/112500 [10:24<00:00, 180.23it/s]



Table 1 — Main classification performance and sensor-ablation results


### Table 1 — Main classification performance and sensor-ablation results

,Setting,CW,FMCW,RF,Accuracy,Balanced Acc.,Macro Precision,Macro Recall,Macro-F1,Weighted-F1,Kappa,Δ Macro-F1 vs All
0,All sensors,Yes,Yes,Yes,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.0000
1,CW + RF,Yes,No,Yes,0.9998,0.9998,0.9998,0.9998,0.9998,0.9998,0.9998,-0.0002
2,FMCW + RF,No,Yes,Yes,0.9998,0.9998,0.9998,0.9998,0.9998,0.9998,0.9998,-0.0002
3,RF only,No,No,Yes,0.9993,0.9993,0.9993,0.9993,0.9993,0.9993,0.9991,-0.0007
4,CW + FMCW,Yes,Yes,No,0.9145,0.9145,0.9166,0.9145,0.9140,0.9140,0.8931,-0.0860
5,FMCW only,No,Yes,No,0.8418,0.8418,0.8431,0.8418,0.8384,0.8384,0.8022,-0.1616
6,CW only,Yes,No,No,0.6171,0.6171,0.6848,0.6171,0.6009,0.6009,0.5213,-0.3991


...

,sensor,mean_macro_f1_with_sensor,mean_macro_f1_without_sensor,delta_with_minus_without
0,RF,0.9997,0.7844,0.2153
1,CW,0.8787,0.9458,-0.0672
2,FMCW,0.9381,0.8667,0.0714


[SAVED] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\_paper_results_tables\table_09_sensor_dependency_summary.csv
[SAVED] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\_paper_results_tables\table_09_sensor_dependency_summary.md
[SAVED] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\_paper_results_tables\table_09_sensor_dependency_summary.tex

Table 10 — Robust evaluation protocols


### Table 10 — Robust evaluation protocols

,Family,Protocol,Held-out distance,Held-out range,Held-out target,Train N,Val N,Test N,OOD Test N,Split file
0,leave_one_distance_out,leave_one_distance_out_2m,2.0,2,NaN,29750,5250,2500,0,leave_one_distance_out_2m.csv
1,leave_one_distance_out,leave_one_distance_out_4m,4.0,4,NaN,29750,5250,2500,0,leave_one_distance_out_4m.csv
2,leave_one_distance_out,leave_one_distance_out_6m,6.0,6,NaN,29750,5250,2500,0,leave_one_distance_out_6m.csv
3,leave_one_distance_out,leave_one_distance_out_8m,8.0,8,NaN,29750,5250,2500,0,leave_one_distance_out_8m.csv
4,leave_one_distance_out,leave_one_distance_out_10m,10.0,10,NaN,29750,5250,2500,0,leave_one_distance_out_10m.csv
5,leave_one_distance_out,leave_one_distance_out_12m,12.0,12,NaN,29750,5250,2500,0,leave_one_distance_out_12m.csv
6,leave_one_distance_out,leave_one_distance_out_14m,14.0,14,NaN,29750,5250,2500,0,leave_one_distance_out_14m.csv
7,leave_one_distance_out,leave_one_distance_out_16m,16.0,16,NaN,29750,5250,2500,0,leave_one_distance_out_16m.csv
8,leave_one_distance_out,leave_one_distance_out_18m,18.0,18,NaN,29750,5250,2500,0,leave_one_distance_out_18m.csv
9,leave_one_distance_out,leave_one_distance_out_20m,20.0,20,NaN,29750,5250,2500,0,leave_one_distance_out_20m.csv


[SAVED] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\_paper_results_tables\table_10_robust_evaluation_protocols.csv
[SAVED] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\_paper_results_tables\table_10_robust_evaluation_protocols.md
[SAVED] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\_paper_results_tables\table_10_robust_evaluation_protocols.tex

Generated audit files:
D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\_paper_results_tables
D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\audit_hash_duplicates.csv
D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\audit_hash_duplicates.md
D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\audit_manifest.json

In [28]:
from pathlib import Path
import pandas as pd

ROOT = Path(r"D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)")
AUDIT_DIR = ROOT / "_ieee_leakage_and_robustness_audit"

dup_path = AUDIT_DIR / "fail_md5_duplicates_across_splits_cw_radar.csv"

dup = pd.read_csv(dup_path)

print("=" * 100)
print("CW Radar MD5 duplicates across splits")
print("=" * 100)
print(f"Rows: {len(dup)}")
print(f"Unique MD5 hashes: {dup['md5'].nunique()}")
print("=" * 100)

cols = [
    "md5",
    "split",
    "tuple_id",
    "target",
    "distance_m",
    "sample_index",
    "sensor",
    "path",
]

display(dup[cols].sort_values(["md5", "split", "target", "distance_m", "sample_index"]))

CW Radar MD5 duplicates across splits
Rows: 17
Unique MD5 hashes: 8


,md5,split,tuple_id,target,distance_m,sample_index,sensor,path
3,0fabc15134cd0bb668e23ed6ab7e1d1a,test,Corner Reflector__14m__172,Corner Reflector,14,172,CW Radar,D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\Dataset\CW Radar\14m\Corner Reflector\Image File\Corner...
2,0fabc15134cd0bb668e23ed6ab7e1d1a,train,Corner Reflector__14m__171,Corner Reflector,14,171,CW Radar,D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\Dataset\CW Radar\14m\Corner Reflector\Image File\Corner...
14,30b09c628d51a712624703b62ed7bebe,test,Corner Reflector__20m__470,Corner Reflector,20,470,CW Radar,D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\Dataset\CW Radar\20m\Corner Reflector\Image File\Corner...
15,30b09c628d51a712624703b62ed7bebe,train,Corner Reflector__20m__471,Corner Reflector,20,471,CW Radar,D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\Dataset\CW Radar\20m\Corner Reflector\Image File\Corner...
16,30b09c628d51a712624703b62ed7bebe,train,Corner Reflector__20m__478,Corner Reflector,20,478,CW Radar,D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\Dataset\CW Radar\20m\Corner Reflector\Image File\Corner...
9,af0ee3b80ac80e63d3ecf3770c3f9a75,train,Corner Reflector__18m__243,Corner Reflector,18,243,CW Radar,D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\Dataset\CW Radar\18m\Corner Reflector\Image File\Corner...
8,af0ee3b80ac80e63d3ecf3770c3f9a75,val,Corner Reflector__18m__240,Corner Reflector,18,240,CW Radar,D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\Dataset\CW Radar\18m\Corner Reflector\Image File\Corner...
13,b21ddbdb2af5f4b8445f44e5b8717add,test,Corner Reflector__20m__405,Corner Reflector,20,405,CW Radar,D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\Dataset\CW Radar\20m\Corner Reflector\Image File\Corner...
12,b21ddbdb2af5f4b8445f44e5b8717add,val,Corner Reflector__20m__399,Corner Reflector,20,399,CW Radar,D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\Dataset\CW Radar\20m\Corner Reflector\Image File\Corner...
10,c084feb5e1b37f237a2a478ab0892c8d,train,Corner Reflector__20m__367,Corner Reflector,20,367,CW Radar,D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\Dataset\CW Radar\20m\Corner Reflector\Image File\Corner...


# ANÁLISE DE DESEMPENHO AO LONGO DAS DISTÂNCIAS

In [29]:
# ============================================================
# DISTANCE-WISE PERFORMANCE ANALYSIS FOR TSMS-DRONE PAPER
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

try:
    from IPython.display import display, Markdown
    HAS_DISPLAY = True
except Exception:
    HAS_DISPLAY = False


# ============================================================
# PATHS
# ============================================================

ROOT = Path(r"D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)")
ADV_DIR = ROOT / "_advanced_fusion_improvements"
AUDIT_DIR = ROOT / "_ieee_leakage_and_robustness_audit"
PRED_DIR = AUDIT_DIR / "prediction_posthoc_analysis"

OUT_DIR = AUDIT_DIR / "_distance_wise_analysis"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("DISTANCE-WISE PERFORMANCE ANALYSIS")
print("=" * 80)
print(f"ADV_DIR : {ADV_DIR}")
print(f"PRED_DIR: {PRED_DIR}")
print(f"OUT_DIR : {OUT_DIR}")
print("=" * 80)


# ============================================================
# HELPERS
# ============================================================

def pretty_method_name(x):
    mapping = {
        "Advanced-Fusion-All": "All sensors",
        "Advanced-Fusion-CW-RF": "CW + RF",
        "Advanced-Fusion-FMCW-RF": "FMCW + RF",
        "Advanced-Fusion-RF-only": "RF only",
        "Advanced-Fusion-CW-FMCW": "CW + FMCW",
        "Advanced-Fusion-FMCW-only": "FMCW only",
        "Advanced-Fusion-CW-only": "CW only",
        "advanced_fusion_all": "All sensors",
        "advanced_fusion_cw_rf": "CW + RF",
        "advanced_fusion_fmcw_rf": "FMCW + RF",
        "advanced_fusion_rf_only": "RF only",
        "advanced_fusion_cw_fmcw": "CW + FMCW",
        "advanced_fusion_fmcw_only": "FMCW only",
        "advanced_fusion_cw_only": "CW only",
    }
    return mapping.get(str(x), str(x).replace("_", " ").title())


def save_table(df, name, caption=None, label=None):
    csv_path = OUT_DIR / f"{name}.csv"
    md_path = OUT_DIR / f"{name}.md"
    tex_path = OUT_DIR / f"{name}.tex"

    df.to_csv(csv_path, index=False, encoding="utf-8-sig")

    try:
        md_path.write_text(df.to_markdown(index=False), encoding="utf-8")
    except Exception:
        md_path.write_text(df.to_csv(index=False), encoding="utf-8")

    latex = df.to_latex(
        index=False,
        escape=False,
        caption=caption,
        label=label,
        float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x),
    )
    tex_path.write_text(latex, encoding="utf-8")

    return csv_path, md_path, tex_path


def show_table(title, df, name, caption=None, label=None):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

    if HAS_DISPLAY:
        display(Markdown(f"### {title}"))
        display(df)
    else:
        print(df.to_string(index=False))

    paths = save_table(df, name, caption, label)
    for p in paths:
        print(f"[SAVED] {p}")


def linear_slope_per_10m(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    mask = ~np.isnan(x) & ~np.isnan(y)
    x = x[mask]
    y = y[mask]

    if len(x) < 2:
        return np.nan

    slope_per_meter = np.polyfit(x, y, 1)[0]
    return slope_per_meter * 10.0


# ============================================================
# LOAD DISTANCE-WISE METRICS
# ============================================================

distance_csv_1 = ADV_DIR / "advanced_distance_wise_metrics_all.csv"
distance_csv_2 = PRED_DIR / "target_distance_metrics_from_predictions.csv"

if distance_csv_1.exists():
    distance_df = pd.read_csv(distance_csv_1)
    source = distance_csv_1
elif distance_csv_2.exists():
    td = pd.read_csv(distance_csv_2)
    distance_df = (
        td.groupby(["method", "distance_m"])
        .agg(
            n=("n", "sum"),
            accuracy=("accuracy", "mean"),
            balanced_accuracy=("balanced_accuracy", "mean"),
            macro_precision=("macro_precision", "mean"),
            macro_recall=("macro_recall", "mean"),
            macro_f1=("macro_f1", "mean"),
            weighted_f1=("weighted_f1", "mean"),
            kappa=("kappa", "mean"),
        )
        .reset_index()
    )
    source = distance_csv_2
else:
    raise FileNotFoundError(
        "No distance-wise metrics found. Expected one of:\n"
        f"{distance_csv_1}\n{distance_csv_2}"
    )

print(f"Loaded distance-wise metrics from: {source}")

distance_df["Setting"] = distance_df["method"].map(pretty_method_name)
distance_df["distance_m"] = distance_df["distance_m"].astype(int)

method_order = [
    "All sensors",
    "CW + RF",
    "FMCW + RF",
    "RF only",
    "CW + FMCW",
    "FMCW only",
    "CW only",
]

distance_df["_method_order"] = distance_df["Setting"].apply(
    lambda x: method_order.index(x) if x in method_order else 999
)
distance_df = distance_df.sort_values(["_method_order", "distance_m"]).drop(columns=["_method_order"])


# ============================================================
# TABLE 1 — DISTANCE-WISE MACRO-F1 MATRIX
# ============================================================

macro_pivot = (
    distance_df
    .pivot_table(index="distance_m", columns="Setting", values="macro_f1")
    .reset_index()
)

ordered_cols = ["distance_m"] + [c for c in method_order if c in macro_pivot.columns]
macro_pivot = macro_pivot[ordered_cols]

macro_pivot_print = macro_pivot.copy()
for c in macro_pivot_print.columns:
    if c != "distance_m":
        macro_pivot_print[c] = macro_pivot_print[c].round(4)

show_table(
    "Table D1 — Macro-F1 by distance and sensor setting",
    macro_pivot_print,
    "table_D1_macro_f1_by_distance",
    caption="Macro-F1 by acquisition distance and sensor configuration.",
    label="tab:distance_macro_f1"
)


# ============================================================
# TABLE 2 — DISTANCE-WISE BALANCED ACCURACY MATRIX
# ============================================================

ba_pivot = (
    distance_df
    .pivot_table(index="distance_m", columns="Setting", values="balanced_accuracy")
    .reset_index()
)

ordered_cols = ["distance_m"] + [c for c in method_order if c in ba_pivot.columns]
ba_pivot = ba_pivot[ordered_cols]

ba_pivot_print = ba_pivot.copy()
for c in ba_pivot_print.columns:
    if c != "distance_m":
        ba_pivot_print[c] = ba_pivot_print[c].round(4)

show_table(
    "Table D2 — Balanced accuracy by distance and sensor setting",
    ba_pivot_print,
    "table_D2_balanced_accuracy_by_distance",
    caption="Balanced accuracy by acquisition distance and sensor configuration.",
    label="tab:distance_balanced_accuracy"
)


# ============================================================
# TABLE 3 — ROBUSTNESS SUMMARY ACROSS DISTANCES
# ============================================================

rows = []

for setting, g in distance_df.groupby("Setting"):
    g = g.sort_values("distance_m")

    macro = g["macro_f1"].to_numpy(dtype=float)
    dist = g["distance_m"].to_numpy(dtype=float)

    best_idx = int(np.nanargmax(macro))
    worst_idx = int(np.nanargmin(macro))

    rows.append({
        "Setting": setting,
        "Mean Macro-F1": np.nanmean(macro),
        "Std Macro-F1": np.nanstd(macro),
        "Min Macro-F1": np.nanmin(macro),
        "Max Macro-F1": np.nanmax(macro),
        "Range": np.nanmax(macro) - np.nanmin(macro),
        "Worst distance (m)": int(dist[worst_idx]),
        "Best distance (m)": int(dist[best_idx]),
        "Slope per 10 m": linear_slope_per_10m(dist, macro),
    })

robust_summary = pd.DataFrame(rows)
robust_summary["_order"] = robust_summary["Setting"].apply(
    lambda x: method_order.index(x) if x in method_order else 999
)
robust_summary = robust_summary.sort_values("_order").drop(columns=["_order"])

for c in [
    "Mean Macro-F1",
    "Std Macro-F1",
    "Min Macro-F1",
    "Max Macro-F1",
    "Range",
    "Slope per 10 m",
]:
    robust_summary[c] = robust_summary[c].round(4)

show_table(
    "Table D3 — Distance robustness summary",
    robust_summary,
    "table_D3_distance_robustness_summary",
    caption="Summary of distance-wise robustness across acquisition ranges.",
    label="tab:distance_robustness_summary"
)


# ============================================================
# TABLE 4 — PERFORMANCE LOSS RELATIVE TO FULL FUSION BY DISTANCE
# ============================================================

if "All sensors" in macro_pivot.columns:
    loss_df = macro_pivot.copy()

    for c in loss_df.columns:
        if c != "distance_m":
            loss_df[c] = macro_pivot["All sensors"] - macro_pivot[c]

    loss_df_print = loss_df.copy()
    for c in loss_df_print.columns:
        if c != "distance_m":
            loss_df_print[c] = loss_df_print[c].round(4)

    show_table(
        "Table D4 — Macro-F1 loss relative to full fusion by distance",
        loss_df_print,
        "table_D4_macro_f1_loss_vs_full_fusion_by_distance",
        caption="Macro-F1 loss relative to full multimodal fusion by distance.",
        label="tab:distance_loss_vs_full"
    )


# ============================================================
# TABLE 5 — TARGET × DISTANCE SUMMARY, IF AVAILABLE
# ============================================================

td_path = PRED_DIR / "target_distance_metrics_from_predictions.csv"

if td_path.exists():
    td = pd.read_csv(td_path)
    td["Setting"] = td["method"].map(pretty_method_name)
    td["distance_m"] = td["distance_m"].astype(int)

    # Keep most informative rows: worst target-distance cases per setting.
    worst_td = (
        td.sort_values(["method", "macro_f1"], ascending=[True, True])
        .groupby("Setting")
        .head(10)
        .copy()
    )

    keep_cols = [
        "Setting",
        "target",
        "distance_m",
        "n",
        "accuracy",
        "balanced_accuracy",
        "macro_f1",
        "weighted_f1",
        "kappa",
    ]
    keep_cols = [c for c in keep_cols if c in worst_td.columns]
    worst_td = worst_td[keep_cols]

    rename = {
        "target": "Target",
        "distance_m": "Distance (m)",
        "n": "N",
        "accuracy": "Accuracy",
        "balanced_accuracy": "Balanced Acc.",
        "macro_f1": "Macro-F1",
        "weighted_f1": "Weighted-F1",
        "kappa": "Kappa",
    }
    worst_td = worst_td.rename(columns=rename)

    for c in ["Accuracy", "Balanced Acc.", "Macro-F1", "Weighted-F1", "Kappa"]:
        if c in worst_td.columns:
            worst_td[c] = worst_td[c].round(4)

    show_table(
        "Table D5 — Worst target-distance cases by setting",
        worst_td,
        "table_D5_worst_target_distance_cases",
        caption="Worst target-distance cases for each sensor setting.",
        label="tab:worst_target_distance_cases"
    )


# ============================================================
# FIGURE 1 — MACRO-F1 CURVES BY DISTANCE
# ============================================================

selected_settings = [
    "All sensors",
    "RF only",
    "CW + RF",
    "FMCW + RF",
    "CW + FMCW",
    "FMCW only",
    "CW only",
]

fig, ax = plt.subplots(figsize=(11, 5))

for setting in selected_settings:
    g = distance_df[distance_df["Setting"] == setting].sort_values("distance_m")
    if len(g) > 0:
        ax.plot(g["distance_m"], g["macro_f1"], marker="o", label=setting)

ax.set_xlabel("Distance (m)")
ax.set_ylabel("Macro-F1")
ax.set_title("Distance-wise Macro-F1 by sensor setting")
ax.set_xticks(sorted(distance_df["distance_m"].unique()))
ax.set_ylim(0, 1.02)
ax.grid(True, linewidth=0.5, alpha=0.4)
ax.legend(fontsize=8, ncol=2)
fig.tight_layout()

fig_path = OUT_DIR / "fig_D1_macro_f1_by_distance.png"
fig.savefig(fig_path, dpi=300)
plt.show()

print(f"[SAVED] {fig_path}")


# ============================================================
# FIGURE 2 — HEATMAP SETTING × DISTANCE
# ============================================================

heat = (
    distance_df
    .pivot_table(index="Setting", columns="distance_m", values="macro_f1")
    .reindex(method_order)
)

fig, ax = plt.subplots(figsize=(11, 4.5))
im = ax.imshow(heat.to_numpy(dtype=float), aspect="auto", vmin=0, vmax=1)

ax.set_title("Macro-F1 heatmap across distances and sensor settings")
ax.set_xlabel("Distance (m)")
ax.set_ylabel("Sensor setting")

ax.set_xticks(range(len(heat.columns)))
ax.set_xticklabels(heat.columns)

ax.set_yticks(range(len(heat.index)))
ax.set_yticklabels(heat.index)

for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        val = heat.iloc[i, j]
        if not pd.isna(val):
            ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=7)

fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()

fig_path = OUT_DIR / "fig_D2_macro_f1_heatmap_setting_distance.png"
fig.savefig(fig_path, dpi=300)
plt.show()

print(f"[SAVED] {fig_path}")


# ============================================================
# FIGURE 3 — TARGET × DISTANCE HEATMAPS FOR KEY SETTINGS
# ============================================================

if td_path.exists():
    key_settings = ["All sensors", "RF only", "CW + FMCW", "FMCW only", "CW only"]

    for setting in key_settings:
        g = td[td["Setting"] == setting].copy()

        if len(g) == 0:
            continue

        pivot = (
            g.pivot_table(index="target", columns="distance_m", values="macro_f1")
            .reindex([
                "Inspire 2",
                "Matrice 30",
                "Mavic 2 Pro",
                "Phantom 4 Pro",
                "Corner Reflector",
            ])
        )

        fig, ax = plt.subplots(figsize=(10, 4.5))
        im = ax.imshow(pivot.to_numpy(dtype=float), aspect="auto", vmin=0, vmax=1)

        ax.set_title(f"Target-distance Macro-F1 — {setting}")
        ax.set_xlabel("Distance (m)")
        ax.set_ylabel("Target")

        ax.set_xticks(range(len(pivot.columns)))
        ax.set_xticklabels(pivot.columns)

        ax.set_yticks(range(len(pivot.index)))
        ax.set_yticklabels(pivot.index)

        for i in range(pivot.shape[0]):
            for j in range(pivot.shape[1]):
                val = pivot.iloc[i, j]
                if not pd.isna(val):
                    ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=7)

        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        fig.tight_layout()

        fig_path = OUT_DIR / f"fig_D3_target_distance_macro_f1_{setting.replace(' ', '_').replace('+', 'plus')}.png"
        fig.savefig(fig_path, dpi=300)
        plt.show()

        print(f"[SAVED] {fig_path}")


# ============================================================
# AUTOMATIC TEXTUAL INTERPRETATION
# ============================================================

print("\n" + "=" * 80)
print("AUTOMATIC DISTANCE-WISE INTERPRETATION")
print("=" * 80)

for _, row in robust_summary.iterrows():
    setting = row["Setting"]
    mean_f1 = row["Mean Macro-F1"]
    min_f1 = row["Min Macro-F1"]
    worst_d = row["Worst distance (m)"]
    slope = row["Slope per 10 m"]

    if slope < -0.01:
        trend = "decreases with distance"
    elif slope > 0.01:
        trend = "increases with distance"
    else:
        trend = "is approximately stable across distance"

    print(
        f"{setting}: mean Macro-F1={mean_f1:.4f}, "
        f"minimum={min_f1:.4f} at {worst_d} m, "
        f"slope per 10 m={slope:.4f}; trend: {trend}."
    )

print("=" * 80)
print("[OK] Distance-wise analysis completed.")
print("=" * 80)

DISTANCE-WISE PERFORMANCE ANALYSIS
ADV_DIR : D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_advanced_fusion_improvements
PRED_DIR: D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\prediction_posthoc_analysis
OUT_DIR : D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\_distance_wise_analysis
Loaded distance-wise metrics from: D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_advanced_fusion_improvements\advanced_distance_wise_metrics_all.csv

Table D1 — Macro-F1 by distance and sensor setting


### Table D1 — Macro-F1 by distance and sensor setting

Setting,distance_m,All sensors,CW + RF,FMCW + RF,RF only,CW + FMCW,FMCW only,CW only
0,2,1.0,1.0000,1.0000,0.9973,0.9947,0.7255,0.9442
1,4,1.0,1.0000,1.0000,1.0000,1.0000,0.9757,0.8475
2,6,1.0,1.0000,1.0000,0.9973,0.8847,0.9128,0.6501
3,8,1.0,1.0000,1.0000,1.0000,0.9813,0.8558,0.7281
4,10,1.0,1.0000,1.0000,1.0000,0.9530,0.9646,0.4638
5,12,1.0,1.0000,1.0000,0.9973,1.0000,0.9920,0.7170
6,14,1.0,1.0000,1.0000,1.0000,0.9623,0.8494,0.3912
7,16,1.0,1.0000,1.0000,1.0000,0.9867,0.9453,0.6578
8,18,1.0,1.0000,1.0000,1.0000,0.9098,0.8840,0.3159
9,20,1.0,1.0000,1.0000,1.0000,0.8765,0.8449,0.4748


[SAVED] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\_distance_wise_analysis\table_D1_macro_f1_by_distance.csv
[SAVED] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\_distance_wise_analysis\table_D1_macro_f1_by_distance.md
[SAVED] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\_distance_wise_analysis\table_D1_macro_f1_by_distance.tex

Table D2 — Balanced accuracy by distance and sensor setting


### Table D2 — Balanced accuracy by distance and sensor setting

...

Setting,distance_m,All sensors,CW + RF,FMCW + RF,RF only,CW + FMCW,FMCW only,CW only
0,2,0.0,0.0000,0.0000,0.0027,0.0053,0.2745,0.0558
1,4,0.0,0.0000,0.0000,0.0000,0.0000,0.0243,0.1525
2,6,0.0,0.0000,0.0000,0.0027,0.1153,0.0872,0.3499
3,8,0.0,0.0000,0.0000,0.0000,0.0187,0.1442,0.2719
4,10,0.0,0.0000,0.0000,0.0000,0.0470,0.0354,0.5362
5,12,0.0,0.0000,0.0000,0.0027,0.0000,0.0080,0.2830
6,14,0.0,0.0000,0.0000,0.0000,0.0377,0.1506,0.6088
7,16,0.0,0.0000,0.0000,0.0000,0.0133,0.0547,0.3422
8,18,0.0,0.0000,0.0000,0.0000,0.0902,0.1160,0.6841
9,20,0.0,0.0000,0.0000,0.0000,0.1235,0.1551,0.5252


[SAVED] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\_distance_wise_analysis\table_D4_macro_f1_loss_vs_full_fusion_by_distance.csv
[SAVED] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\_distance_wise_analysis\table_D4_macro_f1_loss_vs_full_fusion_by_distance.md
[SAVED] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\_distance_wise_analysis\table_D4_macro_f1_loss_vs_full_fusion_by_distance.tex

Table D5 — Worst target-distance cases by setting


### Table D5 — Worst target-distance cases by setting

,Setting,Target,Distance (m),N,Accuracy,Balanced Acc.,Macro-F1,Weighted-F1,Kappa
0,All sensors,Corner Reflector,2,75,1.0,1.0,1.0,1.0,NaN
1,All sensors,Corner Reflector,4,75,1.0,1.0,1.0,1.0,NaN
2,All sensors,Corner Reflector,6,75,1.0,1.0,1.0,1.0,NaN
3,All sensors,Corner Reflector,8,75,1.0,1.0,1.0,1.0,NaN
4,All sensors,Corner Reflector,10,75,1.0,1.0,1.0,1.0,NaN
...,...,...,...,...,...,...,...,...,...
453,RF only,Corner Reflector,8,75,1.0,1.0,1.0,1.0,NaN
454,RF only,Corner Reflector,10,75,1.0,1.0,1.0,1.0,NaN
456,RF only,Corner Reflector,14,75,1.0,1.0,1.0,1.0,NaN
457,RF only,Corner Reflector,16,75,1.0,1.0,1.0,1.0,NaN


[SAVED] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\_distance_wise_analysis\table_D5_worst_target_distance_cases.csv
[SAVED] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\_distance_wise_analysis\table_D5_worst_target_distance_cases.md
[SAVED] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\_distance_wise_analysis\table_D5_worst_target_distance_cases.tex
[SAVED] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\_distance_wise_analysis\fig_D1_macro_f1_by_distance.png
[SAVED] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\_distance_wise_analysis\fig_D2_macro_f1_heatmap_setting_distance.png
[SAVED] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\_distance_wise_analysis\fig_D3_target_distance_macro_f1_A

In [30]:
# ============================================================
# TABLES BY DISTANCE AND SENSOR SCENARIO
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

try:
    from IPython.display import display, Markdown
    HAS_DISPLAY = True
except Exception:
    HAS_DISPLAY = False


# ============================================================
# PATHS
# ============================================================

ROOT = Path(r"D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)")
ADV_DIR = ROOT / "_advanced_fusion_improvements"
AUDIT_DIR = ROOT / "_ieee_leakage_and_robustness_audit"
PRED_DIR = AUDIT_DIR / "prediction_posthoc_analysis"

OUT_DIR = AUDIT_DIR / "_distance_wise_analysis" / "_tables_by_each_distance"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("TABLES BY DISTANCE AND SENSOR SCENARIO")
print("=" * 80)
print(f"ADV_DIR : {ADV_DIR}")
print(f"PRED_DIR: {PRED_DIR}")
print(f"OUT_DIR : {OUT_DIR}")
print("=" * 80)


# ============================================================
# HELPERS
# ============================================================

def pretty_method_name(x):
    mapping = {
        "Advanced-Fusion-All": "All sensors",
        "Advanced-Fusion-CW-RF": "CW + RF",
        "Advanced-Fusion-FMCW-RF": "FMCW + RF",
        "Advanced-Fusion-RF-only": "RF only",
        "Advanced-Fusion-CW-FMCW": "CW + FMCW",
        "Advanced-Fusion-FMCW-only": "FMCW only",
        "Advanced-Fusion-CW-only": "CW only",
        "advanced_fusion_all": "All sensors",
        "advanced_fusion_cw_rf": "CW + RF",
        "advanced_fusion_fmcw_rf": "FMCW + RF",
        "advanced_fusion_rf_only": "RF only",
        "advanced_fusion_cw_fmcw": "CW + FMCW",
        "advanced_fusion_fmcw_only": "FMCW only",
        "advanced_fusion_cw_only": "CW only",
    }
    return mapping.get(str(x), str(x).replace("_", " ").title())


method_order = [
    "All sensors",
    "CW + RF",
    "FMCW + RF",
    "RF only",
    "CW + FMCW",
    "FMCW only",
    "CW only",
]


def save_table(df, name, caption=None, label=None):
    csv_path = OUT_DIR / f"{name}.csv"
    md_path = OUT_DIR / f"{name}.md"
    tex_path = OUT_DIR / f"{name}.tex"

    df.to_csv(csv_path, index=False, encoding="utf-8-sig")

    try:
        md_path.write_text(df.to_markdown(index=False), encoding="utf-8")
    except Exception:
        md_path.write_text(df.to_csv(index=False), encoding="utf-8")

    latex = df.to_latex(
        index=False,
        escape=False,
        caption=caption,
        label=label,
        float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x),
    )
    tex_path.write_text(latex, encoding="utf-8")

    return csv_path, md_path, tex_path


def show_table(title, df, name, caption=None, label=None):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

    if HAS_DISPLAY:
        display(Markdown(f"### {title}"))
        display(df)
    else:
        print(df.to_string(index=False))

    paths = save_table(df, name, caption, label)

    for p in paths:
        print(f"[SAVED] {p}")


# ============================================================
# LOAD DISTANCE-WISE METRICS
# ============================================================

distance_csv_1 = ADV_DIR / "advanced_distance_wise_metrics_all.csv"
distance_csv_2 = PRED_DIR / "target_distance_metrics_from_predictions.csv"

if distance_csv_1.exists():
    distance_df = pd.read_csv(distance_csv_1)
    source = distance_csv_1
elif distance_csv_2.exists():
    td = pd.read_csv(distance_csv_2)
    distance_df = (
        td.groupby(["method", "distance_m"])
        .agg(
            n=("n", "sum"),
            accuracy=("accuracy", "mean"),
            balanced_accuracy=("balanced_accuracy", "mean"),
            macro_precision=("macro_precision", "mean"),
            macro_recall=("macro_recall", "mean"),
            macro_f1=("macro_f1", "mean"),
            weighted_f1=("weighted_f1", "mean"),
            kappa=("kappa", "mean"),
        )
        .reset_index()
    )
    source = distance_csv_2
else:
    raise FileNotFoundError(
        "No distance-wise metrics found. Expected one of:\n"
        f"{distance_csv_1}\n{distance_csv_2}"
    )

print(f"Loaded distance-wise metrics from: {source}")

distance_df["Setting"] = distance_df["method"].map(pretty_method_name)
distance_df["distance_m"] = distance_df["distance_m"].astype(int)

distance_df["_order"] = distance_df["Setting"].apply(
    lambda x: method_order.index(x) if x in method_order else 999
)

distance_df = distance_df.sort_values(["distance_m", "_order"])


# ============================================================
# GENERATE ONE TABLE FOR EACH DISTANCE
# ============================================================

all_distance_tables = []

for distance in sorted(distance_df["distance_m"].unique()):

    table = distance_df[distance_df["distance_m"] == distance].copy()

    table = table[
        [
            "Setting",
            "accuracy",
            "balanced_accuracy",
            "macro_precision",
            "macro_recall",
            "macro_f1",
            "weighted_f1",
            "kappa",
        ]
    ]

    table = table.rename(columns={
        "Setting": "Scenario",
        "accuracy": "Accuracy",
        "balanced_accuracy": "Balanced Acc.",
        "macro_precision": "Macro Precision",
        "macro_recall": "Macro Recall",
        "macro_f1": "Macro-F1",
        "weighted_f1": "Weighted-F1",
        "kappa": "Kappa",
    })

    for c in [
        "Accuracy",
        "Balanced Acc.",
        "Macro Precision",
        "Macro Recall",
        "Macro-F1",
        "Weighted-F1",
        "Kappa",
    ]:
        if c in table.columns:
            table[c] = table[c].astype(float).round(4)

    table.insert(0, "Distance (m)", distance)

    all_distance_tables.append(table)

    show_table(
        f"Table — Performance by scenario at {distance} m",
        table,
        f"table_distance_{distance:02d}m_by_scenario",
        caption=f"Performance by sensor scenario at {distance} m.",
        label=f"tab:distance_{distance:02d}m_by_scenario",
    )


# ============================================================
# MASTER LONG TABLE — ALL DISTANCES × ALL SCENARIOS
# ============================================================

master_long = pd.concat(all_distance_tables, ignore_index=True)

show_table(
    "Master Table — Performance by distance and scenario",
    master_long,
    "table_master_distance_by_scenario",
    caption="Performance by acquisition distance and sensor scenario.",
    label="tab:master_distance_by_scenario",
)


# ============================================================
# COMPACT PIVOT TABLES FOR PAPER
# ============================================================

# Macro-F1 matrix
macro_matrix = (
    master_long
    .pivot_table(index="Distance (m)", columns="Scenario", values="Macro-F1")
    .reset_index()
)

macro_matrix = macro_matrix[
    ["Distance (m)"] + [c for c in method_order if c in macro_matrix.columns]
]

show_table(
    "Compact Table — Macro-F1 by distance and scenario",
    macro_matrix,
    "table_compact_macro_f1_distance_by_scenario",
    caption="Macro-F1 by distance and sensor scenario.",
    label="tab:compact_macro_f1_distance_by_scenario",
)


# Balanced accuracy matrix
ba_matrix = (
    master_long
    .pivot_table(index="Distance (m)", columns="Scenario", values="Balanced Acc.")
    .reset_index()
)

ba_matrix = ba_matrix[
    ["Distance (m)"] + [c for c in method_order if c in ba_matrix.columns]
]

show_table(
    "Compact Table — Balanced accuracy by distance and scenario",
    ba_matrix,
    "table_compact_balanced_accuracy_distance_by_scenario",
    caption="Balanced accuracy by distance and sensor scenario.",
    label="tab:compact_balanced_accuracy_distance_by_scenario",
)


print("\n" + "=" * 80)
print("GENERATED TABLE FILES")
print("=" * 80)

for p in sorted(OUT_DIR.glob("*")):
    print(p)

print("=" * 80)
print("[OK] Tables by distance and scenario generated.")
print("=" * 80)

TABLES BY DISTANCE AND SENSOR SCENARIO
ADV_DIR : D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_advanced_fusion_improvements
PRED_DIR: D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\prediction_posthoc_analysis
OUT_DIR : D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\_distance_wise_analysis\_tables_by_each_distance
Loaded distance-wise metrics from: D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_advanced_fusion_improvements\advanced_distance_wise_metrics_all.csv

Table — Performance by scenario at 2 m


### Table — Performance by scenario at 2 m

,Distance (m),Scenario,Accuracy,Balanced Acc.,Macro Precision,Macro Recall,Macro-F1,Weighted-F1,Kappa
0,2,All sensors,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
30,2,CW + RF,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
45,2,FMCW + RF,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
90,2,RF only,0.9973,0.9973,0.9974,0.9973,0.9973,0.9973,0.9967
15,2,CW + FMCW,0.9947,0.9947,0.9947,0.9947,0.9947,0.9947,0.9933
75,2,FMCW only,0.7893,0.7893,0.6974,0.7893,0.7255,0.7255,0.7367
60,2,CW only,0.9467,0.9467,0.9512,0.9467,0.9442,0.9442,0.9333


[SAVED] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\_distance_wise_analysis\_tables_by_each_distance\table_distance_02m_by_scenario.csv
[SAVED] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\_distance_wise_analysis\_tables_by_each_distance\table_distance_02m_by_scenario.md
[SAVED] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\_distance_wise_analysis\_tables_by_each_distance\table_distance_02m_by_scenario.tex

Table — Performance by scenario at 4 m


### Table — Performance by scenario at 4 m

...

Scenario,Distance (m),All sensors,CW + RF,FMCW + RF,RF only,CW + FMCW,FMCW only,CW only
0,2,1.0,1.0000,1.0000,0.9973,0.9947,0.7255,0.9442
1,4,1.0,1.0000,1.0000,1.0000,1.0000,0.9757,0.8475
2,6,1.0,1.0000,1.0000,0.9973,0.8847,0.9128,0.6501
3,8,1.0,1.0000,1.0000,1.0000,0.9813,0.8558,0.7281
4,10,1.0,1.0000,1.0000,1.0000,0.9530,0.9646,0.4638
5,12,1.0,1.0000,1.0000,0.9973,1.0000,0.9920,0.7170
6,14,1.0,1.0000,1.0000,1.0000,0.9623,0.8494,0.3912
7,16,1.0,1.0000,1.0000,1.0000,0.9867,0.9453,0.6578
8,18,1.0,1.0000,1.0000,1.0000,0.9098,0.8840,0.3159
9,20,1.0,1.0000,1.0000,1.0000,0.8765,0.8449,0.4748


[SAVED] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\_distance_wise_analysis\_tables_by_each_distance\table_compact_macro_f1_distance_by_scenario.csv
[SAVED] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\_distance_wise_analysis\_tables_by_each_distance\table_compact_macro_f1_distance_by_scenario.md
[SAVED] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\_distance_wise_analysis\_tables_by_each_distance\table_compact_macro_f1_distance_by_scenario.tex

Compact Table — Balanced accuracy by distance and scenario


### Compact Table — Balanced accuracy by distance and scenario

Scenario,Distance (m),All sensors,CW + RF,FMCW + RF,RF only,CW + FMCW,FMCW only,CW only
0,2,1.0,1.0000,1.0000,0.9973,0.9947,0.7893,0.9467
1,4,1.0,1.0000,1.0000,1.0000,1.0000,0.9760,0.8640
2,6,1.0,1.0000,1.0000,0.9973,0.8933,0.9147,0.7173
3,8,1.0,1.0000,1.0000,1.0000,0.9813,0.8720,0.7520
4,10,1.0,1.0000,1.0000,1.0000,0.9547,0.9653,0.5360
5,12,1.0,1.0000,1.0000,0.9973,1.0000,0.9920,0.7733
6,14,1.0,1.0000,1.0000,1.0000,0.9627,0.8667,0.4293
7,16,1.0,1.0000,1.0000,1.0000,0.9867,0.9467,0.7147
8,18,1.0,1.0000,1.0000,1.0000,0.9093,0.8853,0.3813
9,20,1.0,1.0000,1.0000,1.0000,0.8853,0.8507,0.5173


[SAVED] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\_distance_wise_analysis\_tables_by_each_distance\table_compact_balanced_accuracy_distance_by_scenario.csv
[SAVED] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\_distance_wise_analysis\_tables_by_each_distance\table_compact_balanced_accuracy_distance_by_scenario.md
[SAVED] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\_distance_wise_analysis\_tables_by_each_distance\table_compact_balanced_accuracy_distance_by_scenario.tex

GENERATED TABLE FILES
D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\_distance_wise_analysis\_tables_by_each_distance\table_compact_balanced_accuracy_distance_by_scenario.csv
D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_ieee_leakage_and_robustness_audit\_distance_wise_analysis\_tables_by_each_distance\t

# ANÁLISES AVANÇADAS

In [33]:
# ============================================================
# tsms_drone_multitask_baseline_fusion_benchmark.py
# ============================================================
# TSMS-Drone: multitask baseline benchmark for simple models and
# sensor-combination fusion scenarios.
#
# Tasks implemented:
#   1. Target classification: 5 classes
#      Inspire 2, Matrice 30, Mavic 2 Pro, Phantom 4 Pro, Corner Reflector
#
#   2. Drone vs non-drone detection: binary
#      Drone = Inspire 2, Matrice 30, Mavic 2 Pro, Phantom 4 Pro
#      Non-drone = Corner Reflector
#
#   3. Drone model identification: 4 classes
#      Inspire 2, Matrice 30, Mavic 2 Pro, Phantom 4 Pro
#      Corner Reflector is excluded.
#
#   4. Distance estimation as classification: 15 distance classes
#      2, 4, ..., 30 m. Also reports distance MAE/RMSE in meters.
#
#   5. Multimodal sensor fusion:
#      isolated sensors, all pairs, and all sensors.
#
# Sensor scenarios:
#   - CW only
#   - FMCW only
#   - RF only
#   - CW + FMCW
#   - CW + RF
#   - FMCW + RF
#   - All sensors
#
# Baseline models:
#   - RidgeClassifier
#   - LinearSVM
#   - LogisticRegression
#   - ExtraTrees
#   - RandomForest, optional in full preset
#
# Data source expected:
#   D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\Dataset
#
# Required structure:
#   Dataset/<Sensor>/<Distance>/<Target>/Image File/*.png
#
# Output:
#   ROOT\_baseline_multitask_fusion_benchmark
#       features/
#       metrics/
#       predictions/
#       figures/
#       tables/
#
# Notes:
#   - Uses PNG image products for simple reproducible baselines.
#   - Builds or reuses a synchronized tuple index.
#   - Reuses the cleaned split if available; otherwise uses the original split;
#     if no split is available, it creates a stratified 70/15/15 split.
#   - Designed to be safe in Jupyter/IPython and PowerShell.
# ============================================================

from __future__ import annotations

import argparse
import json
import math
import re
import sys
import time
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

try:
    from IPython.display import display, Markdown
    HAS_DISPLAY = True
except Exception:
    HAS_DISPLAY = False


# ============================================================
# CONFIGURATION
# ============================================================

SENSORS = ["CW Radar", "FMCW Radar", "RF Receiver"]
SENSOR_SHORT = {
    "CW Radar": "CW",
    "FMCW Radar": "FMCW",
    "RF Receiver": "RF",
}
SHORT_TO_SENSOR = {v: k for k, v in SENSOR_SHORT.items()}

TARGETS = [
    "Inspire 2",
    "Matrice 30",
    "Mavic 2 Pro",
    "Phantom 4 Pro",
    "Corner Reflector",
]
DRONE_TARGETS = ["Inspire 2", "Matrice 30", "Mavic 2 Pro", "Phantom 4 Pro"]
NON_DRONE_TARGET = "Corner Reflector"
DISTANCE_VALUES = list(range(2, 31, 2))

SENSOR_SCENARIOS = [
    ("CW only", ["CW Radar"]),
    ("FMCW only", ["FMCW Radar"]),
    ("RF only", ["RF Receiver"]),
    ("CW + FMCW", ["CW Radar", "FMCW Radar"]),
    ("CW + RF", ["CW Radar", "RF Receiver"]),
    ("FMCW + RF", ["FMCW Radar", "RF Receiver"]),
    ("All sensors", ["CW Radar", "FMCW Radar", "RF Receiver"]),
]

SCENARIO_ORDER = [x[0] for x in SENSOR_SCENARIOS]

TASKS = [
    "target_classification",
    "drone_vs_non_drone",
    "drone_model_identification",
    "distance_estimation",
]

TASK_LABELS = {
    "target_classification": "Target classification",
    "drone_vs_non_drone": "Drone vs non-drone",
    "drone_model_identification": "Drone model identification",
    "distance_estimation": "Distance estimation",
}


@dataclass
class Config:
    root: Path
    dataset_dir: Path
    out_dir: Path
    feature_dir: Path
    metrics_dir: Path
    prediction_dir: Path
    figure_dir: Path
    table_dir: Path
    split_csv: Optional[Path]
    image_size: int = 32
    force_reindex: bool = False
    force_features: bool = False
    model_preset: str = "fast"  # fast, standard, full
    max_train_samples: int = 0   # 0 = no limit
    seed: int = 42
    train_on_train_val: bool = False
    n_jobs: int = -1


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="TSMS-Drone multitask baseline and fusion benchmark.",
        allow_abbrev=False,
    )
    parser.add_argument("--root", type=str, default=r"D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)")
    parser.add_argument("--dataset-dir", type=str, default=None)
    parser.add_argument("--out-dir", type=str, default=None)
    parser.add_argument("--split-csv", type=str, default=None)
    parser.add_argument("--image-size", type=int, default=32)
    parser.add_argument("--force-reindex", action="store_true")
    parser.add_argument("--force-features", action="store_true")
    parser.add_argument("--model-preset", type=str, default="fast", choices=["fast", "standard", "full"])
    parser.add_argument("--max-train-samples", type=int, default=0)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--train-on-train-val", action="store_true")
    parser.add_argument("--n-jobs", type=int, default=-1)

    args, unknown = parser.parse_known_args()
    if unknown:
        print(f"[INFO] Ignored unknown arguments, probably from Jupyter/IPython: {unknown}")
    return args


def build_config(args: argparse.Namespace) -> Config:
    root = Path(args.root)
    dataset_dir = Path(args.dataset_dir) if args.dataset_dir else root / "Dataset"
    out_dir = Path(args.out_dir) if args.out_dir else root / "_baseline_multitask_fusion_benchmark"

    # Prefer leakage-clean split if already created; otherwise original advanced split.
    if args.split_csv:
        split_csv = Path(args.split_csv)
    else:
        clean_split = root / "_advanced_fusion_improvements" / "tuple_splits_70_15_15_full_clean_no_cw_md5_leak.csv"
        original_split = root / "_advanced_fusion_improvements" / "tuple_splits_70_15_15_full.csv"
        split_csv = clean_split if clean_split.exists() else original_split if original_split.exists() else None

    return Config(
        root=root,
        dataset_dir=dataset_dir,
        out_dir=out_dir,
        feature_dir=out_dir / "features",
        metrics_dir=out_dir / "metrics",
        prediction_dir=out_dir / "predictions",
        figure_dir=out_dir / "figures",
        table_dir=out_dir / "tables",
        split_csv=split_csv,
        image_size=args.image_size,
        force_reindex=args.force_reindex,
        force_features=args.force_features,
        model_preset=args.model_preset,
        max_train_samples=args.max_train_samples,
        seed=args.seed,
        train_on_train_val=args.train_on_train_val,
        n_jobs=args.n_jobs,
    )


# ============================================================
# UTILS
# ============================================================

def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def safe_name(text: str) -> str:
    text = str(text).lower().strip()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return text.strip("_")


def display_table(title: str, df: pd.DataFrame, out_dir: Path, name: str, caption: str = "", label: str = "") -> None:
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)
    if HAS_DISPLAY:
        display(Markdown(f"### {title}"))
        display(df)
    else:
        print(df.to_string(index=False))

    ensure_dir(out_dir)
    csv_path = out_dir / f"{name}.csv"
    md_path = out_dir / f"{name}.md"
    tex_path = out_dir / f"{name}.tex"
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    try:
        md_path.write_text(df.to_markdown(index=False), encoding="utf-8")
    except Exception:
        md_path.write_text(df.to_csv(index=False), encoding="utf-8")
    tex = df.to_latex(
        index=False,
        escape=False,
        caption=caption if caption else None,
        label=label if label else None,
        float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x),
    )
    tex_path.write_text(tex, encoding="utf-8")
    print(f"[SAVED] {csv_path}")
    print(f"[SAVED] {md_path}")
    print(f"[SAVED] {tex_path}")


def save_json(obj: dict, path: Path) -> None:
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")


def parse_sample_index(path: Path) -> int:
    # Expected examples: Inspire 2_2m_001.png
    stem = path.stem
    m = re.search(r"_(\d+)$", stem)
    if not m:
        raise ValueError(f"Could not parse sample index from {path}")
    return int(m.group(1))


def scenario_short_name(sensors: List[str]) -> str:
    return "+".join(SENSOR_SHORT[s] for s in sensors)


# ============================================================
# INDEXING
# ============================================================

def build_synchronized_index(cfg: Config) -> pd.DataFrame:
    ensure_dir(cfg.out_dir)
    index_path = cfg.out_dir / "synchronized_tuple_index.csv"

    if index_path.exists() and not cfg.force_reindex:
        print(f"Loading cached synchronized index: {index_path}")
        return pd.read_csv(index_path)

    print("Building synchronized tuple index from Dataset folder...")
    records = {}

    for sensor in SENSORS:
        sensor_dir = cfg.dataset_dir / sensor
        if not sensor_dir.exists():
            raise FileNotFoundError(f"Sensor folder not found: {sensor_dir}")

        for distance in DISTANCE_VALUES:
            distance_token = f"{distance}m"
            for target in TARGETS:
                image_dir = sensor_dir / distance_token / target / "Image File"
                if not image_dir.exists():
                    raise FileNotFoundError(f"Image folder not found: {image_dir}")

                pngs = sorted(image_dir.glob("*.png"))
                if len(pngs) == 0:
                    raise FileNotFoundError(f"No PNG files found in: {image_dir}")

                for p in pngs:
                    sample_index = parse_sample_index(p)
                    tuple_id = f"{target}__{distance_token}__{sample_index:03d}"
                    if tuple_id not in records:
                        records[tuple_id] = {
                            "tuple_id": tuple_id,
                            "target": target,
                            "distance": distance_token,
                            "distance_m": int(distance),
                            "sample_index": int(sample_index),
                        }
                    records[tuple_id][sensor] = str(p)

    df = pd.DataFrame(records.values())

    missing = []
    for sensor in SENSORS:
        n_missing = df[sensor].isna().sum() if sensor in df.columns else len(df)
        if n_missing > 0:
            missing.append((sensor, int(n_missing)))
    if missing:
        raise ValueError(f"Missing sensor paths in synchronized index: {missing}")

    df = df.sort_values(["target", "distance_m", "sample_index"]).reset_index(drop=True)
    df.to_csv(index_path, index=False, encoding="utf-8-sig")
    print(f"Saved synchronized index: {index_path}")
    print(f"Tuples: {len(df)}")
    return df


def attach_or_create_split(cfg: Config, index_df: pd.DataFrame) -> pd.DataFrame:
    if cfg.split_csv is not None and cfg.split_csv.exists():
        print(f"Loading split file: {cfg.split_csv}")
        split_df = pd.read_csv(cfg.split_csv)
        if "split" not in split_df.columns:
            raise ValueError(f"Split CSV has no 'split' column: {cfg.split_csv}")
        # Keep only tuple IDs available in the current index.
        keep_cols = ["tuple_id", "split"]
        split_map = split_df[keep_cols].drop_duplicates("tuple_id")
        df = index_df.merge(split_map, on="tuple_id", how="inner")
        if len(df) == 0:
            raise ValueError("No overlap between synchronized index and split file.")
        if len(df) < len(index_df):
            print(f"[WARN] Split file covers {len(df)} of {len(index_df)} tuples. This can be expected for a cleaned split.")
        out_path = cfg.out_dir / "active_split_index.csv"
        df.to_csv(out_path, index=False, encoding="utf-8-sig")
        print(f"Saved active split index: {out_path}")
        return df

    print("No split file found. Creating 70/15/15 stratified split by target-distance.")
    df = index_df.copy()
    strat = df["target"].astype(str) + "_" + df["distance_m"].astype(str)

    idx = np.arange(len(df))
    train_idx, temp_idx = train_test_split(
        idx,
        test_size=0.30,
        random_state=cfg.seed,
        stratify=strat,
    )
    temp = df.iloc[temp_idx]
    temp_strat = temp["target"].astype(str) + "_" + temp["distance_m"].astype(str)
    val_local, test_local = train_test_split(
        np.arange(len(temp)),
        test_size=0.50,
        random_state=cfg.seed,
        stratify=temp_strat,
    )
    val_idx = temp.index[val_local].to_numpy()
    test_idx = temp.index[test_local].to_numpy()

    df["split"] = ""
    df.loc[train_idx, "split"] = "train"
    df.loc[val_idx, "split"] = "val"
    df.loc[test_idx, "split"] = "test"

    out_path = cfg.out_dir / "tuple_splits_70_15_15_created.csv"
    df.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"Saved created split: {out_path}")
    return df


# ============================================================
# FEATURE EXTRACTION
# ============================================================

def extract_image_features(path: str, image_size: int = 32) -> np.ndarray:
    img = Image.open(path).convert("L").resize((image_size, image_size), Image.BILINEAR)
    arr = np.asarray(img, dtype=np.float32) / 255.0

    flat = arr.reshape(-1)
    stats = np.array([
        arr.mean(), arr.std(), arr.min(), arr.max(),
        np.quantile(arr, 0.01), np.quantile(arr, 0.05), np.quantile(arr, 0.25),
        np.quantile(arr, 0.50), np.quantile(arr, 0.75), np.quantile(arr, 0.95), np.quantile(arr, 0.99),
    ], dtype=np.float32)

    row_mean = arr.mean(axis=1).astype(np.float32)
    col_mean = arr.mean(axis=0).astype(np.float32)
    row_std = arr.std(axis=1).astype(np.float32)
    col_std = arr.std(axis=0).astype(np.float32)

    # Low-frequency FFT magnitude block.
    fft = np.abs(np.fft.rfft2(arr))
    fft = np.log1p(fft)
    fft_block = fft[:8, :8].reshape(-1).astype(np.float32)

    feat = np.concatenate([flat, stats, row_mean, col_mean, row_std, col_std, fft_block]).astype(np.float32)
    return feat


def build_or_load_sensor_features(cfg: Config, split_df: pd.DataFrame, sensor: str) -> Tuple[np.ndarray, np.ndarray]:
    ensure_dir(cfg.feature_dir)
    fpath = cfg.feature_dir / f"features_{safe_name(sensor)}_img{cfg.image_size}.npz"

    if fpath.exists() and not cfg.force_features:
        data = np.load(fpath, allow_pickle=True)
        tuple_ids = data["tuple_ids"]
        X = data["X"].astype(np.float32)
        print(f"Loaded features for {sensor}: {fpath}, shape={X.shape}")
        return tuple_ids, X

    print(f"Extracting image features for {sensor}...")
    rows = split_df[["tuple_id", sensor]].drop_duplicates("tuple_id").reset_index(drop=True)
    tuple_ids = rows["tuple_id"].astype(str).to_numpy()

    feats = []
    for p in tqdm(rows[sensor].astype(str).tolist(), desc=f"Features {sensor}"):
        feats.append(extract_image_features(p, image_size=cfg.image_size))
    X = np.vstack(feats).astype(np.float32)

    np.savez_compressed(fpath, tuple_ids=tuple_ids, X=X)
    print(f"Saved features for {sensor}: {fpath}, shape={X.shape}")
    return tuple_ids, X


def load_all_features(cfg: Config, split_df: pd.DataFrame) -> Dict[str, Dict[str, object]]:
    feature_store = {}
    for sensor in SENSORS:
        tuple_ids, X = build_or_load_sensor_features(cfg, split_df, sensor)
        index = {tid: i for i, tid in enumerate(tuple_ids.astype(str).tolist())}
        feature_store[sensor] = {"tuple_ids": tuple_ids.astype(str), "X": X, "index": index}
    return feature_store


def build_feature_matrix(split_df: pd.DataFrame, feature_store: Dict[str, Dict[str, object]], sensors: List[str]) -> np.ndarray:
    tuple_ids = split_df["tuple_id"].astype(str).tolist()
    matrices = []
    for sensor in sensors:
        Xs = feature_store[sensor]["X"]
        idx_map = feature_store[sensor]["index"]
        indices = [idx_map[tid] for tid in tuple_ids]
        matrices.append(Xs[indices])
    return np.concatenate(matrices, axis=1).astype(np.float32)


# ============================================================
# TASK DATASETS
# ============================================================

def build_task_df(split_df: pd.DataFrame, task: str) -> pd.DataFrame:
    df = split_df.copy()

    if task == "target_classification":
        df["label"] = df["target"]
        df["label_numeric_distance"] = np.nan

    elif task == "drone_vs_non_drone":
        df["label"] = np.where(df["target"] == NON_DRONE_TARGET, "Non-drone", "Drone")
        df["label_numeric_distance"] = np.nan

    elif task == "drone_model_identification":
        df = df[df["target"].isin(DRONE_TARGETS)].copy()
        df["label"] = df["target"]
        df["label_numeric_distance"] = np.nan

    elif task == "distance_estimation":
        df["label"] = df["distance_m"].astype(int).astype(str) + "m"
        df["label_numeric_distance"] = df["distance_m"].astype(int)

    else:
        raise ValueError(f"Unknown task: {task}")

    return df.reset_index(drop=True)


def maybe_subsample_train(df: pd.DataFrame, max_train_samples: int, seed: int) -> pd.DataFrame:
    if max_train_samples <= 0:
        return df
    train = df[df["split"] == "train"].copy()
    other = df[df["split"] != "train"].copy()
    if len(train) <= max_train_samples:
        return df
    strat = train["label"].astype(str)
    train_sub, _ = train_test_split(
        train,
        train_size=max_train_samples,
        random_state=seed,
        stratify=strat,
    )
    out = pd.concat([train_sub, other], ignore_index=True)
    return out


# ============================================================
# MODELS AND METRICS
# ============================================================

def make_models(cfg: Config) -> Dict[str, object]:
    models = {
        "Ridge": make_pipeline(
            StandardScaler(),
            RidgeClassifier(alpha=1.0),
        ),
        "ExtraTrees": ExtraTreesClassifier(
            n_estimators=300 if cfg.model_preset != "fast" else 150,
            max_features="sqrt",
            min_samples_leaf=1,
            class_weight="balanced",
            random_state=cfg.seed,
            n_jobs=cfg.n_jobs,
        ),
    }

    if cfg.model_preset in ["standard", "full"]:
        models["LinearSVM"] = make_pipeline(
            StandardScaler(),
            LinearSVC(C=1.0, class_weight="balanced", random_state=cfg.seed, max_iter=5000),
        )
        models["LogReg"] = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                C=1.0,
                solver="saga",
                penalty="l2",
                class_weight="balanced",
                max_iter=1000,
                random_state=cfg.seed,
                n_jobs=cfg.n_jobs,
            ),
        )

    if cfg.model_preset == "full":
        models["RandomForest"] = RandomForestClassifier(
            n_estimators=300,
            max_features="sqrt",
            min_samples_leaf=1,
            class_weight="balanced",
            random_state=cfg.seed,
            n_jobs=cfg.n_jobs,
        )

    return models


def get_score_or_proba(model, X: np.ndarray) -> Tuple[Optional[np.ndarray], Optional[np.ndarray]]:
    proba = None
    score = None
    if hasattr(model, "predict_proba"):
        try:
            proba = model.predict_proba(X)
        except Exception:
            proba = None
    if hasattr(model, "decision_function"):
        try:
            score = model.decision_function(X)
        except Exception:
            score = None
    return proba, score


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray, labels: List[str], task: str, y_true_distance: Optional[np.ndarray] = None, y_pred_labels: Optional[np.ndarray] = None, proba: Optional[np.ndarray] = None, score: Optional[np.ndarray] = None) -> Dict[str, float]:
    out = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "kappa": cohen_kappa_score(y_true, y_pred),
    }

    # Optional ROC-AUC if score/proba is available.
    try:
        if len(labels) == 2:
            if proba is not None and proba.ndim == 2 and proba.shape[1] == 2:
                out["roc_auc"] = roc_auc_score(y_true, proba[:, 1])
            elif score is not None:
                s = score[:, 1] if getattr(score, "ndim", 1) == 2 else score
                out["roc_auc"] = roc_auc_score(y_true, s)
            else:
                out["roc_auc"] = np.nan
        else:
            if proba is not None and proba.ndim == 2 and proba.shape[1] == len(labels):
                out["roc_auc_ovr"] = roc_auc_score(y_true, proba, multi_class="ovr", average="macro")
            else:
                out["roc_auc_ovr"] = np.nan
    except Exception:
        if len(labels) == 2:
            out["roc_auc"] = np.nan
        else:
            out["roc_auc_ovr"] = np.nan

    # Distance classification auxiliary metrics.
    if task == "distance_estimation" and y_true_distance is not None and y_pred_labels is not None:
        pred_distance = np.array([int(str(x).replace("m", "")) for x in y_pred_labels], dtype=float)
        true_distance = y_true_distance.astype(float)
        out["distance_mae_m"] = mean_absolute_error(true_distance, pred_distance)
        out["distance_rmse_m"] = math.sqrt(mean_squared_error(true_distance, pred_distance))
        out["distance_exact_accuracy"] = out["accuracy"]
        out["distance_within_2m_accuracy"] = float(np.mean(np.abs(true_distance - pred_distance) <= 2.0))
        out["distance_within_4m_accuracy"] = float(np.mean(np.abs(true_distance - pred_distance) <= 4.0))

    return out


# ============================================================
# TRAIN/EVALUATE LOOP
# ============================================================

def train_evaluate_all(cfg: Config, split_df: pd.DataFrame, feature_store: Dict[str, Dict[str, object]]) -> Tuple[pd.DataFrame, pd.DataFrame]:
    ensure_dir(cfg.metrics_dir)
    ensure_dir(cfg.prediction_dir)

    models = make_models(cfg)
    metric_rows = []
    pred_rows_all = []

    for task in TASKS:
        print("\n" + "=" * 100)
        print(f"TASK: {TASK_LABELS[task]} ({task})")
        print("=" * 100)

        task_df0 = build_task_df(split_df, task)
        task_df0 = maybe_subsample_train(task_df0, cfg.max_train_samples, cfg.seed)

        if cfg.train_on_train_val:
            task_df0["fit_split"] = np.where(task_df0["split"].isin(["train", "val"]), "train", task_df0["split"])
        else:
            task_df0["fit_split"] = task_df0["split"]

        for scenario_name, scenario_sensors in SENSOR_SCENARIOS:
            print(f"\n[SCENARIO] {scenario_name} | sensors={scenario_sensors}")

            task_df = task_df0.reset_index(drop=True)
            X = build_feature_matrix(task_df, feature_store, scenario_sensors)

            le = LabelEncoder()
            y = le.fit_transform(task_df["label"].astype(str).to_numpy())
            labels = le.classes_.astype(str).tolist()

            train_mask = task_df["fit_split"].to_numpy() == "train"
            val_mask = task_df["split"].to_numpy() == "val"
            test_mask = task_df["split"].to_numpy() == "test"

            X_train, y_train = X[train_mask], y[train_mask]
            print(f"  X shape={X.shape}, train={X_train.shape}, val={val_mask.sum()}, test={test_mask.sum()}, classes={labels}")

            for model_name, model_proto in models.items():
                print(f"  [MODEL] {model_name}")
                t0 = time.time()
                model = clone(model_proto)
                model.fit(X_train, y_train)
                train_time = time.time() - t0

                for eval_split, mask in [("val", val_mask), ("test", test_mask)]:
                    if mask.sum() == 0:
                        continue
                    X_eval = X[mask]
                    y_eval = y[mask]
                    eval_df = task_df.loc[mask].copy().reset_index(drop=True)

                    t1 = time.time()
                    y_pred = model.predict(X_eval)
                    infer_time = time.time() - t1
                    pred_labels = le.inverse_transform(y_pred)
                    true_labels = le.inverse_transform(y_eval)

                    proba, score = get_score_or_proba(model, X_eval)

                    y_true_distance = eval_df["label_numeric_distance"].to_numpy() if task == "distance_estimation" else None

                    metrics = compute_metrics(
                        y_eval,
                        y_pred,
                        labels=labels,
                        task=task,
                        y_true_distance=y_true_distance,
                        y_pred_labels=pred_labels,
                        proba=proba,
                        score=score,
                    )

                    row = {
                        "task": task,
                        "task_label": TASK_LABELS[task],
                        "scenario": scenario_name,
                        "sensors": scenario_short_name(scenario_sensors),
                        "model": model_name,
                        "split": eval_split,
                        "n_train": int(train_mask.sum()),
                        "n_eval": int(mask.sum()),
                        "n_classes": len(labels),
                        "feature_dim": int(X.shape[1]),
                        "training_time_sec": train_time,
                        "inference_time_sec": infer_time,
                        "latency_ms_per_sample": 1000.0 * infer_time / max(1, int(mask.sum())),
                        **metrics,
                    }
                    metric_rows.append(row)

                    pred_df = pd.DataFrame({
                        "task": task,
                        "task_label": TASK_LABELS[task],
                        "scenario": scenario_name,
                        "sensors": scenario_short_name(scenario_sensors),
                        "model": model_name,
                        "split": eval_split,
                        "tuple_id": eval_df["tuple_id"].to_numpy(),
                        "target": eval_df["target"].to_numpy(),
                        "distance_m": eval_df["distance_m"].to_numpy(),
                        "sample_index": eval_df["sample_index"].to_numpy(),
                        "y_true": true_labels,
                        "y_pred": pred_labels,
                        "correct": true_labels == pred_labels,
                    })
                    pred_rows_all.append(pred_df)

                    pred_path = cfg.prediction_dir / f"pred_{safe_name(task)}__{safe_name(scenario_name)}__{safe_name(model_name)}__{eval_split}.csv"
                    pred_df.to_csv(pred_path, index=False, encoding="utf-8-sig")

                    print(
                        f"    {eval_split}: acc={metrics['accuracy']:.4f}, "
                        f"bal_acc={metrics['balanced_accuracy']:.4f}, macro_f1={metrics['macro_f1']:.4f}, "
                        f"latency={row['latency_ms_per_sample']:.3f} ms/sample"
                    )

    metrics_df = pd.DataFrame(metric_rows)
    preds_df = pd.concat(pred_rows_all, ignore_index=True) if pred_rows_all else pd.DataFrame()

    metrics_path = cfg.metrics_dir / "all_metrics.csv"
    preds_path = cfg.prediction_dir / "all_predictions.csv"
    metrics_df.to_csv(metrics_path, index=False, encoding="utf-8-sig")
    preds_df.to_csv(preds_path, index=False, encoding="utf-8-sig")
    print(f"\nSaved metrics: {metrics_path}")
    print(f"Saved predictions: {preds_path}")
    return metrics_df, preds_df


# ============================================================
# SUMMARY TABLES
# ============================================================

def build_summary_tables(cfg: Config, metrics_df: pd.DataFrame, preds_df: pd.DataFrame) -> None:
    ensure_dir(cfg.table_dir)

    # Test metrics for all model/scenario/task combinations.
    test_metrics = metrics_df[metrics_df["split"] == "test"].copy()
    test_metrics["scenario_order"] = test_metrics["scenario"].apply(lambda x: SCENARIO_ORDER.index(x) if x in SCENARIO_ORDER else 999)
    test_metrics = test_metrics.sort_values(["task", "scenario_order", "model"])

    display_cols = [
        "task_label", "scenario", "sensors", "model", "n_eval", "feature_dim",
        "accuracy", "balanced_accuracy", "macro_precision", "macro_recall", "macro_f1", "weighted_f1", "kappa",
        "distance_mae_m", "distance_rmse_m", "distance_within_2m_accuracy",
        "training_time_sec", "latency_ms_per_sample",
    ]
    display_cols = [c for c in display_cols if c in test_metrics.columns]
    table_all = test_metrics[display_cols].copy()
    for c in table_all.select_dtypes(include=[np.number]).columns:
        table_all[c] = table_all[c].round(4)
    display_table(
        "Table 1 — Test performance for all tasks, sensor scenarios, and baseline models",
        table_all,
        cfg.table_dir,
        "table_01_all_test_metrics",
        caption="Test performance for all tasks, sensor scenarios, and simple baseline models.",
        label="tab:all_test_metrics",
    )

    # Best model selected by validation macro-F1 for each task/scenario.
    val = metrics_df[metrics_df["split"] == "val"].copy()
    if len(val) > 0:
        idx = val.groupby(["task", "scenario"])["macro_f1"].idxmax()
        best_val = val.loc[idx, ["task", "scenario", "model"]].copy()
        best_test = test_metrics.merge(best_val, on=["task", "scenario", "model"], how="inner")
    else:
        idx = test_metrics.groupby(["task", "scenario"])["macro_f1"].idxmax()
        best_test = test_metrics.loc[idx].copy()

    best_test["scenario_order"] = best_test["scenario"].apply(lambda x: SCENARIO_ORDER.index(x) if x in SCENARIO_ORDER else 999)
    best_test = best_test.sort_values(["task", "scenario_order"])
    best_cols = [
        "task_label", "scenario", "sensors", "model", "accuracy", "balanced_accuracy", "macro_f1", "weighted_f1", "kappa",
        "distance_mae_m", "distance_rmse_m", "distance_within_2m_accuracy", "latency_ms_per_sample",
    ]
    best_cols = [c for c in best_cols if c in best_test.columns]
    table_best = best_test[best_cols].copy()
    for c in table_best.select_dtypes(include=[np.number]).columns:
        table_best[c] = table_best[c].round(4)
    display_table(
        "Table 2 — Best validation-selected baseline per task and sensor scenario",
        table_best,
        cfg.table_dir,
        "table_02_best_model_by_task_scenario",
        caption="Best validation-selected baseline model for each task and sensor scenario.",
        label="tab:best_model_by_task_scenario",
    )

    # Compact macro-F1 matrix per task.
    compact = best_test.pivot_table(index="scenario", columns="task_label", values="macro_f1").reset_index()
    compact["scenario_order"] = compact["scenario"].apply(lambda x: SCENARIO_ORDER.index(x) if x in SCENARIO_ORDER else 999)
    compact = compact.sort_values("scenario_order").drop(columns="scenario_order")
    for c in compact.select_dtypes(include=[np.number]).columns:
        compact[c] = compact[c].round(4)
    display_table(
        "Table 3 — Compact Macro-F1 matrix for best baselines",
        compact,
        cfg.table_dir,
        "table_03_compact_macro_f1_matrix",
        caption="Macro-F1 matrix for validation-selected best baselines across tasks and sensor scenarios.",
        label="tab:compact_macro_f1_matrix",
    )

    # Save best test for plotting.
    best_test.to_csv(cfg.metrics_dir / "best_test_metrics_by_val_selection.csv", index=False, encoding="utf-8-sig")


# ============================================================
# PLOTS
# ============================================================

def get_best_predictions(metrics_df: pd.DataFrame, preds_df: pd.DataFrame) -> pd.DataFrame:
    val = metrics_df[metrics_df["split"] == "val"].copy()
    test = metrics_df[metrics_df["split"] == "test"].copy()
    if len(val) > 0:
        idx = val.groupby(["task", "scenario"])["macro_f1"].idxmax()
        best_models = val.loc[idx, ["task", "scenario", "model"]]
    else:
        idx = test.groupby(["task", "scenario"])["macro_f1"].idxmax()
        best_models = test.loc[idx, ["task", "scenario", "model"]]
    out = preds_df[preds_df["split"] == "test"].merge(best_models, on=["task", "scenario", "model"], how="inner")
    return out


def plot_generic_task_accuracy_by_distance(cfg: Config, best_preds: pd.DataFrame) -> None:
    ensure_dir(cfg.figure_dir)
    for task in TASKS:
        g = best_preds[best_preds["task"] == task].copy()
        if len(g) == 0:
            continue
        dist_acc = g.groupby(["scenario", "distance_m"])["correct"].mean().reset_index(name="accuracy")
        dist_acc["scenario_order"] = dist_acc["scenario"].apply(lambda x: SCENARIO_ORDER.index(x) if x in SCENARIO_ORDER else 999)
        dist_acc = dist_acc.sort_values(["scenario_order", "distance_m"])

        fig, ax = plt.subplots(figsize=(11, 5))
        for scenario in SCENARIO_ORDER:
            s = dist_acc[dist_acc["scenario"] == scenario]
            if len(s) > 0:
                ax.plot(s["distance_m"], 100 * s["accuracy"], marker="o", label=scenario)
        ax.set_title(f"Accuracy by distance — {TASK_LABELS[task]}")
        ax.set_xlabel("Distance (m)")
        ax.set_ylabel("Accuracy (%)")
        ax.set_xticks(DISTANCE_VALUES)
        ax.set_ylim(0, 105)
        ax.grid(True, alpha=0.35)
        ax.legend(fontsize=8, ncol=2)
        fig.tight_layout()
        out = cfg.figure_dir / f"accuracy_by_distance__{safe_name(task)}.png"
        fig.savefig(out, dpi=300)
        plt.show()
        print(f"[SAVED] {out}")


def plot_target_accuracy_by_distance_per_scenario(cfg: Config, best_preds: pd.DataFrame) -> None:
    # Like the example figures: per-target accuracy curves for each sensor/fusion scenario.
    task = "target_classification"
    g0 = best_preds[best_preds["task"] == task].copy()
    if len(g0) == 0:
        return

    out_dir = ensure_dir(cfg.figure_dir / "target_accuracy_by_distance")

    for scenario in SCENARIO_ORDER:
        g = g0[g0["scenario"] == scenario].copy()
        if len(g) == 0:
            continue

        acc = g.groupby(["target", "distance_m"])["correct"].mean().reset_index(name="accuracy")

        fig, ax = plt.subplots(figsize=(9, 5))
        for target in TARGETS:
            s = acc[acc["target"] == target].sort_values("distance_m")
            if len(s) > 0:
                ax.plot(s["distance_m"], 100 * s["accuracy"], marker="o", label=target)
        ax.set_title(f"Accuracy of target identification using {scenario}")
        ax.set_xlabel("Distance (m)")
        ax.set_ylabel("Accuracy (%)")
        ax.set_xticks(DISTANCE_VALUES)
        ax.set_ylim(0, 105)
        ax.grid(True, alpha=0.35)
        ax.legend(fontsize=8)
        fig.tight_layout()
        out = out_dir / f"target_accuracy_by_distance__{safe_name(scenario)}.png"
        fig.savefig(out, dpi=300)
        plt.show()
        print(f"[SAVED] {out}")


def plot_average_target_accuracy_bars(cfg: Config, best_preds: pd.DataFrame) -> None:
    task = "target_classification"
    g = best_preds[best_preds["task"] == task].copy()
    if len(g) == 0:
        return

    acc = g.groupby(["scenario", "target"])["correct"].mean().reset_index(name="accuracy")
    pivot = acc.pivot(index="target", columns="scenario", values="accuracy").reindex(TARGETS)
    scenario_cols = [s for s in SCENARIO_ORDER if s in pivot.columns]
    pivot = pivot[scenario_cols]

    fig, ax = plt.subplots(figsize=(12, 5.5))
    x = np.arange(len(pivot.index))
    width = 0.8 / max(1, len(scenario_cols))
    for i, scenario in enumerate(scenario_cols):
        ax.bar(x + (i - len(scenario_cols) / 2) * width + width / 2, 100 * pivot[scenario].to_numpy(), width, label=scenario)
    ax.set_title("Average target identification accuracy by sensor scenario")
    ax.set_xlabel("Target type")
    ax.set_ylabel("Average identification accuracy (%)")
    ax.set_xticks(x)
    ax.set_xticklabels(pivot.index, rotation=0)
    ax.set_ylim(0, 105)
    ax.grid(axis="y", alpha=0.30)
    ax.legend(fontsize=8, ncol=4)
    fig.tight_layout()
    out = cfg.figure_dir / "average_target_accuracy_by_scenario_and_target.png"
    fig.savefig(out, dpi=300)
    plt.show()
    print(f"[SAVED] {out}")

    table = pivot.reset_index()
    for c in scenario_cols:
        table[c] = (100 * table[c]).round(2)
    display_table(
        "Table 4 — Average target identification accuracy by scenario and target (%)",
        table,
        cfg.table_dir,
        "table_04_average_target_accuracy_by_scenario_target",
        caption="Average target identification accuracy by scenario and target.",
        label="tab:average_target_accuracy_by_scenario_target",
    )


def plot_mean_accuracy_by_scenario(cfg: Config, best_preds: pd.DataFrame) -> None:
    task = "target_classification"
    g = best_preds[best_preds["task"] == task].copy()
    if len(g) == 0:
        return

    acc = g.groupby("scenario")["correct"].mean().reset_index(name="accuracy")
    acc["scenario_order"] = acc["scenario"].apply(lambda x: SCENARIO_ORDER.index(x) if x in SCENARIO_ORDER else 999)
    acc = acc.sort_values("scenario_order")

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(acc["scenario"], 100 * acc["accuracy"])
    ax.set_title("Mean target identification accuracy by sensor scenario")
    ax.set_xlabel("Sensor scenario")
    ax.set_ylabel("Mean identification accuracy (%)")
    ax.set_ylim(0, 105)
    ax.grid(axis="y", alpha=0.30)
    ax.tick_params(axis="x", rotation=25)
    fig.tight_layout()
    out = cfg.figure_dir / "mean_target_accuracy_by_scenario.png"
    fig.savefig(out, dpi=300)
    plt.show()
    print(f"[SAVED] {out}")


def plot_confusion_matrices(cfg: Config, best_preds: pd.DataFrame) -> None:
    out_dir = ensure_dir(cfg.figure_dir / "confusion_matrices")
    for task in TASKS:
        g_task = best_preds[best_preds["task"] == task].copy()
        if len(g_task) == 0:
            continue
        for scenario in SCENARIO_ORDER:
            g = g_task[g_task["scenario"] == scenario]
            if len(g) == 0:
                continue
            labels = sorted(g["y_true"].astype(str).unique().tolist())
            # Preserve target order if applicable.
            if task == "target_classification":
                labels = TARGETS
            elif task == "drone_model_identification":
                labels = DRONE_TARGETS
            elif task == "drone_vs_non_drone":
                labels = ["Drone", "Non-drone"]
            elif task == "distance_estimation":
                labels = [f"{d}m" for d in DISTANCE_VALUES]

            cm = confusion_matrix(g["y_true"], g["y_pred"], labels=labels)
            cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)

            fig, ax = plt.subplots(figsize=(7, 6))
            im = ax.imshow(cm_norm, vmin=0, vmax=1, aspect="auto")
            ax.set_title(f"Confusion matrix — {TASK_LABELS[task]} — {scenario}")
            ax.set_xlabel("Predicted")
            ax.set_ylabel("True")
            ax.set_xticks(np.arange(len(labels)))
            ax.set_yticks(np.arange(len(labels)))
            ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
            ax.set_yticklabels(labels, fontsize=8)
            for i in range(cm_norm.shape[0]):
                for j in range(cm_norm.shape[1]):
                    ax.text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center", fontsize=7)
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            fig.tight_layout()
            out = out_dir / f"cm__{safe_name(task)}__{safe_name(scenario)}.png"
            fig.savefig(out, dpi=300)
            plt.close(fig)


def generate_all_plots(cfg: Config, metrics_df: pd.DataFrame, preds_df: pd.DataFrame) -> None:
    best_preds = get_best_predictions(metrics_df, preds_df)
    best_path = cfg.prediction_dir / "best_test_predictions_by_val_selection.csv"
    best_preds.to_csv(best_path, index=False, encoding="utf-8-sig")
    print(f"Saved best predictions: {best_path}")

    plot_generic_task_accuracy_by_distance(cfg, best_preds)
    plot_target_accuracy_by_distance_per_scenario(cfg, best_preds)
    plot_average_target_accuracy_bars(cfg, best_preds)
    plot_mean_accuracy_by_scenario(cfg, best_preds)
    plot_confusion_matrices(cfg, best_preds)


# ============================================================
# MAIN
# ============================================================

def main() -> None:
    args = parse_args()
    cfg = build_config(args)

    for d in [cfg.out_dir, cfg.feature_dir, cfg.metrics_dir, cfg.prediction_dir, cfg.figure_dir, cfg.table_dir]:
        ensure_dir(d)

    print("=" * 100)
    print("TSMS-Drone multitask baseline and multimodal fusion benchmark")
    print("=" * 100)
    print(f"ROOT              : {cfg.root}")
    print(f"DATASET_DIR       : {cfg.dataset_dir}")
    print(f"OUT_DIR           : {cfg.out_dir}")
    print(f"SPLIT_CSV         : {cfg.split_csv}")
    print(f"IMAGE_SIZE        : {cfg.image_size}")
    print(f"MODEL_PRESET      : {cfg.model_preset}")
    print(f"MAX_TRAIN_SAMPLES : {cfg.max_train_samples}")
    print(f"TRAIN_ON_TRAIN_VAL: {cfg.train_on_train_val}")
    print("=" * 100)

    t0 = time.time()
    index_df = build_synchronized_index(cfg)
    split_df = attach_or_create_split(cfg, index_df)

    split_summary = split_df.groupby(["split", "target", "distance_m"]).size().reset_index(name="n")
    split_summary.to_csv(cfg.out_dir / "split_summary_by_target_distance.csv", index=False, encoding="utf-8-sig")

    feature_store = load_all_features(cfg, split_df)
    metrics_df, preds_df = train_evaluate_all(cfg, split_df, feature_store)
    build_summary_tables(cfg, metrics_df, preds_df)
    generate_all_plots(cfg, metrics_df, preds_df)

    manifest = {
        "root": str(cfg.root),
        "dataset_dir": str(cfg.dataset_dir),
        "out_dir": str(cfg.out_dir),
        "split_csv": str(cfg.split_csv),
        "image_size": cfg.image_size,
        "model_preset": cfg.model_preset,
        "max_train_samples": cfg.max_train_samples,
        "train_on_train_val": cfg.train_on_train_val,
        "elapsed_sec": time.time() - t0,
        "tasks": TASKS,
        "sensor_scenarios": [{"name": name, "sensors": sensors} for name, sensors in SENSOR_SCENARIOS],
    }
    save_json(manifest, cfg.out_dir / "run_manifest.json")

    print("\nGenerated outputs:")
    for p in sorted(cfg.out_dir.glob("*")):
        print(p)
    print("=" * 100)
    print("[OK] Multitask baseline and fusion benchmark completed.")
    print("=" * 100)


if __name__ == "__main__":
    main()


[INFO] Ignored unknown arguments, probably from Jupyter/IPython: ['--f=c:\\Users\\Luis\\AppData\\Roaming\\jupyter\\runtime\\kernel-v3040e410708fb7b5bf2eb7c62141f275456ffc0f8.json']
TSMS-Drone multitask baseline and multimodal fusion benchmark
ROOT              : D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)
DATASET_DIR       : D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\Dataset
OUT_DIR           : D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_baseline_multitask_fusion_benchmark
SPLIT_CSV         : D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_advanced_fusion_improvements\tuple_splits_70_15_15_full.csv
IMAGE_SIZE        : 32
MODEL_PRESET      : fast
MAX_TRAIN_SAMPLES : 0
TRAIN_ON_TRAIN_VAL: False
Building synchronized tuple index from Dataset folder...
Saved synchronized index: D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_baseline_multitask_fusion_benchmark\synchronized_tuple_index.csv
Tuples: 37500
Loading

Features CW Radar: 100%|██████████| 37500/37500 [03:46<00:00, 165.85it/s]


Saved features for CW Radar: D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_baseline_multitask_fusion_benchmark\features\features_cw_radar_img32.npz, shape=(37500, 1227)
Extracting image features for FMCW Radar...


Features FMCW Radar: 100%|██████████| 37500/37500 [04:11<00:00, 149.35it/s]


Saved features for FMCW Radar: D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_baseline_multitask_fusion_benchmark\features\features_fmcw_radar_img32.npz, shape=(37500, 1227)
Extracting image features for RF Receiver...


...

task_label,scenario,Distance estimation,Drone model identification,Drone vs non-drone,Target classification
3,CW only,0.3936,0.6848,1.0000,0.7475
5,FMCW only,0.7854,0.9159,0.9958,0.9313
6,RF only,0.4270,0.9991,0.9992,0.9988
1,CW + FMCW,0.8229,0.9311,1.0000,0.9454
2,CW + RF,0.6556,0.9984,1.0000,0.9988
4,FMCW + RF,0.8470,0.9982,0.9997,0.9982
0,All sensors,0.8807,0.9982,1.0000,0.9993


[SAVED] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_baseline_multitask_fusion_benchmark\tables\table_03_compact_macro_f1_matrix.csv
[SAVED] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_baseline_multitask_fusion_benchmark\tables\table_03_compact_macro_f1_matrix.md
[SAVED] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_baseline_multitask_fusion_benchmark\tables\table_03_compact_macro_f1_matrix.tex
Saved best predictions: D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_baseline_multitask_fusion_benchmark\predictions\best_test_predictions_by_val_selection.csv
[SAVED] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_baseline_multitask_fusion_benchmark\figures\accuracy_by_distance__target_classification.png
[SAVED] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_baseline_multitask_fusion_benchmark\figures\accuracy_by_distance__drone_vs_non_drone.png
[SAVED] D:\Time-Synchronized Multi-Sensor Drone Datas

### Table 4 — Average target identification accuracy by scenario and target (%)

scenario,target,CW only,FMCW only,RF only,CW + FMCW,CW + RF,FMCW + RF,All sensors
0,Inspire 2,71.11,94.40,100.00,95.56,100.00,100.00,99.91
1,Matrice 30,66.93,92.00,99.73,94.49,99.64,99.47,99.91
2,Mavic 2 Pro,71.47,92.36,99.91,94.84,99.91,99.91,99.91
3,Phantom 4 Pro,62.49,87.11,99.82,87.82,99.82,99.73,99.91
4,Corner Reflector,100.00,99.82,99.91,100.00,100.00,100.00,100.00


[SAVED] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_baseline_multitask_fusion_benchmark\tables\table_04_average_target_accuracy_by_scenario_target.csv
[SAVED] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_baseline_multitask_fusion_benchmark\tables\table_04_average_target_accuracy_by_scenario_target.md
[SAVED] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_baseline_multitask_fusion_benchmark\tables\table_04_average_target_accuracy_by_scenario_target.tex
[SAVED] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_baseline_multitask_fusion_benchmark\figures\mean_target_accuracy_by_scenario.png

Generated outputs:
D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_baseline_multitask_fusion_benchmark\active_split_index.csv
D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_baseline_multitask_fusion_benchmark\features
D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_baseline_multitask_fusion_benchma

# CÓDIGO PROPOSTO

In [34]:
# ============================================================
# tsms_drone_proposed_multitask_attention_fusion_vscode.py
# ============================================================
# Proposed multimodal attention/gated fusion model for TSMS-Drone.
#
# Tasks:
#   1) target_classification: 5 classes
#   2) drone_vs_non_drone: Drone vs Corner Reflector
#   3) drone_model_identification: 4 drone classes, Corner Reflector excluded
#   4) distance_estimation: 15 distance classes, 2m ... 30m
#
# Sensor scenarios evaluated:
#   CW only, FMCW only, RF only, CW+FMCW, CW+RF, FMCW+RF, All sensors
#
# Proposed model:
#   sensor-specific CNN encoders + cross-modal self-attention + reliability gate
#   + auxiliary head. By default, one full multimodal model is trained per task
#   and then evaluated under all sensor masks. Use --train-per-scenario to train
#   one model for each sensor scenario.
#
# Install:
#   pip install numpy pandas pillow tqdm scikit-learn matplotlib torch tabulate
#
# Run in VSCode terminal:
#   python tsms_drone_proposed_multitask_attention_fusion_vscode.py
#
# Smoke test:
#   python tsms_drone_proposed_multitask_attention_fusion_vscode.py --epochs 2 --max-train-samples 3000 --image-size 96
# ============================================================

from __future__ import annotations

import argparse
import copy
import json
import math
import random
import re
import time
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

SENSORS = ["CW Radar", "FMCW Radar", "RF Receiver"]
SENSOR_SHORT = {"CW Radar": "CW", "FMCW Radar": "FMCW", "RF Receiver": "RF"}
TARGETS = ["Inspire 2", "Matrice 30", "Mavic 2 Pro", "Phantom 4 Pro", "Corner Reflector"]
DRONE_TARGETS = ["Inspire 2", "Matrice 30", "Mavic 2 Pro", "Phantom 4 Pro"]
NON_DRONE_TARGET = "Corner Reflector"
DISTANCE_VALUES = list(range(2, 31, 2))
DISTANCE_LABELS = [f"{d}m" for d in DISTANCE_VALUES]

TASKS = ["target_classification", "drone_vs_non_drone", "drone_model_identification", "distance_estimation"]
TASK_LABELS = {
    "target_classification": "Target classification",
    "drone_vs_non_drone": "Drone vs non-drone",
    "drone_model_identification": "Drone model identification",
    "distance_estimation": "Distance estimation",
}

SENSOR_SCENARIOS = [
    ("CW only", ["CW Radar"]),
    ("FMCW only", ["FMCW Radar"]),
    ("RF only", ["RF Receiver"]),
    ("CW + FMCW", ["CW Radar", "FMCW Radar"]),
    ("CW + RF", ["CW Radar", "RF Receiver"]),
    ("FMCW + RF", ["FMCW Radar", "RF Receiver"]),
    ("All sensors", ["CW Radar", "FMCW Radar", "RF Receiver"]),
]
SCENARIO_ORDER = [s[0] for s in SENSOR_SCENARIOS]


@dataclass
class Config:
    root: Path
    dataset_dir: Path
    out_dir: Path
    split_csv: Optional[Path]
    checkpoint_dir: Path
    metrics_dir: Path
    pred_dir: Path
    fig_dir: Path
    table_dir: Path
    image_size: int = 128
    batch_size: int = 32
    epochs: int = 20
    patience: int = 6
    lr: float = 2e-4
    weight_decay: float = 1e-4
    grad_clip: float = 5.0
    emb_dim: int = 192
    attn_heads: int = 4
    attn_layers: int = 2
    dropout: float = 0.20
    sensor_dropout_prob: float = 0.20
    aux_loss_weight: float = 0.20
    max_train_samples: int = 0
    num_workers: int = 0
    seed: int = 42
    device: str = "auto"
    amp: bool = False
    force_reindex: bool = False
    train_on_train_val: bool = False
    train_per_scenario: bool = False
    weighted_sampler: bool = False


def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(
        description="TSMS-Drone proposed attention/gated fusion model.",
        allow_abbrev=False,
    )
    p.add_argument("--root", type=str, default=r"D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)")
    p.add_argument("--dataset-dir", type=str, default=None)
    p.add_argument("--out-dir", type=str, default=None)
    p.add_argument("--split-csv", type=str, default=None)
    p.add_argument("--image-size", type=int, default=128)
    p.add_argument("--batch-size", type=int, default=32)
    p.add_argument("--epochs", type=int, default=20)
    p.add_argument("--patience", type=int, default=6)
    p.add_argument("--lr", type=float, default=2e-4)
    p.add_argument("--weight-decay", type=float, default=1e-4)
    p.add_argument("--grad-clip", type=float, default=5.0)
    p.add_argument("--emb-dim", type=int, default=192)
    p.add_argument("--attn-heads", type=int, default=4)
    p.add_argument("--attn-layers", type=int, default=2)
    p.add_argument("--dropout", type=float, default=0.20)
    p.add_argument("--sensor-dropout-prob", type=float, default=0.20)
    p.add_argument("--aux-loss-weight", type=float, default=0.20)
    p.add_argument("--max-train-samples", type=int, default=0)
    p.add_argument("--num-workers", type=int, default=0)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--device", type=str, default="auto", choices=["auto", "cpu", "cuda"])
    p.add_argument("--amp", action="store_true")
    p.add_argument("--force-reindex", action="store_true")
    p.add_argument("--train-on-train-val", action="store_true")
    p.add_argument("--train-per-scenario", action="store_true")
    p.add_argument("--weighted-sampler", action="store_true")
    args, unknown = p.parse_known_args()
    if unknown:
        print(f"[INFO] Ignored unknown arguments: {unknown}")
    return args


def build_config(a: argparse.Namespace) -> Config:
    root = Path(a.root)
    dataset_dir = Path(a.dataset_dir) if a.dataset_dir else root / "Dataset"
    out_dir = Path(a.out_dir) if a.out_dir else root / "_proposed_multitask_attention_fusion_vscode"
    if a.split_csv:
        split_csv = Path(a.split_csv)
    else:
        clean = root / "_advanced_fusion_improvements" / "tuple_splits_70_15_15_full_clean_no_cw_md5_leak.csv"
        orig = root / "_advanced_fusion_improvements" / "tuple_splits_70_15_15_full.csv"
        split_csv = clean if clean.exists() else orig if orig.exists() else None
    return Config(
        root=root,
        dataset_dir=dataset_dir,
        out_dir=out_dir,
        split_csv=split_csv,
        checkpoint_dir=out_dir / "checkpoints",
        metrics_dir=out_dir / "metrics",
        pred_dir=out_dir / "predictions",
        fig_dir=out_dir / "figures",
        table_dir=out_dir / "tables",
        image_size=a.image_size,
        batch_size=a.batch_size,
        epochs=a.epochs,
        patience=a.patience,
        lr=a.lr,
        weight_decay=a.weight_decay,
        grad_clip=a.grad_clip,
        emb_dim=a.emb_dim,
        attn_heads=a.attn_heads,
        attn_layers=a.attn_layers,
        dropout=a.dropout,
        sensor_dropout_prob=a.sensor_dropout_prob,
        aux_loss_weight=a.aux_loss_weight,
        max_train_samples=a.max_train_samples,
        num_workers=a.num_workers,
        seed=a.seed,
        device=a.device,
        amp=a.amp,
        force_reindex=a.force_reindex,
        train_on_train_val=a.train_on_train_val,
        train_per_scenario=a.train_per_scenario,
        weighted_sampler=a.weighted_sampler,
    )


# ------------------------- utilities -------------------------

def ensure_dir(p: Path) -> Path:
    p.mkdir(parents=True, exist_ok=True)
    return p


def safe_name(s: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", str(s).lower()).strip("_")


def seed_everything(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    if torch.cuda.is_available():
        torch.backends.cudnn.benchmark = True


def get_device(cfg: Config) -> torch.device:
    if cfg.device == "cpu": return torch.device("cpu")
    if cfg.device == "cuda":
        if not torch.cuda.is_available(): raise RuntimeError("CUDA requested but not available")
        return torch.device("cuda")
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def pretty_seconds(sec: float) -> str:
    if sec < 60: return f"{sec:.1f}s"
    if sec < 3600: return f"{sec/60:.1f}min"
    return f"{sec/3600:.2f}h"


def sensor_mask_np(sensors: List[str]) -> np.ndarray:
    return np.array([1.0 if s in sensors else 0.0 for s in SENSORS], dtype=np.float32)


def scenario_short(sensors: List[str]) -> str:
    return "+".join(SENSOR_SHORT[s] for s in sensors)


def save_table(df: pd.DataFrame, out_dir: Path, name: str, caption: str = "", label: str = "") -> None:
    ensure_dir(out_dir)
    csv = out_dir / f"{name}.csv"; md = out_dir / f"{name}.md"; tex = out_dir / f"{name}.tex"
    df.to_csv(csv, index=False, encoding="utf-8-sig")
    try: md.write_text(df.to_markdown(index=False), encoding="utf-8")
    except Exception: md.write_text(df.to_csv(index=False), encoding="utf-8")
    latex = df.to_latex(index=False, escape=False, caption=caption or None, label=label or None,
                        float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x))
    tex.write_text(latex, encoding="utf-8")
    print(f"[SAVED] {csv}\n[SAVED] {md}\n[SAVED] {tex}")


def print_save(title: str, df: pd.DataFrame, out_dir: Path, name: str, caption: str = "", label: str = "") -> None:
    print("\n" + "=" * 100); print(title); print("=" * 100)
    print(df.to_string(index=False))
    save_table(df, out_dir, name, caption, label)


def save_json(obj: dict, path: Path) -> None:
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")


# ------------------------- index/split -------------------------

def parse_sample_index(path: Path) -> int:
    nums = re.findall(r"\d+", path.stem)
    if not nums: raise ValueError(f"Cannot parse sample index from {path.name}")
    return int(nums[-1])


def build_index(cfg: Config) -> pd.DataFrame:
    index_path = cfg.out_dir / "synchronized_tuple_index.csv"
    if index_path.exists() and not cfg.force_reindex:
        print(f"Loading cached synchronized index: {index_path}")
        return pd.read_csv(index_path)
    if not cfg.dataset_dir.exists(): raise FileNotFoundError(cfg.dataset_dir)
    rec: Dict[str, dict] = {}
    print("Building synchronized tuple index...")
    for sensor in SENSORS:
        for d in DISTANCE_VALUES:
            for target in TARGETS:
                folder = cfg.dataset_dir / sensor / f"{d}m" / target / "Image File"
                if not folder.exists(): raise FileNotFoundError(folder)
                for p in sorted(folder.glob("*.png")):
                    si = parse_sample_index(p)
                    tid = f"{target}__{d}m__{si:03d}"
                    rec.setdefault(tid, {"tuple_id": tid, "target": target, "distance": f"{d}m", "distance_m": d, "sample_index": si})
                    rec[tid][sensor] = str(p)
    df = pd.DataFrame(rec.values())
    missing = df[df[SENSORS].isna().any(axis=1)] if all(s in df.columns for s in SENSORS) else df
    if len(missing) > 0:
        missing.to_csv(cfg.out_dir / "incomplete_synchronized_tuples.csv", index=False, encoding="utf-8-sig")
        raise ValueError("Incomplete synchronized tuples found")
    df = df.sort_values(["target", "distance_m", "sample_index"]).reset_index(drop=True)
    df.to_csv(index_path, index=False, encoding="utf-8-sig")
    print(f"Saved index: {index_path}; tuples={len(df)}")
    return df


def attach_or_create_split(cfg: Config, index_df: pd.DataFrame) -> pd.DataFrame:
    if cfg.split_csv and cfg.split_csv.exists():
        print(f"Loading split file: {cfg.split_csv}")
        sp = pd.read_csv(cfg.split_csv)
        if "tuple_id" not in sp.columns or "split" not in sp.columns:
            raise ValueError("split csv must contain tuple_id and split")
        df = index_df.merge(sp[["tuple_id", "split"]].drop_duplicates("tuple_id"), on="tuple_id", how="inner")
        if len(df) == 0: raise ValueError("No overlap between index and split")
        if len(df) < len(index_df): print(f"[INFO] split covers {len(df)} of {len(index_df)} tuples")
        df.to_csv(cfg.out_dir / "active_split_index.csv", index=False, encoding="utf-8-sig")
        return df
    print("Creating stratified 70/15/15 split")
    df = index_df.copy(); strat = df["target"].astype(str) + "_" + df["distance_m"].astype(str)
    idx = np.arange(len(df))
    tr, tmp = train_test_split(idx, test_size=0.30, random_state=cfg.seed, stratify=strat)
    tmp_df = df.iloc[tmp]
    tmp_strat = tmp_df["target"].astype(str) + "_" + tmp_df["distance_m"].astype(str)
    val_local, test_local = train_test_split(np.arange(len(tmp_df)), test_size=0.50, random_state=cfg.seed, stratify=tmp_strat)
    df["split"] = "train"; df.loc[tmp_df.index[val_local], "split"] = "val"; df.loc[tmp_df.index[test_local], "split"] = "test"
    df.to_csv(cfg.out_dir / "tuple_splits_70_15_15_created.csv", index=False, encoding="utf-8-sig")
    return df


def summarize_split(cfg: Config, df: pd.DataFrame) -> None:
    summary = df.groupby(["split", "target"]).size().unstack(fill_value=0).reset_index()
    print_save("Split summary by target", summary, cfg.table_dir, "split_summary_by_target")
    df.groupby(["split", "target", "distance_m"]).size().reset_index(name="n").to_csv(cfg.out_dir / "split_summary_by_target_distance.csv", index=False, encoding="utf-8-sig")


# ------------------------- tasks -------------------------

def build_task_df(df: pd.DataFrame, task: str) -> pd.DataFrame:
    d = df.copy()
    if task == "target_classification":
        d["label"] = d["target"].astype(str)
    elif task == "drone_vs_non_drone":
        d["label"] = np.where(d["target"] == NON_DRONE_TARGET, "Non-drone", "Drone")
    elif task == "drone_model_identification":
        d = d[d["target"].isin(DRONE_TARGETS)].copy(); d["label"] = d["target"].astype(str)
    elif task == "distance_estimation":
        d["label"] = d["distance_m"].astype(int).astype(str) + "m"
    else:
        raise ValueError(task)
    d["distance_label"] = d["distance_m"].astype(int).astype(str) + "m"
    d["target_label"] = d["target"].astype(str)
    return d.reset_index(drop=True)


def maybe_subsample_train(df: pd.DataFrame, max_n: int, seed: int) -> pd.DataFrame:
    if max_n <= 0: return df
    tr = df[df["split"] == "train"].copy(); rest = df[df["split"] != "train"].copy()
    if len(tr) <= max_n: return df
    tr_sub, _ = train_test_split(tr, train_size=max_n, random_state=seed, stratify=tr["label"].astype(str))
    return pd.concat([tr_sub, rest], ignore_index=True)


@dataclass
class Encoders:
    main: LabelEncoder
    distance: LabelEncoder
    target: LabelEncoder


def make_encoders(df: pd.DataFrame) -> Encoders:
    main = LabelEncoder().fit(df["label"].astype(str).to_numpy())
    distance = LabelEncoder().fit(DISTANCE_LABELS)
    target = LabelEncoder().fit(TARGETS)
    return Encoders(main, distance, target)


# ------------------------- dataset -------------------------

def read_image(path: str, image_size: int, augment: bool) -> torch.Tensor:
    img = Image.open(path).convert("L").resize((image_size, image_size), Image.BILINEAR)
    arr = np.asarray(img, dtype=np.float32) / 255.0
    if augment:
        if random.random() < 0.5: arr = np.fliplr(arr).copy()
        if random.random() < 0.20: arr = np.clip(arr + np.random.normal(0, 0.015, arr.shape).astype(np.float32), 0, 1)
        if random.random() < 0.20: arr = np.clip(arr * np.random.uniform(0.90, 1.10), 0, 1)
    arr = (arr - 0.5) / 0.5
    return torch.from_numpy(arr[None, :, :].astype(np.float32))


class TSMSDataset(Dataset):
    def __init__(self, df: pd.DataFrame, enc: Encoders, image_size: int, augment: bool):
        self.df = df.reset_index(drop=True); self.enc = enc; self.image_size = image_size; self.augment = augment
        self.y = enc.main.transform(self.df["label"].astype(str).to_numpy()).astype(np.int64)
        self.yd = enc.distance.transform(self.df["distance_label"].astype(str).to_numpy()).astype(np.int64)
        self.yt = enc.target.transform(self.df["target_label"].astype(str).to_numpy()).astype(np.int64)
    def __len__(self): return len(self.df)
    def __getitem__(self, i: int):
        r = self.df.iloc[i]
        x = torch.stack([read_image(str(r[s]), self.image_size, self.augment) for s in SENSORS], dim=0)
        return {"x": x, "y": torch.tensor(self.y[i]), "yd": torch.tensor(self.yd[i]), "yt": torch.tensor(self.yt[i]),
                "tuple_id": str(r["tuple_id"]), "target": str(r["target"]), "distance_m": int(r["distance_m"]),
                "sample_index": int(r["sample_index"]), "label": str(r["label"])}


def weighted_sampler(labels: np.ndarray) -> WeightedRandomSampler:
    counts = np.bincount(labels); w_cls = 1.0 / np.maximum(counts, 1); weights = w_cls[labels]
    return WeightedRandomSampler(torch.as_tensor(weights, dtype=torch.double), num_samples=len(weights), replacement=True)


def make_loaders(cfg: Config, df: pd.DataFrame, enc: Encoders) -> Tuple[DataLoader, DataLoader, DataLoader]:
    tr_df = df[df["split"].isin(["train", "val"] if cfg.train_on_train_val else ["train"])].copy()
    va_df = df[df["split"] == "val"].copy(); te_df = df[df["split"] == "test"].copy()
    tr_ds = TSMSDataset(tr_df, enc, cfg.image_size, True); va_ds = TSMSDataset(va_df, enc, cfg.image_size, False); te_ds = TSMSDataset(te_df, enc, cfg.image_size, False)
    sampler = weighted_sampler(tr_ds.y) if cfg.weighted_sampler else None
    return (
        DataLoader(tr_ds, batch_size=cfg.batch_size, shuffle=sampler is None, sampler=sampler, num_workers=cfg.num_workers, pin_memory=torch.cuda.is_available()),
        DataLoader(va_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=torch.cuda.is_available()),
        DataLoader(te_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=torch.cuda.is_available()),
    )


# ------------------------- model -------------------------

class SensorEncoder(nn.Module):
    def __init__(self, emb_dim: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 5, 2, 2, bias=False), nn.BatchNorm2d(32), nn.SiLU(inplace=True),
            nn.Conv2d(32, 64, 3, 2, 1, bias=False), nn.BatchNorm2d(64), nn.SiLU(inplace=True),
            nn.Conv2d(64, 96, 3, 2, 1, bias=False), nn.BatchNorm2d(96), nn.SiLU(inplace=True),
            nn.Conv2d(96, 128, 3, 2, 1, bias=False), nn.BatchNorm2d(128), nn.SiLU(inplace=True),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(128, emb_dim), nn.LayerNorm(emb_dim), nn.SiLU(inplace=True), nn.Dropout(dropout),
        )
    def forward(self, x): return self.net(x)


class ProposedFusionNet(nn.Module):
    def __init__(self, n_main: int, n_dist: int, n_target: int, emb_dim: int, heads: int, layers: int, dropout: float):
        super().__init__()
        self.n_sensors = 3
        self.enc = nn.ModuleList([SensorEncoder(emb_dim, dropout) for _ in range(3)])
        self.mod_emb = nn.Parameter(torch.zeros(1, 3, emb_dim)); nn.init.normal_(self.mod_emb, std=0.02)
        layer = nn.TransformerEncoderLayer(d_model=emb_dim, nhead=heads, dim_feedforward=4*emb_dim, dropout=dropout, activation="gelu", batch_first=True, norm_first=True)
        self.attn = nn.TransformerEncoder(layer, num_layers=layers)
        self.gate = nn.Sequential(nn.Linear(emb_dim, emb_dim//2), nn.SiLU(inplace=True), nn.Dropout(dropout), nn.Linear(emb_dim//2, 1))
        self.norm = nn.LayerNorm(emb_dim)
        self.main_head = nn.Sequential(nn.Linear(emb_dim, emb_dim), nn.SiLU(inplace=True), nn.Dropout(dropout), nn.Linear(emb_dim, n_main))
        self.dist_head = nn.Sequential(nn.Linear(emb_dim, emb_dim//2), nn.SiLU(inplace=True), nn.Dropout(dropout), nn.Linear(emb_dim//2, n_dist))
        self.target_head = nn.Sequential(nn.Linear(emb_dim, emb_dim//2), nn.SiLU(inplace=True), nn.Dropout(dropout), nn.Linear(emb_dim//2, n_target))

    def _drop_sensors(self, mask: torch.Tensor, p: float) -> torch.Tensor:
        if not self.training or p <= 0: return mask
        keep = (torch.rand_like(mask) > p).float(); m = mask * keep
        bad = m.sum(1) < 0.5
        for i in torch.where(bad)[0]:
            avail = torch.where(mask[i] > 0.5)[0]
            if len(avail) == 0: m[i, 0] = 1
            else: m[i, avail[torch.randint(len(avail), (1,), device=mask.device)]] = 1
        return m

    def forward(self, x, mask=None, train_sensor_dropout_p: float = 0.0):
        # x [B,3,1,H,W]
        b = x.size(0)
        if mask is None: mask = torch.ones(b, 3, dtype=x.dtype, device=x.device)
        else: mask = mask.to(device=x.device, dtype=x.dtype)
        mask = self._drop_sensors(mask, train_sensor_dropout_p)
        toks = torch.stack([self.enc[i](x[:, i]) for i in range(3)], dim=1) + self.mod_emb
        toks = toks * mask.unsqueeze(-1)
        pad = mask < 0.5
        toks = self.attn(toks, src_key_padding_mask=pad)
        g_logits = self.gate(toks).squeeze(-1).masked_fill(mask < 0.5, -1e9)
        gates = torch.softmax(g_logits, dim=1)
        fused = self.norm((toks * gates.unsqueeze(-1)).sum(1))
        return {"logits": self.main_head(fused), "distance_logits": self.dist_head(fused), "target_logits": self.target_head(fused), "gate": gates}


# ------------------------- train/eval -------------------------

def batch_mask(bs: int, sensors: List[str], device: torch.device) -> torch.Tensor:
    return torch.from_numpy(sensor_mask_np(sensors)).float().unsqueeze(0).repeat(bs, 1).to(device)


def loss_fn(out: dict, batch: dict, task: str, aux_w: float) -> torch.Tensor:
    main = F.cross_entropy(out["logits"], batch["y"])
    aux = F.cross_entropy(out["target_logits"], batch["yt"]) if task == "distance_estimation" else F.cross_entropy(out["distance_logits"], batch["yd"])
    return main + aux_w * aux


def train_epoch(model, loader, opt, scaler, device, cfg, task, train_sensors):
    model.train(); losses = []
    for batch in tqdm(loader, desc="train", leave=False):
        x = batch["x"].to(device, non_blocking=True)
        for k in ["y", "yd", "yt"]: batch[k] = batch[k].to(device, non_blocking=True)
        mask = batch_mask(x.size(0), train_sensors, device)
        opt.zero_grad(set_to_none=True)
        if cfg.amp and device.type == "cuda":
            with torch.cuda.amp.autocast():
                out = model(x, mask, cfg.sensor_dropout_prob if len(train_sensors) > 1 else 0.0)
                loss = loss_fn(out, batch, task, cfg.aux_loss_weight)
            scaler.scale(loss).backward()
            if cfg.grad_clip > 0:
                scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(opt); scaler.update()
        else:
            out = model(x, mask, cfg.sensor_dropout_prob if len(train_sensors) > 1 else 0.0)
            loss = loss_fn(out, batch, task, cfg.aux_loss_weight)
            loss.backward()
            if cfg.grad_clip > 0: nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            opt.step()
        losses.append(float(loss.detach().cpu()))
    return float(np.mean(losses))


def compute_metrics(y_true, y_pred, proba, classes, task, dist_true=None, dist_pred=None) -> dict:
    out = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "kappa": cohen_kappa_score(y_true, y_pred),
    }
    try:
        if len(classes) == 2: out["roc_auc"] = roc_auc_score(y_true, proba[:, 1])
        else: out["roc_auc_ovr"] = roc_auc_score(y_true, proba, multi_class="ovr", average="macro")
    except Exception:
        out["roc_auc" if len(classes) == 2 else "roc_auc_ovr"] = np.nan
    if task == "distance_estimation" and dist_true is not None and dist_pred is not None:
        out["distance_mae_m"] = mean_absolute_error(dist_true, dist_pred)
        out["distance_rmse_m"] = math.sqrt(mean_squared_error(dist_true, dist_pred))
        out["distance_exact_accuracy"] = out["accuracy"]
        out["distance_within_2m_accuracy"] = float(np.mean(np.abs(dist_true - dist_pred) <= 2))
        out["distance_within_4m_accuracy"] = float(np.mean(np.abs(dist_true - dist_pred) <= 4))
    return out


@torch.no_grad()
def evaluate(model, loader, device, task, enc: Encoders, scenario_name: str, sensors: List[str]):
    model.eval(); rows=[]; ys=[]; ps=[]; probs=[]; dt=[]; dp=[]
    for batch in tqdm(loader, desc=f"eval {scenario_name}", leave=False):
        x = batch["x"].to(device, non_blocking=True)
        out = model(x, batch_mask(x.size(0), sensors, device), 0.0)
        proba = torch.softmax(out["logits"], dim=1).cpu().numpy()
        pred = proba.argmax(1); true = batch["y"].numpy()
        ylab = enc.main.inverse_transform(true); plab = enc.main.inverse_transform(pred)
        gates = out["gate"].cpu().numpy()
        ys.append(true); ps.append(pred); probs.append(proba)
        if task == "distance_estimation":
            td = np.array([int(str(v).replace("m", "")) for v in ylab], dtype=float)
            pd_ = np.array([int(str(v).replace("m", "")) for v in plab], dtype=float)
            dt.append(td); dp.append(pd_)
        for i in range(len(true)):
            r = {"task": task, "task_label": TASK_LABELS[task], "eval_scenario": scenario_name, "eval_sensors": scenario_short(sensors),
                 "tuple_id": batch["tuple_id"][i], "target": batch["target"][i], "distance_m": int(batch["distance_m"][i]),
                 "sample_index": int(batch["sample_index"][i]), "y_true": str(ylab[i]), "y_pred": str(plab[i]), "correct": bool(ylab[i] == plab[i])}
            for j, s in enumerate(SENSORS): r[f"gate_{safe_name(s)}"] = float(gates[i, j])
            for j, c in enumerate(enc.main.classes_): r[f"proba_{safe_name(c)}"] = float(proba[i, j])
            rows.append(r)
    y = np.concatenate(ys); p = np.concatenate(ps); pr = np.concatenate(probs)
    dist_true = np.concatenate(dt) if dt else None; dist_pred = np.concatenate(dp) if dp else None
    metrics = compute_metrics(y, p, pr, enc.main.classes_.astype(str).tolist(), task, dist_true, dist_pred)
    return pd.DataFrame(rows), metrics


def fit_task_model(cfg: Config, task: str, task_df: pd.DataFrame, enc: Encoders, train_sensors: List[str], name: str, device: torch.device):
    train_loader, val_loader, _ = make_loaders(cfg, task_df, enc)
    model = ProposedFusionNet(len(enc.main.classes_), len(enc.distance.classes_), len(enc.target.classes_), cfg.emb_dim, cfg.attn_heads, cfg.attn_layers, cfg.dropout).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, cfg.epochs))
    scaler = torch.cuda.amp.GradScaler(enabled=(cfg.amp and device.type == "cuda"))
    best_state=None; best_f1=-1; best_epoch=0; wait=0; hist=[]; ckpt = cfg.checkpoint_dir / f"{name}.pt"
    print("\n" + "="*100); print(f"Training proposed model: task={task}, sensors={scenario_short(train_sensors)}"); print("="*100)
    for ep in range(1, cfg.epochs+1):
        t0=time.time(); loss=train_epoch(model, train_loader, opt, scaler, device, cfg, task, train_sensors); sch.step()
        _, vm = evaluate(model, val_loader, device, task, enc, "validation", train_sensors)
        hist.append({"epoch": ep, "train_loss": loss, "val_macro_f1": vm["macro_f1"], "val_accuracy": vm["accuracy"], "lr": opt.param_groups[0]["lr"], "epoch_time_sec": time.time()-t0})
        print(f"Epoch {ep:03d}/{cfg.epochs}: loss={loss:.4f}, val_f1={vm['macro_f1']:.4f}, val_acc={vm['accuracy']:.4f}, time={pretty_seconds(time.time()-t0)}")
        if vm["macro_f1"] > best_f1 + 1e-8:
            best_f1 = vm["macro_f1"]; best_epoch = ep; wait = 0; best_state = copy.deepcopy(model.state_dict())
            torch.save({"model_state": best_state, "epoch": ep, "best_val_f1": best_f1, "task": task, "train_sensors": train_sensors,
                        "main_classes": enc.main.classes_.tolist()}, ckpt)
            print(f"[CHECKPOINT] {ckpt}")
        else:
            wait += 1
        if wait >= cfg.patience:
            print(f"Early stopping at epoch {ep}. Best epoch: {best_epoch}."); break
    if best_state is not None: model.load_state_dict(best_state)
    hist_df = pd.DataFrame(hist); hist_df.to_csv(cfg.metrics_dir / f"history_{name}.csv", index=False, encoding="utf-8-sig")
    return model, hist_df


def run_experiments(cfg: Config, split_df: pd.DataFrame, device: torch.device):
    metrics=[]; pred_tables=[]
    for task in TASKS:
        tdf = maybe_subsample_train(build_task_df(split_df, task), cfg.max_train_samples, cfg.seed)
        enc = make_encoders(tdf)
        training_plan = SENSOR_SCENARIOS if cfg.train_per_scenario else [("All sensors", SENSORS)]
        _, _, test_loader = make_loaders(cfg, tdf, enc)
        for train_name, train_sensors in training_plan:
            model_name = f"{safe_name(task)}__trained_{safe_name(train_name)}"
            model, hist = fit_task_model(cfg, task, tdf, enc, train_sensors, model_name, device)
            eval_plan = [(train_name, train_sensors)] if cfg.train_per_scenario else SENSOR_SCENARIOS
            for eval_name, eval_sensors in eval_plan:
                t0=time.time(); pred_df, m = evaluate(model, test_loader, device, task, enc, eval_name, eval_sensors); elapsed=time.time()-t0
                row = {"task": task, "task_label": TASK_LABELS[task], "train_scenario": train_name, "train_sensors": scenario_short(train_sensors),
                       "eval_scenario": eval_name, "eval_sensors": scenario_short(eval_sensors), "model": "Proposed-Attention-Gated-Fusion",
                       "n_eval": len(pred_df), "n_classes": len(enc.main.classes_), "image_size": cfg.image_size, "emb_dim": cfg.emb_dim,
                       "best_epoch": int(hist["val_macro_f1"].idxmax()+1) if len(hist) else np.nan,
                       "eval_time_sec": elapsed, "latency_ms_per_sample": 1000*elapsed/max(1,len(pred_df)), **m}
                metrics.append(row)
                pred_df["train_scenario"] = train_name; pred_df["train_sensors"] = scenario_short(train_sensors); pred_df["model"] = "Proposed-Attention-Gated-Fusion"
                pred_tables.append(pred_df)
                pred_df.to_csv(cfg.pred_dir / f"pred_{safe_name(task)}__train_{safe_name(train_name)}__eval_{safe_name(eval_name)}.csv", index=False, encoding="utf-8-sig")
                print(f"[TEST] {task} | train={train_name} | eval={eval_name}: acc={m['accuracy']:.4f}, bal_acc={m['balanced_accuracy']:.4f}, macro_f1={m['macro_f1']:.4f}")
    met = pd.DataFrame(metrics); preds = pd.concat(pred_tables, ignore_index=True) if pred_tables else pd.DataFrame()
    met.to_csv(cfg.metrics_dir / "all_metrics.csv", index=False, encoding="utf-8-sig")
    preds.to_csv(cfg.pred_dir / "all_predictions.csv", index=False, encoding="utf-8-sig")
    return met, preds


# ------------------------- tables/figures -------------------------

def build_tables(cfg: Config, met: pd.DataFrame) -> None:
    m = met.copy(); order={s:i for i,s in enumerate(SCENARIO_ORDER)}; m["scenario_order"] = m["eval_scenario"].map(order).fillna(999)
    m = m.sort_values(["task", "train_scenario", "scenario_order"])
    cols = ["task_label","train_scenario","eval_scenario","model","n_eval","accuracy","balanced_accuracy","macro_precision","macro_recall","macro_f1","weighted_f1","kappa","roc_auc","roc_auc_ovr","distance_mae_m","distance_rmse_m","distance_within_2m_accuracy","distance_within_4m_accuracy","latency_ms_per_sample"]
    cols = [c for c in cols if c in m.columns]
    t = m[cols].copy();
    for c in t.select_dtypes(include=[np.number]).columns: t[c]=t[c].round(4)
    print_save("Table 1 — Proposed model performance across tasks and sensor scenarios", t, cfg.table_dir, "table_01_proposed_model_all_metrics")
    compact = m.pivot_table(index="eval_scenario", columns="task_label", values="macro_f1", aggfunc="mean").reset_index()
    compact["scenario_order"] = compact["eval_scenario"].map(order).fillna(999); compact=compact.sort_values("scenario_order").drop(columns="scenario_order").rename(columns={"eval_scenario":"Scenario"})
    for c in compact.select_dtypes(include=[np.number]).columns: compact[c]=compact[c].round(4)
    print_save("Table 2 — Compact Macro-F1 matrix for proposed model", compact, cfg.table_dir, "table_02_proposed_compact_macro_f1_matrix")
    d = m[m["task"]=="distance_estimation"].copy()
    if len(d):
        dc=["eval_scenario","accuracy","macro_f1","distance_mae_m","distance_rmse_m","distance_within_2m_accuracy","distance_within_4m_accuracy"]
        dc=[c for c in dc if c in d.columns]; dt=d[dc].rename(columns={"eval_scenario":"Scenario"}).copy()
        for c in dt.select_dtypes(include=[np.number]).columns: dt[c]=dt[c].round(4)
        print_save("Table 3 — Proposed model distance-estimation performance", dt, cfg.table_dir, "table_03_proposed_distance_estimation")


def plot_accuracy_by_distance(cfg: Config, preds: pd.DataFrame) -> None:
    for task in TASKS:
        g = preds[preds["task"]==task].copy()
        if len(g)==0: continue
        acc = g.groupby(["eval_scenario","distance_m"])["correct"].mean().reset_index(name="accuracy")
        fig, ax = plt.subplots(figsize=(11,5.5))
        for s in SCENARIO_ORDER:
            z=acc[acc["eval_scenario"]==s].sort_values("distance_m")
            if len(z): ax.plot(z["distance_m"], 100*z["accuracy"], marker="o", label=s)
        ax.set_title(f"Accuracy by distance — {TASK_LABELS[task]} — Proposed model")
        ax.set_xlabel("Distance (m)"); ax.set_ylabel("Accuracy (%)"); ax.set_xticks(DISTANCE_VALUES); ax.set_ylim(0,105); ax.grid(True, alpha=.35); ax.legend(fontsize=8,ncol=2)
        fig.tight_layout(); out=cfg.fig_dir / f"proposed_accuracy_by_distance__{safe_name(task)}.png"; fig.savefig(out,dpi=300); plt.show(); print(f"[SAVED] {out}")
        tab = acc.pivot_table(index="distance_m", columns="eval_scenario", values="accuracy").reset_index()
        sc=[s for s in SCENARIO_ORDER if s in tab.columns]; tab=tab[["distance_m"]+sc].rename(columns={"distance_m":"Distance (m)"})
        for c in sc: tab[c]=(100*tab[c]).round(2)
        save_table(tab, cfg.table_dir, f"table_proposed_accuracy_by_distance_{safe_name(task)}")


def plot_target_curves(cfg: Config, preds: pd.DataFrame) -> None:
    g0 = preds[preds["task"]=="target_classification"].copy()
    if len(g0)==0: return
    od=ensure_dir(cfg.fig_dir / "target_accuracy_by_distance")
    for sc in SCENARIO_ORDER:
        g=g0[g0["eval_scenario"]==sc].copy()
        if len(g)==0: continue
        acc=g.groupby(["target","distance_m"])["correct"].mean().reset_index(name="accuracy")
        fig,ax=plt.subplots(figsize=(9.5,5.2))
        for target in TARGETS:
            z=acc[acc["target"]==target].sort_values("distance_m")
            if len(z): ax.plot(z["distance_m"],100*z["accuracy"],marker="o",label=target)
        ax.set_title(f"Accuracy of target identification using {sc} — Proposed model")
        ax.set_xlabel("Distance (m)"); ax.set_ylabel("Accuracy (%)"); ax.set_xticks(DISTANCE_VALUES); ax.set_ylim(0,105); ax.grid(True,alpha=.35); ax.legend(fontsize=8)
        fig.tight_layout(); out=od / f"proposed_target_accuracy_by_distance__{safe_name(sc)}.png"; fig.savefig(out,dpi=300); plt.show(); print(f"[SAVED] {out}")


def plot_bars_and_gates(cfg: Config, preds: pd.DataFrame) -> None:
    g=preds[preds["task"]=="target_classification"].copy()
    if len(g)==0: return
    acc=g.groupby(["eval_scenario","target"])["correct"].mean().reset_index(name="accuracy")
    piv=acc.pivot(index="target", columns="eval_scenario", values="accuracy").reindex(TARGETS)
    sc=[s for s in SCENARIO_ORDER if s in piv.columns]
    fig,ax=plt.subplots(figsize=(12.5,5.8)); x=np.arange(len(piv.index)); width=.82/max(1,len(sc))
    for i,s in enumerate(sc): ax.bar(x+(i-len(sc)/2)*width+width/2,100*piv[s].to_numpy(),width,label=s)
    ax.set_title("Average target identification accuracy by scenario — Proposed model"); ax.set_xlabel("Target type"); ax.set_ylabel("Accuracy (%)"); ax.set_xticks(x); ax.set_xticklabels(piv.index); ax.set_ylim(0,105); ax.grid(axis="y",alpha=.30); ax.legend(fontsize=8,ncol=4)
    fig.tight_layout(); out=cfg.fig_dir / "proposed_average_target_accuracy_by_scenario_and_target.png"; fig.savefig(out,dpi=300); plt.show(); print(f"[SAVED] {out}")
    tab=piv.reset_index().rename(columns={"target":"Target"})
    for c in sc: tab[c]=(100*tab[c]).round(2)
    print_save("Table 4 — Proposed model average target accuracy by scenario and target (%)", tab, cfg.table_dir, "table_04_proposed_average_target_accuracy_by_scenario_target")
    # gate by distance for all sensors
    gate_cols=[f"gate_{safe_name(s)}" for s in SENSORS]
    gf=g[g["eval_scenario"]=="All sensors"].copy()
    if len(gf) and all(c in gf.columns for c in gate_cols):
        gs=gf.groupby("distance_m")[gate_cols].mean().reset_index().rename(columns={gate_cols[0]:"Gate CW", gate_cols[1]:"Gate FMCW", gate_cols[2]:"Gate RF"})
        fig,ax=plt.subplots(figsize=(10,5))
        for c in ["Gate CW","Gate FMCW","Gate RF"]: ax.plot(gs["distance_m"],gs[c],marker="o",label=c)
        ax.set_title("Reliability-gate weights by distance — Proposed model"); ax.set_xlabel("Distance (m)"); ax.set_ylabel("Mean gate weight"); ax.set_xticks(DISTANCE_VALUES); ax.set_ylim(0,1); ax.grid(True,alpha=.35); ax.legend()
        fig.tight_layout(); out=cfg.fig_dir / "proposed_gate_weights_by_distance_all_sensors.png"; fig.savefig(out,dpi=300); plt.show(); print(f"[SAVED] {out}")
        for c in ["Gate CW","Gate FMCW","Gate RF"]: gs[c]=gs[c].round(4)
        print_save("Table 5 — Proposed model gate weights by distance", gs, cfg.table_dir, "table_05_proposed_gate_weights_by_distance")


def plot_confusions(cfg: Config, preds: pd.DataFrame) -> None:
    od=ensure_dir(cfg.fig_dir / "confusion_matrices")
    for task in TASKS:
        gt=preds[preds["task"]==task].copy()
        if len(gt)==0: continue
        labels = TARGETS if task=="target_classification" else DRONE_TARGETS if task=="drone_model_identification" else ["Drone","Non-drone"] if task=="drone_vs_non_drone" else DISTANCE_LABELS
        for sc in SCENARIO_ORDER:
            g=gt[gt["eval_scenario"]==sc].copy()
            if len(g)==0: continue
            cm=confusion_matrix(g["y_true"].astype(str), g["y_pred"].astype(str), labels=labels); cmn=cm/np.maximum(cm.sum(1,keepdims=True),1)
            fig,ax=plt.subplots(figsize=(7.5,6.5)); im=ax.imshow(cmn,vmin=0,vmax=1,aspect="auto")
            ax.set_title(f"Confusion matrix — {TASK_LABELS[task]} — {sc}"); ax.set_xlabel("Predicted"); ax.set_ylabel("True")
            ax.set_xticks(np.arange(len(labels))); ax.set_yticks(np.arange(len(labels))); ax.set_xticklabels(labels,rotation=45,ha="right",fontsize=8); ax.set_yticklabels(labels,fontsize=8)
            for i in range(cmn.shape[0]):
                for j in range(cmn.shape[1]): ax.text(j,i,f"{cmn[i,j]:.2f}",ha="center",va="center",fontsize=7)
            fig.colorbar(im, ax=ax, fraction=.046, pad=.04); fig.tight_layout(); fig.savefig(od / f"proposed_cm__{safe_name(task)}__{safe_name(sc)}.png", dpi=300); plt.close(fig)


def generate_figures(cfg: Config, preds: pd.DataFrame) -> None:
    plot_accuracy_by_distance(cfg, preds); plot_target_curves(cfg, preds); plot_bars_and_gates(cfg, preds); plot_confusions(cfg, preds)


# ------------------------- main -------------------------

def main() -> None:
    args=parse_args(); cfg=build_config(args)
    for d in [cfg.out_dir,cfg.checkpoint_dir,cfg.metrics_dir,cfg.pred_dir,cfg.fig_dir,cfg.table_dir]: ensure_dir(d)
    seed_everything(cfg.seed); device=get_device(cfg)
    print("="*100); print("TSMS-Drone proposed multitask attention-gated fusion model — VSCode version"); print("="*100)
    print(f"ROOT               : {cfg.root}")
    print(f"DATASET_DIR        : {cfg.dataset_dir}")
    print(f"OUT_DIR            : {cfg.out_dir}")
    print(f"SPLIT_CSV          : {cfg.split_csv}")
    print(f"DEVICE             : {device}")
    print(f"IMAGE_SIZE         : {cfg.image_size}")
    print(f"BATCH_SIZE         : {cfg.batch_size}")
    print(f"EPOCHS             : {cfg.epochs}")
    print(f"TRAIN_PER_SCENARIO : {cfg.train_per_scenario}")
    print("="*100)
    t0=time.time()
    index=build_index(cfg); split=attach_or_create_split(cfg,index); summarize_split(cfg,split)
    metrics,preds=run_experiments(cfg,split,device); build_tables(cfg,metrics); generate_figures(cfg,preds)
    save_json({"root":str(cfg.root),"dataset_dir":str(cfg.dataset_dir),"out_dir":str(cfg.out_dir),"split_csv":str(cfg.split_csv),"device":str(device),"image_size":cfg.image_size,"batch_size":cfg.batch_size,"epochs":cfg.epochs,"elapsed_sec":time.time()-t0,"tasks":TASKS,"sensor_scenarios":[{"name":n,"sensors":s} for n,s in SENSOR_SCENARIOS],"n_active_tuples":int(len(split))}, cfg.out_dir / "run_manifest.json")
    print("\nGenerated output folders:")
    for p in sorted(cfg.out_dir.glob("*")): print(p)
    print("="*100); print(f"[OK] Completed in {pretty_seconds(time.time()-t0)}"); print("="*100)


if __name__ == "__main__":
    main()


[INFO] Ignored unknown arguments: ['--f=c:\\Users\\Luis\\AppData\\Roaming\\jupyter\\runtime\\kernel-v3040e410708fb7b5bf2eb7c62141f275456ffc0f8.json']
TSMS-Drone proposed multitask attention-gated fusion model — VSCode version
ROOT               : D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)
DATASET_DIR        : D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\Dataset
OUT_DIR            : D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_proposed_multitask_attention_fusion_vscode
SPLIT_CSV          : D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_advanced_fusion_improvements\tuple_splits_70_15_15_full.csv
DEVICE             : cpu
IMAGE_SIZE         : 128
BATCH_SIZE         : 32
EPOCHS             : 20
TRAIN_PER_SCENARIO : False
Building synchronized tuple index...
Saved index: D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_proposed_multitask_attention_fusion_vscode\synchronized_tuple_index.csv; tuples=37500
Loading sp

Epoch 001/20: loss=0.6555, val_f1=0.9966, val_acc=0.9966, time=48.6min
[CHECKPOINT] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_proposed_multitask_attention_fusion_vscode\checkpoints\target_classification__trained_all_sensors.pt


Epoch 002/20: loss=0.4722, val_f1=0.9991, val_acc=0.9991, time=47.6min
[CHECKPOINT] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_proposed_multitask_attention_fusion_vscode\checkpoints\target_classification__trained_all_sensors.pt


Epoch 003/20: loss=0.4086, val_f1=0.9995, val_acc=0.9995, time=48.3min
[CHECKPOINT] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_proposed_multitask_attention_fusion_vscode\checkpoints\target_classification__trained_all_sensors.pt


Epoch 004/20: loss=0.3698, val_f1=0.9998, val_acc=0.9998, time=47.9min
[CHECKPOINT] D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)\_proposed_multitask_attention_fusion_vscode\checkpoints\target_classification__trained_all_sensors.pt


...

[TEST] distance_estimation | train=All sensors | eval=RF only: acc=0.2836, bal_acc=0.2836, macro_f1=0.2837


[TEST] distance_estimation | train=All sensors | eval=CW + FMCW: acc=0.9138, bal_acc=0.9138, macro_f1=0.9141


[TEST] distance_estimation | train=All sensors | eval=CW + RF: acc=0.6777, bal_acc=0.6777, macro_f1=0.6785


[TEST] distance_estimation | train=All sensors | eval=FMCW + RF: acc=0.9273, bal_acc=0.9273, macro_f1=0.9274


[TEST] distance_estimation | train=All sensors | eval=All sensors: acc=0.9612, bal_acc=0.9612, macro_f1=0.9612

Table 1 — Proposed model performance across tasks and sensor scenarios
                task_label train_scenario eval_scenario                           model  n_eval  accuracy  balanced_accuracy  macro_precision  macro_recall  macro_f1  weighted_f1  kappa  roc_auc  roc_auc_ovr  distance_mae_m  distance_rmse_m  distance_within_2m_accuracy  distance_within_4m_accuracy  latency_ms_per_sample
       Distance estimation    All sensors       CW only Proposed-Attention-Gated-Fusion    5625    0.4123             0.4123           0.4532        0.4123    0.4143       0.4143 0.3703      NaN       0.8778          4.9252           7.7956                       0.5168                       0.6288                37.4279
       Distance estimation    All sensors     FMCW only Proposed-Attention-Gated-Fusion    5625    0.8674             0.8674           0.8690        0.8674    0.8672       0

In [35]:
from pathlib import Path
import numpy as np
import pandas as pd
import math

ROOT = Path(r"D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)")

BASELINE_PRED = ROOT / "_baseline_multitask_fusion_benchmark" / "predictions" / "all_predictions.csv"
PROPOSED_PRED = ROOT / "_proposed_multitask_attention_fusion_vscode" / "predictions" / "all_predictions.csv"

OUT_DIR = ROOT / "_comparative_baseline_vs_proposed_tables"
OUT_DIR.mkdir(parents=True, exist_ok=True)

def label_to_m(x):
    return int(str(x).replace("m", "").strip())

def distance_metrics(df, model_name, scenario_col, scenario_value):
    g = df[
        (df["task"] == "distance_estimation") &
        (df[scenario_col] == scenario_value)
    ].copy()

    if "model" in g.columns:
        # For baseline, keep the best baseline used in the previous analysis.
        if model_name == "Baseline-ExtraTrees":
            g = g[g["model"] == "ExtraTrees"].copy()

    g["true_m"] = g["y_true"].apply(label_to_m)
    g["pred_m"] = g["y_pred"].apply(label_to_m)
    g["abs_error_m"] = (g["true_m"] - g["pred_m"]).abs()
    g["sq_error_m"] = (g["true_m"] - g["pred_m"]) ** 2

    out = (
        g.groupby("distance_m")
        .agg(
            n=("correct", "size"),
            accuracy=("correct", "mean"),
            mae_m=("abs_error_m", "mean"),
            rmse_m=("sq_error_m", lambda x: math.sqrt(float(np.mean(x)))),
            within_2m_accuracy=("abs_error_m", lambda x: float(np.mean(x <= 2.0))),
            within_4m_accuracy=("abs_error_m", lambda x: float(np.mean(x <= 4.0))),
        )
        .reset_index()
        .rename(columns={"distance_m": "Distance (m)"})
    )

    out.insert(1, "Model", model_name)
    return out

baseline = pd.read_csv(BASELINE_PRED)
proposed = pd.read_csv(PROPOSED_PRED)

baseline_dist = distance_metrics(
    baseline,
    model_name="Baseline-ExtraTrees",
    scenario_col="scenario",
    scenario_value="All sensors"
)

proposed_dist = distance_metrics(
    proposed,
    model_name="Proposed-Attention-Gated-Fusion",
    scenario_col="eval_scenario",
    scenario_value="All sensors"
)

long_table = pd.concat([baseline_dist, proposed_dist], ignore_index=True)

# Wide comparison table
b = baseline_dist.drop(columns=["Model"]).add_prefix("Baseline ")
p = proposed_dist.drop(columns=["Model"]).add_prefix("Proposed ")

comp = pd.concat(
    [
        baseline_dist[["Distance (m)", "n"]].rename(columns={"n": "n"}),
        b[["Baseline accuracy", "Baseline mae_m", "Baseline rmse_m", "Baseline within_2m_accuracy", "Baseline within_4m_accuracy"]],
        p[["Proposed accuracy", "Proposed mae_m", "Proposed rmse_m", "Proposed within_2m_accuracy", "Proposed within_4m_accuracy"]],
    ],
    axis=1
)

comp["Delta accuracy"] = comp["Proposed accuracy"] - comp["Baseline accuracy"]
comp["Delta MAE"] = comp["Proposed mae_m"] - comp["Baseline mae_m"]
comp["Delta RMSE"] = comp["Proposed rmse_m"] - comp["Baseline rmse_m"]
comp["Delta within 2m"] = comp["Proposed within_2m_accuracy"] - comp["Baseline within_2m_accuracy"]

# Round for reporting
for c in comp.select_dtypes(include=[np.number]).columns:
    if c not in ["Distance (m)", "n"]:
        comp[c] = comp[c].round(4)

csv_path = OUT_DIR / "table_distance_by_range_baseline_vs_proposed.csv"
tex_path = OUT_DIR / "table_distance_by_range_baseline_vs_proposed.tex"
md_path = OUT_DIR / "table_distance_by_range_baseline_vs_proposed.md"

comp.to_csv(csv_path, index=False, encoding="utf-8-sig")
comp.to_markdown(md_path, index=False)

latex = comp.to_latex(
    index=False,
    escape=False,
    caption="Distance-wise comparison between the best baseline and the proposed model in the all-sensor setting.",
    label="tab:distance_wise_baseline_vs_proposed",
    float_format="%.4f"
)
tex_path.write_text(latex, encoding="utf-8")

print(comp)
print(f"[SAVED] {csv_path}")
print(f"[SAVED] {tex_path}")
print(f"[SAVED] {md_path}")

    Distance (m)    n  Baseline accuracy  Baseline mae_m  Baseline rmse_m  Baseline within_2m_accuracy  Baseline within_4m_accuracy  Proposed accuracy  Proposed mae_m  Proposed rmse_m  Proposed within_2m_accuracy  \
0              2  750             0.9987          0.0027           0.0730                       1.0000                       1.0000             1.0000          0.0000           0.0000                       1.0000   
1              4  750             0.9573          0.1867           1.2649                       0.9787                       0.9840             0.9627          0.2080           1.0979                       0.9653   
2              6  750             0.9333          0.2613           1.5697                       0.9840                       0.9907             0.9840          0.0320           0.2530                       1.0000   
3              8  750             0.8880          0.3813           1.7080                       0.9747                       0.9880     

# GRÁFICOS

In [36]:
# ============================================================
# extract_all_scenario_metrics_and_plots.py
# ============================================================
# Extract metrics for all sensor-scenario combinations from:
#   1) Baseline predictions
#   2) Proposed model predictions
#
# Outputs:
#   - Overall metrics by task / model / scenario
#   - Distance-estimation metrics by scenario
#   - Distance-wise metrics
#   - Target-wise accuracy by distance
#   - Publication-style plots similar to the TSMS-Drone reference figures
# ============================================================

from pathlib import Path
import math
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    cohen_kappa_score,
)

warnings.filterwarnings("ignore")


# ============================================================
# CONFIG
# ============================================================

ROOT = Path(r"D:\Time-Synchronized Multi-Sensor Drone Dataset (TSMS-Drone)")

BASELINE_PRED = (
    ROOT
    / "_baseline_multitask_fusion_benchmark"
    / "predictions"
    / "all_predictions.csv"
)

PROPOSED_PRED = (
    ROOT
    / "_proposed_multitask_attention_fusion_vscode"
    / "predictions"
    / "all_predictions.csv"
)

OUT_DIR = ROOT / "_comparative_all_scenario_metrics"
TABLE_DIR = OUT_DIR / "tables"
FIG_DIR = OUT_DIR / "figures"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

SCENARIO_ORDER = [
    "CW only",
    "FMCW only",
    "RF only",
    "CW + FMCW",
    "CW + RF",
    "FMCW + RF",
    "All sensors",
]

TASK_ORDER = [
    "target_classification",
    "drone_vs_non_drone",
    "drone_model_identification",
    "distance_estimation",
]

TASK_LABELS = {
    "target_classification": "Target classification",
    "drone_vs_non_drone": "Drone vs non-drone",
    "drone_model_identification": "Drone model identification",
    "distance_estimation": "Distance estimation",
}

TARGET_ORDER = [
    "Inspire 2",
    "Matrice 30",
    "Mavic 2 Pro",
    "Phantom 4 Pro",
    "Corner Reflector",
]

DISTANCE_ORDER = list(range(2, 31, 2))


# ============================================================
# UTILS
# ============================================================

def safe_name(text: str) -> str:
    text = str(text).strip().lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return text.strip("_")


def label_to_distance_m(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().replace("m", "")
    return int(float(s))


def save_table(df: pd.DataFrame, name: str, caption: str = "", label: str = ""):
    csv_path = TABLE_DIR / f"{name}.csv"
    md_path = TABLE_DIR / f"{name}.md"
    tex_path = TABLE_DIR / f"{name}.tex"

    df.to_csv(csv_path, index=False, encoding="utf-8-sig")

    try:
        md_path.write_text(df.to_markdown(index=False), encoding="utf-8")
    except Exception:
        md_path.write_text(df.to_csv(index=False), encoding="utf-8")

    latex = df.to_latex(
        index=False,
        escape=False,
        caption=caption if caption else None,
        label=label if label else None,
        float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x),
    )
    tex_path.write_text(latex, encoding="utf-8")

    print(f"[SAVED] {csv_path}")
    print(f"[SAVED] {md_path}")
    print(f"[SAVED] {tex_path}")


def scenario_sort_key(x):
    if x in SCENARIO_ORDER:
        return SCENARIO_ORDER.index(x)
    return 999


def task_sort_key(x):
    if x in TASK_ORDER:
        return TASK_ORDER.index(x)
    return 999


# ============================================================
# LOAD AND NORMALIZE PREDICTIONS
# ============================================================

def normalize_predictions(path: Path, source_name: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Prediction file not found: {path}")

    df = pd.read_csv(path)

    # Normalize scenario columns.
    if "scenario" in df.columns:
        scenario_col = "scenario"
    elif "eval_scenario" in df.columns:
        scenario_col = "eval_scenario"
    else:
        raise ValueError(f"No scenario/eval_scenario column found in {path}")

    if "sensors" in df.columns:
        sensors_col = "sensors"
    elif "eval_sensors" in df.columns:
        sensors_col = "eval_sensors"
    else:
        sensors_col = None

    out = df.copy()
    out["source"] = source_name
    out["scenario_norm"] = out[scenario_col].astype(str)

    if sensors_col is not None:
        out["sensors_norm"] = out[sensors_col].astype(str)
    else:
        out["sensors_norm"] = out["scenario_norm"]

    if "model" not in out.columns:
        out["model"] = source_name

    out["model_norm"] = out["model"].astype(str)

    # Ensure correct column exists.
    if "correct" not in out.columns:
        out["correct"] = out["y_true"].astype(str) == out["y_pred"].astype(str)

    # Ensure distance_m exists.
    if "distance_m" not in out.columns:
        if "y_true" in out.columns:
            # Only valid for distance_estimation. Otherwise remains NaN.
            out["distance_m"] = np.nan
        else:
            raise ValueError("No distance_m column found.")

    out["task"] = out["task"].astype(str)
    out["y_true"] = out["y_true"].astype(str)
    out["y_pred"] = out["y_pred"].astype(str)

    return out


baseline_df = normalize_predictions(BASELINE_PRED, "Baseline")
proposed_df = normalize_predictions(PROPOSED_PRED, "Proposed")

all_pred = pd.concat([baseline_df, proposed_df], ignore_index=True)

print("=" * 100)
print("Loaded predictions")
print("=" * 100)
print(all_pred[["source", "task", "scenario_norm", "model_norm"]].drop_duplicates().to_string(index=False))


# ============================================================
# METRIC FUNCTIONS
# ============================================================

def compute_classification_metrics(g: pd.DataFrame) -> dict:
    y_true = g["y_true"].astype(str).to_numpy()
    y_pred = g["y_pred"].astype(str).to_numpy()

    metrics = {
        "n": int(len(g)),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "kappa": cohen_kappa_score(y_true, y_pred),
    }

    if g["task"].iloc[0] == "distance_estimation":
        true_m = np.array([label_to_distance_m(x) for x in g["y_true"]], dtype=float)
        pred_m = np.array([label_to_distance_m(x) for x in g["y_pred"]], dtype=float)
        abs_err = np.abs(true_m - pred_m)

        metrics.update({
            "distance_mae_m": float(np.mean(abs_err)),
            "distance_rmse_m": float(math.sqrt(np.mean((true_m - pred_m) ** 2))),
            "distance_within_2m_accuracy": float(np.mean(abs_err <= 2.0)),
            "distance_within_4m_accuracy": float(np.mean(abs_err <= 4.0)),
        })

    return metrics


# ============================================================
# 1. OVERALL METRICS FOR ALL TASKS / SCENARIOS / MODELS
# ============================================================

overall_rows = []

group_cols = [
    "source",
    "task",
    "scenario_norm",
    "sensors_norm",
    "model_norm",
]

for keys, g in all_pred.groupby(group_cols, dropna=False):
    source, task, scenario, sensors, model = keys
    m = compute_classification_metrics(g)
    overall_rows.append({
        "source": source,
        "task": task,
        "task_label": TASK_LABELS.get(task, task),
        "scenario": scenario,
        "sensors": sensors,
        "model": model,
        **m,
    })

overall_metrics = pd.DataFrame(overall_rows)
overall_metrics["task_order"] = overall_metrics["task"].map(task_sort_key)
overall_metrics["scenario_order"] = overall_metrics["scenario"].map(scenario_sort_key)
overall_metrics = overall_metrics.sort_values(
    ["task_order", "scenario_order", "source", "model"]
).drop(columns=["task_order", "scenario_order"])

save_table(
    overall_metrics,
    "01_overall_metrics_all_tasks_all_scenarios",
    caption="Overall metrics for all tasks, models, and sensor scenarios.",
    label="tab:overall_metrics_all_scenarios",
)

print("\nOverall metrics:")
print(overall_metrics.to_string(index=False))


# ============================================================
# 2. DISTANCE-ESTIMATION METRICS ONLY
# ============================================================

distance_metrics = overall_metrics[
    overall_metrics["task"] == "distance_estimation"
].copy()

distance_metrics = distance_metrics[
    [
        "source",
        "scenario",
        "sensors",
        "model",
        "n",
        "accuracy",
        "balanced_accuracy",
        "macro_precision",
        "macro_recall",
        "macro_f1",
        "weighted_f1",
        "kappa",
        "distance_mae_m",
        "distance_rmse_m",
        "distance_within_2m_accuracy",
        "distance_within_4m_accuracy",
    ]
].copy()

save_table(
    distance_metrics,
    "02_distance_estimation_metrics_all_scenarios",
    caption="Distance-estimation metrics for all models and sensor scenarios.",
    label="tab:distance_estimation_all_scenarios",
)

print("\nDistance-estimation metrics:")
print(distance_metrics.to_string(index=False))


# ============================================================
# 3. DISTANCE-WISE METRICS FOR DISTANCE ESTIMATION
# ============================================================

dist_pred = all_pred[all_pred["task"] == "distance_estimation"].copy()

dist_pred["true_distance_m"] = dist_pred["y_true"].apply(label_to_distance_m)
dist_pred["pred_distance_m"] = dist_pred["y_pred"].apply(label_to_distance_m)
dist_pred["abs_error_m"] = (dist_pred["true_distance_m"] - dist_pred["pred_distance_m"]).abs()
dist_pred["sq_error_m"] = (dist_pred["true_distance_m"] - dist_pred["pred_distance_m"]) ** 2

distance_wise_rows = []

for keys, g in dist_pred.groupby(
    ["source", "scenario_norm", "sensors_norm", "model_norm", "true_distance_m"],
    dropna=False
):
    source, scenario, sensors, model, true_d = keys

    y_true = g["y_true"].astype(str).to_numpy()
    y_pred = g["y_pred"].astype(str).to_numpy()

    distance_wise_rows.append({
        "source": source,
        "scenario": scenario,
        "sensors": sensors,
        "model": model,
        "distance_m": int(true_d),
        "n": int(len(g)),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "mae_m": float(g["abs_error_m"].mean()),
        "rmse_m": float(math.sqrt(g["sq_error_m"].mean())),
        "within_2m_accuracy": float((g["abs_error_m"] <= 2.0).mean()),
        "within_4m_accuracy": float((g["abs_error_m"] <= 4.0).mean()),
    })

distance_wise_metrics = pd.DataFrame(distance_wise_rows)
distance_wise_metrics["scenario_order"] = distance_wise_metrics["scenario"].map(scenario_sort_key)
distance_wise_metrics = distance_wise_metrics.sort_values(
    ["source", "model", "scenario_order", "distance_m"]
).drop(columns=["scenario_order"])

save_table(
    distance_wise_metrics,
    "03_distance_wise_metrics_all_scenarios",
    caption="Distance-wise metrics for distance estimation across all scenarios.",
    label="tab:distance_wise_all_scenarios",
)

print("\nDistance-wise metrics:")
print(distance_wise_metrics.head(30).to_string(index=False))


# ============================================================
# 4. TARGET-CLASSIFICATION ACCURACY BY TARGET AND DISTANCE
# ============================================================

target_pred = all_pred[all_pred["task"] == "target_classification"].copy()

target_accuracy_rows = []

for keys, g in target_pred.groupby(
    ["source", "scenario_norm", "sensors_norm", "model_norm", "target", "distance_m"],
    dropna=False
):
    source, scenario, sensors, model, target, distance_m = keys
    target_accuracy_rows.append({
        "source": source,
        "scenario": scenario,
        "sensors": sensors,
        "model": model,
        "target": target,
        "distance_m": int(distance_m),
        "n": int(len(g)),
        "accuracy": float(g["correct"].mean()),
        "accuracy_percent": float(100.0 * g["correct"].mean()),
    })

target_distance_accuracy = pd.DataFrame(target_accuracy_rows)
target_distance_accuracy["scenario_order"] = target_distance_accuracy["scenario"].map(scenario_sort_key)
target_distance_accuracy["target_order"] = target_distance_accuracy["target"].apply(
    lambda x: TARGET_ORDER.index(x) if x in TARGET_ORDER else 999
)

target_distance_accuracy = target_distance_accuracy.sort_values(
    ["source", "model", "scenario_order", "target_order", "distance_m"]
).drop(columns=["scenario_order", "target_order"])

save_table(
    target_distance_accuracy,
    "04_target_classification_accuracy_by_target_distance",
    caption="Target-classification accuracy by target and distance.",
    label="tab:target_accuracy_by_distance",
)

print("\nTarget-classification accuracy by target and distance:")
print(target_distance_accuracy.head(30).to_string(index=False))


# ============================================================
# 5. PIVOT TABLES FOR EASIER INSPECTION
# ============================================================

# 5.1 Distance estimation exact accuracy by scenario and distance.
distance_accuracy_pivot = (
    distance_wise_metrics
    .pivot_table(
        index=["source", "model", "scenario"],
        columns="distance_m",
        values="accuracy",
        aggfunc="mean",
    )
    .reset_index()
)

distance_accuracy_pivot.columns = [
    str(c) if not isinstance(c, int) else f"{c}m"
    for c in distance_accuracy_pivot.columns
]

save_table(
    distance_accuracy_pivot,
    "05_pivot_distance_estimation_accuracy_by_distance",
    caption="Distance-estimation accuracy by true distance for all scenarios.",
    label="tab:pivot_distance_accuracy",
)

# 5.2 Target classification accuracy by target and distance.
target_accuracy_pivot = (
    target_distance_accuracy
    .pivot_table(
        index=["source", "model", "scenario", "target"],
        columns="distance_m",
        values="accuracy_percent",
        aggfunc="mean",
    )
    .reset_index()
)

target_accuracy_pivot.columns = [
    str(c) if not isinstance(c, int) else f"{c}m"
    for c in target_accuracy_pivot.columns
]

save_table(
    target_accuracy_pivot,
    "06_pivot_target_accuracy_by_target_distance",
    caption="Target-classification accuracy by target and distance.",
    label="tab:pivot_target_accuracy",
)


# ============================================================
# 6. SELECT BEST BASELINE PER TASK / SCENARIO
# ============================================================
# For each task and scenario, pick the baseline model with highest Macro-F1.

baseline_overall = overall_metrics[overall_metrics["source"] == "Baseline"].copy()

best_baseline_rows = []
for keys, g in baseline_overall.groupby(["task", "scenario"], dropna=False):
    idx = g["macro_f1"].idxmax()
    best_baseline_rows.append(g.loc[idx].to_dict())

best_baseline = pd.DataFrame(best_baseline_rows)
best_baseline["task_order"] = best_baseline["task"].map(task_sort_key)
best_baseline["scenario_order"] = best_baseline["scenario"].map(scenario_sort_key)
best_baseline = best_baseline.sort_values(
    ["task_order", "scenario_order"]
).drop(columns=["task_order", "scenario_order"])

save_table(
    best_baseline,
    "07_best_baseline_by_task_and_scenario",
    caption="Best baseline model selected by Macro-F1 for each task and scenario.",
    label="tab:best_baseline_by_task_scenario",
)


# ============================================================
# 7. COMPARISON: BEST BASELINE VS PROPOSED
# ============================================================

proposed_overall = overall_metrics[overall_metrics["source"] == "Proposed"].copy()

comparison_rows = []

for _, p in proposed_overall.iterrows():
    task = p["task"]
    scenario = p["scenario"]

    b_candidates = best_baseline[
        (best_baseline["task"] == task) &
        (best_baseline["scenario"] == scenario)
    ]

    if len(b_candidates) == 0:
        continue

    b = b_candidates.iloc[0]

    row = {
        "task": task,
        "task_label": TASK_LABELS.get(task, task),
        "scenario": scenario,
        "baseline_model": b["model"],
        "proposed_model": p["model"],
        "baseline_accuracy": b["accuracy"],
        "proposed_accuracy": p["accuracy"],
        "delta_accuracy": p["accuracy"] - b["accuracy"],
        "baseline_macro_f1": b["macro_f1"],
        "proposed_macro_f1": p["macro_f1"],
        "delta_macro_f1": p["macro_f1"] - b["macro_f1"],
    }

    if task == "distance_estimation":
        for metric in [
            "distance_mae_m",
            "distance_rmse_m",
            "distance_within_2m_accuracy",
            "distance_within_4m_accuracy",
        ]:
            row[f"baseline_{metric}"] = b.get(metric, np.nan)
            row[f"proposed_{metric}"] = p.get(metric, np.nan)
            row[f"delta_{metric}"] = p.get(metric, np.nan) - b.get(metric, np.nan)

    comparison_rows.append(row)

baseline_vs_proposed = pd.DataFrame(comparison_rows)
baseline_vs_proposed["task_order"] = baseline_vs_proposed["task"].map(task_sort_key)
baseline_vs_proposed["scenario_order"] = baseline_vs_proposed["scenario"].map(scenario_sort_key)

baseline_vs_proposed = baseline_vs_proposed.sort_values(
    ["task_order", "scenario_order"]
).drop(columns=["task_order", "scenario_order"])

save_table(
    baseline_vs_proposed,
    "08_best_baseline_vs_proposed_all_scenarios",
    caption="Comparison between the best baseline and the proposed model across all tasks and scenarios.",
    label="tab:best_baseline_vs_proposed_all_scenarios",
)

print("\nBest baseline vs proposed:")
print(baseline_vs_proposed.to_string(index=False))


# ============================================================
# 8. PLOTS IN REFERENCE STYLE
# ============================================================

def plot_target_accuracy_reference_style(
    df: pd.DataFrame,
    source: str,
    model: str,
    scenario: str,
    save_path: Path,
):
    g = df[
        (df["source"] == source) &
        (df["model"] == model) &
        (df["scenario"] == scenario)
    ].copy()

    if len(g) == 0:
        return

    plt.figure(figsize=(10.5, 6.2))

    markers = {
        "Inspire 2": "o",
        "Matrice 30": "s",
        "Mavic 2 Pro": "^",
        "Phantom 4 Pro": "v",
        "Corner Reflector": "D",
    }

    for target in TARGET_ORDER:
        gt = g[g["target"] == target].sort_values("distance_m")
        if len(gt) == 0:
            continue

        plt.plot(
            gt["distance_m"],
            gt["accuracy_percent"],
            marker=markers.get(target, "o"),
            linewidth=2.0,
            markersize=7,
            label=target,
        )

    plt.title(
        f"Accuracy of Target Identification using {scenario}",
        fontsize=16,
        fontweight="bold",
        pad=12,
    )
    plt.xlabel("Distance (m)", fontsize=14)
    plt.ylabel("Accuracy (%)", fontsize=14)

    plt.xticks(
        DISTANCE_ORDER,
        [f"{d}m" for d in DISTANCE_ORDER],
        rotation=45,
        fontsize=11,
    )
    plt.yticks(np.arange(0, 101, 20), fontsize=11)
    plt.ylim(0, 105)

    plt.grid(True, linestyle="--", alpha=0.35)
    plt.legend(
        loc="lower left",
        frameon=True,
        facecolor="white",
        edgecolor="black",
        fontsize=11,
    )
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()

    print(f"[SAVED] {save_path}")


# Generate target-identification plots for all source/model/scenario combinations.
for (source, model, scenario), _ in target_distance_accuracy.groupby(
    ["source", "model", "scenario"]
):
    name = (
        f"target_accuracy_by_distance__"
        f"{safe_name(source)}__{safe_name(model)}__{safe_name(scenario)}.png"
    )
    plot_target_accuracy_reference_style(
        target_distance_accuracy,
        source=source,
        model=model,
        scenario=scenario,
        save_path=FIG_DIR / name,
    )


def plot_distance_estimation_by_scenario(
    df: pd.DataFrame,
    source: str,
    model: str,
    metric: str,
    ylabel: str,
    title: str,
    save_path: Path,
):
    g = df[
        (df["source"] == source) &
        (df["model"] == model)
    ].copy()

    if len(g) == 0:
        return

    plt.figure(figsize=(10.5, 6.2))

    for scenario in SCENARIO_ORDER:
        gs = g[g["scenario"] == scenario].sort_values("distance_m")
        if len(gs) == 0:
            continue

        plt.plot(
            gs["distance_m"],
            gs[metric],
            marker="o",
            linewidth=2.0,
            markersize=5,
            label=scenario,
        )

    plt.title(title, fontsize=16, fontweight="bold", pad=12)
    plt.xlabel("Distance (m)", fontsize=14)
    plt.ylabel(ylabel, fontsize=14)

    plt.xticks(
        DISTANCE_ORDER,
        [f"{d}m" for d in DISTANCE_ORDER],
        rotation=45,
        fontsize=11,
    )

    if "accuracy" in metric or metric == "macro_f1":
        plt.ylim(0, 1.05)

    plt.grid(True, linestyle="--", alpha=0.35)
    plt.legend(
        loc="best",
        frameon=True,
        facecolor="white",
        edgecolor="black",
        fontsize=9,
    )
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()

    print(f"[SAVED] {save_path}")


# Distance-estimation plots.
for (source, model), _ in distance_wise_metrics.groupby(["source", "model"]):
    plot_distance_estimation_by_scenario(
        distance_wise_metrics,
        source=source,
        model=model,
        metric="accuracy",
        ylabel="Accuracy",
        title=f"Distance-Estimation Accuracy by Scenario ({source}: {model})",
        save_path=FIG_DIR / f"distance_estimation_accuracy__{safe_name(source)}__{safe_name(model)}.png",
    )

    plot_distance_estimation_by_scenario(
        distance_wise_metrics,
        source=source,
        model=model,
        metric="mae_m",
        ylabel="MAE (m)",
        title=f"Distance-Estimation MAE by Scenario ({source}: {model})",
        save_path=FIG_DIR / f"distance_estimation_mae__{safe_name(source)}__{safe_name(model)}.png",
    )

    plot_distance_estimation_by_scenario(
        distance_wise_metrics,
        source=source,
        model=model,
        metric="rmse_m",
        ylabel="RMSE (m)",
        title=f"Distance-Estimation RMSE by Scenario ({source}: {model})",
        save_path=FIG_DIR / f"distance_estimation_rmse__{safe_name(source)}__{safe_name(model)}.png",
    )


# ============================================================
# 9. ALL-SENSOR BEST BASELINE VS PROPOSED DISTANCE-WISE PLOT
# ============================================================

def plot_best_baseline_vs_proposed_distance_metric(metric, ylabel, title, save_path):
    # Use best baseline for All sensors distance estimation.
    b_row = best_baseline[
        (best_baseline["task"] == "distance_estimation") &
        (best_baseline["scenario"] == "All sensors")
    ]

    if len(b_row) == 0:
        print("[WARN] No best baseline found for distance_estimation / All sensors")
        return

    baseline_model = b_row.iloc[0]["model"]

    b = distance_wise_metrics[
        (distance_wise_metrics["source"] == "Baseline") &
        (distance_wise_metrics["scenario"] == "All sensors") &
        (distance_wise_metrics["model"] == baseline_model)
    ].sort_values("distance_m")

    p = distance_wise_metrics[
        (distance_wise_metrics["source"] == "Proposed") &
        (distance_wise_metrics["scenario"] == "All sensors")
    ].copy()

    if len(p) == 0:
        print("[WARN] No proposed all-sensors result found.")
        return

    # If several proposed model names exist, use first.
    proposed_model = p["model"].iloc[0]
    p = p[p["model"] == proposed_model].sort_values("distance_m")

    plt.figure(figsize=(10.5, 6.2))

    plt.plot(
        b["distance_m"],
        b[metric],
        marker="s",
        linewidth=2.2,
        markersize=6,
        label=f"Best baseline ({baseline_model})",
    )

    plt.plot(
        p["distance_m"],
        p[metric],
        marker="o",
        linewidth=2.2,
        markersize=6,
        label="Proposed",
    )

    plt.title(title, fontsize=16, fontweight="bold", pad=12)
    plt.xlabel("Distance (m)", fontsize=14)
    plt.ylabel(ylabel, fontsize=14)

    plt.xticks(
        DISTANCE_ORDER,
        [f"{d}m" for d in DISTANCE_ORDER],
        rotation=45,
        fontsize=11,
    )

    if "accuracy" in metric or metric == "macro_f1":
        plt.ylim(0, 1.05)

    plt.grid(True, linestyle="--", alpha=0.35)
    plt.legend(
        loc="best",
        frameon=True,
        facecolor="white",
        edgecolor="black",
        fontsize=11,
    )
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()

    print(f"[SAVED] {save_path}")


plot_best_baseline_vs_proposed_distance_metric(
    metric="accuracy",
    ylabel="Accuracy",
    title="Distance-Estimation Accuracy: Best Baseline vs Proposed",
    save_path=FIG_DIR / "all_sensors_distance_accuracy_best_baseline_vs_proposed.png",
)

plot_best_baseline_vs_proposed_distance_metric(
    metric="mae_m",
    ylabel="MAE (m)",
    title="Distance-Estimation MAE: Best Baseline vs Proposed",
    save_path=FIG_DIR / "all_sensors_distance_mae_best_baseline_vs_proposed.png",
)

plot_best_baseline_vs_proposed_distance_metric(
    metric="rmse_m",
    ylabel="RMSE (m)",
    title="Distance-Estimation RMSE: Best Baseline vs Proposed",
    save_path=FIG_DIR / "all_sensors_distance_rmse_best_baseline_vs_proposed.png",
)


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 100)
print("DONE")
print("=" * 100)
print(f"Tables saved in : {TABLE_DIR}")
print(f"Figures saved in: {FIG_DIR}")

Loaded predictions
  source                       task scenario_norm                      model_norm
Baseline      target_classification       CW only                           Ridge
Baseline      target_classification       CW only                      ExtraTrees
Baseline      target_classification     FMCW only                           Ridge
Baseline      target_classification     FMCW only                      ExtraTrees
Baseline      target_classification       RF only                           Ridge
Baseline      target_classification       RF only                      ExtraTrees
Baseline      target_classification     CW + FMCW                           Ridge
Baseline      target_classification     CW + FMCW                      ExtraTrees
Baseline      target_classification       CW + RF                           Ridge
Baseline      target_classification       CW + RF                      ExtraTrees
Baseline      target_classification     FMCW + RF                           Rid